# R6 own-report final-stage adaptation
Private hypothesis screen; NOT a submission. Original R6/source/medical locator attribution: Heliosli and bundled upstream authors. Existing model-use terms and OAI/NDA acknowledgement retained. No new OrthoFoundation reader.


In [ ]:
import time, json, sys
from pathlib import Path
NB_T0 = time.time()
OWN = Path('/kaggle/working/own_report_adaptation')
OWN.mkdir(exist_ok=False)
FILES = {'report_rules.py': '"""New conservative report-state supervision; no teacher or published labels.\n\nThese literal rules are a weak report proxy, not expert pathology annotations.\nAbsence of a mention NEVER means normal. No organizer report is inspected.\n"""\nimport re\nimport unicodedata\n\nVERSION = "own_literal_states_v1_1"\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n          "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nENTITY = {\n    "ACL": r"\\bacl\\b|\\blca\\b|anterior cruciate|cruzado anterior|croise anterieur|vorder\\w* kreuzband|voorste kruisband",\n    "MCL": r"\\bmcl\\b|medial collateral|colateral (?:medial|interno)|mediale kollateral|mediale collaterale",\n    "Medial Meniscus": r"medial menisc\\w*|menisc\\w* (?:medial|intern[oa])|menisco intern[oa]|innenmenisk\\w*|mediale meniscus",\n    "Lateral Meniscus": r"lateral menisc\\w*|menisc\\w* (?:lateral|extern[oa])|menisco extern[oa]|aussenmenisk\\w*|laterale meniscus",\n    "Effusion": r"\\beffusion\\b|\\bderrame\\b|gelenkerguss|hydrops|epanchement",\n    "Synovitis": r"\\bsynovitis\\b|\\bsinovitis\\b|\\bsinovite\\b",\n    "Baker\'s": r"baker|popliteal cyst|quiste popliteo|cisto popliteo",\n    "Fracture": r"\\bfractur\\w*|\\bfraktur\\w*",\n}\nUNCERTAIN = r"\\?|\\b(?:possible|possibly|probable|suspect\\w*|equivocal|cannot exclude|rule out|query|sospech\\w*|posible|probabl\\w*|verdacht|mogelijk|eventueel)\\b"\nNORMAL = r"\\b(?:normal|intact|preserved|intakt\\w*|integro|integra|conservad[oa]|unauffallig|normaal|intacte)\\b"\nNO = r"\\b(?:no|without|absent|sin|sem|kein\\w*|geen|aucun\\w*|pas de)\\b"\nTEAR = r"\\b(?:tear|torn|ruptur\\w*|rotur\\w*|desgarr\\w*|riss|scheur)\\b"\nCOMPLETE = r"\\b(?:complete|completa|completo|full.thickness|vollstandig\\w*|komplett\\w*|volledige)\\b"\nSMALL = r"\\b(?:small|pequen\\w*|klein\\w*|petit\\w*)\\b"\nMODERATE = r"\\b(?:moderate|moderad\\w*|massig\\w*|matig\\w*|modere\\w*)\\b"\nLARGE = r"\\b(?:large|grande|gross\\w*|groot|important)\\b"\nPOSTOP = r"\\b(?:postop\\w*|post.operativ\\w*|meniscect\\w*|reconstruct\\w*|graft|injerto|plastik)\\b"\nBODY = r"(?:findings|impression|results|resultados|befund|beurteilung|bevindingen|conclusion|conclusiones|conclusie|impressie|diagnostico)\\s*[:\\n]"\nABNORMAL_STATE = r"\\b(?:degener\\w*|sprain|esguince|edema|oedema|partial|parcial|partiell|injur\\w*|lesion\\w*|abnormal)\\b|increased signal"\n\n\ndef norm(text):\n    text = unicodedata.normalize("NFKD", text.casefold()).replace("ß", "ss")\n    return "".join(c for c in text if not unicodedata.combining(c))\n\n\ndef state_for(label, clause):\n    if re.search(UNCERTAIN, clause):\n        return None\n    entity = re.search(ENTITY[label], clause)\n    before = clause[max(0, entity.start()-55):entity.start()]\n    negators = list(re.finditer(NO, before))\n    neg = bool(negators and len(before[negators[-1].end():].split()) <= 5)\n    neg = neg or bool(re.match(r".{0,12}\\b(?:is absent|are absent|not seen)\\b", clause[entity.end():]))\n    normal = bool(re.search(NORMAL, clause))\n    tear = bool(re.search(TEAR, clause))\n    if label in ("ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"):\n        if re.search(POSTOP, clause):\n            return None\n        # Do not associate a generic negation with an unrelated positive tear.\n        if re.search(NO, clause) and tear:\n            # Absence of a tear does NOT establish the fine-grained normal\n            # state (e.g. degeneration without tear). Leave it unlabelled.\n            return None\n        if normal and tear:\n            return None\n        if normal and not re.search(ABNORMAL_STATE, clause):\n            return "normal"\n        if not tear or neg:\n            return None\n        if label in ("Medial Meniscus", "Lateral Meniscus"):\n            return "tear"\n        if re.search(COMPLETE, clause):\n            return "complete_tear"\n        # No forced grade mapping from an unspecified MCL injury.\n        return "tear_unspecified" if label == "ACL" and not re.search(r"partial|parcial|partiell|grade|grado", clause) else None\n    if label in ("Effusion", "Baker\'s"):\n        if neg:\n            if any(re.search(p, clause) for p in (SMALL, MODERATE, LARGE)) or re.search(r"significant|sizeable", clause):\n                return None  # "no large effusion" does not mean no effusion.\n            return "none_or_trace" if label == "Effusion" else "none"\n        if label == "Baker\'s" and re.search(r"ruptur\\w*|rotur\\w*|leak\\w*", clause):\n            if re.search(r"\\b(?:not|nicht|no|geen)\\b", clause):\n                return None\n            return "ruptured_or_leaking"\n        grades = [name for name, pattern in (("small", SMALL), ("moderate", MODERATE), ("large", LARGE)) if re.search(pattern, clause)]\n        if len(grades) == 1:\n            return grades[0]\n        if label == "Effusion" and re.search(r"\\b(?:trace|trivial|minimal)\\b", clause) and not grades:\n            return "none_or_trace"\n        return None\n    if label == "Synovitis":\n        return "none" if neg else "synovitis"\n    if label == "Fracture":\n        if neg:\n            if re.search(r"\\b(?:acute|agud\\w*|akut\\w*|old|healed|chronic)\\b", clause):\n                return None\n            return "none"\n        if re.search(r"\\b(?:old|healed|chronic|antigu\\w*|consolidat\\w*|sequela\\w*)\\b", clause):\n            return "old_or_healed_fracture"\n        if re.search(r"\\b(?:acute|agud\\w*|akut\\w*|recent)\\b", clause):\n            return "acute_fracture"\n    return None\n\n\ndef extract(report):\n    text = norm(report)\n    start = re.search(BODY, text)\n    if start is None:\n        return {}  # Never train on a clinical question masquerading as findings.\n    text = text[start.end():]\n    found = {}\n    for clause in re.split(r"[.;\\n]+", text):\n        labels = [l for l, p in ENTITY.items() if re.search(p, clause)]\n        if len(labels) != 1:\n            continue  # No cross-entity attribution of severity/negation.\n        label = labels[0]\n        state = state_for(label, clause)\n        if state is not None:\n            found.setdefault(label, set()).add(state)\n    return {label: next(iter(states)) for label, states in found.items() if len(states) == 1}\n', 'prepare_splits.py': '"""Freeze report groups and own state targets without parsing organizer gold."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom report_rules import LABELS, VERSION, extract\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef prepare(train_path, spec_path):\n    if sha(train_path) != "8ca2203c0e9d61c080c7a314c7cdb51c1b03a1d9eb4770819f7f34af53ef4e33":\n        raise ValueError("official metadata identity drift")\n    if sha(spec_path) != "3914242b0c404aee625ea65801f14f60adfee326c1ddd6fb96c48ed7050b7ff3":\n        raise ValueError("source state specification drift")\n    with Path(train_path).open(encoding="utf-8-sig", newline="") as stream:\n        rows = list(csv.DictReader(stream))\n    if len(rows) != 4407 or len({r["StudyInstanceUID"] for r in rows}) != 4407:\n        raise ValueError("official population mismatch")\n    spec = json.loads(Path(spec_path).read_text())\n    groups = defaultdict(list)\n    gold = set()\n    fold = {}\n    for row in rows:\n        uid = row["StudyInstanceUID"]\n        group = hashlib.sha256(" ".join(row["Report"].casefold().split()).encode()).hexdigest()\n        groups[group].append(row)\n        fold[uid] = int(hashlib.sha1(uid.encode()).hexdigest()[:8], 16) % 4\n        if all(row[l].strip() for l in LABELS):\n            gold.add(uid)\n    if len(gold) != 58:\n        raise ValueError("gold exclusion count mismatch")\n    excluded = {g for g, rs in groups.items() if any(r["StudyInstanceUID"] in gold for r in rs)}\n    fit, hold = [], []\n    counts = {"gold": len(gold), "gold_report_duplicate_non_gold": sum(len(groups[g]) for g in excluded) - len(gold),\n              "eligible_source_fit_studies": 0, "eligible_guarded_holdout_studies": 0, "parsed_gold_reports": 0}\n    for group, members in groups.items():\n        if group in excluded:\n            continue\n        outside = [r for r in members if fold[r["StudyInstanceUID"]] != 0]\n        guarded = all(fold[r["StudyInstanceUID"]] == 0 for r in members)\n        counts["eligible_source_fit_studies"] += len(outside)\n        counts["eligible_guarded_holdout_studies"] += len(members) if guarded else 0\n        states = extract(members[0]["Report"])\n        if not states:\n            continue\n        target = [spec["categories"][l].index(states[l]) if l in states else -1 for l in LABELS]\n        candidates = outside if outside else members\n        uid = min(r["StudyInstanceUID"] for r in candidates)\n        record = {"id": uid, "report_group": group, "state_targets": target}\n        if outside:\n            fit.append(record)\n        elif guarded:\n            hold.append(record)\n    if counts["eligible_source_fit_studies"] != 3317 or counts["eligible_guarded_holdout_studies"] != 977:\n        raise ValueError("source exclusion contract drift")\n    order = lambda prefix, x: hashlib.sha256((prefix + x["report_group"]).encode()).hexdigest()\n    fit.sort(key=lambda x: order("fit_v1", x))\n    hold.sort(key=lambda x: order("hold_v1", x))\n    result = {"version": VERSION, "metadata_sha256": sha(train_path), "state_spec_sha256": sha(spec_path),\n              "rules_sha256": sha(Path(__file__).with_name("report_rules.py")), "exclusion_counts": counts,\n              "covered_fit_groups": len(fit), "covered_guarded_holdout_groups": len(hold),\n              "status": "READY" if len(fit) >= 192 and len(hold) >= 128 else "COVERAGE_REJECTION",\n              "fit": fit[:192], "tune": hold[:64], "confirmation": hold[64:128],\n              "pathology_truth": False, "submission_created": False}\n    all_records = result["fit"] + result["tune"] + result["confirmation"]\n    if len({r["id"] for r in all_records}) != len(all_records) or len({r["report_group"] for r in all_records}) != len(all_records):\n        raise ValueError("partition overlap")\n    result["state_observation_counts"] = {\n        partition: {l: sum(r["state_targets"][j] >= 0 for r in result[partition]) for j, l in enumerate(LABELS)}\n        for partition in ("fit", "tune", "confirmation")}\n    return result\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--train", required=True); p.add_argument("--spec", required=True); p.add_argument("--out", required=True)\n    a = p.parse_args()\n    result = prepare(a.train, a.spec)\n    Path(a.out).write_text(json.dumps(result, indent=2) + "\\n")\n    print(json.dumps({k: v for k, v in result.items() if k not in ("fit", "tune", "confirmation")}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'train.py': '"""Bounded own-report adaptation of actual R6 final encoder stage. No submission."""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport math\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nSEED = 20261020\nAMP_INITIAL_SCALE = 1024.  # v1 default65536 overflow reproduced without optimization.\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(8 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n\n\ndef prefix(model, images):\n    x = (images.float() / 255. - model.mean) / model.std\n    x = model.enc.stem(x.contiguous(memory_format=torch.channels_last))\n    for stage in model.enc.stages[:3]:\n        x = stage(x)\n    return x\n\n\ndef tail(model, early, meta):\n    x = model.enc.stages[3](early)\n    x = model.enc.forward_head(model.enc.norm_pre(x))\n    tokens = model.spatial_pool(x) if model.spatial else x\n    mask = torch.ones((1, len(meta)), dtype=torch.bool, device=meta.device)\n    return model.head(tokens, meta[None], mask, return_aux=True)\n\n\ndef aligned_state_prob(model, plain, reflected, perm):\n    p = plain[1].float().softmax(-1)\n    q = reflected[1][:, perm].float().softmax(-1)\n    out = .5 * (p + q)\n    out[:, 1] = p[:, 1]  # Original MCL plain-only exception.\n    return out[0]\n\n\ndef training_objective(outputs, teacher, target, class_counts, state_valid):\n    """Unchanged v1 objective, shared with the scaled actual-device preflight."""\n    z, sl, bl = outputs\n    known = target >= 0\n    ce = F.cross_entropy(sl[0, known].float(), target[known], reduction="none")\n    weights = torch.tensor(1. / class_counts[known.cpu().numpy()], dtype=torch.float32, device=z.device)\n    supervision = (ce * weights).sum() * (192. / int(np.sum(class_counts > 0)))\n    kl_z = F.binary_cross_entropy_with_logits(z.float(), teacher[0].sigmoid()) - F.binary_cross_entropy_with_logits(teacher[0], teacher[0].sigmoid())\n    log_state = sl.float().log_softmax(-1).masked_fill(~state_valid[None], 0.)\n    kl_s = F.kl_div(log_state, teacher[1].softmax(-1), reduction="none").sum(-1).mean()\n    kl_b = F.binary_cross_entropy_with_logits(bl.float(), teacher[2].sigmoid()) - F.binary_cross_entropy_with_logits(teacher[2], teacher[2].sigmoid())\n    return supervision + .5*kl_z + .5*kl_s + .1*kl_b\n\n\ndef state_losses(probabilities, targets):\n    losses = np.full((len(targets), 12), np.nan, dtype=np.float64)\n    for i, target in enumerate(targets):\n        for j, state in enumerate(target):\n            if state >= 0:\n                losses[i, j] = -math.log(max(float(probabilities[i][j, state]), 1e-12))\n    return losses\n\n\ndef gates(control, candidate, before, after):\n    counts = np.isfinite(control).sum(0)\n    eligible = counts >= 5\n    gains = np.full(12, np.nan)\n    for j in np.flatnonzero(eligible):\n        gains[j] = np.nanmean(control[:, j] - candidate[:, j])\n    delta = float(np.nanmean(gains)) if eligible.any() else 0.\n    # Cluster bootstrap at the study level, retaining all known states together.\n    rng = np.random.default_rng(SEED)\n    boot = []\n    difference = control[:, eligible] - candidate[:, eligible]\n    for _ in range(2000):\n        sampled = difference[rng.integers(0, len(control), len(control))]\n        col = [float(np.mean(sampled[np.isfinite(sampled[:, j]), j])) for j in range(sampled.shape[1])\n               if np.isfinite(sampled[:, j]).any()]\n        boot.append(float(np.mean(col)) if col else 0.)\n    positive = float(np.mean(np.asarray(boot) > 0))\n    q025 = float(np.quantile(boot, .025))\n    drift = np.abs(before - after)\n    checks = {"class_coverage": int(eligible.sum()) >= 4, "macro_state_nll_gain": delta >= .01,\n              "four_nonnegative_classes": int(np.sum(gains[eligible] >= 0)) >= 4,\n              "bootstrap_positive": positive >= .90, "q025": q025 > -.01,\n              "mean_probability_drift": float(drift.mean()) <= .05,\n              "max_probability_drift": float(drift.max()) <= .30}\n    return {"passed": all(checks.values()), "checks": checks, "macro_state_nll_gain": delta,\n            "per_class_gain": [float(v) if np.isfinite(v) else None for v in gains],\n            "known_state_counts": counts.tolist(), "bootstrap_positive": positive, "q025": q025,\n            "mean_probability_drift": float(drift.mean()), "max_probability_drift": float(drift.max())}\n\n\ndef run(assets_path):\n    a = json.loads(Path(assets_path).read_text())\n    out = Path(a["out"]); out.mkdir(exist_ok=True, parents=True)\n    deadline = a["started_epoch"] + 3 * 3600\n    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\n    if not torch.cuda.is_available():\n        raise RuntimeError("actual training GPU preflight required")\n    torch.set_num_threads(2)\n    dev = "cuda:0"\n    source, bundle_dir = Path(a["source"]), Path(a["bundle"])\n    sys.path[:0] = [str(source), a["code"]]\n    import compact_train as trainer\n    import compact_infer as worker\n    from postprocess import load_bundle, combine_passes, apply_postprocessors\n    from prepare_splits import prepare\n    split = prepare(a["train"], bundle_dir / "state_spec.json")\n    if split["status"] != "READY":\n        write(out / "coverage_rejection.json", split)\n        return\n    if sha(a["splits"]) != a["split_sha256"] or json.loads(Path(a["splits"]).read_text()) != split:\n        raise ValueError("frozen local vs training metadata/splits differ")\n    write(out / "contract.json", {"splits_sha256": a["split_sha256"], "epochs": 3, "seed": SEED,\n          "train_groups": 192, "tune_groups": 64, "sealed_groups": 64, "windows": 32,\n          "lr": 1e-5, "wd": .01, "budget_seconds": 10800, "gold_evaluation": False,\n          "caveat": "report proxy and conditional image-gradient holdout, NOT whole-pipeline OOF"})\n    checkpoint = bundle_dir / "E_lrprior_s0_f0.pt"\n    if sha(checkpoint) != "59cc88a197bc80c7fa5faa89e69ae7bec04f69cab5259a23a0d0ceb9431e53f6":\n        raise ValueError("released fold0 checkpoint changed")\n    write(out / "input_identity_receipt.json", {\n          "checkpoint_sha256": sha(checkpoint), "metadata_sha256": sha(a["train"]),\n          "series_metadata_sha256": sha(Path(a["train"]).with_name("train_series.csv")),\n          "source_sha256": {n: sha(source/n) for n in ("compact_train.py", "compact_infer.py", "px_cache.py", "knee_anchor_bank_v2.py", "side_common.py")},\n          "auxiliary_sha256": {n: sha(bundle_dir/n) for n in ("state_spec.json", "side_cls.pt", "state_stacker.json", "fluid_stacker.json")},\n          "anatomy_sha256": sha(a["anatomy"]), "split_sha256": a["split_sha256"],\n          "own_source_sha256": {n: sha(Path(__file__).with_name(n)) for n in ("train.py", "prepare_splits.py", "report_rules.py")}})\n    state = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    spec = json.loads((bundle_dir / "state_spec.json").read_text())\n    reference = trainer.Net("convnext_small", None, crop=352, state_spec=spec, in_chans=3,\n                            spatial="sp_q" in state, extent="ext_u" in state, syn_decomp="syn_beta" in state)\n    reference.load_state_dict(state, strict=True)\n    reference = reference.eval().requires_grad_(False).to(dev)\n    candidate = copy.deepcopy(reference)\n    candidate.enc.stages[3].requires_grad_(True)\n    names = [n for n, p in candidate.named_parameters() if p.requires_grad]\n    if len(names) != 31 or any(not n.startswith("enc.stages.3.") for n in names):\n        raise ValueError("trainable parameter scope changed")\n    post = load_bundle(source / "compact_infer.py", bundle_dir / "state_spec.json",\n                       bundle_dir / "fluid_stacker.json", bundle_dir / "state_stacker.json")\n    records = split["fit"] + split["tune"] + split["confirmation"]\n    class_counts = np.sum(np.asarray([r["state_targets"] for r in split["fit"]]) >= 0, 0)\n    cache = Path(a["cache"]); cache.mkdir(parents=True, exist_ok=True)\n    ds = worker.CStudies(a["eval_root"], "test", [r["id"] for r in records], cache / "render", a["code"],\n                         384, eval_stride=1, views="v4", slices=3, anchor_weights=a["anatomy"], phys_mm=3.5,\n                         side_model=bundle_dir / "side_cls.pt", extent_meta=True, fluid=True)\n    ds._setup()\n    anchor_counts = {}\n    original_record = ds.bank.record\n\n    def checked_anchor(*args, **kwargs):\n        rec = original_record(*args, **kwargs)\n        if any(str(w).startswith("error:") for w in rec["why"]) or rec["method"] is None:\n            raise RuntimeError("anchor error/no geometry, no silent fallback")\n        anchor_counts[rec["method"]] = anchor_counts.get(rec["method"], 0) + 1\n        return rec\n\n    ds.bank.record = checked_anchor\n    cache_receipts, elapsed = [], []\n    parity = None\n    plain_targets = {}\n    for i, record in enumerate(records):\n        if time.time() > deadline - 600:\n            write(out / "budget_rejection.json", {"stage": "cache", "cached": i, "optimizer_steps": 0})\n            return\n        start = time.time()\n        item = ds[i]\n        if item["error"]:\n            raise RuntimeError("study render failed: " + item["error"])\n        n = len(item["images"])\n        if n < 1:\n            raise RuntimeError("empty windows")\n        indices = np.rint(np.linspace(0, n - 1, 32)).astype(np.int64)\n        images = item["images"][indices, :, 16:368, 16:368].to(dev)\n        meta = item["meta"][indices].to(dev)\n        reflected, mirror_meta = worker.mirror_windows(images[None], meta[None])\n        reflected = reflected[0]\n        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):\n            early = torch.cat([prefix(reference, images[j:j+8]) for j in range(0, 32, 8)])\n            mir = torch.cat([prefix(reference, reflected[j:j+8]) for j in range(0, 32, 8)])\n            # ConvNeXt residuals can remain float32 under AMP. Test and use the\n            # actual half-precision storage representation, not an unrounded surrogate.\n            early = early.half(); mir = mir.half()\n            teacher = tuple(v.float().detach() for v in tail(reference, early, meta))\n            if i == 0:\n                direct = reference(images[None], meta[None], torch.ones((1, 32), dtype=torch.bool, device=dev), chunk=8, return_aux=True)\n                errors = [float((direct[0].float()-teacher[0]).abs().max()),\n                          float((direct[1].float()[:, reference.state_valid]-teacher[1][:, reference.state_valid]).abs().max()),\n                          float((direct[2].float()-teacher[2]).abs().max())]\n                # Native AMP/batch choices can differ slightly; require strict finite bounded error.\n                if any(not math.isfinite(v) or v > .02 for v in errors):\n                    raise RuntimeError("cached prefix vs full reader parity failed: " + str(errors))\n                parity = errors\n            for v in (teacher[0], teacher[1][:, reference.state_valid], teacher[2], early, mir):\n                if not torch.isfinite(v).all():\n                    raise RuntimeError("nonfinite feature/head output")\n        name = hashlib.sha256(record["id"].encode()).hexdigest()[:24] + ".npz"\n        path = cache / name\n        np.savez(path, early=early.cpu().numpy().astype(np.float16), mirror=mir.cpu().numpy().astype(np.float16),\n                 meta=meta.cpu().numpy(), mirror_meta=mirror_meta[0].cpu().numpy(),\n                 z=teacher[0].cpu().numpy(), sl=teacher[1].cpu().numpy(), bl=teacher[2].cpu().numpy())\n        plain_targets[record["id"]] = path\n        cache_receipts.append({"id": record["id"], "sha256": sha(path), "fluid": item["fluid"], "path": str(path),\n                               "source_windows": n, "selected_window_indices": indices.tolist(),\n                               "side": item["side"], "anchor": item["anchor"],\n                               "feature_shape": list(early.shape), "storage_dtype": "float16"})\n        torch.cuda.synchronize(); elapsed.append(time.time() - start)\n        if i == 7:\n            # Full forward/backward throughput measured without taking an optimizer step.\n            start_step = time.time()\n            with torch.autocast("cuda", dtype=torch.float16):\n                probe = tail(candidate, early.detach(), meta)\n            objective = training_objective(probe, teacher, torch.tensor(record["state_targets"], device=dev),\n                                           class_counts, candidate.state_valid)\n            (objective * AMP_INITIAL_SCALE).backward(); torch.cuda.synchronize()\n            if not torch.isfinite(objective) or any(p.grad is None or not torch.isfinite(p.grad).all()\n                                                   for p in candidate.parameters() if p.requires_grad):\n                raise RuntimeError("scaled objective/gradient preflight failed before full cache")\n            step_seconds = time.time() - start_step\n            candidate.zero_grad(set_to_none=True)\n            projected = time.time() + 1.5 * np.mean(elapsed) * (len(records) - 8) + 1.5 * step_seconds * (3*192 + 4*128) + 300\n            preflight = {"actual_studies": 8, "study_seconds": elapsed.copy(), "training_step_seconds": step_seconds,\n                         "projected_completion_epoch": projected, "deadline_epoch": deadline,\n                         "optimizer_steps": 0, "prefix_full_max_abs_errors": parity,\n                         "amp_initial_scale": AMP_INITIAL_SCALE, "scaled_objective_gradients_finite": True,\n                         "trainable_names": names, "peak_cuda_bytes": torch.cuda.max_memory_allocated()}\n            write(out / "preflight.json", preflight)\n            if projected > deadline:\n                write(out / "budget_rejection.json", {"stage": "preflight", **preflight})\n                return\n        if (i + 1) % 8 == 0:\n            write(out / "cache_progress.json", {"cached": i+1, "total": len(records), "elapsed_seconds": time.time()-a["started_epoch"]})\n            print("CACHE", i+1, len(records), "seconds", round(time.time()-a["started_epoch"]), flush=True)\n        del images, reflected, early, mir, meta, mirror_meta, teacher, item\n    write(out / "cache_receipt.json", {"entries": cache_receipts, "explicit_anchor_paths": anchor_counts, "reader_failures": 0})\n    lookup = {r["id"]: r for r in cache_receipts}\n\n    def data(record):\n        with np.load(plain_targets[record["id"]], allow_pickle=False) as z:\n            return {k: torch.from_numpy(z[k]).to(dev) for k in z.files}\n\n    optimizer = torch.optim.AdamW([p for p in candidate.parameters() if p.requires_grad], lr=1e-5, weight_decay=.01)\n    scaler = torch.amp.GradScaler("cuda", init_scale=AMP_INITIAL_SCALE)\n    training_log = []\n    for epoch in range(3):\n        candidate.eval(); candidate.enc.stages[3].train()\n        order = np.random.default_rng(SEED+epoch).permutation(192)\n        losses = []\n        for index in order:\n            if time.time() > deadline - 300:\n                write(out / "budget_rejection.json", {"stage": "training", "completed_epochs": epoch,\n                      "completed_steps_this_epoch": len(losses), "incomplete_candidate": True})\n                return\n            record = split["fit"][int(index)]; d = data(record)\n            target = torch.tensor(record["state_targets"], device=dev)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast("cuda", dtype=torch.float16):\n                z, sl, bl = tail(candidate, d["early"], d["meta"])\n            loss = training_objective((z, sl, bl), (d["z"], d["sl"], d["bl"]), target, class_counts, candidate.state_valid)\n            if not torch.isfinite(loss):\n                write(out / "runtime_failure.json", {"stage": "objective", "epoch": epoch+1,\n                      "completed_steps_this_epoch": len(losses), "optimizer_steps": epoch*192+len(losses)})\n                raise RuntimeError("nonfinite training objective")\n            scaler.scale(loss).backward(); scaler.unscale_(optimizer)\n            norm = torch.nn.utils.clip_grad_norm_([p for p in candidate.parameters() if p.requires_grad], 1.)\n            if not torch.isfinite(norm):\n                write(out / "runtime_failure.json", {"stage": "gradient", "epoch": epoch+1,\n                      "completed_steps_this_epoch": len(losses), "optimizer_steps": epoch*192+len(losses),\n                      "amp_scale": scaler.get_scale(), "finite_objective": float(loss.detach())})\n                raise RuntimeError("nonfinite gradient, refusing silent scaler-skipped step")\n            scaler.step(optimizer); scaler.update()\n            losses.append(float(loss.detach()))\n        ckpt = out / f"epoch_{epoch+1}.pt"\n        torch.save({"state_dict": candidate.cpu().state_dict(), "optimizer": optimizer.state_dict(),\n                    "scaler": scaler.state_dict(), "epoch": epoch+1, "torch_rng": torch.get_rng_state(),\n                    "cuda_rng": torch.cuda.get_rng_state_all(),\n                    "numpy_rng": (np.random.get_state()[0], np.random.get_state()[1].tolist(),\n                                  *np.random.get_state()[2:]),\n                    "python_rng": random.getstate()}, ckpt)\n        candidate.to(dev)\n        training_log.append({"epoch": epoch+1, "mean_loss": float(np.mean(losses)), "checkpoint_sha256": sha(ckpt)})\n        write(out / "training_progress.json", training_log)\n        print("EPOCH_COMPLETE", training_log[-1], flush=True)\n    candidate.eval()\n    # Assert every frozen tensor/buffer is still unchanged after all optimization.\n    changed = [k for k, v in reference.state_dict().items() if not torch.equal(v, candidate.state_dict()[k])]\n    if any(not k.startswith("enc.stages.3.") for k in changed):\n        raise RuntimeError("frozen head/prefix/buffer changed")\n\n    @torch.no_grad()\n    def evaluate(partition):\n        base_s, cand_s, base_p, cand_p = [], [], [], []\n        for record in split[partition]:\n            if time.time() > deadline - 30:\n                raise RuntimeError("evaluation deadline exceeded; no partial advancement")\n            d = data(record)\n            values = []\n            for model in (reference, candidate):\n                with torch.autocast("cuda", dtype=torch.float16):\n                    p = tail(model, d["early"], d["meta"])\n                    m = tail(model, d["mirror"], d["mirror_meta"])\n                p = tuple(x.float() for x in p); m = tuple(x.float() for x in m)\n                sp = aligned_state_prob(model, p, m, post["perm"]).cpu().numpy()\n                z, st = combine_passes(post, model, p, m)\n                probs = apply_postprocessors(post, z, st, lookup[record["id"]]["fluid"])\n                values.append((sp, probs))\n            base_s.append(values[0][0]); base_p.append(values[0][1]); cand_s.append(values[1][0]); cand_p.append(values[1][1])\n        targets = [r["state_targets"] for r in split[partition]]\n        before = state_losses(base_s, targets); after = state_losses(cand_s, targets)\n        result = gates(before, after, np.asarray(base_p), np.asarray(cand_p))\n        np.savez_compressed(out / f"{partition}_predictions.npz", ids=np.asarray([r["id"] for r in split[partition]], dtype="U80"),\n                            state_targets=np.asarray(targets, dtype=np.int16), control_states=np.asarray(base_s),\n                            candidate_states=np.asarray(cand_s), control_probabilities=np.asarray(base_p), candidate_probabilities=np.asarray(cand_p))\n        return result\n\n    tuning = evaluate("tune")\n    write(out / "tuning_report.json", tuning)\n    confirmation = evaluate("confirmation") if tuning["passed"] else None\n    final = {"status": "PASSED_PROXY_SCREEN" if confirmation and confirmation["passed"] else "REJECTED",\n             "tuning": tuning, "confirmation": confirmation, "confirmation_opened": confirmation is not None,\n             "training": training_log, "changed_tensors": changed, "reader_failures": 0,\n             "gold_evaluated": False, "pathology_improvement_established": False,\n             "submission_created": False, "larger_training_authorized": False,\n             "wall_seconds": time.time()-a["started_epoch"]}\n    write(out / "screen_report.json", final)\n    print("OWN_REPORT_ADAPTATION_COMPLETE_NO_SUBMISSION", json.dumps(final), flush=True)\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser(); p.add_argument("--assets", required=True)\n    run(p.parse_args().assets)\n', 'splits.json': '{\n  "version": "own_literal_states_v1_1",\n  "metadata_sha256": "8ca2203c0e9d61c080c7a314c7cdb51c1b03a1d9eb4770819f7f34af53ef4e33",\n  "state_spec_sha256": "3914242b0c404aee625ea65801f14f60adfee326c1ddd6fb96c48ed7050b7ff3",\n  "rules_sha256": "e10ed7ed45841ab058ffe6926e8bddf01e46a67da2fcf149c6dae0f49569b736",\n  "exclusion_counts": {\n    "gold": 58,\n    "gold_report_duplicate_non_gold": 4,\n    "eligible_source_fit_studies": 3317,\n    "eligible_guarded_holdout_studies": 977,\n    "parsed_gold_reports": 0\n  },\n  "covered_fit_groups": 999,\n  "covered_guarded_holdout_groups": 303,\n  "status": "READY",\n  "fit": [\n    {\n      "id": "1.2.826.0.1.3680043.8.498.53797236075631231042400235117445767744",\n      "report_group": "1fe117dcac6d2c38aaa8f55ac687416f69973b892d9c604180843adc823673e2",\n      "state_targets": [\n        0,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709",\n      "report_group": "2b95d756a9b248e0ab77bfdd3b72b47b3406f25894068a169f5ed76a1ffe70aa",\n      "state_targets": [\n        0,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        5,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.56814987912934301686833132930648230999",\n      "report_group": "3f312d7a7841bb5509e4bd7d5380f525fe7d18e8b0a4ba101569d4a061965b58",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935",\n      "report_group": "74ed70b2de0889e6e8cde56c623d1c5f09f7638b0c3f361ca4b7a974a71b3b17",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.95497572581157858069692837655072209020",\n      "report_group": "d786e995e43f70a63eea4c487e7670de673684a122b7aa4e4dd8a275fedffe7e",\n      "state_targets": [\n        4,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949",\n      "report_group": "016cd3bc390c7f64c75c330cc51e451673e228b76c2bae146b105726f3ba69c4",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.30800677497901063887487934675082066316",\n      "report_group": "23d6fc5e6cb8c3bfd249485b58d364fcdbe0350723ba19d8a05a372b39eb2ad4",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12836702174475748025129787471026627810",\n      "report_group": "9e21c455b39ecff86dc062c382fcc78984794a09c7d30bfcb61952b0f7fcdb92",\n      "state_targets": [\n        -1,\n        5,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412",\n      "report_group": "e90be384b601a1ec18cb57576f2a585be6b2038aba585a20ebc581284dfffffe",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.75523541256169382504121567002441615798",\n      "report_group": "16dfb1cf2b66290ca6086f0b4d15a055383a54192c4dec214e01ada350ac17a1",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11150597567791052354257165699079348456",\n      "report_group": "bf4b07bc23ed9eaa14d22d53ddcbe130aab0e80d96e790b56fc5d19793fbe717",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893",\n      "report_group": "b53e012f34ca8670469161dc1ad6fa00d763dfbd6a895de6363c5fc219737dae",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.16538738179840493095338324009830673843",\n      "report_group": "4db667bbbdfe8b167b19685754141e1cc93482c4d94cdf2ef54359caaac7dfb5",\n      "state_targets": [\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545",\n      "report_group": "e0f4dfdd609366f7434c2415217fd30bb25545caf9896606f9da1cc81cadb7d3",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.20852351697358589606507109300359572624",\n      "report_group": "20b18a1a9f64c711e80178b7e19330501ed62fcb1648d216897dcf8fd8dc7ddf",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.71507059468810869004142324822642900237",\n      "report_group": "75333d0188f62937a8e38da9f9207aa80d00603ebd053405b3bdf35ac6cdcdb0",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.21634792491972658759276025394685307053",\n      "report_group": "758430da8ee33d738471306eb4f80de56bd44210ab2965b4d6cf155fef64cd61",\n      "state_targets": [\n        5,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        3,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.16397020531009156965821242117417547331",\n      "report_group": "210ec9f865db0b54b21a833fa1a0aa18ae7a367b16ae2a0b891a3b41e5688a82",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10492484093480313596330475837639172625",\n      "report_group": "6955272df808e433d34cc051696debe071792b4a3ddc8ff71067b0df102010bb",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.98027050915885651900992366068887550187",\n      "report_group": "27a05e4268ad38bc155f860f89753160d1621f01bf1ef02d2d6eba0d15c40974",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13505671694940677161128783863912245059",\n      "report_group": "52c08cce7cef6af5c14da11b3645bbaac34476000478df87321c45373c92228d",\n      "state_targets": [\n        4,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11651755040149549966484720727140190853",\n      "report_group": "4354a009f558321fed6d1cb9b0d93f44cf91e206ded2da895bda0dcff7077675",\n      "state_targets": [\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.42719592405241698450757802131459694665",\n      "report_group": "6ec4f5c5debc18e2d8669b748cafc34ebda054326dbd3540df7e2a98194136bd",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409",\n      "report_group": "4383126846a4e1e730b4f847c48a9c9c8daa1424d874c188e371f2aa4c81b6be",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827",\n      "report_group": "477164271c8d45d6968d5297cb846be5435ea51b323c49fee80f030aa1cd8f1a",\n      "state_targets": [\n        5,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13128352910661672966143173193241171100",\n      "report_group": "1df70719ed87cdd3a8b4ec7d1421ef08a7dbef92b21c2b3a47adbe375d011bf7",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157",\n      "report_group": "3de2c9e3984df9570426971ee67381e4139875cc33e377ba895d7a415af8f9ee",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.35991881811942962264382690224767291006",\n      "report_group": "c31514897987f4585c72659f3aed4365169bc2fc715df55ef9a46277e0818af4",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.90758414832541790868268922794274257839",\n      "report_group": "6cd677dca31ef26f3de50944cfc3f526da7716a89532a8647454748434629c63",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411",\n      "report_group": "2dee5dd237e2f480e88f226d69bfbec249b129e06de90bd3af7efabb92458591",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10491012711700179118010977263805828326",\n      "report_group": "8821cd5ad83739cb35567410bb77f373b3db1aa1cc8af70714be05fbee60d4c5",\n      "state_targets": [\n        0,\n        0,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.22684505799921376181913295621462268931",\n      "report_group": "b950d7cb92c4693432dd480852faf7738081bbd30ff144530a8cb818120041ec",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.56839374698050163738039760889156066445",\n      "report_group": "5ad3d08933a093c30a944fabacd9e02e18e3b3454f631d247f6617955500a162",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.67397756161011684282778294948252031539",\n      "report_group": "5bf2d16a1b507da84dd248a8a5db40a20bd376f16b6594e98148eb206e927d55",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.37447511763482014027579634343525200849",\n      "report_group": "00b0a6a9161cc3590eb77f157b0112adf107428893a01e1a4f2a658883fe72f4",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.86405969333062807387894608315897106350",\n      "report_group": "f5100e18b487ab1caa67eab1f5dd08699e0b97fb6e9357e62dddcf865cde948c",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.30206987686375669521451035370290737934",\n      "report_group": "2d3dc3ec52d2314bdeccb86780575a4247a2c415516f6ce3809965cb08cb0341",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775",\n      "report_group": "a69e6b94d41ae37476d8d15de5598121ad8656c8d8866c5b02c989f0eb078f68",\n      "state_targets": [\n        0,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.77546769004367585830455474984769277713",\n      "report_group": "2289dde5d927685bce4c66448451e920dafd216c0b3cfd6395b39d0ac77876d5",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.75773447415943537938555952303103557864",\n      "report_group": "9d285c1b32eb6ce5ecf13875343008f2bed9a65b6d140e03c7f2f3f119b6affc",\n      "state_targets": [\n        5,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875",\n      "report_group": "aebc0e3d9413ab677962b9fa876c73ac83f2de472188515097e5e83ffb83213a",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054",\n      "report_group": "9863269dec036c5c04ccf765e1a498d1cb2d2dd81f6fc0674c31373a5f538683",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12689874339887047433232628649886917274",\n      "report_group": "f60297a2b2338dccf1254733b35e6ce8361a9294c96bd6c5bf033ac19f946fc0",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065",\n      "report_group": "87cd450eeadbbd4521ea5e43c20edd1059223564541a184bc6351204399003c1",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010",\n      "report_group": "dc9ae227fefed9d1bdc4790b226ce4b580027c732de4170a59bf445c13e7e4d9",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.93188773180403601524488403580611534053",\n      "report_group": "c73a81af2cea4be2af0c5b0e5150568132c59db4be20725bfd403c1906c892c3",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.22057596860528658818833903685827585805",\n      "report_group": "cfb825f817e8297049252e5c85f85619063e560f0240d045632381486a085ea0",\n      "state_targets": [\n        0,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.56991590802231436015843619633497607680",\n      "report_group": "f93c535f30047af37641110deea2bc8b124c6b3f262742c650558e77bec51301",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286",\n      "report_group": "6044027ad965fb25113caefa3c25a5e987d5b8bfefa74622e2b559ed903b2127",\n      "state_targets": [\n        -1,\n        -1,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10190506315646895064803936408630113507",\n      "report_group": "68be469c39fee06ab388469a10a355adcb3945232112c18413bfbd87dee307f1",\n      "state_targets": [\n        5,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745",\n      "report_group": "3bfe66439cfb6666d064626a322a0303b5c0768a56ef6ee2af7f3813abb9fe77",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.49984662916977328110175445895515541267",\n      "report_group": "a98c398b91569f7c4c6238febfc174cf78d5aeeadaeba0a5596a1859abe875e7",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039",\n      "report_group": "07681563a69828cfd93e8b22da7a375ed8e6e3b71c3f179d103c11d71ce4ed66",\n      "state_targets": [\n        0,\n        0,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052",\n      "report_group": "adb5a0ee087a88efff21a1773002006396074932a3f7371fbae9137a94becdb7",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12221892384948220776502422531414659289",\n      "report_group": "b9d76b44580473c734f0b3e2750988a2eb10b4b2eb00b5661761dac7adf425ce",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11587989960888698769912899464470116193",\n      "report_group": "edc67267e8721b978572c6cc24d52adc8dd90097cd88d6f65eb5236c914344f2",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.47094981922544354899183959858078311972",\n      "report_group": "5665938590fe0949961378339f548ea2fad274536e772f721039dc261d68b0ef",\n      "state_targets": [\n        -1,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12225975251304101975065271521710735735",\n      "report_group": "0b3cfdbce9729f57015a30f7e52944be59d2fe5d3810f8210260e7c9c825dfab",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.68304853223764870957854101604223809218",\n      "report_group": "6681901d456f818aea51f5feb71f2712fb725d381d848be7e2d4fce6c4560f21",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.43065391907984821806263204343792008938",\n      "report_group": "f651612b68dd0721c33f3297571e0c7e3cbdca73268836260bb8b12c3d8c43cf",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.75893915698353338311947268382839152877",\n      "report_group": "11056e5491c94dc6fc537101e7e08571674c3d68f48ea798081f8bc0fd08c978",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.68777783869253673804970857981124324335",\n      "report_group": "537e1c3f012a21fe24c02a9c808b1f82565e86d4cccb3ac5b0800af5a6ba7caf",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.54662745951809363774369202018332309887",\n      "report_group": "93c96733f77ea656b644ceb859f3786c4f30eb364f06c41ff3e4672693bf5169",\n      "state_targets": [\n        -1,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.14679140388328648617386311272361396555",\n      "report_group": "21ff48bf2caf9e3439a38fc893efad253314a211facc3285ee81910ddf16d137",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121",\n      "report_group": "2b98645b3dd250d49bbe83009617369b5157339699dafe2f115c083cd64b9c77",\n      "state_targets": [\n        5,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13332521365069604440230185026865466121",\n      "report_group": "88c7f0da970467bd0e9144eeede32b1ec0320f8dc96a1db877e686485df315e5",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.96094001324999448553107805455529187871",\n      "report_group": "a6b3ef5d166256a2b1c565ba02ee9bd2f247640f9a5f5e40ec7cf63195117a51",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12260661730837055872432103268905182876",\n      "report_group": "d4c6f9d499756223b5603673bb55515d23fec9f034c94edcd3bcaeeea477bbb4",\n      "state_targets": [\n        0,\n        0,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.59476657510212258499047356518058209126",\n      "report_group": "980e75c6ac8c1207d13c60e936f348c8c007c18bf8fb7f583484505a32635b56",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.49884243466758407760126403944842240776",\n      "report_group": "26a741b1c37a30e31aba8deed45b20d20e261a840508567161ee2d8d9c67cda9",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.90206716031776606956011363950625953310",\n      "report_group": "eac265c6635db5eacfd5cfce97cfd3663792be0f9468bdcbc690e440ea5e4e42",\n      "state_targets": [\n        4,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.44365814642869065241166313231424697474",\n      "report_group": "d0736e1dcb851e8efb4cb1969be6f4fb3533b5ed43e752794b5c0f896a36dd1b",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        3,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11219813299300915347039206446596901205",\n      "report_group": "29f469233cc80684b81b05829d31ad95c3dd0913d319f9c63952dd02b68ab882",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12965787822216495014894819598440431626",\n      "report_group": "df049b9dd83eb5543687e474ef74021c9a7a3cc4b360c847e176cc9270fd2c44",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.38039631093067555454700266397073080430",\n      "report_group": "f09690d4bd17ed2fd5e0e851c404e775806edba762602d5ba8bd29ef8160f9f8",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.19722687746450343001601428638894755031",\n      "report_group": "dfb78e1d0c817dc5629d5ebeeebd3bec8e80513965eccb54d8f89643dcf36958",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604",\n      "report_group": "8f38f604248bcbe2602bfce8f286e53d5ba37ee0f9d83121682043e7d5a14a70",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.80492324225032367918842505447056622561",\n      "report_group": "f36be75b1a48d10c991f318316b6ec6f290425bc1a96afdd2749eebe957541fa",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196",\n      "report_group": "5e1ea6a13d0b307ef62a2b0eaf38a7221a6a792383f866a11c18f5e17cbfd40c",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670",\n      "report_group": "47f14aa0a1df0bb28825398747a099cbce3544809fbd3adcdd99740114ed40b5",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.29090158364700006304493717187346344597",\n      "report_group": "9a671d6e16fe764968c3a4e3e74fde0520297f3f8d61072b6031e40e70d95e48",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451",\n      "report_group": "45cc7f40b9ac39e71c41903427ea825f4e9aeb928f8e133448248a1a441ec682",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.77183368292104509335580564463404842046",\n      "report_group": "d7fce9d8c2ff0b76c23e73c165d9b8639e1fbe5d5cbf4b19c3bb22c482b6f96b",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11448865770537436004548718998804636234",\n      "report_group": "75e6ec3a827b042f10a13fb23e85a9b1cfca5a57705628e6141ff57dcd45e115",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103",\n      "report_group": "69c43313d2e45c77afe83e75d7f7830151ec2183e0e7b8905a75a1593744ebf0",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12185615973620890770158935333033805789",\n      "report_group": "e9a643115d0f20286773189b84d8a93519475e8ba4d12dc4ff5b4718298efeaf",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745",\n      "report_group": "b640918dd285c61ba35c309a437326df231f21dbb6cf805630cfe4bd2a7fa953",\n      "state_targets": [\n        0,\n        0,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760",\n      "report_group": "efaac3c1b1ebdb8319a507c5ae11b2446ad4fc47fe0671f6eb16f351b0f52662",\n      "state_targets": [\n        5,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.80319546205646617603546887163539306146",\n      "report_group": "248732708ba793f5978bc23a5693d473d1ee89380c925a5dc23cfc8b1447703b",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032",\n      "report_group": "8e7f8b1de69516248eee9aa183e3d4384505a84f543447e784ba20ee4fdb8988",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.75670996863043899245632198549551296292",\n      "report_group": "e89c4c0e17c53f4fa2bf32ba3cd6e40bb842cb9abc63bc2311cee9444ea954c5",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826",\n      "report_group": "21463b756b10b33d4be20e4e2850e793a217bcc840ae22bd77eb31eafbd36a22",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.82967086493280898295669930903240677729",\n      "report_group": "543e6e845c6c066d4724d3f7fe23b9f58065792ae1c591e4a48b9ddb20f246c5",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319",\n      "report_group": "26624613e71dc3cf37b61d294751db322b36c9ee6a2da0c54e27143190813e97",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10327608571317884761792196888985963332",\n      "report_group": "6535cc11a121431fff50c812584ee27195c892caf1a58621f599240f4709b7c5",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.42896117106271721157113072562739356381",\n      "report_group": "63c44e97718befe58ecc2c1e196fd14a5410cc802f4e7c7bc8ed8e9f94ddd765",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11706378290185110677863653788588307917",\n      "report_group": "182d0ee318afdd779528f5533ebcf331125cfdf5053751d75cc8385ccf6bef38",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.47855330524708374523226349383799712473",\n      "report_group": "75757eaaad74785662a308d5901f70005acae6678163e62bfe825c2550c468dd",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509",\n      "report_group": "9d269e7107d988ffb88be3127b508346fdee51708777275ac804cb331814b717",\n      "state_targets": [\n        0,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.99864558747514747686151806909794288262",\n      "report_group": "51244873333c606b70ed7e64b8026b2837bb2c7e7b236f0cdd115ad7b462d521",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.70909154434106251910698870110315237266",\n      "report_group": "578ccfb20dd5e5669cd01dce0391a7e849901a6de547edc4ca270b46e115b99b",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        3,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960",\n      "report_group": "876b4fa8aa3271a40d0ef0eb2e44a9ef252e9fa6415b5fdcd9f80d966b428d1a",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.45251306032215692953561852960585282987",\n      "report_group": "2c8e2f9bc853ef2fe9dca5135fa2587fcffed9dbe496ef20807a4fd734366ec1",\n      "state_targets": [\n        5,\n        5,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.39959632509842689058626880477779808473",\n      "report_group": "54aa74d23aed761c80ac6afb9681fe8025e1e4df5d18fc1d53cdd3e9bd0690a4",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.65013994414976103275014874284064145461",\n      "report_group": "a0ecf235543d946ee162d91dac41cff11645445af010a78d7cf02f68a234f1ef",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063",\n      "report_group": "a87ad3db29cf97ad047e12c2f8a8e52829a13d645ff13edfbd3bf6d78d30808d",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11901474252960403779405194385221103673",\n      "report_group": "088c0b33dde66d23bae97f9f5d2df43eb6d319b20aadf5b30487ff91281ebd1f",\n      "state_targets": [\n        0,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.52138967568492774495485454357106269286",\n      "report_group": "fe9220ba8056da9b9028dd860b9a8a73e057e01c25bcf787e8627b215543d11b",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067",\n      "report_group": "bb5bbd9b4f400d98ab8b241a016f2de8b6fca430ab0b818502ffbe694ecec488",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.90172333170788218623219231170360850947",\n      "report_group": "3e64b8c23fd9b78af402b298934cdde80c5611b65eeb6559af17dffcd4eb3357",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        4,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528",\n      "report_group": "4701d48e2eec9fa7a1173a47897c195e0a677c70da07b09684b68fa2875dee7f",\n      "state_targets": [\n        0,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409",\n      "report_group": "7f3295453ff175f3e424308943420baf73d509b6e7f371eb1d68979adab3215a",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860",\n      "report_group": "3de6a583bdfd6e420d2b2604232c00454ff41999ddfbb7cc877dd81f7ccc6821",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.44333089398623004498721836325723633455",\n      "report_group": "bf9f4698b50c23f4949a8bd6f2015e072dcae2b57ab676b2c39adc7e4fb3abb3",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290",\n      "report_group": "f0d4136aa45d78e011a7674f334e935c3ad8976d83bbb652cfee7f4522a677f0",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.92449204325697143336618138322280397765",\n      "report_group": "0d4513ff7cd9ed1ce61c2798cdc7cc5ed7d12655f8840604c72e59662732619d",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.73478948866451045346677275674140339803",\n      "report_group": "09bf9a582310d8c39cd913f1cf78a3728ca6b06c1f87b3f9c59452c5c058ed73",\n      "state_targets": [\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930",\n      "report_group": "ebf1a0eccfcbda9612d2c488c001f6fe0cf51864048c9965b0752e78127b4fd2",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11052656375391331522834364601050945860",\n      "report_group": "7e582bdbd396a6220e7c37838c06742beb2a76139855b0838454e4de2b91443d",\n      "state_targets": [\n        0,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12547308734950397389859712060653254111",\n      "report_group": "5e942576829569912d1001745fac2a1e6166ac7dac0da7b5339b908047f5bbd9",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.52937412433120578046842194545406797236",\n      "report_group": "247d6c7445c7b8aa2bb10a6d67a9a3f6c9422b4a718deb03b153a03102cd4e3d",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.15297444019822621504849851092382560980",\n      "report_group": "d256985e8007a5d536b1e924183072fff39ca59e34e9e891423fc2f8759e6c1b",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597",\n      "report_group": "cf37e26a74abd98edc8ddfce31f776ca359bd8020733011916a0a357304e2d58",\n      "state_targets": [\n        0,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.40154708689850542672289658249059660788",\n      "report_group": "3998a0d484d317c057f4df8a34a7e582f9526a93eee843e2b5a7137817579b94",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        4,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.27644321991318957631114727895883331622",\n      "report_group": "db57714f188415796e34b3eae6ddf8a111b9cb8796fdfd75253f1a85df45eeac",\n      "state_targets": [\n        0,\n        0,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.85326542895076582985200613540648336047",\n      "report_group": "72363a9ae24c6630f129296ee96280a08af7772138743088854961277ead232d",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908",\n      "report_group": "b5b7667de5fec007d292860fc8228a0c516d10e3232d6f89058c6eb5f819507d",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        7\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.67941678374744711267840655419606994796",\n      "report_group": "c294189309bd6ca4a20e5a3422053cef5f3d28dec4059f50d9a948805afcf27c",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.45118217412917282413100913670807141634",\n      "report_group": "0e3dab7af400f6921fbab9ddf73ebef9ff3e640ce89b86a6064c1c0228d9ac62",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778",\n      "report_group": "c1b284da3dc263bb8cd40d80e2319e3e331fa76429a004f5843d537d9610adbe",\n      "state_targets": [\n        0,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.63223220827342079068491460847713457238",\n      "report_group": "f47fbe64231e6815c61fd33103240fae3afd405cb394e9bec9f0eb1302486278",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.37969790195791584440906553207150666923",\n      "report_group": "b7d822f7f3c884196d29f87d748be01919aad58f66638058404628a9c9aeb8a4",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990",\n      "report_group": "ef6638c716bf264554f981dda199d208e7cc017d29c7c824c71ce8fab18bffb5",\n      "state_targets": [\n        5,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.82655005222804459605894403147145992714",\n      "report_group": "5ba7479dd40c843d053340dff98c8ca5074f9950a77ab108eb3654ccb9e08652",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.39070169128115370701189151138307035053",\n      "report_group": "a23ea9806f7d5c4f25fff07db5fd42cef62b413d5202c351eac6f4d07f2190f4",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367",\n      "report_group": "20c33c74823637a8558549ffb52db3ff79337f4947384a7a1aa33a8b1f8f12ba",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.73021428171736137311489171988512484948",\n      "report_group": "12147f81b02819e7162de619b2a9863661c284406ad3c8c6a5b18bfa16dc7bc4",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.67001329390153622359718581388215312710",\n      "report_group": "41c6631ec5338c75ec3e288c106ce6285738bdb4c0b0f15037d6557fc651b658",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952",\n      "report_group": "ab0d0f5f0db3d857e928558721a8904635793da632c39ecf0ca789d5e1b4866e",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169",\n      "report_group": "6b30414bf77eee4b8e23094d963bef5cb75137ba78bc21d48e03b13f10a23dbf",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12697859103448296208701081683487338111",\n      "report_group": "ad694e39b253a9fe4cc5b93a847faed3fdd272e9d3d09ac89df5b42cf4185e8f",\n      "state_targets": [\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.91385694666101904780012335954322545042",\n      "report_group": "9c492ad75c0538cbb502916bd74dc0534f9c3eaa0aaa7f368ef266c0f515244a",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.95555755747044090477918262640906805885",\n      "report_group": "68190ac9a69e22051af3203d22837b0d90a35f51fdb6813e2790b02110d5fe7b",\n      "state_targets": [\n        5,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.47784065888386216103165400551121771174",\n      "report_group": "ec00545e1e55a910fb85f4e1f5b53c2b6ef0ee9c36cb628511056517dd4f0e40",\n      "state_targets": [\n        5,\n        -1,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628",\n      "report_group": "caf8fe00d2282e02fe1ac7d8e6a9cf1773099e232a0784c1e3c1d207e9a31fc3",\n      "state_targets": [\n        4,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.68576465611266620513305434082656807028",\n      "report_group": "35ab96688ab5a569ae05557d8520c31e304e036d2258c05f3dc471fdd5043cd1",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.76903562636417038907113894538863784737",\n      "report_group": "21d4f1e71beb66e5743360c5fd084d0685aa36d5a54fdb2289724532bccef7bb",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668",\n      "report_group": "72c7c31821b2a5d6f251e9d5b0081da7145aa41d22538a12e38056e648560070",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13072908567275370866813748886351904593",\n      "report_group": "87d966e3a6ff81fba26042fdac136795aa858eb7de95b42f26896e8153c96e11",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.39954567501513957418422043089587980092",\n      "report_group": "7ab071b8f3cf2719d364b81d0b295961bcbfcbfb803755967655c3b66f9c938d",\n      "state_targets": [\n        0,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460",\n      "report_group": "e7a6d2f70d7fe7f4397ee770080327e19c6ac6aaee39d592e80e8460b406893c",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552",\n      "report_group": "14e4eea016ba26418e5843b7c10cde449be497ca32e38bfb51af2c1faefeb439",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.80726797831040227285328789648680519664",\n      "report_group": "2fda203898d3d88c4f372b0a4d38218b37dc24943bbead2271c980755ceef9e4",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585",\n      "report_group": "1740cfc52e435957f37cd019c11962df98d41f62caae9d4602b9f68502ef6dc5",\n      "state_targets": [\n        0,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568",\n      "report_group": "0e3c2bc9c8687bd26b3895100dc02d53dc4160f045f4496fca58c3f75849498a",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.20953536352252551462901259297974956430",\n      "report_group": "eb85c7e70ea560e0ab7e4d297cc29561e84a3b475765ec044a181e23f91eeef4",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13114693814781737495494700902699619138",\n      "report_group": "d1eca1d47c607baeaefbbd384b7d19ff7b3181b737f6597a95f7ca7c9224df4a",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801",\n      "report_group": "d2ee35c0a75a9bffbbb16691557511a91cddef240686742c9fa2e8f209aa1646",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11753067309373104365558887398441364695",\n      "report_group": "44366183ddec4583cf4f62daee1cfdcf9f0bdcfc382919da00d0bdd971316710",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13353966209257569249620562863006006441",\n      "report_group": "e3e3afb46610f5390975a8fa3c3c40380b8917dc5e5b5938a5624184348d3f59",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10284446538566255572004044331831825072",\n      "report_group": "77c6914f9cf660add54b35844c82fc196b3482214ca08831b8e56e16137019ba",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.14117402902169664329719927863268363596",\n      "report_group": "57d51572b3015941e868553fe6830c6c63e2323046a9ea636ffd478296ff2a3e",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12762900657007480138202359874575403361",\n      "report_group": "ed56a668b6582343d10feba553bed7417da6e11278b548b9b01cc640a84187cc",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260",\n      "report_group": "5cc85c513b1fbe67b90d514625b6be5bc44b3f6cacd340bd6e6033c4713e1083",\n      "state_targets": [\n        0,\n        0,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876",\n      "report_group": "e2eb0b6b95a352f6c04d29aaddfc86e48f21321499dd4641a449cc77707ce839",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.69429384137362531712760113705595820981",\n      "report_group": "8a6311c60448a05597ce8b02f695d7e0f017d4c0fa5433893a3f5cff9ae32dfd",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397",\n      "report_group": "f6b865a6f60fc3038e4533db96724fefbc10ccee8b65f22b6cd4a990395c9919",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197",\n      "report_group": "a996666543b4011a8e0e5a101bd454100a11f6c2995ebd9948d0d08a33760dbf",\n      "state_targets": [\n        -1,\n        0,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194",\n      "report_group": "d45ea333b623c92c0a80010f55c6ebf479761ba6088b31e3c559010206654d4b",\n      "state_targets": [\n        -1,\n        0,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101",\n      "report_group": "3ee54ed716dff3185f90348537483f472e50cb7418c1fe574996738120f3b668",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260",\n      "report_group": "c10fe944414aae559b16a4d81f45ce2e94d5e050eb8464a7b7efee9301601877",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.85212353344613461454490252928274988058",\n      "report_group": "cb449cc6e78a891c934d226332cae7e92f1740f6241cc6b4df340895da27d628",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.56188634971310070481285525830793231849",\n      "report_group": "18ea79669fcf9864725c3a6d6bdfdb99c71f900a30cb3f07527feb83a71bc77b",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.54000782656400326230944526201670909133",\n      "report_group": "580d87f5ccf268cbd84877cb46b188a659f0a26ecb1d093e5f9b32fee5fc097a",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455",\n      "report_group": "0f154014da5380929f297323e26db8a40420ccfce7550a5af69e54371f8016e8",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578",\n      "report_group": "5ebf4ff59a5c57770aede2b9c7626e2f0e8609ac04612b605c762a9aeed12d05",\n      "state_targets": [\n        0,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775",\n      "report_group": "fd6adc9a73eabfae438452fc95f9706280d5b4169673ef85326d729855f3d312",\n      "state_targets": [\n        5,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11262944211104069231500934582412003273",\n      "report_group": "b3e4ca04591d6ade05126f01e3228caa819525087edd081ae1819eda6db54240",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093",\n      "report_group": "55cceeda25a1aa194a4c9a188ca0b0e39d2b30d33f069004ca8a4b02306d5cd8",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13391735462825209934041441767886593395",\n      "report_group": "a74fda308e45b80d04aa3f6865d1288e8b04723c3794ede4bb21043a2000261e",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315",\n      "report_group": "00dc84c0707fce755f8632799965705f47312c10e36d024dcabef266bffb99e6",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973",\n      "report_group": "e64f2d0b597640c9114a20f88d4a7e9352f1a14ab4d83638a90a6209a1e40729",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089",\n      "report_group": "3337c7f5fcd5436d85a776f4b0dfa4356119cd9dfbb7a5618abbe8472e5efd87",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.14963472809950919852176661973255059069",\n      "report_group": "5c9b2c6b6c95b3ea96acc0d10f21b731adf4974596df2846d048bc93470ac9c4",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.19788612491138642031531719942303619648",\n      "report_group": "dfd7ac69576a48cdc0c3044044402b4782b931697aef9406bc56b1f1242d69cb",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461",\n      "report_group": "5b4b1fe4998bab15066b6b144394a2343f6eaee29c7d137517a98fa2c7f5f32b",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265",\n      "report_group": "a176d7a846ad51e517d2f63396968155f1ffb44a666f8cc6dcac5b4fa32645d7",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.40419759559020255481034237210540871312",\n      "report_group": "5a89fbfc2c79342ad04cd7bca150b0fd8d2f717c66e92f0e0d5e8d191e08ef7b",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.80886525412417948114478368374120704834",\n      "report_group": "edc15b849b4ce0444deaaae46e45bbc585a607a0fb15bdb379b7a5ab9c9e40c5",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514",\n      "report_group": "b81032660105052d293614874d780646f5b776ad3e5cc62736c44fa6a783680f",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154",\n      "report_group": "dd8f29b5386be6e9430f83871a93c62abf7968333d85c93317784b4748b206de",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803",\n      "report_group": "61831eccf8762e98697766c8a8e1bb0fbeb787a33d1723813ad56b05a5eb113f",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    }\n  ],\n  "tune": [\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12573518342767030025851169814440863785",\n      "report_group": "c8cf54fdd8b651e7679221eb92c23a4e6c1d9585ca0ee8bad33d261735459c5b",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587",\n      "report_group": "8cd48d6a4ea30c04a570df58b32af8931f25b5aadf86da8f42b89c2834f92d16",\n      "state_targets": [\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.59938475193950583662125713924156951751",\n      "report_group": "be805e2e9831cc645d238c7635f58105dfd9980391a72fc5a3e42881cc948c42",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365",\n      "report_group": "400935cffba24bee7d662494f29d41895fc918251a5455ca4199272ae93c8157",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.93718451128867963123946970296603639068",\n      "report_group": "d93d361e89c328ed2b59ea004fd4aeb198267cc62234139deb7b2646351f4b42",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.38005916106577944091708997377832932938",\n      "report_group": "64a24ee105984709f3fe8e5666201a1a2dc220d79aa17498a3f1955777bf62c3",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.16158812105029662172517682642229718094",\n      "report_group": "94c0c75522ea1a4e68bfe54f389419e029e08faaec5a6a0f22a2cd76d5a5aa49",\n      "state_targets": [\n        0,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075",\n      "report_group": "c23a4f99672b749f36f2518f3c308df4574d5bd4000e776e0b1be20617ae99ef",\n      "state_targets": [\n        5,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961",\n      "report_group": "bf91beb3a05d38b88b4f5d3cd5b204e9cee2edce386e97bf59ec4d92f53075d0",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.93985093305024344873337249433396141838",\n      "report_group": "a3b6ad31b4c84886753e0fbb577c475df6b64ebf1b5bfa680e4d62f842c0d120",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13309299049346932814529962458932580753",\n      "report_group": "a1e08e519dce50e16abad96353fed89ca992eff7bdac580a3390fc58a0e57e68",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12242172267695588655664456078003635837",\n      "report_group": "7f45dd70512789eb25054ef81e016db64cdf03489af98be1aa8f3276aed58bf8",\n      "state_targets": [\n        5,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12431272351851438889289190308550697909",\n      "report_group": "9f1407bc2cb08697274be62dd5420994b7eb962883f27faad5f814547a2baf43",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.23939313916974849526459717576344063277",\n      "report_group": "38e3cc39b598ccc4709ea04c1931e4a087a85ffb546c7789653ef55e472e1c03",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12837030643153472711780894512565629507",\n      "report_group": "f2b285cf0e1c024ca58884081acd322b1e8a9306ff3d361e11b6794494b6b308",\n      "state_targets": [\n        0,\n        5,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11406304618685790128709325242250648417",\n      "report_group": "4d8a95935051d7f43637b26cb6ee57f7ab98d963374906c50552c3231fdb3316",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.18230847745299902998425838090653825745",\n      "report_group": "3914924ea3b40206ea7cc1e5d7e2c9989d4ff44a962cead9be244b2ac17994f4",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13022266764623819477228425331705640702",\n      "report_group": "8d9985a5708b63c3bb9c7e6e6dfbf4ff6c7275cfb795d42b9b34ccb4052cb126",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.25006652485159423287561656594171884310",\n      "report_group": "1d62a2cb64276998e30773cadfef578bdb9e0d8730b3b390aad86b9666eeafce",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.57316522861709598765962486949846744486",\n      "report_group": "6d00a060025ba8708217f670ba6a57d50c9d53b9bd4f327b9c3de50b4f73ad05",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.47391919345230380216864709248400573953",\n      "report_group": "760f26c5edf47c78080b07019f09f17b706490302c0b951ba07dd0e2e8c43533",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.74952769952682175467667812958859010329",\n      "report_group": "af42e048b4f8edb8cb96952bf9db9b78d2bb6fba3a8293c9c3e4a9afb1aaed5d",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        7\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12817753008281742621758605755308036035",\n      "report_group": "9d50c4df3572db4a6058e0151faebee70b5c566aa553be805fb6bf168cdebeda",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.23081294963277525274551516232134221416",\n      "report_group": "e0e41b6ef61b0430b81ee9287740bf33e7ed850c3023012bbc87e3cac344956a",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762",\n      "report_group": "86007aa535133a458fffcb84201662ba6eb00caf1774e8bf9ffdcc359238a6cb",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.52435631723314823200374985065756182593",\n      "report_group": "4055de47ac2912eebe5b109ad041a89fce15ea10bb4d68b91113470a3297ec40",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.77164129615096222757740198026718214733",\n      "report_group": "2d5e89a491337d3c754229178b0fa3ffec32b0b015bda2ddde8949e9ab9a6f65",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.45894475516096841241467193775629362381",\n      "report_group": "f3cd744df80005d4f86606aa7457a724f513f115bbd26d37d9d0357029fe8d93",\n      "state_targets": [\n        -1,\n        0,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.28351884945918062351436698411322252525",\n      "report_group": "786a44826be349a86f17daa8127707aaa0a804a8afbad693df413b108930c6f7",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        3,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.30104491808697888642424160917715722238",\n      "report_group": "5f3d5046f0f44816cee0124d13f61c8dd5ed2e00d2059feeb045da155df3d9b8",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.37046961643473249831189497076699393868",\n      "report_group": "961965ba72e2303ec79940cdcc41c4285d5e6e457e759872843aab73c5be7fb2",\n      "state_targets": [\n        5,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.76742750470853901575934097885382242952",\n      "report_group": "fc216881cc7d6281d86d21bd5834c594d05c091ef5fe32e60c31262f398a6caa",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.56385473079196655120993854770623075553",\n      "report_group": "1c9fde4d74a359f352aebecd0c8316a23f46149ff5e10438fb35c603225d5d70",\n      "state_targets": [\n        5,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11836929166973549727078374306175992474",\n      "report_group": "a8ef54f1dae8e8d0a95c69d07eee39020744743af7a6b7b6897354ede61c5ee6",\n      "state_targets": [\n        5,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.16364873390453802195568603605191835957",\n      "report_group": "f52567cb8c5f162935a477a908abe999d13a06ef5433967680a2311c332c4282",\n      "state_targets": [\n        -1,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.37438631765770807892914198396852544668",\n      "report_group": "bfc370f11a3412aaddf429a750d26c288c0399df2b0effc52616cc37df22e7d9",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10902166579916526176697193295886992412",\n      "report_group": "d3e302e67140bb6384f3bc529ff2f5b4745774f22af84b5a4d1239ed81ea41a2",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819",\n      "report_group": "06275dd195223abab6d569e587ef46617d0c5d58059792edbf5ee1bd5845fbe8",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        4,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.86142930956005716169649168306759482915",\n      "report_group": "f28b4bc6b4e3232e10e83f074647af1d438c1690d62ea17bfe7354af863de512",\n      "state_targets": [\n        0,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269",\n      "report_group": "2915e7609d859eeb2c3346b84967e6e478edd82b93029d589ddb2639f8ddee5c",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.65597375829884314152221851871550349856",\n      "report_group": "5d1baa7f8030d8f7e6b930086926ba53b839afef32df7d82edbff4f50920312b",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994",\n      "report_group": "1562c4389cd3a8133a2a543c660a920831f45fe8f5bcce54a1cd795e725963d1",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        6\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.45072896942121803896760563154385918032",\n      "report_group": "7050658349faa1abe1b09800aa6462498a5ced896580a8ab402faabd998d32e9",\n      "state_targets": [\n        -1,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.37257728513136112802363958568529758895",\n      "report_group": "abc0a3d0e4bb8608a65a77992e329dac5c6a62d540b8266fc46e49bad12efe7f",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466",\n      "report_group": "478f6500f4bd4f90df087176e2d41fd56d6be8d3f7b53fae698693b507dc6842",\n      "state_targets": [\n        -1,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.70271042033022764422915852922035168712",\n      "report_group": "80c15966f5600e2de58426755ca3ff591671acdc0317a1169a75af0ca756e090",\n      "state_targets": [\n        4,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11440214477653629899108986507154583794",\n      "report_group": "bbd041f5fdd0cec5603f695952d7fa45ffde4e89e55fd168b95e5a22a114b753",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.90976227335921831531105845602655777296",\n      "report_group": "242d268f65d91569ccaad501c79525e5fe32e14a2644762f044b9579b5d3d2b6",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119",\n      "report_group": "217142eb927e6d4060a977a12f48b617f773bafa75b96b5e067409b26f1911e6",\n      "state_targets": [\n        0,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        3,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930",\n      "report_group": "4d6f9f1b52ec114f1c823bce1d6be63597cdacfa785b00cf063793d1d5fb233f",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456",\n      "report_group": "5c09feec5860e742bc9296ed382356c6aa68f49274b903af3f0c48044a1f0d52",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315",\n      "report_group": "676eb1ee2c0d2c9934eb832bab35c98ba12ec723b9c4077c3072a029a11aed46",\n      "state_targets": [\n        0,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        3,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569",\n      "report_group": "50f68e7b7cdd1479002415e172cb8f8af400b3516e5a5c736e3f3b0789ee8741",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.62205799963299902160268565319924399775",\n      "report_group": "f089478d685cc9363fdf052393593132b5173e5b002cdc0d8cf956a2e1b98646",\n      "state_targets": [\n        5,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12625250919597111075072308824921101918",\n      "report_group": "f493f9c5e3eb6e13e8f7143641e13873ad538cb5bce0650117cc15a333ce8309",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530",\n      "report_group": "ae0fb4e79a783824a26b67c1e6bbf71434f3c2aa4f1d8e4ef3a881fc13101e73",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968",\n      "report_group": "b39673df6e772ee51558bd74b11fab7b2eeb7cd9e9365928a01597aba36033d3",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720",\n      "report_group": "d256d36cc36e88dc7b9443c5bcc96d2e116831a46e711570f829062536460c87",\n      "state_targets": [\n        0,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.45625191292968830335971647388746041117",\n      "report_group": "b76860ee8d1c6641170be9a3e3131e4d3d797b97c0dbae972c38ceb88b6b7925",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689",\n      "report_group": "e542d28aef22ec611063cd2b25f0645e506f7a911c01b979f4c7575bc846e28d",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.97004483591331502283638578516018997228",\n      "report_group": "b97dc65b6728cbb104390adbe15793ea0d4f58c478d6576667fc49ab0c55d147",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.48521453765449688946535921546652765399",\n      "report_group": "d54e005036d9ea71688c9c0853eae3cabe7bb4c20dbab819744a9ecd15d1030e",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12724250000565012779853884787450109956",\n      "report_group": "4442c7db12d9fa73314dc194bbc61baf44af3227829d91cf99c909f56caf6832",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734",\n      "report_group": "56753b007028ebfff61d82afbf5737a54c4c3cab1393c70bc3e26ea0c135a60b",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    }\n  ],\n  "confirmation": [\n    {\n      "id": "1.2.826.0.1.3680043.8.498.93623421864452751381778712192879121271",\n      "report_group": "8690e0a010a665095d1257f429ede766a68f166fc788247cf5283c6a93d9cbac",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367",\n      "report_group": "a112785abb8cf8328c1e8ef50f50e940143d26ea419a2420acfea963d7a52c04",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.67772676243889350087068112324015156848",\n      "report_group": "f128457bbf4842d4c31254af49ad2dd5a48214c4de09282665bb96837ac22066",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.40723835064354784133289444204866484476",\n      "report_group": "5397efe08f4fb26faa9efac05ae6db18490b41dc86c93f8da3cd7201021eafdc",\n      "state_targets": [\n        0,\n        0,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.30947630416867960170317316514422933066",\n      "report_group": "010422c625f83e6f921f9e518ccb459c02ee35c53a28666cb5932ce4f04520f3",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.25909078145370189195643713208718315003",\n      "report_group": "8017a33963e35ceea4d5c7c054bb6618514f34fb62a2f863897b521b105a9bfd",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431",\n      "report_group": "d2c5f8c122e4b89d1b78785ed37d67867a6f5acabcf02caff704cf8393a04902",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.38701927697650476092773702704871872598",\n      "report_group": "7b0207c837569939ec338bdedb4bdcfea79431b8a12b6940e18d99ff46fc4ccf",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719",\n      "report_group": "ff42f0251075a15f931d6322bbc2ba68984dffadb2e727e9ff55f02985d80be6",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.37713008080610827760634302982073393454",\n      "report_group": "95bf29fc2ffcde2d034e6ba32a4e605ea2883255d9126839e83f30bb4f04b512",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347",\n      "report_group": "68bbd59adf83f7d39234573e8673fbbd5845ab993786f33a5296f004916be524",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382",\n      "report_group": "4d284de6faabb4a10676cc4bf9a12d4f7e1e8843bdfde71dbe0caef541438718",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292",\n      "report_group": "930aaf6f94e7d47f022b3756b65a3949114a168fa98567525df9e6ed121ce8e5",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.19977273190231806393528180968361858122",\n      "report_group": "7c63ac235acee3be7815861f822b57b0af82009309e82dbd29feb107e2b6440e",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.80180578697901833764761405542790185180",\n      "report_group": "b7d1beeef7ea224e2e5945f3478e4a2a2fd7deda9f71a5c5101e27da7f925bde",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.28425280718252632810092859612928047098",\n      "report_group": "49e77bf71b07c5820f2772613241a15026adde8280354c78a04876c80c3f3005",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12626383124142146001532197239671686442",\n      "report_group": "dec73b58123a6016c0604a2db03128993605928d8e62889332e7c812b264a602",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11150751959089942633754454715290410758",\n      "report_group": "70245ef9a39141d15bf4be92791dafbc23380e4e0abf5083417de2024f96699c",\n      "state_targets": [\n        0,\n        0,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868",\n      "report_group": "190c584c9465589909eca1ca13bac255cee0c51fbbedd97ea862742b6147a75f",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781",\n      "report_group": "7c3e2680f32c61ebf85a821a08e328fe4157d197b18eb074321d4a374bd5dd2e",\n      "state_targets": [\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.19832308119617081676615236082965891208",\n      "report_group": "9cd8ccf5eabee32b280360d66ee54a49c35f2694b94be9d370da5011b85932c1",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12051169109587120248372991913002670813",\n      "report_group": "516d5265c025056f59a09935e3e866946dc9d79f4a89827d372db0a00085e4cd",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.23107903961514228607587963030481481744",\n      "report_group": "c74814526550b61a52b55c58f0e0c7a3826d2a7ba3aab8355d5cc0cedb9e8363",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.50687743970686088872471288288960579306",\n      "report_group": "fedac49c978cfaf75461eab43c17505ceaba6e932dba89c0285711d96174cf30",\n      "state_targets": [\n        -1,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885",\n      "report_group": "546ba48a744e0fcd121405c1fd6b94e0f2a6b48ba9be175484c595536a286437",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        7\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607",\n      "report_group": "0d54573d9334df283ba3af0dc5441bca5180f18f8ff6a5f2805125a90acf523b",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032",\n      "report_group": "fa8a82f39cebed8d398735a53eae6a4e081052717c97c3e7a0b2941dcf6f79c9",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13176146790594027813190776550419533937",\n      "report_group": "ea5daed47a19f68f662faa7a57cff629a32a2e18bcc814fb3886f1a60af3eee1",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129",\n      "report_group": "abe22676a65cddf87ba08795fd0da7a5509dfae70050a1be6c39edca957a9496",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.64172939203926034064618447795131041647",\n      "report_group": "2270d48eb0ac76341b7fbebee4c3d506f5b040a17c342c28b9f36efd5a9e7350",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715",\n      "report_group": "465fd2be08e30943d614da12effcf361e923316fd7d114f094577c80f6197707",\n      "state_targets": [\n        5,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121",\n      "report_group": "f8c9e93345a537fbb4e0a88ce6b701e4217baa6d94b8b720bf050bf318a32ac6",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11988551795896617815207749665102049499",\n      "report_group": "f1d8ef291e49cdf7bf4ff73b63584b687baca3090268bf19feda599297bbb187",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120",\n      "report_group": "62faa40516a83e629e3db4f276faab9cef474d9196cf60a1162fb1908d16624f",\n      "state_targets": [\n        5,\n        5,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779",\n      "report_group": "d3fdb2e60b14e0d92eb9090e62d2ecee324c8ca0545813f23ad3d10cfd386575",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803",\n      "report_group": "8329a08fea446aac1d6c48c3db5b53bb603da43043b87f6b2eb021456f4fcea2",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047",\n      "report_group": "4105f2ced596c2228e75f023786a12448d6507004c639bc1450429c6dde0512c",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839",\n      "report_group": "0847a6179cd5d001af90c98be58fae18743459e964fb4b951f34e22e8370d503",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11592056376890319271545710970155310824",\n      "report_group": "cd6ca0c0d4a850b039a5c4f12b33e78b2218d4f3e21af99d828c1ec6283a3b48",\n      "state_targets": [\n        5,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.71157888664602228768823145752952837565",\n      "report_group": "2d655a289e55b1f9bb4dc5b0eb085c4d8979041f813c0cd595d3ff4ad26b15c8",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.14325424371811230497993759959862738838",\n      "report_group": "b179884d743a2840217244afe15b869d6f6c7e00264cf46ee16f1494aee86b14",\n      "state_targets": [\n        0,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        5,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018",\n      "report_group": "fe1c4dd8793d821828a363f94ac038bab8a1254d5ff77b4fa293a1b60094d911",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185",\n      "report_group": "432bf2be6b4a1b048e3ea3a59eb1a79ff4d51bbd1adc643b2c0bdfdf7fa7091b",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.11629659616217352895144469980856259097",\n      "report_group": "dc26986a049df1d595e865968137f841120619eafcac2ec875de51ea31425119",\n      "state_targets": [\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451",\n      "report_group": "cb883f329cbf2d62ad2e4594a3e91329267e57232fde135722f72c919a1afc7e",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10454875772176439944597553400963295397",\n      "report_group": "e1a9d3056c31a6a2bca846c0a01ee2b3994a65cf84e2f3f0797f348958013bec",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        1,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.26946980494141305258556994153410856428",\n      "report_group": "15f53e01a25afe3bbbb10b700264150a881aeb18f72ce05c782de4fcd49fceeb",\n      "state_targets": [\n        0,\n        0,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962",\n      "report_group": "cf09ab966c583581daad03ef0bff89d15d943cfb293724b8d23b4ebb0c19ac25",\n      "state_targets": [\n        5,\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044",\n      "report_group": "b397dc1b86ab5cabd167e3659a95f5187f1039e73eda40328eed64d7624732e0",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882",\n      "report_group": "494e08813100efc55f20035b1febb70917e8489e2465a28717208710ac7b9f41",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164",\n      "report_group": "1046e6cfc3bb9ee5390966547f7863dc4441e9762662c4b468ec576cbb1a9ab7",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.23713320796530316370039202981138376061",\n      "report_group": "fbe64267bb36a282b1ba549130104c171796a20fac1bd2b911f598e190706340",\n      "state_targets": [\n        4,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.61581513083114643407028382481113097015",\n      "report_group": "31405ba2a0dd1c5fb86ca0a9bc4db859ccb04d069f726debf512f96a201851a8",\n      "state_targets": [\n        5,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080",\n      "report_group": "3ebecaeb22f6538277db6e7eb4800851dd8ae6a1e4c3e493f95026c90e1ee222",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.19592989024994864431870593504844309885",\n      "report_group": "b065551175befb7628b0fd12f6e577bf6236f4c6a65734dafebba6f4f6b094be",\n      "state_targets": [\n        -1,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.60259828181878278590614805828898992241",\n      "report_group": "71b11798ba966ea962a91269e9bea638c5e260f193c693a148c0fc1c557be9b3",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.99052905406751383674032587504235261874",\n      "report_group": "8adfa75bf63d9fe45e3679eaa45b109da84298f94c9a9e3db0341ad2d434a34a",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        3,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.82447100001417065190678617013741085481",\n      "report_group": "d7f80c4bf969b8b40ffa051fb30592414703eb49edc5414b87cd4ccfe430f5b6",\n      "state_targets": [\n        -1,\n        -1,\n        3,\n        0,\n        -1,\n        -1,\n        -1,\n        4,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.37580209294522316445963449455348371815",\n      "report_group": "e7de286e2f1189fa5e57adb230d3ff5a1ebaceab430f8f37af932a65de94927d",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        1,\n        -1,\n        -1,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.38796519786809225694393617946244012065",\n      "report_group": "bbb53ed17d9530b7f2f4a282e5e172264ce68c19187043c77a60fae6e0b2ead3",\n      "state_targets": [\n        5,\n        5,\n        3,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.12987428318535758055829100395092244687",\n      "report_group": "a5f52c40c2d05d9e7f651a1831fa31caa10491e0fef92e8cd35dec8523dadc78",\n      "state_targets": [\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139",\n      "report_group": "1f7dd212e47db88b0e3cf41f7910885773f3b6d9ee05a57763d43f46b4fbe256",\n      "state_targets": [\n        -1,\n        0,\n        3,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158",\n      "report_group": "94f79378f960039acc0ae5583e247b965762d14bc5e8ec32521a802d5e2809dc",\n      "state_targets": [\n        0,\n        0,\n        0,\n        0,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        0,\n        -1,\n        -1\n      ]\n    },\n    {\n      "id": "1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386",\n      "report_group": "7426c4527c3e6a3ccaf3e5df741a3621ca46c60c90aede4cb63ae721493e64f4",\n      "state_targets": [\n        -1,\n        0,\n        -1,\n        -1,\n        -1,\n        -1,\n        -1,\n        0,\n        -1,\n        1,\n        -1,\n        -1\n      ]\n    }\n  ],\n  "pathology_truth": false,\n  "submission_created": false,\n  "state_observation_counts": {\n    "fit": {\n      "ACL": 84,\n      "MCL": 60,\n      "Medial Meniscus": 124,\n      "Lateral Meniscus": 112,\n      "Medial OA": 0,\n      "Lateral OA": 0,\n      "PF OA": 0,\n      "Effusion": 86,\n      "Synovitis": 12,\n      "Baker\'s": 53,\n      "Contusion": 0,\n      "Fracture": 5\n    },\n    "tune": {\n      "ACL": 24,\n      "MCL": 17,\n      "Medial Meniscus": 37,\n      "Lateral Meniscus": 36,\n      "Medial OA": 0,\n      "Lateral OA": 0,\n      "PF OA": 0,\n      "Effusion": 28,\n      "Synovitis": 5,\n      "Baker\'s": 23,\n      "Contusion": 0,\n      "Fracture": 3\n    },\n    "confirmation": {\n      "ACL": 33,\n      "MCL": 30,\n      "Medial Meniscus": 34,\n      "Lateral Meniscus": 36,\n      "Medial OA": 0,\n      "Lateral OA": 0,\n      "PF OA": 0,\n      "Effusion": 29,\n      "Synovitis": 7,\n      "Baker\'s": 24,\n      "Contusion": 0,\n      "Fracture": 5\n    }\n  }\n}\n', 'postprocess.py': '"""Pinned R6 TTA and deployed readout assembly; fail closed, never refit weights."""\nimport importlib.util\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom strict_reconstruct import require, SPEC_SHA\n\nINFER_SHA = "8401812cd63737ec99839d317c4037c4efb445d1baec892fa7481bb30399cc2d"\nFLUID_SHA = "9fbd6f652470c5ef131f997edb4e358038937dd45e6f6b65a4afed80e10cf35d"\nSTATE_SHA = "b05147bb463a389a66a4a0d7ff0555efd2bde6f486718f79e9d7fa3d90b24522"\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n          "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\n\n\ndef load_bundle(worker_path, spec_path, fluid_path, state_path):\n    require(worker_path, INFER_SHA); require(spec_path, SPEC_SHA)\n    require(fluid_path, FLUID_SHA); require(state_path, STATE_SHA)\n    module_spec = importlib.util.spec_from_file_location("pinned_r6_worker", worker_path)\n    worker = importlib.util.module_from_spec(module_spec); module_spec.loader.exec_module(worker)\n    spec = json.loads(Path(spec_path).read_text())\n    layout = worker.state_layout(spec, LABELS)\n    fluid = worker.fluid_stacker_load(fluid_path, 1)\n    import hashlib\n    state = worker.state_stacker_load(state_path, LABELS, layout, hashlib.md5(Path(spec_path).read_bytes()).hexdigest())\n    if set(fluid["classes"]) & set(state["classes"]):\n        raise ValueError("postprocessors overlap")\n    bins = [(b, l) for b, labs in spec["binaries"].items() for l in labs]\n    perm, bperm = worker.mirror_perm(LABELS, spec, bins)\n    mcl_columns = []\n    for column in layout["columns"]:\n        if column["class"] == "MCL":\n            if "categories" in column:\n                mcl_columns.extend(range(column["start"], column["stop"]))\n            else:\n                mcl_columns.append(column["column"])\n    return {"worker": worker, "spec": spec, "layout": layout, "fluid": fluid, "state": state,\n            "perm": perm, "bin_perm": bperm, "bin_names": bins, "mcl_columns": mcl_columns}\n\n\ndef combine_passes(bundle, model, plain, mirrored):\n    """One fold, batched studies: deployed logits plus deployed state readout."""\n    z, sl, bl = plain; zm, slm, blm = mirrored\n    if z.shape != zm.shape or z.ndim != 2 or z.shape[1] != 12:\n        raise ValueError("class logit shape mismatch")\n    if not torch.isfinite(z).all() or not torch.isfinite(zm).all():\n        raise ValueError("nonfinite class logits")\n    worker = bundle["worker"]; p = bundle["perm"]\n    combined = .5 * (z + zm[:, p]); combined[:, 1] = z[:, 1]\n    states = worker.state_readout(model, sl, bl)\n    reflected = worker.state_readout(model, slm[:, p], blm[:, bundle["bin_perm"]])\n    deployed = .5 * (states + reflected)\n    deployed[:, bundle["mcl_columns"]] = states[:, bundle["mcl_columns"]]\n    if not torch.isfinite(deployed).all():\n        raise ValueError("nonfinite state probabilities")\n    return combined, deployed\n\n\ndef sigmoid(value):\n    return 1 / (1 + math.exp(-value)) if value >= 0 else math.exp(value) / (1 + math.exp(value))\n\n\n@torch.inference_mode()\ndef apply_postprocessors(bundle, fold_logits, fold_states, fluid_record):\n    """One study; float32 fold/TTA mean before float64 stackers, as deployed."""\n    if fold_logits.ndim != 2 or fold_logits.shape[1] != 12 or not torch.isfinite(fold_logits).all():\n        raise ValueError("invalid fold logits")\n    if fold_states.shape != (len(fold_logits), bundle["layout"]["width"]) or not torch.isfinite(fold_states).all():\n        raise ValueError("invalid state readout")\n    z = fold_logits.mean(0)\n    probabilities = torch.sigmoid(z).double().cpu().numpy().copy()\n    z64 = z.double().cpu().numpy()\n    state64 = fold_states.double().mean(0).cpu().numpy()\n    for name, value in bundle["worker"].fluid_stack(bundle["fluid"], LABELS, z64, fluid_record)[0].items():\n        if not math.isfinite(value):\n            raise ValueError("nonfinite fluid stacker result")\n        probabilities[LABELS.index(name)] = sigmoid(value)\n    for name, co in bundle["state"]["classes"].items():\n        features = []\n        for feature, column in zip(co["features"], co["columns"]):\n            if feature == "z":\n                features.append(float(z64[LABELS.index(name)]))\n            elif feature in ("lv", "miss"):\n                value = None if fluid_record is None else fluid_record.get(co["volume_column"])\n                missing = value is None or not math.isfinite(float(value)) or float(value) < 0\n                features.append((0. if missing else float(np.log1p(float(value)))) if feature == "lv" else float(missing))\n            else:\n                features.append(float(state64[column]))\n        value = co["b0"] + float(np.dot(np.asarray(features, np.float64), np.asarray(co["coef"], np.float64)))\n        if not math.isfinite(value):\n            raise ValueError("nonfinite state stacker result")\n        probabilities[LABELS.index(name)] = sigmoid(value)\n    if not np.isfinite(probabilities).all() or (probabilities < 0).any() or (probabilities > 1).any():\n        raise ValueError("invalid final probabilities")\n    return probabilities\n', 'strict_reconstruct.py': '"""CPU-only R6 reconstruction and final-stage parameter selection; no optimizer."""\nimport argparse\nimport copy\nimport hashlib\nimport importlib.util\nimport json\nimport sys\nfrom pathlib import Path\n\nimport torch\n\nCHECKPOINT_SHA = "59cc88a197bc80c7fa5faa89e69ae7bec04f69cab5259a23a0d0ceb9431e53f6"\nSPEC_SHA = "3914242b0c404aee625ea65801f14f60adfee326c1ddd6fb96c48ed7050b7ff3"\nTRAINER_SHA = "04df7e2e154b2c2521696908f272940c63ce17e9b6337b85847fee64504eeaba"\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as stream:\n        for block in iter(lambda: stream.read(8 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef require(path, expected):\n    if sha(path) != expected:\n        raise ValueError(f"input identity drift: {path}")\n\n\ndef last_stage_only(model):\n    model.requires_grad_(False)\n    if not hasattr(model.enc, "stages") or len(model.enc.stages) != 4:\n        raise ValueError("unexpected ConvNeXt stage layout")\n    model.enc.stages[3].requires_grad_(True)\n    names = [name for name, p in model.named_parameters() if p.requires_grad]\n    if not names or not all(name.startswith("enc.stages.3.") for name in names):\n        raise ValueError("incorrect trainable scope")\n    return names\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--checkpoint", required=True)\n    parser.add_argument("--state-spec", required=True)\n    parser.add_argument("--trainer", required=True)\n    parser.add_argument("--timm-root", required=True, help="directory containing the audited timm package")\n    parser.add_argument("--out", required=True)\n    args = parser.parse_args()\n    require(args.checkpoint, CHECKPOINT_SHA); require(args.state_spec, SPEC_SHA); require(args.trainer, TRAINER_SHA)\n    sys.path.insert(0, str(Path(args.timm_root).resolve()))\n    import timm\n    if timm.__version__ != "1.0.25":\n        raise ValueError("review timm implementation before changing the pinned runtime")\n    torch.set_num_threads(4)\n    module_spec = importlib.util.spec_from_file_location("r6_original_trainer", args.trainer)\n    trainer = importlib.util.module_from_spec(module_spec); module_spec.loader.exec_module(trainer)\n    state = torch.load(args.checkpoint, map_location="cpu", weights_only=True)\n    if not isinstance(state, dict) or not all(isinstance(t, torch.Tensor) for t in state.values()):\n        raise ValueError("expected plain whole-model tensor state")\n    assert all(torch.isfinite(t).all().item() for t in state.values() if t.is_floating_point())\n    state_spec = json.loads(Path(args.state_spec).read_text())\n    # Exactly the flag inference used by the released inference worker.\n    flags = {"spatial": "sp_q" in state, "extent": any(k in state for k in ("ext_u", "ext_view", "ext_gamma")),\n             "syn_decomp": "syn_beta" in state}\n    reference = trainer.Net("convnext_small", None, crop=352, state_spec=state_spec, in_chans=3, **flags)\n    reference.load_state_dict(state, strict=True)\n    reference.eval().requires_grad_(False)\n    if set(reference.state_dict()) != set(state):\n        raise ValueError("strict state identity failure")\n    assert all(torch.equal(reference.state_dict()[k], v) for k, v in state.items())\n    assert all(reference.state_dict()[k].dtype == v.dtype for k, v in state.items())\n    # A missing head tensor must fail; no strict=False/backbone-only escape.\n    broken = dict(state); broken.pop("cls_w")\n    try:\n        reference.load_state_dict(broken, strict=True)\n    except RuntimeError:\n        negative_load_rejected = True\n    else:\n        raise ValueError("missing head tensor was accepted")\n    reference.load_state_dict(state, strict=True)\n    candidate = copy.deepcopy(reference)\n    trainable = last_stage_only(candidate)\n    assert all(torch.equal(candidate.state_dict()[k], v) for k, v in state.items())\n    # This synthetic head-only check is not raw-image/preprocessing equivalence.\n    generator = torch.Generator().manual_seed(20261020)\n    features = torch.randn(2, 768, 11, 11, generator=generator) if flags["spatial"] else torch.randn(2, 768, generator=generator)\n    meta = torch.zeros(1, 2, 9)\n    meta[0, :, 0] = 0; meta[0, :, 1] = torch.tensor([0, 4])\n    meta[..., 3] = 1.; meta[..., 8] = 2.\n    mask = torch.ones(1, 2, dtype=torch.bool)\n    with torch.inference_mode():\n        def output(model):\n            tokens = model.spatial_pool(features) if flags["spatial"] else features\n            return model.head(tokens, meta, mask, return_aux=True)\n        before, after = output(reference), output(candidate)\n    for a, b in zip(before, after):\n        assert torch.equal(a, b), "freezing changed initial outputs"\n    assert before[0].shape == (1, 12) and torch.isfinite(before[0]).all()\n    assert torch.isfinite(before[1][:, reference.state_valid]).all()\n    assert torch.isfinite(before[2]).all()\n    state_probs = before[1].softmax(-1)\n    assert torch.isfinite(state_probs).all() and torch.allclose(state_probs.sum(-1), torch.ones(1, 12))\n    timm_dir = Path(timm.__file__).resolve().parent\n    report = {"checkpoint_sha256": CHECKPOINT_SHA, "state_spec_sha256": SPEC_SHA,\n              "checkpoint_bytes": Path(args.checkpoint).stat().st_size,\n              "trainer_sha256": TRAINER_SHA, "timm_version": timm.__version__,\n              "timm_source_hashes": {name: sha(timm_dir / name) for name in\n                                     ("__init__.py", "models/convnext.py", "models/_factory.py")},\n              "torch_version": torch.__version__, "device": "cpu", "checkpoint_tensors": len(state),\n              "all_checkpoint_tensors_exact_after_load": True, "all_tensor_dtypes_preserved": True, "flags": flags,\n              "missing_head_negative_test_rejected": negative_load_rejected,\n              "candidate_state_unchanged_at_initialization": True,\n              "synthetic_head_outputs_bitwise_equal": True, "state_and_binary_logits_checked": True,\n              "trainable_parameter_tensors": len(trainable),\n              "trainable_parameters": sum(p.numel() for p in candidate.parameters() if p.requires_grad),\n              "total_parameters": sum(p.numel() for p in candidate.parameters()),\n              "trainable_names": trainable, "frozen_heads": True,\n              "gpu_used": False, "optimizer_steps": 0, "study_predictions": 0,\n              "raw_pixel_preprocessing_parity_tested": False, "postprocessor_parity_tested": False,\n              "promotion": False}\n    Path(args.out).write_text(json.dumps(report, indent=2) + "\\n")\n    print(json.dumps({k: v for k, v in report.items() if k != "trainable_names"}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, content in FILES.items():
    (OWN/name).write_text(content)


In [ ]:
ARM = 'E_lrprior'
BUNDLE_ID = 'fe055406fad6a12f'
DEADLINE_HOURS = 8.3
CODE_SLUG = 'rsna-orthofoundation-fold0-epoch7-20261004'
COMPETITION = 'rsna-knee-abnormality-detection'
CODE_SHA = {'native_cache.py': 'c53ce8985406b3b918d9e821d43518479e89d9f946daebd817f378e3949718b9', 'native_data.py': 'f59afe205c8e193369fa3ccd22edcce8df3b293205fc48ddb1dd45169ee43b36', 'input_provenance.py': 'b3a22500c7f976f6ee040239ada5cacecf2326861cb49af1c7d8cfdeeb55fcda', 'intensity_mapping.py': '19be2d8cbd6f8971c1288d7512a8cca41d143470c2ba6e9b9096fde035fa9d41', 'coronal_views.py': '841f216d2682ae2c0f8afd82fc902b9fa74924d7145b952069dea8b0bbd03c40', 'exact_duplicates.py': '51b38db21ce9c5fe2b6278a835983414756b61ac6af81aba380b44b41b075897', 'dicom_geometry.py': '12aa501f9704aef15b7a3bbd560951acb6d95b89b7de0141d6cf89fc3f051ab1', 'common.py': '8283fa1f8e7ef8cfacbba52df1715636be3c9ad688e39912814e7d5bcf57105f'}
SOURCE_SHA = {'px_cache.py': 'b13e38a7ad3bd496f8027e02209653e224b0926db4f0853eeb0d076a2d6e2632', 'compact_train.py': '04df7e2e154b2c2521696908f272940c63ce17e9b6337b85847fee64504eeaba', 'compact_infer.py': '8401812cd63737ec99839d317c4037c4efb445d1baec892fa7481bb30399cc2d', 'probe_infer.py': 'a6cc31448019c84be5868d0a2ff1d99719b23b0f86110efc03efff3ce6bd6980', 'probe_dual.py': 'a6155eb08701f9bb746f838bc8ef8e2129c85f76a5c58e379692ffe2e76bbef0', 'knee_anchor_bank_v2.py': 'e564807a129dfcd3c984348917436ee6083066a4d685340587ebd1d3ea14f309', 'side_common.py': 'bbcc5d027834e6a6fd48c4a8cf228e25661b7522c792587c840fe4d7cd5c0566'}
PROBE_CLASS = None
C_SLUG = 'rsna-knee-r6-frozen-fe055406-w1'   # [d957] the frozen copy of the R6 bundle dataset (manifest sha256 6508ce6921eda986.. = 57068142's)
FLIP_TTA = True   # [d957] package E: v4c's flip TTA on R6
W_SLUG = 'rsna-knee-ofl-ssl-weights'
SOURCES = {'px_cache.py': '"""Per-slice pixel cache on the fast disk, for training compact image models without re-reading F:.\n\nData preparation only: no model, no optimizer.  Reads F:/rsnaknee/cache_native (read-only) and writes\nONLY under --out:\n  px.u8.npy    [N, S, S] uint8   one row per (series, view, slice), slices in physical order\n  meta.f32.npy [N, 12]\n  index.json   study -> [start, n_rows, n_series], per-series blocks, config, progress\nEach row is exactly one channel of the mainline\'s validation-mode window (same crop geometry, same\nINTER_AREA resize, same series_companded intensity mapping), at S pixels instead of 384.\nViews: full field + 100 mm local; coronal uses the mainline\'s bilateral views.\n"""\nimport sys, os, json, time, hashlib, argparse\nsys.dont_write_bytecode = True\nCODE_ROOT = os.environ.get("RSNA_CODE_ROOT", "D:/rsnaknee")      # mainline checkout (Linux box: /root/autodl-tmp/rsnaknee)\nsys.path.insert(0, CODE_ROOT)\nimport numpy as np\nimport torch\n\nLOCAL_MM = 100.\nTHIN_MM, SLAB_MM = 1.75, 3.5       # stacks thinner than THIN_MM are averaged into SLAB_MM slabs (see StudySlices.build)\nPOLICY = "series_companded"\nPLANE_ID = {"Sagittal": 0, "Coronal": 1, "Axial": 2}\nMETA_COLS = ["series_idx", "slot", "plane", "view_id", "rel_pos", "log_spacing", "extent_130", "fluid",\n             "fatsat", "slice_order", "n_slices", "real_series_idx"]\nUIDK = "StudyInstanceUID"\n\n\ndef study_json(cache, uid):\n    return os.path.join(cache, "studies", hashlib.sha256(uid.encode()).hexdigest()[:24] + ".json")\n\n\n# View sets.  v1: full field + 100 mm local at the field centre (coronal: bilateral locals).  v2 adds, for sagittal and axial\n# series, a 100 mm "patellofemoral" local (view id 4; ids 2/3 are the coronal bilateral locals) centred anterior-superior: in the canonical cache anterior is +column\n# for sagittal and -row for axial, and a centre-of-field local cuts the patella at its edge in every study inspected\n# (10-06; PF OA is the class with the largest report-vs-image gap on the 58 official studies, 0.92 vs 0.82).\n# Centre tuples are (row fraction, column fraction) as view_geometry expects.\nPF_CENTER = {"Sagittal": (.38, .60), "Axial": (.38, .50)}\n# v3 (high-resolution set, meant for a 384 px cache): NO full-field view; sagittal/axial = centre local (1) + patellofemoral\n# local (4) + posterior local (5: popliteal fossa / Baker\'s cyst / PCL; posterior = -column for sagittal, +row for axial);\n# coronal = the two bilateral locals only.  Three overlapping 100 mm locals cover what the 1.8 px/mm full view showed.\nPOST_CENTER = {"Sagittal": (.55, .35), "Axial": (.62, .50)}\n\n\ndef knee_anchor(vol, plane):\n    """(row_frac, col_frac) of the knee in the series\' in-plane frame from the foreground of the 3 middle slices.\n    Falls back to the field centre when the leg fills the field or nothing is found."""\n    import cv2\n    m = len(vol) // 2\n    mid = np.asarray(vol[max(0, m - 1):m + 2]).astype(np.float32).mean(0)\n    H, W = mid.shape\n    hi = float(np.percentile(mid, 99.5))\n    if hi <= 0:\n        return (.5, .5), "empty"\n    fg = (mid > 0.08 * hi).astype(np.uint8)\n    n, lab, stats, _ = cv2.connectedComponentsWithStats(fg, 8)\n    if n <= 1:\n        return (.5, .5), "no_fg"\n    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    x, y, w, h = [int(v) for v in stats[k, :4]]\n    if w >= 0.97 * W and h >= 0.97 * H:\n        return (.5, .5), "fills_field"\n    comp = (lab == k)\n    if plane in ("Sagittal", "Coronal"):\n        width = comp[y:y + h].sum(1).astype(np.float32)\n        if len(width) >= 15:\n            kern = np.ones(15, np.float32) / 15\n            width = np.convolve(width, kern, mode="same")\n        row = (y + int(np.argmax(width))) / max(1, H - 1)\n    else:\n        row = (y + h / 2.) / max(1, H - 1)\n    col = (x + w / 2.) / max(1, W - 1)\n    return (float(row), float(col)), "ok"\n\n\ndef anchored(anchor, spacing_mm, shape, d_row_mm, d_col_mm):\n    """centre = anchor shifted by physical offsets (rows: +down, cols: +right), as (row_frac, col_frac)."""\n    sy, sx = float(spacing_mm[0]), float(spacing_mm[1])\n    H, W = shape\n    return (min(1., max(0., anchor[0] + d_row_mm / (sy * max(1, H - 1)))),\n            min(1., max(0., anchor[1] + d_col_mm / (sx * max(1, W - 1)))))\n\n\ndef series_views(plane, native_shape, spacings, size, views="v1", anchor=None, spacing_mm=None, patella=None):\n    if views in ("v3k", "v4"):\n        # knee-centred set from the anchor bank (SKM-TEA joint centre projected through the geometry bank):\n        # v4 adds the full field of view (id 0) of every series in front of the locals, so nothing is out of sight.\n        full = [(0, None, (.5, .5))] if views == "v4" else []\n        # centre local (1) at the joint; PF local (4) at the patella anchor when the segmentation found it, else\n        # anterior-superior of the joint; posterior local (5); coronal medial (2) / lateral (3) 30 mm either side.\n        # Anterior = +col (sagittal), -row (axial) in the canonical cache.\n        shp = tuple(native_shape[-2:])\n        a = anchor if anchor is not None else (.5, .5)\n        sp = spacing_mm if spacing_mm is not None else (1., 1.)\n        if plane == "Sagittal":\n            pf = patella if patella is not None else anchored(a, sp, shp, -25., +40.)   # patella centre ~40 mm anterior of the joint centre (gold montages: +30 cut it)\n            return full + [(1, LOCAL_MM, a), (4, LOCAL_MM, pf), (5, LOCAL_MM, anchored(a, sp, shp, +5., -35.))]\n        if plane == "Axial":\n            pf = patella if patella is not None else anchored(a, sp, shp, -40., 0.)\n            return full + [(1, LOCAL_MM, a), (4, LOCAL_MM, pf), (5, LOCAL_MM, anchored(a, sp, shp, +35., 0.))]\n        return full + [(2, LOCAL_MM, anchored(a, sp, shp, 0., -30.)), (3, LOCAL_MM, anchored(a, sp, shp, 0., +30.))]\n    if plane == "Coronal":\n        if views == "v2add":\n            return []\n        from training.coronal_views import coronal_bilateral_views\n        out = coronal_bilateral_views(tuple(native_shape[-2:]), spacings, size, LOCAL_MM)\n        if views == "v3":                     # bilateral locals only (view id 0 is the full field)\n            return [v for v in out if int(v[0]) != 0]\n        return out\n    if views == "v2add":                      # add-on cache: ONLY the rows v2 adds to v1 (coronal adds none)\n        return [(4, LOCAL_MM, PF_CENTER[plane])] if plane in PF_CENTER else []\n    if views == "v3":\n        return [(1, LOCAL_MM, (.5, .5)), (4, LOCAL_MM, PF_CENTER[plane]), (5, LOCAL_MM, POST_CENTER[plane])]\n    out = [(0, None, (.5, .5)), (1, LOCAL_MM, (.5, .5))]\n    if views == "v2":\n        out.append((4, LOCAL_MM, PF_CENTER[plane]))\n    elif views != "v1":\n        raise ValueError("unknown view set " + str(views))\n    return out\n\n\nclass StudySlices(torch.utils.data.Dataset):\n    def __init__(self, cache, uids, size, views="v1", anchors=None):\n        self.cache, self.uids, self.size, self.views = cache, uids, size, views\n        self.anchors = anchors                   # directory of <sha24(study)>.json from knee_anchor_bank_v2.py (views v3k)\n        self._resolver = None\n\n    def __len__(self):\n        return len(self.uids)\n\n    def __getitem__(self, i):\n        uid = self.uids[i]\n        try:\n            return self.build(uid)\n        except Exception as exc:\n            return {"uid": uid, "error": repr(exc)[:400], "px": torch.zeros((0, self.size, self.size), dtype=torch.uint8),\n                    "meta": torch.zeros((0, len(META_COLS)), dtype=torch.float32), "blocks": [], "removed": 0}\n\n    def build(self, uid):\n        import cv2\n        cv2.setNumThreads(1)\n        from training.input_provenance import view_geometry\n        from training.intensity_mapping import normalize_intensity\n        from training.native_data import select_display_window\n        from training.exact_duplicates import ExactDuplicateResolver\n        if self._resolver is None:\n            self._resolver = ExactDuplicateResolver(None, None, False)\n        S = self.size\n        with open(study_json(self.cache, uid), encoding="utf-8") as fh:\n            study = json.load(fh)\n        # study JSONs carry the builder\'s (Windows) path of every series: relocate to this cache before anything reads them;\n        # a series the native cache never rendered (12 studies / 31 series) is dropped so the study keeps its other series\n        series, skipped = [], []\n        for s in study["series"]:\n            path = s["path"]\n            if not os.path.exists(path):\n                path = os.path.join(self.cache, "series", os.path.basename(path.replace(chr(92), "/")))\n            if not os.path.exists(path):\n                skipped.append(s["SeriesInstanceUID"]); continue\n            series.append(dict(s, path=path))\n        if not series:\n            raise ValueError("no series file present for this study (%d missing)" % len(skipped))\n        study = dict(study, series=series)\n        plan = self._resolver.resolve(study)\n        self._resolver.cache.clear()\n        px, meta, blocks = [], [], []\n        for real_idx in plan["representative_indices"]:\n            s = study["series"][real_idx]\n            path = s["path"]\n            volume = np.load(path, mmap_mode="r", allow_pickle=False)\n            pos = np.asarray(s["positions"], float)\n            if volume.ndim != 3 or len(pos) != len(volume) or not len(pos) or not np.isfinite(pos).all():\n                raise ValueError("native volume/slice-position mapping is incomplete")\n            if s["plane"] == "Sagittal" and s.get("side") == "R":\n                pos = -pos\n            order = np.argsort(pos, kind="stable")\n            pos = pos[order]\n            spacing = float(np.median(np.diff(pos))) if len(pos) > 1 else 0.\n            window = select_display_window(s, POLICY)\n            fluid = s.get("fluid_sensitive", -1)\n            fat = s.get("fat_suppression", -1)\n            sidx = len(blocks)\n            vol = np.asarray(volume[order])                      # slices in physical order, read once\n            order_rep = order                                     # native slice index behind each stored slice (in-plane spacing lookups)\n            if 0 < spacing < THIN_MM:\n                # thin 3-D stacks (0.4-1.5 mm): average groups of slices into ~SLAB_MM slabs, so every slice contributes\n                # (a per-slice window sampler at a physical stride would skip ~90 % of a 0.4 mm stack) and the series\n                # carries the same slab statistics as a 2-D acquisition.  Same code at Kaggle inference (StudySlices).\n                g = max(1, int(round(SLAB_MM / spacing)))\n                n_new = len(pos) // g + (1 if len(pos) % g >= max(1, g // 2) else 0)\n                if n_new >= 2:\n                    chunks = [vol[j * g:min(len(vol), (j + 1) * g)] for j in range(n_new)]\n                    vol = np.stack([np.rint(c.astype(np.float32).mean(0)).astype(vol.dtype) for c in chunks])\n                    pos = np.array([pos[j * g:min(len(pos), (j + 1) * g)].mean() for j in range(n_new)])\n                    order_rep = np.array([order[j * g] for j in range(n_new)])\n                    spacing = float(np.median(np.diff(pos)))\n            block = {"series": s["SeriesInstanceUID"], "plane": s["plane"], "n_slices": int(len(pos)), "views": []}\n            anchor = pat = None\n            sp0 = None\n            if self.views in ("v3k", "v4"):\n                if self.anchors:\n                    ap_ = os.path.join(self.anchors, hashlib.sha256(uid.encode()).hexdigest()[:24] + ".json")\n                    ent = (json.load(open(ap_)).get("series", {}) if os.path.exists(ap_) else {}).get(s["SeriesInstanceUID"])\n                    if ent and ent.get("in_field", True):\n                        anchor = tuple(float(v) for v in ent["joint"])\n                        pat = tuple(float(v) for v in ent["patella"]) if ent.get("patella") else None\n                if anchor is None:\n                    anchor = (.5, .5)                                # no anchor entry: field centre (the foreground heuristic was worse on g02)\n                sp0 = s["spacing_mm"][int(order_rep[0])]\n            for view_id, crop, center_xy in series_views(s["plane"], volume.shape, s["spacing_mm"], S, self.views, anchor, sp0, pat):\n                block["views"].append([int(view_id), len(px)])\n                for k in range(len(pos)):\n                    sp = s["spacing_mm"][int(order_rep[k])]\n                    g = view_geometry(vol.shape[1:], sp, S, crop_mm=crop, center=center_xy)\n                    x, y, x1, y1 = g.crop_xyxy\n                    pxo, pyo, _, _ = g.valid_xyxy\n                    oh, ow = g.resized_shape\n                    resized = cv2.resize(vol[k, y:y1, x:x1].astype(np.float32), (ow, oh), interpolation=cv2.INTER_AREA)\n                    resized = normalize_intensity(resized, window, POLICY)\n                    out = np.zeros((S, S), np.uint8)\n                    out[pyo:pyo + oh, pxo:pxo + ow] = np.rint(np.clip(resized, 0, 1) * 255).astype(np.uint8)\n                    px.append(out)\n                    relative = 2 * (pos[k] - pos[0]) / max(pos[-1] - pos[0], 1e-6) - 1\n                    meta.append([sidx, s["slot"], PLANE_ID[s["plane"]], view_id, relative, np.log1p(abs(spacing)),\n                                 g.extent / 130., -1 if fluid is None else fluid, -1 if fat is None else fat,\n                                 k, len(pos), real_idx])\n            blocks.append(block)\n            del volume, vol\n        if px:\n            px = torch.from_numpy(np.stack(px))\n            meta = torch.tensor(meta, dtype=torch.float32)\n        else:\n            px = torch.zeros((0, S, S), dtype=torch.uint8)\n            meta = torch.zeros((0, len(META_COLS)), dtype=torch.float32)\n        return {"uid": uid, "error": None, "px": px, "meta": meta, "blocks": blocks, "skipped_series": skipped,\n                "removed": int(plan.get("removed_series", 0))}\n\n\ndef count_upper(cache, uid, size, views="v1"):\n    with open(study_json(cache, uid), encoding="utf-8") as fh:\n        study = json.load(fh)\n    n = 0\n    for s in study["series"]:\n        try:\n            v = len(series_views(s["plane"], s["native_shape"], s["spacing_mm"], size, "v3" if views in ("v3k", "v4") else views)) + (1 if views == "v4" else 0)\n        except Exception:\n            v = 3\n        n += v * len(s["positions"])\n    return n\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--cache", default="F:/rsnaknee/cache_native")\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--first", default=None)\n    ap.add_argument("--size", type=int, default=288)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--workers", type=int, default=10)\n    ap.add_argument("--seed", type=int, default=20261005)\n    ap.add_argument("--margin-gb", type=float, default=60.)\n    ap.add_argument("--anchors", default=None, help="anchor bank directory (knee_anchor_bank.py) for --views v3k")\n    ap.add_argument("--views", default="v1", choices=["v1", "v2", "v2add", "v3", "v3k", "v4"],\n                    help="v2 = v1 + patellofemoral local view for sagittal/axial; v2add = only that added view (an add-on cache "\n                         "that compact_train reads next to a v1 cache with --px-extra)")\n    a = ap.parse_args()\n    os.makedirs(a.out, exist_ok=True)\n    t0 = time.time()\n    import pandas as pd\n    uids = pd.read_csv(os.path.join(a.cache, "patients.csv"), dtype=str)[UIDK].tolist()\n    known = set(uids)\n    first = json.load(open(a.first, encoding="utf-8")) if a.first else []\n    first = [u for u in first if u in known]\n    rest = sorted(known - set(first))\n    rng = np.random.default_rng(a.seed)\n    rest = [rest[i] for i in rng.permutation(len(rest))]\n    order = first + rest\n    if a.limit:\n        order = order[:a.limit]\n    index_path = os.path.join(a.out, "index.json")\n    if os.path.exists(index_path):\n        index = json.load(open(index_path, encoding="utf-8"))\n        if index["order"] != order or index["config"]["size"] != a.size or index["config"].get("views", "v1") != a.views:\n            raise SystemExit("existing index was built for another order/size/views; use a new --out")\n    else:\n        counts = [count_upper(a.cache, u, a.size, a.views) for u in order]\n        index = {"order": order, "n_upper": int(sum(counts)), "done": {}, "blocks": {}, "cursor": 0, "errors": {},\n                 "removed_series": 0, "meta_cols": META_COLS,\n                 "config": {"size": a.size, "local_mm": LOCAL_MM, "policy": POLICY,\n                            "views": a.views, "anchors": a.anchors,\n                            "view_recipe": "global_local+coronal_bilateral" + ("+pf_local" if a.views == "v2" else ""),\n                            "duplicates": "collapse_exact",\n                            "rows": "one per (series, view, slice), physical order", "gold_first": len(first)}}\n        json.dump(index, open(index_path, "w", encoding="utf-8"))\n    N = max(1, index["n_upper"])\n    gb = N * a.size * a.size / 2 ** 30\n    print("studies", len(order), "row upper bound", N, "pixel file GB %.1f" % gb, "count seconds %.1f" % (time.time() - t0), flush=True)\n    import shutil\n    free = shutil.disk_usage(a.out).free / 2 ** 30\n    if not os.path.exists(os.path.join(a.out, "px.u8.npy")) and free < gb + a.margin_gb:\n        raise SystemExit("not enough free space: need %.0f GB + %.0f GB margin, have %.0f GB" % (gb, a.margin_gb, free))\n\n    def open_mm(path, shape, dtype):\n        if os.path.exists(path):\n            mm = np.load(path, mmap_mode="r+")\n            if tuple(mm.shape) != tuple(shape):\n                raise SystemExit("existing memmap has another shape: " + path)\n            return mm\n        return np.lib.format.open_memmap(path, mode="w+", dtype=dtype, shape=shape)\n\n    px_mm = open_mm(os.path.join(a.out, "px.u8.npy"), (N, a.size, a.size), np.uint8)\n    meta_mm = open_mm(os.path.join(a.out, "meta.f32.npy"), (N, len(META_COLS)), np.float32)\n    todo = [u for u in order if u not in index["done"] and u not in index["errors"]]\n    loader = torch.utils.data.DataLoader(StudySlices(a.cache, todo, a.size, a.views, a.anchors), batch_size=None, shuffle=False,\n                                         num_workers=a.workers, prefetch_factor=2 if a.workers else None,\n                                         persistent_workers=bool(a.workers))\n    state = {"t": time.time(), "studies": 0, "last": time.time()}\n\n    def save(final=False):\n        px_mm.flush(); meta_mm.flush()\n        tmp = index_path + ".tmp"\n        json.dump(index, open(tmp, "w", encoding="utf-8"))\n        os.replace(tmp, index_path)\n        el = time.time() - state["t"]\n        sps = state["studies"] / max(el, 1e-6)\n        prog = {"done_studies": len(index["done"]), "errors": len(index["errors"]), "total_studies": len(order),\n                "rows_written": index["cursor"], "elapsed_s": round(el, 1), "studies_per_s": round(sps, 3),\n                "eta_min": round((len(todo) - state["studies"]) / max(sps, 1e-6) / 60, 1), "final": final,\n                "time": time.strftime("%Y-%m-%d %H:%M:%S")}\n        json.dump(prog, open(os.path.join(a.out, "progress.json"), "w"))\n        print(json.dumps(prog), flush=True)\n\n    for item in loader:\n        uid = item["uid"]\n        state["studies"] += 1\n        if item["error"]:\n            index["errors"][uid] = item["error"]\n            continue\n        n = len(item["px"])\n        cur = index["cursor"]\n        if cur + n > N:\n            raise RuntimeError("row count exceeds the allocated upper bound")\n        if n:\n            px_mm[cur:cur + n] = item["px"].numpy()\n            meta_mm[cur:cur + n] = item["meta"].numpy()\n        for b in item["blocks"]:\n            b["views"] = [[v, cur + off] for v, off in b["views"]]\n        index["done"][uid] = [cur, n, len(item["blocks"])]\n        index["blocks"][uid] = item["blocks"]\n        index["cursor"] = cur + n\n        index["removed_series"] += item["removed"]\n        if time.time() - state["last"] > 90:\n            save(); state["last"] = time.time()\n    save(final=True)\n    print("DONE rows", index["cursor"], "studies", len(index["done"]), "errors", len(index["errors"]),\n          "minutes %.1f" % ((time.time() - t0) / 60), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'compact_train.py': '"""Compact fine-tuned image model on the per-slice pixel cache.  The user authorised fine-tuning a compact model on 2026-10-05 evening\n# v9.8 (10-11 01:00, spec card plans/spec_trainer_v98_fmaux_20261011.md md5 fc451b3e, sections B and E): --fm-aux DIR = the fastMRI+\n#   lesion-box auxiliary task on the shared encoder.  Every optimiser step, after the main path\'s loss.backward() (v9.7\'s batch, windows,\n#   augmentation, random streams, untouched), one auxiliary batch of M = --fm-aux-windows (96) fastMRI 3-slice windows from the fm384_v1\n#   store (format: FmStore) -- 4 volumes per step (75% drawn from volumes with boxes, 25% from reviewed volumes without findings), per boxed\n#   volume half the windows centred on slices with a box (any class) and half on slices without, per window a view uniform in {0, 2, 3},\n#   neighbours at phys step 1 clamped as window_rows -- is cropped to the run\'s --crop (offset uniform on 0..384-crop, augment()\'s\n#   distribution, own stream), translated by an integer (dy, dx) uniform on -N..N when --shift-px N > 0 (zero fill, v9.5\'s convention),\n#   flipped left-right with probability --fm-aux-flip and gamma / contrast jittered as augment() does (per window), and goes through the\n#   SAME model.enc (train mode, bf16 autocast, the compiled module under --compile): map [M, 768, G, G], G = crop / 32 (asserted equal to the\n#   main batch\'s spatial size / 32 and to the map\'s grid).  An independent head aux_* (per-position LayerNorm(768) -> linear 768 -> 11,\n#   float32) gives dense logits [M, 11, G, G]; presence = per class log-mean-exp over the G^2 positions.  L_aux = BCE(dense, per-cell\n#   coverage fraction of the class\'s boxes after crop / shift / flip; pos_weight 8) + BCE(presence, a box of the class keeping >= 25% of its\n#   original area in the window, any of the 3 slices); means over windows x positions x classes / windows x classes.  lambda * L_aux\n#   (--fm-aux-weight 0.3) is back-propagated into the same .grad (two backward passes; = the gradient of L_main + lambda * L_aux), then ONE\n#   clip over every parameter (fm_clip: v9.7\'s norm over v9.7\'s parameters combined with the head\'s), one AdamW step, one EMA update.  The\n#   auxiliary step (fetch, forward, backward) runs inside torch.random.fork_rng with the seed fm_fork_seed(seed, fold, epoch, step): the\n#   global CPU / CUDA random states after it are the ones before it, bit for bit.  Sampling and augmentation draws: two private\n#   torch.Generators seeded per epoch from 9800 + 17*seed + fold + 1000*epoch (augmentation: + 2**32), so a resume at an epoch boundary\n#   repeats the uninterrupted run\'s auxiliary batches; 2 loader workers of their own (--fm-aux-workers).  The aux_* head is created last\n#   from a private generator (the global stream is untouched), trains in the head group, joins the EMA and the checkpoint, and is\n#   stripped from deploy_state_dict (fold models keep v9.7\'s key set).  Logs: every --fm-aux-log-every (100) steps |lambda grad L_aux| /\n#   |grad L_main| over the encoder (before the clip); per epoch L_dense, L_presence, the sample digest and the per-class presence AUC on\n#   the store\'s aux_holdout volumes (window level; volume level as information), at the fold\'s end with the shipped (EMA) weights, plus the\n#   B5 gate (meniscus_tear >= 0.75, effusion >= 0.80, cart_full >= 0.75) -> fm_aux_<arm>_s<seed>_f<k>.json.  Refused: a non-ConvNeXt\n#   backbone (BatchNorm backbones: the aux batch would move the statistics; ViT / OFL: no map), no --spatial-attn, --reeval-run, a crop that\n#   is not a multiple of 32 up to 384, M not a multiple of 8, M < 0.25 x the main windows per step, a store failing FmStore\'s checks.\n#   warm_load also accepts a source without the aux_* keys (missing keys must be exactly that set).  E (v9.7 audit notes): E1 --reeval-run\n#   with a state spec whose md5 differs from the one in the source run_config is refused at start-up (both paths and md5s in the message);\n#   E2 an --out that already holds a done marker ((partial_)preds_<arm>_s<seed>.json, the state read-out\'s preds_state_* excepted) is\n#   refused before any file is written, and run_config.json / pool_uids.json are written only after every start-up check (now including\n#   the arms\' targets, the reeval pool and E1) has passed.  Off (no --fm-aux, the default) = v9.7 bit for bit in every output and printed\n#   line, except E1 / E2\'s refusals (base: compact_train_v97.py md5 0b7bc5d8).\n# v9.7 (10-10 21:00, spec card spec_trainer_v97_state_oof_20261010.md): --save-state-oof = the state head\'s read-out written next to the\n#   class OOF (R6 state-level fusion in the label builder).  The forward that gives a study\'s class logits in predict() (now called\n#   with return_aux) also returns the state head\'s logits; for every held-out study a fold model predicts (its own fold; with\n#   --holdout-fold also the holdout fold\'s studies) and for every gold study it predicts: per class the probability vector over the\n#   class\'s categories = the softmax over the class\'s valid slots, the very p of Net.state_evidence, and per binary head\n#   sigmoid(logit).  Files (the partial_ prefix with --only-folds as for the class files; also under --reeval-run):\n#   oof_state_<arm>_s<seed>.npy  float32 [pool, D], rows = pool_uids.json, columns = per class (LABELS order) its categories in the\n#     spec\'s order, then the binaries in the spec\'s order; NaN = a row this process did not predict (another fold\'s study);\n#     holdout-fold rows = sum over the fold models of p / (K - 1) (the class OOF\'s convention);\n#   oof_state_<arm>_s<seed>.json  the column layout (class, category names, column range; binary, class, column) + provenance;\n#   preds_state_<arm>_s<seed>.json  gold_uids + per fold model and per class the gold studies\' probability lists (+ their mean\n#     over the fold models once the file holds every fold model).\n#   Written before the preds json (the run\'s done marker); a done run without them is refused, not skipped.  Needs the state head\n#   (--state-targets).  Merge fold-parallel runs with merge_partial_v2.py.  Training, losses, sampling, augmentation, EMA and the\n#   class outputs are untouched; off (default) = v9.6 bit for bit (base: compact_train_v96.py md5 d0d362c3).\n# v9.6 (10-10 18:00, spec card spec_r5_repair_training_20261010.md): --warm-start DIR = repair training.  Fold k starts from the source\n#   run\'s fold-k model DIR/models/<arm>_s<seed>_f<k>.pt: the WHOLE Net state dict (encoder and every head, parameters and buffers) loaded\n#   with strict=True after the initialisation and the bias0 copy, before the optimizer and the EMA are built -- so the EMA starts from the\n#   loaded weights.  The optimizer and the LR schedule start fresh over --epochs (same warm-up + cosine shape).  --warm-lr-scale s\n#   (default 1.0) multiplies every parameter group\'s LR (encoder, the layer-wise groups, head) before the schedule records it.  Refused at\n#   start-up unless DIR\'s run_config equals this run in K (--folds), the holdout fold, backbone, crop, slices, spatial attention, extent\n#   (and its classes) and syn-decomp, the source\'s state spec has this run\'s categories and binaries, pool_uids.json equals the pool,\n#   every fold model to load exists and --out is not the source; at load time the configuration-derived buffers of the fresh Net must\n#   equal the file\'s.  The state spec\'s prior / base values MAY differ (coordinator correction, 10-10: R5 trains with a re-calibrated\n#   spec): after the strict load, state_prior / state_valid / state_base_logit are re-set from this run\'s spec (per-class change logged);\n#   every learned parameter keeps the loaded value.  A fold-k model never saw fold k\'s studies and the hash folds are the same, so the\n#   continued model\'s out-of-fold predictions stay out-of-fold.  ofl_l is refused (its fold files omit the frozen public weights).\n#   --aug-grid-fp32 (coordinator addition, 10-10): under --aug v2 the rotation / scale grid\n#   (F.affine_grid, whose matmul autocast runs in bf16 inside the training autocast region: ~0.7 px grid error) and its bilinear\n#   resampling (F.grid_sample) run in float32 with autocast off for just those two calls; same draws, same order, same gamma / contrast.\n#   --draw-seed (coordinator, SCAN_A D1): the training samplers draw with replacement, but Studies seeded each study\'s window choice by\n#   (seed, epoch, index), so a study drawn k times in an epoch got the same windows k times; with the flag every training draw carries a\n#   nonce (its position in the epoch) and seeds [seed, epoch, index, nonce]; the drawn index sequence is unchanged; int indices\n#   (evaluation, BN recalibration) keep the old seeding.  --rare-boost-names (SCAN_A D4): the rare-boost terms by state name -- MCL +2 no\n#   longer for chronic_or_healed_change (v9.5\'s "MCL >= 2"); the other four terms asserted to select the same studies as before.  Each\n#   of the three switches is recorded in the checkpoint (a resumed fold cannot switch mode).\n#   Without any v9.6 flag (the defaults) = v9.5 bit for bit (base: compact_train_v95.py md5 64358dcb).\n# v9.5 (10-10 12:00, spec card spec_trainer_v95_shift_20261010.md): --shift-px N = batch translation jitter, one draw per TRAINING\n#   batch: inside augment(), after the crop and before the --aug v2 affine, every window of the batch (every study, window, plane and\n#   slice channel) is translated by the same integer (dy, dx), each uniform on -N..N, with zero fill -- the batch is padded with N\n#   zeros on both pixel axes and cropped back at offset (N + dy, N + dx) (never torch.roll: nothing wraps around).  The draws come\n#   from their own torch.Generator (9500 + 17*seed + fold, checkpointed as shift_gen), so the crop offsets, rotation / scale,\n#   resolution, gamma / contrast, mirror, sampler and dropout draws are v9.4\'s with the flag on.  Purpose: --crop 384 gets the\n#   +-16 px translation that --crop 352 takes from its random crop offset (the R4 field-of-view screen changed both at once).\n#   Never in predict / reeval / bn_recalibrate (centre crop).  --shift-px 0 (default) = v9.4 bit for bit: no draw, no tensor touched\n#   (base: compact_train_v94.py md5 450b5fc6).\n# v9.4 (10-10 10:30, spec card spec_trainer_v94_mirror_20261010.md): --mirror p = anatomical mirror + output-permutation augmentation,\n#   one draw per TRAINING batch (both studies of a batch, so both studies of a matched pair, share it): coronal / axial windows are\n#   flipped on the width axis (the patient\'s left-right axis), sagittal windows are NOT flipped in-plane (in-plane left-right is\n#   anterior-posterior) but reversed along the slice axis -- the K slice channels [s-r .. s+r] -> [s+r .. s-r] and rel_pos -> -rel_pos\n#   (the cache stores sagittal slices in a side-canonical order) -- and the coronal bilateral locals change sides (view id 2 <-> 3).\n#   The model keeps the canonical class order; for a mirrored batch its outputs are re-indexed by P = swap(Medial, Lateral Meniscus)\n#   x swap(Medial OA, Lateral OA) before EVERY loss (12 logits, state logits and binaries per class, extent area; the syn-decomp\n#   columns are fixed points), targets / weights / state targets / pair signatures untouched.  MCL is not trained on a mirrored\n#   batch (main, state and pair weights 0: the mirrored MCL position has no truth).  Never in predict / reeval / bn_recalibrate.\n#   --extent-classes = the OA classes whose logit receives the extent head\'s area term (default the three OA classes; the area,\n#   its supervision and the area families are unchanged).  Both flags at their defaults = v9.3 bit for bit (base:\n#   compact_train_v93.py md5 2c523804).\n# v9.3 (10-10 02:30, spec card spec_trainer_v93_20261010.md; designs mechanisms_r4d_20261010.md 1.4 and 3.5): the P1 families are\n#   indexed by FAMILY (role / sig / pw are [N, F]; a class may own several families; a family may rank the class logit or the\n#   extent head\'s area; optional per-study masks) -- with the v9 list this is the v9.2 computation column for column.  Four\n#   families behind four flags: --oa-area-rank (high_grade_large > high_grade_small on log1p(area/100), + the extent head\'s\n#   soft target 0.2 for high_grade_small; needs the opusv11c spec), --oa-sign-decoy (high grade with no bone sign written >\n#   low grade with a sign written), --acl-acuity-decoy (chronic / calm-unstated ACL tear > ACL normal with acute co-findings;\n#   age from --lig-age <csv>), --mcl-boundary (high-grade partial / complete, not chronic > grade 2).  All off = v9.2b bit for bit\n#   (base: compact_train_v92.py md5 29e681e7, the audited v9.2b).\n# v9.2b (10-10 01:50, audit D1 fixed: beta logged per epoch inside the fold loop): --syn-decomp = synovitis logit decomposed into a specific part + beta * stop-grad(effusion logit) (design mechanisms_r4d_20261010.md 2.4); without it identical to v9.1.\n# v9.1 (10-09 23:50): --grad-ckpt-blocks N = partial gradient checkpointing (memory only, same maths); without it identical to v9.\n(the mainline OrthoFoundation training hold on the 5090 is untouched).  v1.1: fold models are saved.\nv9 (10-09 evening, spec card plans/spec_trainer_v9_grade_pairs_20261009.md): optional --pair-extra adds four within-class grade\nfamilies to the P1 matched-pair ranking -- Effusion moderate/large over none/small, Baker\'s moderate/large over none/small,\nContusion contusion/traumatic oedema over degenerative/reactive oedema, Fracture acute/impaction/avulsion over trabecular\nmicrofracture/old fracture -- the grade boundaries where both R2 students err together (shared-error map, 10-09).  These\nfamilies match no co-finding (signature 0).  Without --pair-extra the code path is v8\'s.\n\nSame contract as probe_train.py (so probe_eval.py and fuse_eval.py read its outputs unchanged):\n  - never reads official labels; gold studies are never trained on; targets come from the same CSVs\n  - K-fold over the non-gold pool with the same hash folds; writes gold predictions (mean of K models)\n    and out-of-fold predictions for every pool study\n  - arms: S_strict, P_presence, E_prior, E_lrprior, or name=bce:<column> / name=lr:<column> from --extra\n\nModel: 2.5D windows (three physically adjacent slices as channels) -> CNN encoder -> per-class attention\nover the windows of a study -> 12 logits.  Training samples a fixed number of windows per study; evaluation\nuses every third slice as a centre (the mainline\'s cover sampling) in both views.\n"""\nimport argparse, collections, contextlib, csv, hashlib, json, math, os, re, sys, time\nsys.dont_write_bytecode = True\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",\n          "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nC = len(LABELS)\n\n\nPREDICT_CHUNK = int(os.environ.get("CT_PREDICT_CHUNK", "512"))   # v7: evaluation windows per encoder call (512 = v6)\n\n\ndef covering_centers(n, phase=1, stride=3):\n    if stride <= 1:\n        return list(range(n))\n    c = list(range(min(phase, n - 1), n, stride))\n    if c[0] > 1:\n        c.insert(0, 0)\n    if c[-1] < n - 2:\n        c.append(n - 1)\n    return c\n\n\ndef phys_step(spacing_mm, unit_mm):\n    """slices per ~unit_mm of anatomy: 1 for an ordinary 3-4 mm series, ~9 for a 0.4 mm 3-D stack (unit_mm <= 0: always 1)."""\n    if unit_mm <= 0:\n        return 1\n    return max(1, int(round(unit_mm / max(float(spacing_mm), 0.1))))\n\n\ndef window_rows(blocks, meta_of, mode, slices, eval_stride=3, cover_stride=3, phys_mm=0.0, rng=None, slice_jitter=0.0,\n                train_sampling="cover"):\n    """The window list of one study.  blocks: [{\'n_slices\': n, \'views\': [(file, view_id, start), ...]}]; meta_of(file, row)\n    -> the px meta row of that window (column 5 = log1p(slice spacing in mm)).  Returns tuples\n    (file, centre row, row of slice -r, ..., row of slice +r).  \'train\' covers every (series, view) at stride\n    cover_stride x phys_step with a random phase (and takes the K neighbours two steps apart with probability slice_jitter);\n    \'eval\' covers at stride eval_stride x phys_step.  phys_mm = 0 reproduces the old per-slice behaviour."""\n    cand = []\n    r_ = slices // 2\n    for b in blocks:\n        n = b["n_slices"]\n        for f, view_id, start in b["views"]:\n            ps = phys_step(math.expm1(float(meta_of(f, start)[5])), phys_mm)\n            if mode == "train":\n                if train_sampling == "cover":\n                    st = cover_stride * ps\n                    ks = covering_centers(n, int(rng.integers(0, st)), st)\n                else:\n                    ks = range(n)\n            else:\n                ks = covering_centers(n, 1, eval_stride * ps)\n            for k in ks:\n                step_ = ps * (2 if (mode == "train" and slice_jitter > 0 and rng.random() < slice_jitter) else 1)\n                cand.append((f, start + k) + tuple(start + min(n - 1, max(0, k + d * step_)) for d in range(-r_, r_ + 1)))\n    return cand\n\n\nclass Studies(torch.utils.data.Dataset):\n    """Returns uint8 windows [W,3,S,S] and meta [W,6] for one study.  mode \'train\' samples, \'eval\' covers."""\n\n    def __init__(self, px_dir, uids, n_win, mode, seed, eval_stride=3, train_sampling="uniform", cover_stride=3, px_extra=None, slices=3,\n                 slice_jitter=0.0, phys_mm=0.0, px_override=None, extent_meta=False):\n        self.dir, self.uids, self.n_win, self.mode, self.seed = px_dir, uids, n_win, mode, seed\n        self.extent_meta = bool(extent_meta)\n        self.eval_stride, self.train_sampling, self.cover_stride = eval_stride, train_sampling, cover_stride\n        self.slices = int(slices)\n        self.slice_jitter = float(slice_jitter)\n        self.phys_mm = float(phys_mm)\n        assert self.slices % 2 == 1 and self.slices >= 1\n        index = json.load(open(os.path.join(px_dir, "index.json"), encoding="utf-8"))\n        # blocks carry (file, view_id, start): file 0 = the main cache, file 1 = the add-on cache (e.g. the v2 patellofemoral\n        # view built with px_cache --views v2add); a study missing from the add-on simply has no add-on windows\n        self.blocks = {u: [dict(b, views=[(0, v, st) for v, st in b["views"]]) for b in index["blocks"][u]] for u in uids}\n        self.dirs = [px_dir]\n        if px_extra:\n            extra = json.load(open(os.path.join(px_extra, "index.json"), encoding="utf-8"))\n            self.dirs.append(px_extra)\n            n_add = 0\n            for u in uids:\n                eb = extra["blocks"].get(u)\n                if not eb:\n                    continue\n                main = self.blocks[u]\n                if [b["series"] for b in eb] != [b["series"] for b in main]:\n                    raise SystemExit("add-on cache series order differs from the main cache for study " + u)\n                for b, e in zip(main, eb):\n                    b["views"] = b["views"] + [(1, v, st) for v, st in e["views"]]\n                    n_add += len(e["views"])\n            print("add-on cache %s: %d extra (series, view) blocks for %d studies" % (px_extra, n_add, len(uids)), flush=True)\n        if px_override:\n            # override cache (sidefix): the listed studies are read entirely from it (same format, own row space), e.g.\n            # the studies whose knee side the image check corrected, re-rendered through the inference path\n            if px_extra:\n                raise SystemExit("--px-override cannot be combined with --px-extra")\n            ov = json.load(open(os.path.join(px_override, "index.json"), encoding="utf-8"))\n            f_ov = len(self.dirs)\n            self.dirs.append(px_override)\n            n_ov = 0\n            for u in uids:\n                ob = ov["blocks"].get(u)\n                if ob is not None:\n                    self.blocks[u] = [dict(b, views=[(f_ov, v, st) for v, st in b["views"]]) for b in ob]\n                    n_ov += 1\n            print("override cache %s: %d of %d studies read from it" % (px_override, n_ov, len(uids)), flush=True)\n        self.px = self.meta = None\n        self.epoch = 0\n\n    def __len__(self):\n        return len(self.uids)\n\n    def __getitem__(self, i):\n        if self.px is None:\n            self.px = [np.load(os.path.join(d, "px.u8.npy"), mmap_mode="r") for d in self.dirs]\n            self.meta = [np.load(os.path.join(d, "meta.f32.npy"), mmap_mode="r") for d in self.dirs]\n        nonce = None\n        if isinstance(i, tuple):                    # v9.6 --draw-seed: (index, nonce) from a training sampler (an int index = v9.5\'s path)\n            i, nonce = i\n        uid = self.uids[i]\n        rng = np.random.default_rng(self.seed + 1000003 * self.epoch + i) if self.mode == "train" else None\n        if nonce is not None and self.mode == "train":   # v9.6 --draw-seed: this draw\'s own stream (repeat draws of a study differ)\n            rng = np.random.default_rng([int(self.seed), int(self.epoch), int(i), int(nonce)])\n        # \'cover\': every series/view is covered at stride cover_stride (x the physical step) with a random phase, so a\n        # focal finding spanning >= one stride is in the sampled set of every epoch (uniform sampling of ~10% of the\n        # windows misses a 3-slice finding in about half of the epochs).  Rows: (file, centre row, K slice rows k-r..k+r).\n        cand = window_rows(self.blocks[uid], lambda f, r: self.meta[f][r], self.mode, self.slices, self.eval_stride,\n                           self.cover_stride, self.phys_mm, rng, self.slice_jitter, self.train_sampling)\n        if self.mode == "train" and len(cand) > self.n_win:\n            cand = [cand[j] for j in np.sort(rng.choice(len(cand), self.n_win, replace=False))]\n        rows = np.array(cand, np.int64)\n        S = self.px[0].shape[-1]\n        img = np.empty((len(rows), self.slices, S, S), np.uint8)\n        m = np.empty((len(rows), self.meta[0].shape[1]), np.float32)\n        for f in range(len(self.dirs)):\n            sel = np.flatnonzero(rows[:, 0] == f)\n            if not len(sel):\n                continue\n            r = rows[sel]\n            img[sel] = np.stack([np.asarray(self.px[f][r[:, 2 + d]]) for d in range(self.slices)], 1)\n            m[sel] = np.asarray(self.meta[f][r[:, 1]])\n        meta = np.stack([m[:, 1], m[:, 3], m[:, 4], m[:, 5], m[:, 6], m[:, 7]], 1)   # slot, view, rel_pos, log_spacing, extent, fluid\n        if self.extent_meta:                                   # + series index, plane (0 sag / 1 cor / 2 ax), slices in the series\n            meta = np.concatenate([meta, np.stack([m[:, 0], m[:, 2], m[:, 10]], 1)], 1)\n        return torch.from_numpy(img), torch.from_numpy(meta.astype(np.float32)), i\n\n\ndef collate(batch):\n    W = max(len(b[0]) for b in batch)\n    S = batch[0][0].shape[-1]\n    img = torch.zeros((len(batch), W, batch[0][0].shape[1], S, S), dtype=torch.uint8)\n    meta = torch.zeros((len(batch), W, batch[0][1].shape[1]), dtype=torch.float32)\n    mask = torch.zeros((len(batch), W), dtype=torch.bool)\n    for j, (x, m, _) in enumerate(batch):\n        img[j, :len(x)] = x\n        meta[j, :len(x)] = m\n        mask[j, :len(x)] = True\n    return img, meta, mask, torch.tensor([b[2] for b in batch])\n\n\nOFL_BACKBONE = "ofl_l"\n\n\nclass OFLEncoder(nn.Module):\n    """OrthoFoundation-L (public self-supervised weights) as the window encoder, run exactly as the feature extractor\n    runs it: timm Eva vit_large_patch16_dinov3_qkvb, RoPE removed, 5 prefix tokens, ImageNet normalisation (done by\n    Net).  Feature = [CLS token, mean patch token] (2048).  The patch embedding, the tokens, norm_pre and the first\n    `freeze` of the 24 blocks are frozen and run without a graph; the remaining blocks and the final norm are trained.\n    `deploy_state_dict` returns only what differs from the public weights, so a fold model can be shipped small and\n    rebuilt on top of the public weights."""\n\n    def __init__(self, init, freeze=16, img_size=256):\n        super().__init__()\n        import timm\n        self.vit = timm.create_model("vit_large_patch16_dinov3_qkvb", pretrained=False, num_classes=0, global_pool="",\n                                     img_size=img_size)\n        if init:\n            if str(init).endswith(".safetensors"):\n                from safetensors.torch import load as load_bytes\n                with open(str(init), "rb") as fh:          # one sequential read; the mmap loader page-faults 4 KB at a time\n                    sd = load_bytes(fh.read())             # (0.4 MB/s from a USB disk: 48 min per fold on the 5090)\n            else:\n                sd = torch.load(init, map_location="cpu")\n            self.vit.load_state_dict(sd, strict=True)\n        self.vit.rope = None\n        self.freeze = int(freeze)\n        self.num_features = 2 * self.vit.embed_dim\n        self.frozen_keys = set()\n        frozen_modules = [self.vit.patch_embed, self.vit.norm_pre] + list(self.vit.blocks[: self.freeze])\n        for mod in frozen_modules:\n            for p in mod.parameters():\n                p.requires_grad_(False)\n        for name in ("cls_token", "reg_token", "pos_embed"):\n            t = getattr(self.vit, name, None)\n            if isinstance(t, nn.Parameter):\n                t.requires_grad_(False)\n        for k, p in self.vit.named_parameters():\n            if not p.requires_grad:\n                self.frozen_keys.add("vit." + k)\n\n    def trunk(self, x):\n        v = self.vit\n        with torch.no_grad():\n            x = v.patch_embed(x)\n            x, _ = v._pos_embed(x)\n            x = v.norm_pre(x)\n            for blk in v.blocks[: self.freeze]:\n                x = blk(x, rope=None)\n        return x\n\n    def tail(self, x):\n        v = self.vit\n        for blk in v.blocks[self.freeze:]:\n            x = blk(x, rope=None)\n        x = v.norm(x)\n        return torch.cat([x[:, 0], x[:, v.num_prefix_tokens:].mean(1)], -1)\n\n    def forward(self, x):\n        return self.tail(self.trunk(x))\n\n\ndef partial_grad_checkpointing(enc, n):\n    """v9.1: recompute the first n transformer blocks of a timm ViT encoder in backward (torch.utils.checkpoint,\n    use_reentrant=False, as timm\'s own grad checkpointing).  Memory saved ~ n/len(blocks) of full checkpointing, extra compute\n    the same fraction.  Outputs and gradients are those of the plain forward (a checkpoint only changes WHEN activations are\n    computed).  Eval / no_grad passes run the plain forward."""\n    import torch.utils.checkpoint as _ckpt\n    blocks = getattr(enc, "blocks", None)\n    if not isinstance(blocks, (nn.Sequential, nn.ModuleList)):\n        raise SystemExit("--grad-ckpt-blocks needs a ViT-style timm encoder with .blocks (use --grad-ckpt for others)")\n    n = min(int(n), len(blocks))\n\n    def wrap(f):\n        def fwd(x, *args, **kw):\n            if torch.is_grad_enabled() and x.requires_grad:\n                return _ckpt.checkpoint(f, x, *args, use_reentrant=False, **kw)\n            return f(x, *args, **kw)\n        return fwd\n    for blk in list(blocks)[:n]:\n        blk.forward = wrap(blk.forward)\n    return n\n\n\nclass Net(nn.Module):\n    def __init__(self, backbone, init, hidden=384, drop=0.1, freeze_blocks=16, crop=256, state_spec=None, in_chans=3, spatial=False,\n                 extent=False, syn_decomp=False, extent_classes=None, fm_aux=0):\n        super().__init__()\n        import timm\n        self.backbone = backbone\n        self.crop = int(crop)\n        self.in_chans = int(in_chans)\n        self.spatial = bool(spatial)\n        self.extent = bool(extent)\n        self._ext_area = None\n        self.syn_decomp = bool(syn_decomp)      # v9.2 (10-10, mechanisms_r4d_20261010.md 2.4): z_syn = z_spec + beta * sg(z_eff)\n        self._syn_spec = None\n        if self.syn_decomp:                     # created only when asked: state_dict keys unchanged otherwise\n            self.syn_beta = nn.Parameter(torch.zeros(()))   # init 0 -> the baseline network at step 0\n        if self.spatial and backbone == OFL_BACKBONE:\n            raise SystemExit("--spatial-attn needs a timm backbone (the OFL encoder returns pooled features)")\n        if backbone == OFL_BACKBONE:\n            assert self.in_chans == 3, "the OFL patch embedding is a public 3-channel weight"\n            self.enc = OFLEncoder(init, freeze=freeze_blocks, img_size=crop)\n        else:\n            self.enc = self._timm_encoder(backbone, init, self.in_chans)\n        self.has_bn = any(isinstance(m, nn.modules.batchnorm._BatchNorm) for m in self.enc.modules())\n        self._build_head(hidden, drop)\n        self.state_spec = state_spec\n        self.has_state = state_spec is not None\n        if self.has_state:\n            self._build_state_head(hidden, state_spec)\n        if self.spatial:                        # created last: every baseline parameter keeps the baseline\'s initial values\n            d = self.enc.num_features\n            self.sp_key = nn.Linear(d, 128)     # keys from per-position normalised features\n            self.sp_pos = nn.Linear(16, 128, bias=False)\n            nn.init.zeros_(self.sp_pos.weight)\n            self.sp_q = nn.Parameter(torch.zeros(C, 128))   # zero query = uniform weights = the baseline average\n        if self.extent:                         # OA defect extent (after every other parameter: same init as the baseline)\n            self.ext_u = nn.Parameter(torch.zeros(C, hidden))\n            self.ext_b = nn.Parameter(torch.full((C,), -4.0))   # sigmoid(-4) = 0.018: area starts ~4 mm^2, not ~3,000 (audit 3)\n            self.ext_gamma = nn.Parameter(torch.zeros(C))    # zero = the baseline logit at init\n            self.ext_view = nn.Parameter(torch.zeros(C, 6))  # per (class, view) weight inside a series; 0 = uniform (audit 4)\n            em = torch.zeros(C)\n            # v9.4: --extent-classes picks the OA classes whose logit receives the area term (None = the three = v9.3); the area\n            # itself is still computed and supervised for all three\n            for l in (("Medial OA", "Lateral OA", "PF OA") if extent_classes is None else extent_classes):\n                em[LABELS.index(l)] = 1.0\n            self.register_buffer("ext_mask", em)\n        self.fm_aux = int(fm_aux)\n        if self.fm_aux:                         # v9.8 --fm-aux: the auxiliary head, created LAST (every v9.7 parameter keeps its initial\n            # value and its position in parameters() / state_dict()) from a private generator (the global stream is not consumed);\n            # per-position LayerNorm(d) -> linear d -> A on the encoder\'s map; never part of a competition output, stripped from the\n            # fold model by deploy_state_dict\n            d = self.enc.num_features\n            g_aux = torch.Generator().manual_seed(9800)\n            self.aux_ln_w = nn.Parameter(torch.ones(d))\n            self.aux_ln_b = nn.Parameter(torch.zeros(d))\n            self.aux_w = nn.Parameter(torch.randn(self.fm_aux, d, generator=g_aux) * .02)\n            self.aux_b = nn.Parameter(torch.full((self.fm_aux,), FM_BIAS0))\n\n    def aux_dense(self, fmap):\n        """v9.8: encoder map [M, d, G, G] (the live map, NOT detached: the auxiliary gradient reaches every encoder parameter) -> dense\n        logits [M, A, G, G] float32: LayerNorm over d at every position, then the linear map d -> A at every position (no pooling first).\n        Run in float32 with autocast off."""\n        assert fmap.ndim == 4, "v9.8 --fm-aux needs the encoder\'s spatial map [M, d, G, G]"\n        with torch.autocast("cuda", enabled=False):\n            f = fmap.permute(0, 2, 3, 1).float()\n            f = F.layer_norm(f, (f.shape[-1],), self.aux_ln_w, self.aux_ln_b, 1e-6)\n            z = torch.matmul(f, self.aux_w.t()) + self.aux_b\n        return z.permute(0, 3, 1, 2)\n\n    def _build_state_head(self, hidden, spec):\n        """Report-state heads: for class c a K_c-way read-out of the class token (which report state the images show),\n        OA-sign / acuity binaries on the parent class token, and the frozen prior vector per state (mean report prior of the state\'s category over the pool, not fitted on gold).\n        The class logit gains  gate_c * (logit(sum_s p(s|x) prior_c(s)) - logit(base_c)): the \'align to the official\n        label\' mapping applied to predicted states; gate starts at 0 so the direct path is untouched at init."""\n        cats = spec["categories"]\n        self.state_k = [len(cats[l]) for l in LABELS]\n        kmax = max(self.state_k)\n        self.state_w = nn.Parameter(torch.randn(C, kmax, hidden) * .02)\n        self.state_b = nn.Parameter(torch.zeros(C, kmax))\n        prior = torch.full((C, kmax), float("nan"))\n        valid = torch.zeros((C, kmax), dtype=torch.bool)\n        for i, l in enumerate(LABELS):\n            for j, pv in enumerate(spec["prior"][l]):\n                prior[i, j] = float(pv); valid[i, j] = True\n        self.register_buffer("state_prior", torch.nan_to_num(prior, nan=0.5).clamp(1e-3, 1 - 1e-3))\n        self.register_buffer("state_valid", valid)\n        self.register_buffer("state_base_logit", torch.logit(torch.tensor([float(spec["base"][l]) for l in LABELS]).clamp(1e-3, 1 - 1e-3)))\n        self.state_gate = nn.Parameter(torch.zeros(C))\n        self.bin_names = [(b, l) for b, labs in spec["binaries"].items() for l in labs]\n        self.bin_parent = [LABELS.index(l) for _, l in self.bin_names]\n        self.bin_w = nn.Parameter(torch.randn(len(self.bin_names), hidden) * .02)\n        self.bin_b = nn.Parameter(torch.zeros(len(self.bin_names)))\n\n    def state_logits(self, study):\n        """study: [B, C, hidden] class tokens -> state logits [B, C, Kmax] (invalid slots -inf), binary logits [B, nbin]."""\n        sl = torch.einsum("bch,ckh->bck", study, self.state_w).float() + self.state_b\n        sl = sl.masked_fill(~self.state_valid[None], float("-inf"))\n        bl = torch.einsum("bjh,jh->bj", study[:, self.bin_parent], self.bin_w).float() + self.bin_b\n        return sl, bl\n\n    def state_evidence(self, sl):\n        """logit of the official probability implied by the predicted state distribution, relative to the class base rate."""\n        p = sl.softmax(-1)                                          # -inf slots -> 0\n        e = (p * self.state_prior[None]).sum(-1).clamp(1e-3, 1 - 1e-3)\n        return torch.logit(e) - self.state_base_logit[None]\n\n    def deploy_state_dict(self):\n        """Everything except the frozen public OFL weights (identical to state_dict() for other backbones).  v9.8: and except the\n        auxiliary head aux_* (--fm-aux), so a fold model has v9.7\'s key set (none of its keys starts with aux_)."""\n        frozen = {"enc." + k for k in getattr(self.enc, "frozen_keys", set())}\n        return {k: v.detach().cpu() for k, v in self.state_dict().items() if k not in frozen and not k.startswith("aux_")}\n\n    def load_deploy_state_dict(self, sd):\n        missing, unexpected = self.load_state_dict(sd, strict=False)\n        frozen = {"enc." + k for k in getattr(self.enc, "frozen_keys", set())}\n        if unexpected or set(missing) - frozen:\n            raise ValueError("deploy state does not match: missing %s unexpected %s" % (sorted(set(missing) - frozen)[:5], unexpected[:5]))\n\n    def _timm_encoder(self, backbone, init, in_chans=3):\n        import timm\n        # init == "timm": public ImageNet weights from the local hub cache (HF_HOME); a path: a state dict file\n        is_vit = backbone.startswith("vit_")\n        kw = dict(pretrained=(init == "timm"), num_classes=0, in_chans=in_chans,\n                  drop_path_rate=0.1 if ("convnext" in backbone or is_vit) else 0.0)\n        if is_vit:\n            kw["dynamic_img_size"] = True       # keeps the pretrained grid (DINOv2 518 / DINOv3 256); position embeddings or RoPE\n        else:                                   # are interpolated to the crop at forward; pooling = the model\'s own (token / avg)\n            kw["global_pool"] = "avg"\n        if backbone.startswith(("coatnet", "maxvit", "maxxvit")):\n            kw["img_size"] = self.crop          # relative-position tables are built for this input size (CoAtNet / MaxViT)\n        if self.spatial:                        # keep the spatial grid: CNN -> normed map [N, d, h, w], ViT -> all tokens [N, T, d]\n            kw["global_pool"] = ""\n        enc = timm.create_model(backbone, **kw)\n        if init and init != "timm":\n            if str(init).endswith(".safetensors"):\n                from safetensors.torch import load as load_bytes\n                with open(str(init), "rb") as fh:\n                    sd = load_bytes(fh.read())\n            else:\n                sd = torch.load(init, map_location="cpu")\n                sd = sd.get("state_dict", sd.get("model", sd)) if isinstance(sd, dict) else sd\n            if in_chans != 3:                       # inflate the pretrained RGB stem: mean over RGB, replicated K times, scaled 3/K\n                for k, w in list(sd.items()):\n                    if w.ndim == 4 and w.shape[1] == 3 and k in enc.state_dict() and enc.state_dict()[k].shape[1] == in_chans:\n                        sd[k] = w.mean(1, keepdim=True).repeat(1, in_chans, 1, 1) * (3.0 / in_chans)\n                        print("inflated %s %s -> %s" % (k, tuple(w.shape), tuple(sd[k].shape)), flush=True)\n            missing, unexpected = enc.load_state_dict(sd, strict=False)\n            print("init %s: missing %d unexpected %d" % (init, len(missing), len(unexpected)), flush=True)\n        self.sp_head_norm = None\n        if self.spatial and isinstance(getattr(getattr(enc, "head", None), "norm", None), nn.Module) \\\n                and not isinstance(enc.head.norm, nn.Identity):\n            # CNN heads pool THEN normalise: keep that order per class (pool the raw map, then this norm on each class\'s\n            # pooled vector), so a zero spatial query reproduces the baseline feature LN(mean_p x_p) exactly\n            self.sp_head_norm = enc.head.norm\n            enc.head.norm = nn.Identity()\n        return enc\n\n    def _build_head(self, hidden, drop):\n        d = self.enc.num_features\n        self.proj = nn.Sequential(nn.Linear(d, hidden), nn.LayerNorm(hidden))\n        self.slot = nn.Embedding(10, hidden)\n        self.view = nn.Embedding(6, hidden)\n        self.geom = nn.Linear(4, hidden, bias=False)\n        for m in (self.slot, self.view):\n            nn.init.zeros_(m.weight)\n        nn.init.zeros_(self.geom.weight)\n        self.drop = nn.Dropout(drop)\n        self.win_proj = nn.Linear(hidden, hidden)\n        self.q_win = nn.Parameter(torch.randn(C, hidden) * .02)\n        self.out_norm = nn.LayerNorm(hidden)\n        self.cls_w = nn.Parameter(torch.randn(C, hidden) * .02)\n        self.bias = nn.Parameter(torch.zeros(C))\n        if self.in_chans == 3:\n            self.register_buffer("mean", torch.tensor([.485, .456, .406]).view(1, 3, 1, 1))\n            self.register_buffer("std", torch.tensor([.229, .224, .225]).view(1, 3, 1, 1))\n        else:                                       # grey slices: one ImageNet-mean/std per channel\n            self.register_buffer("mean", torch.full((1, self.in_chans, 1, 1), .449))\n            self.register_buffer("std", torch.full((1, self.in_chans, 1, 1), .226))\n\n    def spatial_pool(self, x):\n        """Encoder output with its spatial grid ([N, d, h, w] CNN map or [N, T, d] ViT tokens) -> per-class window features\n        [N, C, d]: class c weighs the window\'s positions with its own query (sp_q[c]; zero at init = the plain average)\n        over keys tanh(W f(p) + W_pos pe(p)), so a class can read its own structure inside a window shared with others."""\n        if x.ndim == 4:\n            h, w = x.shape[-2:]\n            f = x.flatten(2).transpose(1, 2)                                      # [N, h*w, d]\n        else:\n            f = x[:, getattr(self.enc, "num_prefix_tokens", 0):]\n            h = w = int(round(f.shape[1] ** 0.5))\n            assert h * w == f.shape[1], "non-square ViT token grid"\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h, device=f.device), torch.linspace(-1, 1, w, device=f.device), indexing="ij")\n        fr = torch.tensor([1., 2., 4., 8.], device=f.device) * (math.pi / 2)\n        yy, xx = yy.reshape(-1, 1) * fr, xx.reshape(-1, 1) * fr\n        pe = torch.cat([yy.sin(), yy.cos(), xx.sin(), xx.cos()], 1)              # [h*w, 16] fixed 2-D position features\n        kin = F.layer_norm(f.float(), (f.shape[-1],)).to(f.dtype)                # keys: per-position normalised features\n        k = torch.tanh(self.sp_key(kin) + self.sp_pos(pe.to(f.dtype))[None])    # [N, P, 128]\n        att = torch.einsum("npk,ck->npc", k, self.sp_q.to(k.dtype)).float().softmax(1).to(f.dtype)\n        g = torch.einsum("npc,npd->ncd", att, f)                                 # [N, C, d]\n        if self.sp_head_norm is not None:                                        # CNN: the head norm after pooling, per class\n            n_ = self.sp_head_norm\n            g = F.layer_norm(g.float(), (g.shape[-1],), n_.weight, n_.bias, n_.eps).to(f.dtype)\n        return g\n\n    def extent_area(self, e, meta, mask):\n        """e [B, W, C] window defect logits -> A [B, C] (mm^2, an area proxy): inside each series the mean sigmoid(e) of\n        each view\'s windows, combined over the views present by a learned per-(class, view) softmax (uniform at init: a\n        compartment sits inside only some of the locals), times the series\' slab thickness (slices x spacing) = mm of\n        slab with defect evidence; per plane the max over series; then sagittal x coronal (tibiofemoral OA) or axial x\n        sagittal (PF OA); other classes 0."""\n        V = 6\n        sid = meta[..., 6].long()\n        # one slot per series, never merged: 16 covers every training study (at most 14 series); a deployed study with more\n        # series gets more slots (lossless -- an empty slot has mean 0 and the per-plane max runs over values >= 0)\n        S_MAX = max(16, int(sid.max()) + 1)\n        vid = meta[..., 1].long().clamp(0, V - 1)\n        q = torch.sigmoid(e.float()) * mask[..., None]\n        ohv = F.one_hot(sid * V + vid, S_MAX * V).float() * mask[..., None]     # [B, W, S*V]\n        cntv = ohv.sum(1)                                                          # [B, S*V]\n        mean_v = (torch.einsum("bwk,bwc->bkc", ohv, q) / cntv.clamp_min(1)[..., None]).view(len(e), S_MAX, V, -1)\n        present = (cntv > 0).view(len(e), S_MAX, V, 1)\n        wv = self.ext_view.float().t()[None, None].masked_fill(~present, -1e4).softmax(2) * present   # [B, S, V, C]\n        mean = (wv * mean_v).sum(2)                                                # [B, S, C]; 0 for a series without windows\n        oh = F.one_hot(sid, S_MAX).float() * mask[..., None]                     # [B, W, S]\n        cnt = oh.sum(1)                                                            # [B, S]\n        span = torch.expm1(meta[..., 3].float()).clamp_min(0) * meta[..., 8].float()   # slab thickness (mm) of each window\'s series\n        span_s = torch.einsum("bws,bw->bs", oh, span) / cnt.clamp_min(1)\n        plane_s = (torch.einsum("bws,bw->bs", oh, meta[..., 7].float()) / cnt.clamp_min(1)).round()\n        E = mean * span_s[..., None]                                               # [B, S, C] mm of slab with defect evidence\n        per_plane = []\n        for pl in (0, 1, 2):\n            ok = ((plane_s == pl) & (cnt > 0))[..., None]\n            per_plane.append(E.masked_fill(~ok, 0.0).amax(1))                     # [B, C]; 0 when the plane is absent\n        sag, cor, ax = per_plane\n        area = torch.zeros_like(sag)\n        for l in ("Medial OA", "Lateral OA"):\n            c = LABELS.index(l)\n            area[:, c] = sag[:, c] * cor[:, c]\n        c = LABELS.index("PF OA")\n        area[:, c] = ax[:, c] * sag[:, c]\n        return area\n\n    def head(self, tok, meta, mask, return_aux=False):\n        """Encoder output of the real windows [N, d] (or [N, C, d] with --spatial-attn) -> class logits [B, C] (and the\n        state/binary logits when asked)."""\n        B, W = mask.shape\n        f = self.proj(tok.float())\n        m = meta\n        if self.spatial:                                                          # class-specific window tokens [B, W, C, H]\n            emb = self.slot(m[..., 0].long().clamp(0, 9)) + self.view(m[..., 1].long().clamp(0, 5)) + self.geom(m[..., 2:6])\n            t = f.new_zeros((B, W) + tuple(f.shape[1:]))\n            t[mask] = f\n            t = self.drop(t + emb[:, :, None, :])\n            s2 = torch.einsum("bwch,ch->bwc", torch.tanh(self.win_proj(t)), self.q_win).float()\n            s2 = s2.masked_fill(~mask[:, :, None], float("-inf"))\n            a2 = s2.softmax(1).to(t.dtype)\n            study = self.out_norm(torch.einsum("bwc,bwch->bch", a2, t))\n            if self.extent:\n                e_win = (t.float() * self.ext_u[None, None]).sum(-1) + self.ext_b\n        else:\n            t = f.new_zeros((B, W, f.shape[-1]))\n            t[mask] = f\n            t = t + self.slot(m[..., 0].long().clamp(0, 9)) + self.view(m[..., 1].long().clamp(0, 5)) + self.geom(m[..., 2:6])\n            t = self.drop(t)\n            s2 = torch.einsum("bwh,ch->bwc", torch.tanh(self.win_proj(t)), self.q_win).float()\n            s2 = s2.masked_fill(~mask[:, :, None], float("-inf"))\n            a2 = s2.softmax(1).to(t.dtype)\n            study = self.out_norm(torch.einsum("bwc,bwh->bch", a2, t))\n            if self.extent:\n                e_win = torch.einsum("bwh,ch->bwc", t.float(), self.ext_u) + self.ext_b\n        z = (study * self.cls_w).sum(-1).float() + self.bias\n        if self.extent:\n            area = self.extent_area(e_win, meta, mask)\n            self._ext_area = area\n            z = z + self.ext_gamma * self.ext_mask * torch.log1p(area / 100.0)\n        if not self.has_state:\n            z = self._decompose_syn(z)\n            return (z, None, None) if return_aux else z\n        sl, bl = self.state_logits(study)\n        z = z + self.state_gate[None] * self.state_evidence(sl)\n        z = self._decompose_syn(z)\n        return (z, sl, bl) if return_aux else z\n\n    def _decompose_syn(self, z):\n        """v9.2: the synovitis logit = a synovium-specific part + beta * the effusion logit taken as a fixed covariate\n        (detached: the synovitis loss must not reshape the effusion logit).  Off: z unchanged.  The specific part is kept\n        for the out-of-fold read-out (oof_synspec) and, later, as the teacher evidence for silent synovitis cells."""\n        if not self.syn_decomp:\n            return z\n        SYN, EFF = LABELS.index("Synovitis"), LABELS.index("Effusion")\n        self._syn_spec = z[:, SYN].detach().float()\n        ze = z[:, EFF].detach().clone()                            # own storage: no shared version counter with z\n        onehot = torch.zeros(z.shape[1], device=z.device, dtype=z.dtype)\n        onehot[SYN] = 1\n        return z + (self.syn_beta * ze)[:, None] * onehot[None, :]  # out of place: only the synovitis column changes\n\n    def forward(self, img, meta, mask, chunk=0, return_aux=False):\n        x = img[mask]                                               # only real windows go through the encoder\n        n = len(x)\n        if n % 32 and (not self.training or not self.has_bn):     # pad the encoder batch to a multiple of 32: a handful of shapes\n            x = torch.cat([x, x.new_zeros((32 - n % 32,) + tuple(x.shape[1:]))])   # instead of one per step (allocator, cuDNN autotune);\n        feats = []\n        step = chunk or len(x)\n        for s in range(0, len(x), step):\n            xs = x[s:s + step].float().div_(255.)\n            xs = ((xs - self.mean) / self.std).contiguous(memory_format=torch.channels_last)\n            e = self.enc(xs)\n            feats.append(self.spatial_pool(e) if self.spatial else e)\n        return self.head(torch.cat(feats)[:n], meta, mask, return_aux=return_aux)\n\n\ndef bce_soft(z, target, weight):\n    loss = F.binary_cross_entropy_with_logits(z, target, reduction="none") * weight\n    den = weight.sum(0)\n    ok = den > 0\n    return (loss.sum(0)[ok] / den[ok]).mean()\n\n\ndef lr_loss(z, lr, weight=None):\n    """Likelihood-ratio loss on soft targets.  v8: optional per-cell weight [B, C]; the mean stays over cells, so a cell\n    with weight 0.5 pushes half as hard (weight None = the v7 loss, bit for bit)."""\n    per = -torch.logaddexp(F.logsigmoid(z) + torch.log(lr), F.logsigmoid(-z))\n    if weight is None:\n        return per.mean(0).mean()\n    return (per * weight).mean(0).mean()\n\n\ndef state_loss(sl, bl, st_idx, st_w, bin_t):\n    """Masked cross-entropy over report states (per class, weight-normalised) + masked BCE over the binaries.\n    st_idx [B, C] long (-1 unknown), st_w [B, C], bin_t [B, nbin] (nan unknown)."""\n    total, n = sl.new_zeros(()), 0\n    for c in range(sl.shape[1]):\n        ok = st_idx[:, c] >= 0\n        if ok.any():\n            ce = F.cross_entropy(sl[ok, c], st_idx[ok, c], reduction="none") * st_w[ok, c]\n            total = total + ce.sum() / st_w[ok, c].sum().clamp_min(1e-6); n += 1\n    known = ~torch.isnan(bin_t)\n    if known.any():\n        t = torch.nan_to_num(bin_t)\n        bce = F.binary_cross_entropy_with_logits(bl, t, reduction="none") * known\n        per = bce.sum(0) / known.sum(0).clamp_min(1)\n        total = total + per[known.any(0)].mean(); n += 1\n    return total / max(n, 1)\n\n\n# ---- v9.3: P1 families indexed by FAMILY (shared refactor) and the four R4d families ---------------------------------------\nAGE_CODES = {"age_unstated": 0, "chronic": 1, "acute": 2, "graft": 3, "no_tear": 4}   # --lig-age csv values (the E8 LIG protocol)\nOA_SMALL_TARGET = 0.2       # --oa-area-rank: extent-head soft target of a high_grade_small cell (prior 0.33-0.36: < 1 cm^2 more often than not)\nOA_L = ("Medial OA", "Lateral OA", "PF OA")\n\n\ndef load_lig_age(path, pool):\n    """v9.3: tear-age attribute per (study, class) from a csv with columns study, class, age (AGE_CODES keys): the E8 re-read\n    (reader A, provisional) or the adjudicated r4d_inputs/ligament_age.csv.  Cells absent from the csv = age_unstated (0),\n    as the coordinator\'s rule says (the state table\'s own acuity binary is NOT consulted)."""\n    Li = {l: c for c, l in enumerate(LABELS)}\n    idx = {u: i for i, u in enumerate(pool)}\n    out = np.zeros((len(pool), C), np.int64)\n    n_in = n_out = 0\n    counts = collections.Counter()\n    for r in csv.DictReader(open(path, encoding="utf-8")):\n        age, cls = (r.get("age") or "").strip(), (r.get("class") or "").strip()\n        if age not in AGE_CODES:\n            raise SystemExit("--lig-age: unknown age value %r for %s / %s" % (age, r.get("study"), cls))\n        if cls not in Li:\n            raise SystemExit("--lig-age: unknown class %r" % cls)\n        i = idx.get((r.get("study") or "").strip())\n        if i is None:\n            n_out += 1\n            continue\n        out[i, Li[cls]] = AGE_CODES[age]\n        n_in += 1\n        counts[(cls, age)] += 1\n    print("lig-age %s: %d cells on pool studies (%d rows outside the pool) %s" % (path, n_in, n_out, dict(sorted(counts.items()))), flush=True)\n    return out\n\n\ndef build_pair_families(spec, sti, bint, bin_names, pair_extra, oa_area_rank=False, oa_sign_decoy=False,\n                        acl_acuity_decoy=False, mcl_boundary=False, lig_age=None):\n    """v9.3: the P1 families as a list indexed by FAMILY (v8 / v9 indexed them by class: one family per class).  A family is\n    {label, pos / neg (state names), cof (co-finding arrays, -1 = unknown -> the study leaves the family), pos_mask / neg_mask\n    (optional bool[N]: an extra per-study membership condition beyond the state), quantity ("z" = class logit, "area" =\n    log1p(extent area / 100)), name}.  Returns (fams, role[N, F] in {+1, -1, 0}, sig[N, F], pw[N, F] = report prior of the\n    study\'s state in the family\'s class, fam_class[F]).  With the v9 list and no v9.3 flag, column f of role / sig / pw is\n    v9.2\'s column of that family\'s class, in the same family order (v93_identity_test.py checks it on the real tables)."""\n    cats, N = spec["categories"], len(sti)\n    Li = {l: c for c, l in enumerate(LABELS)}\n    bni = {(b, l): j for j, (b, l) in enumerate(bin_names)}\n\n    def sset(l, *names):\n        return [cats[l].index(x) for x in names]\n\n    def inset(l, idx_):                     # 1 / 0 per study, -1 when the report is silent on l\n        v = sti[:, Li[l]]\n        return np.where(v < 0, -1, np.isin(v, idx_).astype(np.int64))\n\n    def osteo(l):                           # "osteophytes written or not" (the table has 0 only for a normal compartment)\n        v = bint[:, bni[("sign_osteophytes", l)]]\n        return (v == 1).astype(np.int64)\n\n    TORN_M, HIGH = ("tear", "macerated_or_destroyed"), ("high_grade_size_unstated", "high_grade_large")\n    # v9.3: with a spec that carries high_grade_partial_tear (opusv11c) that officially positive state is an ACL tear both as a\n    # co-finding (signature of the meniscus families) and as a positive of the ACL family; with opusv11b the tuple is v9.2\'s\n    acl_torn = tuple(s for s in ("partial_tear_unspecified", "tear_unspecified", "complete_tear", "high_grade_partial_tear") if s in cats["ACL"])\n    acl_t = inset("ACL", sset("ACL", *acl_torn))\n    mm_t = inset("Medial Meniscus", sset("Medial Meniscus", *TORN_M))\n    lm_t = inset("Lateral Meniscus", sset("Lateral Meniscus", *TORN_M))\n    hi = {l: inset(l, sset(l, *HIGH)) for l in OA_L}\n    cont = inset("Contusion", sset("Contusion", "contusion_or_bruise", "traumatic_marrow_edema"))\n    ev = sti[:, Li["Effusion"]]\n    eff_b = np.select([ev < 0, ev == 0, ev == 1, np.isin(ev, [2, 3])], [-1, 0, 1, 2], 3)\n    fams = [("Lateral Meniscus", TORN_M, ("normal", "degeneration_no_tear"), [acl_t, mm_t, hi["Lateral OA"]]),\n            ("Medial Meniscus", TORN_M, ("normal", "degeneration_no_tear"), [acl_t, lm_t, hi["Medial OA"]]),\n            ("Lateral OA", HIGH, ("normal", "low_grade_chondral"), [osteo("Lateral OA"), hi["Medial OA"], hi["PF OA"]]),\n            ("Medial OA", HIGH, ("normal", "low_grade_chondral"), [osteo("Medial OA"), hi["Lateral OA"], hi["PF OA"]]),\n            ("PF OA", HIGH, ("normal", "low_grade_chondral"), [osteo("PF OA"), hi["Medial OA"], hi["Lateral OA"]]),\n            ("Synovitis", ("synovitis", "synovial_thickening"), ("other_synovial_finding", "none"), [eff_b]),\n            ("MCL", ("partial_tear_or_grade2", "high_grade_partial_tear", "complete_tear"),\n             ("normal", "periligamentous_edema_or_grade1", "chronic_or_healed_change"), [mm_t, cont, acl_t]),\n            ("ACL", acl_torn,\n             ("normal", "signal_or_degeneration", "low_grade_sprain_or_partial"), [cont, lm_t, mm_t])]\n    if pair_extra:                          # v9: within-class grade boundaries, no co-finding matching\n        fams += [("Effusion", ("moderate", "large"), ("none_or_trace", "small"), []),\n                 ("Baker\'s", ("moderate", "large"), ("none", "small"), []),\n                 ("Contusion", ("contusion_or_bruise", "traumatic_marrow_edema"), ("degenerative_or_reactive_edema",), []),\n                 ("Fracture", ("acute_fracture", "impaction_or_osteochondral_fracture", "avulsion_fracture"),\n                  ("trabecular_microfracture_only", "old_or_healed_fracture"), [])]\n    fams = [dict(label=l, pos=p, neg=n, cof=cf, pos_mask=None, neg_mask=None, quantity="z", name=l) for l, p, n, cf in fams]\n    if oa_sign_decoy:                       # v9.3: the bone-sign proxy crossed with the grade (signature 0: the crossing is the point)\n        for l in OA_L:\n            written = np.zeros(N, bool)\n            for b in ("sign_osteophytes", "sign_subchondral_marrow", "sign_bone_on_bone"):\n                written |= bint[:, bni[(b, l)]] == 1\n            fams.append(dict(label=l, pos=HIGH, neg=("low_grade_chondral",), cof=[], pos_mask=~written, neg_mask=written,\n                             quantity="z", name=l + "/sign_decoy"))\n    if oa_area_rank:                        # v9.3: the official area boundary inside the high grade, asked of the extent head\n        for l in OA_L:\n            if "high_grade_small" not in cats[l]:\n                raise SystemExit("--oa-area-rank needs the state high_grade_small in the spec of %s (state_spec_opusv11c.json)" % l)\n            fams.append(dict(label=l, pos=("high_grade_large",), neg=("high_grade_small",), cof=[], pos_mask=None, neg_mask=None,\n                             quantity="area", name=l + "/area_rank"))\n    if acl_acuity_decoy:                    # v9.3: a tear without acute company > an intact ACL with acute company\n        if lig_age is None:\n            raise SystemExit("--acl-acuity-decoy needs --lig-age")\n        age = lig_age[:, Li["ACL"]]\n        tears = tuple(s for s in ("tear_unspecified", "complete_tear", "high_grade_partial_tear") if s in cats["ACL"])\n        acute_cof = (cont == 1) | np.isin(ev, sset("Effusion", "moderate", "large"))\n        calm = (cont == 0) & np.isin(ev, sset("Effusion", "none_or_trace", "small"))\n        pos_mask = (age == AGE_CODES["chronic"]) | ((age == AGE_CODES["age_unstated"]) & calm)\n        fams.append(dict(label="ACL", pos=tears, neg=("normal", "signal_or_degeneration", "low_grade_sprain_or_partial"), cof=[],\n                         pos_mask=pos_mask, neg_mask=acute_cof, quantity="z", name="ACL/acuity_decoy"))\n    if mcl_boundary:                        # v9.3: the official MCL boundary (high-grade partial / complete acute > grade 2)\n        if lig_age is None:\n            raise SystemExit("--mcl-boundary needs --lig-age")\n        age = lig_age[:, Li["MCL"]]\n        pos_mask = ~np.isin(age, [AGE_CODES["chronic"], AGE_CODES["graft"], AGE_CODES["no_tear"]])\n        fams.append(dict(label="MCL", pos=("high_grade_partial_tear", "complete_tear"), neg=("partial_tear_or_grade2",), cof=[],\n                         pos_mask=pos_mask, neg_mask=None, quantity="z", name="MCL/boundary"))\n    Fn = len(fams)\n    role = np.zeros((N, Fn), np.int64)             # +1 positive / -1 hard negative of the family / 0\n    sig = np.zeros((N, Fn), np.int64)              # co-finding signature: pairs only within one signature\n    pw = np.zeros((N, Fn), np.float32)              # report prior of the study\'s state in the family\'s class (pair weight)\n    fam_class = []\n    for f, fm in enumerate(fams):\n        l = fm["label"]\n        c = Li[l]\n        v = sti[:, c]\n        pm = np.isin(v, sset(l, *fm["pos"]))\n        if fm["pos_mask"] is not None:\n            pm &= fm["pos_mask"]\n        nm = np.isin(v, sset(l, *fm["neg"]))\n        if fm["neg_mask"] is not None:\n            nm &= fm["neg_mask"]\n        role[pm, f] = 1\n        role[nm, f] = -1\n        unknown_cof = np.zeros(N, bool)\n        for arr in fm["cof"]:\n            unknown_cof |= np.asarray(arr) < 0\n        role[unknown_cof, f] = 0                 # an unknown co-finding cannot be matched: the study leaves this family\n        code = np.zeros(N, np.int64)\n        for arr in fm["cof"]:\n            code = code * 8 + (np.asarray(arr, np.int64) + 1)\n        sig[:, f] = code\n        ok = v >= 0\n        pw[ok, f] = np.asarray(spec["prior"][l], np.float32)[v[ok]]\n        fam_class.append(c)\n    return fams, role, sig, pw, fam_class\n\n\ndef pair_buckets_of(role, sig, pi):\n    """(positive dataset indices, negative dataset indices, probability) per (family, signature) with both present; every\n    family gets the same total probability, buckets within it ~ sqrt(n_pos * n_neg).  v9.2\'s pair_buckets with the family\n    axis in place of the class axis: the same list in the same order with the same probabilities for the v9 list."""\n    Fn = role.shape[1]\n    out = []\n    for f in range(Fn):\n        r, sg = role[pi, f], sig[pi, f]\n        fam = []\n        for v in np.unique(sg[r != 0]):\n            P_ = np.flatnonzero((r == 1) & (sg == v)); N_ = np.flatnonzero((r == -1) & (sg == v))\n            if len(P_) and len(N_):\n                fam.append((torch.from_numpy(P_), torch.from_numpy(N_), math.sqrt(len(P_) * len(N_))))\n        tot = sum(b[2] for b in fam)\n        out += [(p_, n_, w_ / tot / Fn) for p_, n_, w_ in fam]\n    return out\n\n\ndef family_summary(fams, role, sig):\n    """per family: positives, negatives, buckets (signatures with both sides) and pairable (positive, negative) pairs over the rows given."""\n    out = []\n    for f, fm in enumerate(fams):\n        r, sg = role[:, f], sig[:, f]\n        nb = npairs = 0\n        for v in np.unique(sg[r != 0]):\n            p_, n_ = int(((r == 1) & (sg == v)).sum()), int(((r == -1) & (sg == v)).sum())\n            if p_ and n_:\n                nb += 1; npairs += p_ * n_\n        out.append("%s[%s] pos %d neg %d buckets %d pairs %d" % (fm["name"], fm["quantity"], int((r == 1).sum()), int((r == -1).sum()), nb, npairs))\n    return out\n\n\ndef pair_loss_of(z, area, r, sg, pr, cw, fam_class, fam_q):\n    """Matched-pair ranking over families: for each ordered (positive i, hard negative j) of family f in the batch with the\n    same co-finding signature, w * softplus(q_j - q_i), w = (prior_i - prior_j) * min(sw_i, sw_j); q = the class logit\n    (quantity "z") or log1p(area / 100) of the extent head (quantity "area", gradient through the extent read-out and the\n    encoder, nothing detached); mean over active families.  v9.2\'s pair_loss with the family axis in place of the class axis:\n    the same operations in the same order for the v9 list."""\n    zf = z.float()\n    la = None if area is None else torch.log1p(area.float() / 100.0)\n    tot, cnt = zf.new_zeros(()), 0\n    for f, c in enumerate(fam_class):\n        P_ = torch.nonzero(r[:, f] == 1).flatten()\n        N_ = torch.nonzero(r[:, f] == -1).flatten()\n        if len(P_) == 0 or len(N_) == 0:\n            continue\n        same = (sg[P_, f][:, None] == sg[N_, f][None, :]).float()\n        w = (pr[P_, f][:, None] - pr[N_, f][None, :]).clamp_min(0) * torch.minimum(cw[P_, c][:, None], cw[N_, c][None, :]) * same\n        if float(w.sum()) <= 0:\n            continue\n        q = zf if fam_q[f] == "z" else la\n        if q is None:\n            raise SystemExit("an \'area\' family needs the extent area (--extent)")\n        # mean over the valid pairs with w as each pair\'s weight (dividing by sum(w) would cancel w for a single pair)\n        tot = tot + (w * F.softplus(q[N_, c][None, :] - q[P_, c][:, None])).sum() / same.sum()\n        cnt += 1\n    return tot / max(cnt, 1), cnt\n\n\nclass PairBatchSampler:\n    """Training batches of `batch` dataset indices.  With probability `frac` a batch opens with a matched pair -- a positive\n    and a hard negative of one pair-family class that share the co-finding signature (bucket drawn with the given\n    probabilities) -- and is filled with base draws; otherwise all draws are base draws (weighted with replacement, as\n    the rare-boost sampler).  len = n // batch: the same number of steps per epoch as the plain loader."""\n\n    def __init__(self, n, batch, weights, buckets, frac, seed):\n        self.n, self.batch, self.frac = int(n), int(batch), float(frac)\n        self.w = torch.ones(self.n, dtype=torch.double) if weights is None else torch.as_tensor(weights, dtype=torch.double)\n        self.buckets = buckets\n        self.bprob = torch.tensor([b[2] for b in buckets], dtype=torch.double) if buckets else None\n        self.generator = torch.Generator().manual_seed(int(seed))\n        assert self.batch >= 2\n\n    def __len__(self):\n        return self.n // self.batch\n\n    def _base(self, k):\n        return torch.multinomial(self.w, k, replacement=True, generator=self.generator).tolist() if k > 0 else []\n\n    def __iter__(self):\n        g = self.generator\n        for _ in range(len(self)):\n            if self.buckets and float(torch.rand(1, generator=g)) < self.frac:\n                pos, neg, _ = self.buckets[int(torch.multinomial(self.bprob, 1, generator=g))]\n                i = pos[int(torch.randint(0, len(pos), (1,), generator=g))]\n                j = neg[int(torch.randint(0, len(neg), (1,), generator=g))]\n                yield [int(i), int(j)] + self._base(self.batch - 2)\n            else:\n                yield self._base(self.batch)\n\n\n# ---- v9.6: --draw-seed (SCAN_A D1) -- every training draw its own window stream ---------------------------------------------------\n# Both training samplers draw WITH replacement (pair sampler base draws, rare-boost WeightedRandomSampler), but Studies seeded its rng with\n# (seed, epoch, index) only, so a study drawn k times in one epoch was served the same windows k times.  With --draw-seed the training\n# samplers below yield (index, nonce), nonce = the draw\'s position in the epoch (batch * batch size + slot), and Studies seeds\n# np.random.default_rng([seed, epoch, index, nonce]).  The subclasses only re-wrap what their parent yields: the parent\'s generator calls\n# -- and so the drawn index sequence -- are unchanged, and the isinstance checks of the checkpoint (sampler stream) keep working.\nclass NoncePairBatchSampler(PairBatchSampler):\n    def __iter__(self):\n        for b, batch in enumerate(super().__iter__()):\n            yield [(int(i), b * self.batch + s) for s, i in enumerate(batch)]\n\n\nclass NonceWeightedSampler(torch.utils.data.WeightedRandomSampler):\n    def __iter__(self):\n        for pos, i in enumerate(super().__iter__()):\n            yield (int(i), pos)\n\n\nclass NonceRandomSampler(torch.utils.data.RandomSampler):\n    def __iter__(self):\n        for pos, i in enumerate(super().__iter__()):\n            yield (int(i), pos)\n\n\n# ---- v9.6: --rare-boost-names (SCAN_A D4) -- the rare-boost terms by state name ----------------------------------------------------\n# v9.5\'s rare_weights tests MCL by index (state >= 2), which also boosts index 3 = chronic_or_healed_change, a state the MCL pair family\n# treats as a negative.  By name: MCL +2 only for the three tear states below (or the acute-MCL binary, unchanged); the other four terms\n# are the names of exactly the indices v9.5 tests (Fracture >= 1, PF OA in {3, 4}, Baker\'s in {3, 4, 5}, Contusion in {4, 5} under\n# opusv11b / opusv11c), asserted at start-up to select the same studies as the index rules under the run\'s spec, so only MCL changes.\nRARE_NAME_TERMS = (("MCL", 2.0, ("partial_tear_or_grade2", "high_grade_partial_tear", "complete_tear")),\n                   ("Fracture", 2.0, ("trabecular_microfracture_only", "insufficiency_or_stress_fracture", "avulsion_fracture",\n                                      "impaction_or_osteochondral_fracture", "fracture_age_unstated", "acute_fracture", "old_or_healed_fracture")),\n                   ("PF OA", 1.0, ("high_grade_size_unstated", "high_grade_large")),\n                   ("Baker\'s", 1.0, ("moderate", "large", "ruptured_or_leaking")),\n                   ("Contusion", 1.0, ("traumatic_marrow_edema", "contusion_or_bruise")))\nRARE_INDEX_RULES = {"MCL": lambda s: s >= 2, "Fracture": lambda s: s >= 1, "PF OA": lambda s: np.isin(s, (3, 4)),\n                    "Baker\'s": lambda s: np.isin(s, (3, 4, 5)), "Contusion": lambda s: np.isin(s, (4, 5))}   # v9.5\'s rare_weights, per term\n\n\ndef rare_term_masks(st, cats, acute, by_name):\n    """Per rare-boost term a bool [N] over the rows of the state-index matrix st [N, C] (-1 = unknown): by_name -> the RARE_NAME_TERMS name\n    sets looked up in the spec\'s categories, else v9.5\'s index rules; the MCL term also when the acute-MCL binary is 1 (acute: bool [N])."""\n    out = {}\n    for l, _, names in RARE_NAME_TERMS:\n        s = st[:, LABELS.index(l)]\n        m = np.isin(s, [cats[l].index(n_) for n_ in names]) if by_name else np.asarray(RARE_INDEX_RULES[l](s), bool)\n        out[l] = (m | acute) if l == "MCL" else m\n    return out\n\n\ndef rare_weights_of(masks):\n    """v9.5\'s weight formula on term masks: 1 + 2 MCL + 2 Fracture + PF OA + Baker\'s + Contusion, capped at 4."""\n    v = np.ones(len(masks["MCL"]))\n    for l, wt, _ in RARE_NAME_TERMS:\n        v = v + wt * masks[l]\n    return np.minimum(v, 4.0)\n\n\n# ---- v9.4: anatomical mirror + output permutation (spec card spec_trainer_v94_mirror_20261010.md) ------------------------------------\nMIRROR_PAIRS = (("Medial Meniscus", "Lateral Meniscus"), ("Medial OA", "Lateral OA"))   # the class pairs P exchanges; the rest fixed\n\n\ndef mirror_perm(spec=None, bin_names=None):\n    """The output permutation P of a mirrored batch as index lists: perm over the 12 classes (Medial <-> Lateral Meniscus, Medial OA\n    <-> Lateral OA, every other class a fixed point; P is its own inverse) and, with a state spec, bin_perm over the binaries (the\n    same exchange on the binary\'s parent class).  Refuses a spec whose two classes of a pair do not share the state category list\n    (the state head\'s per-class K-way outputs would not correspond slot for slot) and a binary that exists for one side only."""\n    swap = {}\n    for m_, l_ in MIRROR_PAIRS:\n        swap[m_], swap[l_] = l_, m_\n    perm = [LABELS.index(swap.get(l, l)) for l in LABELS]\n    bin_perm = None\n    if spec is not None:\n        for m_, l_ in MIRROR_PAIRS:\n            if spec["categories"][m_] != spec["categories"][l_]:\n                raise SystemExit("--mirror needs identical state category lists for %s and %s; the spec has %s vs %s" % (\n                    m_, l_, spec["categories"][m_], spec["categories"][l_]))\n        if bin_names is not None:\n            bin_perm = []\n            for b, l in bin_names:\n                partner = (b, swap.get(l, l))\n                if partner not in bin_names:\n                    raise SystemExit("--mirror: the binary %s exists for %s but not for %s" % (b, l, swap[l]))\n                bin_perm.append(bin_names.index(partner))\n    return perm, bin_perm\n\n\ndef mirror_batch(img, meta):\n    """Every window of a training batch as seen in the virtual contralateral knee.  img [B, W, K, S, S] (after augment), meta\n    [B, W, >= 8] with the plane in column 7 (0 sagittal / 1 coronal / 2 axial; the loader adds the extent columns for --mirror as\n    it does for --extent).  Coronal and axial windows are flipped on the width axis (the patient\'s left-right axis in both\n    planes).  A sagittal window is NOT flipped in-plane (its in-plane axes are anterior-posterior and superior-inferior): its mirror\n    runs along the slice axis, which the cache stores in a side-canonical order (px_cache: a right knee\'s positions are sign-\n    reversed before sorting), so the K slice channels [s-r .. s+r] become [s+r .. s-r] and rel_pos (column 2, 2(pos-pos0)/span - 1\n    in [-1, 1]) becomes -rel_pos.  The coronal bilateral locals change sides: view id 2 <-> 3 (column 1; coronal_views.py puts them\n    at column .25 / .75 of the field, which the flip exchanges).  Slot, spacing, extent, fluid and the extent-head columns are\n    unchanged; padded windows (zero meta) are flipped as sagittal zeros and stay zero.  Training forward only: refuses without\n    autograd, which is how predict / reeval / bn_recalibrate run their forwards."""\n    assert torch.is_grad_enabled(), "v9.4 mirror reached without autograd: the evaluation paths never mirror"\n    assert meta.shape[-1] >= 8, "v9.4 mirror needs the plane column (meta column 7)"\n    sag = meta[..., 7].round() == 0\n    img = torch.where(sag[:, :, None, None, None], img.flip(2), img.flip(-1))\n    meta = meta.clone()\n    meta[..., 2] = torch.where(sag, -meta[..., 2], meta[..., 2])\n    v = meta[..., 1]\n    meta[..., 1] = torch.where(v == 2, torch.full_like(v, 3.), torch.where(v == 3, torch.full_like(v, 2.), v))\n    return img, meta\n\n\ndef mirror_outputs(z, sl, bl, area, w_main, st_idx, st_w, bin_t, perm, bin_perm, mcl_bins):\n    """A mirrored batch\'s model outputs re-indexed to the canonical class order: z[:, perm] puts the model\'s lateral output under\n    the medial target and vice versa (menisci, OA compartments), the state logits, the binaries and the extent area the same way\n    (the syn-decomp columns Synovitis / Effusion are fixed points of P), so the canonical targets, weights, state targets and\n    pair-family signatures apply unchanged.  MCL leaves every loss of the batch: main weight 0 (a weight of ones is created when\n    the arm carries no cell weights), state weight 0 with the state index unknown, its binaries unknown; the pair loss receives\n    the zeroed state weight as its cell weight, so every MCL family\'s pair weight is 0.  The selections carry gradient as is."""\n    MCL = LABELS.index("MCL")\n    z = z[:, perm]\n    if sl is not None:\n        sl, bl = sl[:, perm], bl[:, bin_perm]\n    if area is not None:\n        area = area[:, perm]\n    w_main = torch.ones_like(z) if w_main is None else w_main.clone()\n    w_main[:, MCL] = 0\n    if st_w is not None:\n        st_idx, st_w, bin_t = st_idx.clone(), st_w.clone(), bin_t.clone()\n        st_idx[:, MCL] = -1\n        st_w[:, MCL] = 0\n        if mcl_bins:\n            bin_t[:, mcl_bins] = float("nan")\n    return z, sl, bl, area, w_main, st_idx, st_w, bin_t\n\n\n# ---- v9.5: batch translation jitter (spec card spec_trainer_v95_shift_20261010.md) ---------------------------------------------------\ndef shift_draw(n, gen):\n    """One training batch\'s translation (dy, dx), each uniform on the integers -n .. n, from the shift stream `gen` (its own\n    generator: no other augmentation draw moves).  Two draws per batch, dy first."""\n    dy = int(torch.randint(-n, n + 1, (1,), generator=gen))\n    dx = int(torch.randint(-n, n + 1, (1,), generator=gen))\n    return dy, dx\n\n\ndef shift_batch(img, dy, dx, n):\n    """Every window of a training batch translated by the same integer (dy, dx) with zero fill.  img [..., H, W] (the cropped uint8\n    batch, [B, W, K, crop, crop]) is padded with n zeros on both pixel axes and cropped back at offset (n + dy, n + dx):\n    out[..., y, x] = img[..., y + dy, x + dx] where that pixel exists, 0 where it does not (what leaves the field is dropped and the\n    entering border is zero -- not torch.roll, which would wrap the opposite side\'s anatomy in).  One (dy, dx) for every study,\n    window, plane and slice channel of the batch.  Pure indexing (no autocast op, no interpolation, the dtype is kept).  Training\n    forward only: refuses without autograd, which is how predict / reeval / bn_recalibrate run their forwards (they take the centre\n    crop and never reach augment)."""\n    assert torch.is_grad_enabled(), "v9.5 shift reached without autograd: the evaluation paths never shift"\n    assert -n <= dy <= n and -n <= dx <= n, "v9.5 shift (%d, %d) outside -%d..%d" % (dy, dx, n, n)\n    H, W_ = img.shape[-2:]\n    return F.pad(img, (n, n, n, n))[..., n + dy:n + dy + H, n + dx:n + dx + W_]\n\n\n# ---- v9.6: warm start = repair training (spec card spec_r5_repair_training_20261010.md) ----------------------------------------------\n# run_config fields that must be equal in the source run (--warm-start DIR) and this run: K and the holdout fold (which studies each fold\n# model never saw: the hash folds) and every flag that changes what Net holds (backbone, crop, slices = input channels, spatial attention,\n# the extent head and its logit classes, syn-decomp).  A key absent from an older run_config (v9.3 has no extent_classes) is that flag\'s\n# default.  The state spec (by content) and the pool (pool_uids.json) are compared on their own.\nWARM_FIELDS = ("folds", "holdout_fold", "backbone", "crop", "slices", "spatial_attn", "extent", "extent_classes", "syn_decomp")\n# Net\'s buffers fall in two groups (Net registers exactly these six).  CONFIG buffers come from flags asserted equal above (input\n# normalisation from --slices, the extent logit mask from --extent-classes): a fresh Net of this run must hold exactly the file\'s values\n# (refused otherwise).  SPEC buffers come from the state spec\'s prior / base values (state prior, valid slots, base logit): this run may\n# bring a spec whose categories and binaries are the source\'s but whose prior / base values are new (R5: re-calibrated labels), so after\n# the strict load they are re-set from THIS run\'s spec -- the loaded values would otherwise silently replace the new spec\'s -- while every\n# learned parameter (state_gate, state_w, state_b, bin_w, bin_b included) keeps the loaded value.  Nothing else in Net comes from the\n# spec; the pair-family weights and the extent targets are rebuilt from this run\'s spec in main() and are not part of the model.\nWARM_CONFIG_BUFFERS = ("mean", "std", "ext_mask")\nWARM_SPEC_BUFFERS = ("state_prior", "state_valid", "state_base_logit")\n\n\ndef file_md5(path):\n    h = hashlib.md5()\n    with open(path, "rb") as fh:\n        for blk in iter(lambda: fh.read(1 << 24), b""):\n            h.update(blk)\n    return h.hexdigest()\n\n\ndef warm_load(model, path):\n    """v9.6 --warm-start: a source run\'s fold model (the plain tensor dict that Net.deploy_state_dict wrote -- the whole state dict for\n    every timm backbone) loaded into the freshly built `model` with strict=True: the encoder, every head, every parameter and buffer; a\n    missing, unexpected or reshaped key refuses (never a backbone-only load).  Before loading, each CONFIG buffer of the fresh model must\n    equal the file\'s bit for bit and each SPEC buffer must have the file\'s dtype and shape (WARM_CONFIG_BUFFERS / WARM_SPEC_BUFFERS); after\n    loading, the SPEC buffers are put back to the fresh model\'s values = this run\'s state spec.  With the source\'s own spec that write-back\n    restores the very values just loaded (bit for bit a plain strict load).  Returns (tensors loaded, md5 of the file, {spec buffer: per-class\n    change vs the loaded value -- max |d| over the slots for state_prior, the number of changed slots for state_valid, |d| for\n    state_base_logit}).\n    """\n    import io\n    if not os.path.isfile(path):\n        raise SystemExit("--warm-start: fold model %s does not exist" % path)\n    with open(path, "rb") as fh:                         # one read: the md5 is that of the very bytes loaded\n        raw = fh.read()\n    md5 = hashlib.md5(raw).hexdigest()\n    sd = torch.load(io.BytesIO(raw), map_location="cpu")\n    del raw\n    own = model.state_dict()\n    for name in WARM_CONFIG_BUFFERS + WARM_SPEC_BUFFERS:\n        if (name in own) != (name in sd):\n            raise SystemExit("--warm-start: buffer %s is %s this run\'s Net but %s %s" % (\n                name, "in" if name in own else "not in", "in" if name in sd else "not in", path))\n        if name in own:\n            b_own, b_src = own[name].detach().cpu(), sd[name]\n            if b_own.dtype != b_src.dtype or b_own.shape != b_src.shape:\n                raise SystemExit("--warm-start: %s holds a %s of dtype / shape %s %s; this run\'s Net builds %s %s (another state spec "\n                                 "category layout, extent classes or slices)" % (path, name, b_src.dtype, tuple(b_src.shape), b_own.dtype,\n                                                                                 tuple(b_own.shape)))\n            if name in WARM_CONFIG_BUFFERS and not torch.equal(b_own, b_src):\n                raise SystemExit("--warm-start: %s holds another %s than this run\'s Net builds (other extent classes or slices)" % (path, name))\n            # with identical categories (asserted at start-up) the valid slots cannot differ unless a spec\'s prior lists do not match its\n            # categories or the source\'s spec file was edited after the source run: refused (a valid target slot masked to -inf\n            # would give an infinite state cross-entropy)\n            if name == "state_valid" and not torch.equal(b_own, b_src):\n                raise SystemExit("--warm-start: %s holds other state_valid slots than this run\'s spec builds (a spec whose prior lists do "\n                                 "not match its categories, or a source spec file edited after the source run)" % path)\n    spec_now = {n_: own[n_].detach().clone() for n_ in WARM_SPEC_BUFFERS if n_ in own}   # this run\'s spec, before the load overwrites it\n    aux_keys = {k_ for k_ in own if k_.startswith("aux_")}   # v9.8: the --fm-aux head (no fold model carries it: deploy_state_dict strips it)\n    if not aux_keys:                                       # v9.7\'s strict load, unchanged\n        try:\n            model.load_state_dict(sd, strict=True)\n        except RuntimeError as e:\n            raise SystemExit("--warm-start: %s does not match this run\'s Net (strict load): %s" % (path, str(e)[:3000]))\n    else:                                                  # v9.8: strict except that the missing keys must be EXACTLY the aux_* set\n        try:\n            missing, unexpected = model.load_state_dict(sd, strict=False)\n        except RuntimeError as e:                          # a reshaped key still raises with strict=False\n            raise SystemExit("--warm-start: %s does not match this run\'s Net: %s" % (path, str(e)[:3000]))\n        if set(missing) != aux_keys or unexpected:\n            raise SystemExit("--warm-start: %s does not match this run\'s Net: the missing keys must be exactly the auxiliary head %s; "\n                             "missing %s, unexpected %s" % (path, sorted(aux_keys), sorted(missing)[:20], sorted(unexpected)[:20]))\n    change = {}\n    if spec_now:\n        bufs = dict(model.named_buffers())\n        with torch.no_grad():\n            for n_, v_ in spec_now.items():\n                old = bufs[n_].detach().clone()                  # the source run\'s value, as loaded\n                bufs[n_].copy_(v_)\n                if v_.dtype == torch.bool:\n                    change[n_] = (v_ != old).sum(-1).tolist()\n                else:\n                    d_ = (v_.float() - old.float()).abs()\n                    change[n_] = (d_.amax(-1) if d_.ndim > 1 else d_).tolist()\n    return len(sd), md5, change\n\n\ndef file_ids(d):\n    """{(device, inode): path} of the files directly in d, d/models and d/ckpt (a hard link shares its inode)"""\n    out = {}\n    for sub in ("", "models", "ckpt"):\n        dd = os.path.join(d, sub)\n        if os.path.isdir(dd):\n            for f in sorted(os.listdir(dd)):\n                p = os.path.join(dd, f)\n                if os.path.isfile(p):\n                    st = os.stat(p)\n                    out[(st.st_dev, st.st_ino)] = p\n    return out\n\n\n# ---- v9.7: --save-state-oof -- the state head\'s read-out next to the class OOF (spec card spec_trainer_v97_state_oof_20261010.md) ---------\ndef state_layout(spec):\n    """The column layout of the state read-out, fixed by the state spec: per class (LABELS order) a block of K_c columns = its categories in\n    the spec\'s order (the state head\'s slots 0 .. K_c-1), then one column per binary head in Net\'s order (the spec\'s binaries dict order, each\n    binary\'s class list order = Net.bin_names).  Refuses a spec whose prior list length differs from its category count (Net takes the\n    valid slots from the prior lists and K_c from the categories: the slots would not be the categories)."""\n    bad = [l for l in LABELS if len(spec["prior"][l]) != len(spec["categories"][l])]\n    if bad:\n        raise SystemExit("--save-state-oof: the state spec\'s prior list length differs from its category count for %s (the state head\'s "\n                         "valid slots would not be the categories)" % bad)\n    cols, j = [], 0\n    for l in LABELS:\n        k_ = len(spec["categories"][l])\n        cols.append({"class": l, "categories": list(spec["categories"][l]), "start": j, "stop": j + k_})\n        j += k_\n    for b, labs in spec["binaries"].items():\n        for l in labs:\n            cols.append({"binary": b, "class": l, "column": j})\n            j += 1\n    return {"columns": cols, "width": j}\n\n\ndef state_slots_check(model, layout):\n    """v9.7: the model\'s state head must have exactly the layout\'s slots -- per class valid slots 0 .. K_c-1 and no other, K_c = the\n    layout\'s block width -- and the layout\'s binaries (name, class) in the same order; refused otherwise (never a silent column shift)."""\n    blocks = [c_ for c_ in layout["columns"] if "categories" in c_]\n    valid = model.state_valid.detach().cpu()\n    for c, b_ in enumerate(blocks):\n        kc = b_["stop"] - b_["start"]\n        if b_["class"] != LABELS[c] or model.state_k[c] != kc or not bool(valid[c, :kc].all()) or bool(valid[c, kc:].any()):\n            raise SystemExit("--save-state-oof: the state head\'s slots for %s (K %d, valid %s) are not the layout\'s %d categories" % (\n                LABELS[c], model.state_k[c], valid[c].tolist(), kc))\n    if [(c_["binary"], c_["class"]) for c_ in layout["columns"] if "binary" in c_] != [tuple(x) for x in model.bin_names]:\n        raise SystemExit("--save-state-oof: the state head\'s binaries %s are not the layout\'s" % (model.bin_names,))\n\n\ndef state_probs(model, sl, bl):\n    """v9.7: the state read-out of one evaluation batch, [B, D] float32.  sl [B, C, Kmax] / bl [B, nbin] are the state / binary logits that\n    Net.head computed in the forward that gave the batch\'s class logits (return_aux).  Per class: p = sl.softmax(-1) -- exactly the p of\n    Net.state_evidence (the -inf invalid slots get 0) -- restricted to its K_c valid slots 0 .. K_c-1; then sigmoid of every binary logit."""\n    p = sl.softmax(-1)\n    return torch.cat([p[:, c, :kc] for c, kc in enumerate(model.state_k)] + [torch.sigmoid(bl)], 1).float()\n\n\ndef state_files(out_dir, pre, arm, seed):\n    """v9.7: the three read-out files of one (arm, seed): pool array, its layout sidecar, gold json."""\n    return [os.path.join(out_dir, pre + n_ % (arm, seed)) for n_ in ("oof_state_%s_s%d.npy", "oof_state_%s_s%d.json", "preds_state_%s_s%d.json")]\n\n\ndef write_state_readout(out_dir, pre, arm, seed, layout, spec_path, pool_n, st_pool, gold, gold_fold, st_gold, K, H, only, reeval_of):\n    """v9.7: writes the three files of state_files().  st_pool: float32 [pool, D] in pool order (NaN = not predicted here; holdout-fold rows\n    = sum over this file\'s fold models of p / (K - 1)); st_gold: {fold k: float32 [gold, D]} = fold model k\'s read-out of every gold\n    study.  The gold mean (arithmetic, float64, over the fold models in fold order) is written only when the file holds every fold model\n    of the run (merge_partial_v2.py writes it for a merged run)."""\n    p_npy, p_side, p_gold = state_files(out_dir, pre, arm, seed)\n    folds_here = sorted(st_gold)\n    expected = [k_ for k_ in range(K) if k_ != H]\n    blocks = [c_ for c_ in layout["columns"] if "categories" in c_]\n    bcols = [c_["column"] for c_ in layout["columns"] if "binary" in c_]\n    common = {"arm": arm, "seed": seed, "labels": LABELS, "state_spec": spec_path, "state_spec_md5": file_md5(spec_path), "folds": K,\n              "holdout_fold": H, "only_folds": sorted(only) if only else None, "fold_models": folds_here, "reeval_of": reeval_of,\n              "values": "per class: softmax of the state head\'s logits over the class\'s valid slots (the p of Net.state_evidence); per binary "\n                        "head: sigmoid of its logit; float32, from the bf16-autocast evaluation forward that gives the class logits"}\n\n    def lists(arr):                             # [n, D] -> per class [n][K_c] lists + the binaries [n][nbin] (float32 values, unrounded)\n        return {"state": {b_["class"]: arr[:, b_["start"]:b_["stop"]].tolist() for b_ in blocks}, "binary": arr[:, bcols].tolist()}\n    np.save(p_npy, st_pool.astype(np.float32))\n    side = dict(common, kind="v9.7 state-head read-out: pool, out-of-fold", array=os.path.basename(p_npy), shape=list(st_pool.shape),\n                dtype="float32", rows="pool order = pool_uids.json (%d studies)" % pool_n, columns=layout["columns"], width=layout["width"],\n                nan_rows="a pool study this process did not predict (another fold\'s study under --only-folds) is a NaN row",\n                holdout_rows=None if H is None else "holdout fold %d: sum over this file\'s fold models of p / (K - 1) (the class OOF\'s "\n                                                    "convention: the mean over the K - 1 fold models once every fold model is in)" % H)\n    json.dump(side, open(p_side, "w"), indent=1)\n    g = dict(common, kind="v9.7 state-head read-out: gold", gold_uids=gold, gold_fold=gold_fold,\n             categories={b_["class"]: b_["categories"] for b_ in blocks},\n             binaries=[[c_["binary"], c_["class"]] for c_ in layout["columns"] if "binary" in c_],\n             per_fold_model={str(k_): lists(st_gold[k_]) for k_ in folds_here},\n             mean=lists(np.stack([st_gold[k_] for k_ in folds_here]).astype(np.float64).mean(0)) if folds_here == expected else None,\n             mean_note="arithmetic mean over the fold models (fold order) of each model\'s probability vectors; null while the file lacks "\n                       "a fold model of the run (a partial file)")\n    json.dump(g, open(p_gold, "w"))\n\n\n# ---- v9.8: --fm-aux -- the fastMRI+ lesion-box auxiliary task on the shared encoder (spec card spec_trainer_v98_fmaux_20261011.md B) ------\nFM_CLASSES = ("meniscus_tear", "cart_partial", "cart_full", "bone_edema", "bone_fx_contusion", "effusion", "cyst", "acl_high", "acl_low",\n              "collateral", "other")       # A4: auxiliary class ids 0 .. 10 (never merged)\nFM_A = len(FM_CLASSES)\nFM_VIEWS = (0, 2, 3)            # px_cache v4\'s coronal views: 0 = the whole field, 2 / 3 = the 100 mm locals (medial / lateral)\nFM_SIZE = 384                   # the store\'s rendering size (as px384_v4)\nFM_CELL = 32                    # the encoder\'s stride: G = crop / 32 cells per side (11 at 352, 12 at 384)\nFM_VOLS_PER_STEP = 4            # B2: 4 volumes x M / 4 windows per optimiser step\nFM_BOXED_FRAC = 0.75            # B2: share of the volume draws from volumes with boxes (the rest: reviewed volumes without findings)\nFM_POS_WEIGHT = 8.0             # B1: pos_weight of the dense BCE (pre-registered, fixed)\nFM_PRESENCE_MIN = 0.25          # B1: a box makes its class present when >= 25% of its original area is inside the window\nFM_BIAS0 = -4.0                 # aux head bias at init: sigmoid(-4) = 0.018 for sparse targets (ext_b\'s choice)\nFM_GATE = (("meniscus_tear", 0.75), ("effusion", 0.80), ("cart_full", 0.75))   # B5: the "the head learned" diagnostic gate\nFM_DONE_RE = re.compile(r"^(partial_)?preds_(?!state_).+_s\\d+\\.json$")      # E2: a run\'s done marker (not the v9.7 state read-out\'s gold json)\n\n\nclass FmStore:\n    """v9.8: the fm384_v1 store (spec A4), read and checked at start-up; any inconsistency is refused.  Files in the directory:\n      px.u8.npy     uint8 [N, 384, 384] (np.save; read with mmap)\n      index.json    {"format": "fm384_v1", "size": 384, "classes": FM_CLASSES (id order), "n_rows": N,\n                     "volumes": {<volume>: {"n_slices": n, "views": [[0, s0], [2, s2], [3, s3]], "patient_id": str, "aux_holdout": bool,\n                                            optional "excluded": true (skipped, counted); other keys (acquisition, side score / decision,\n                                            anchor) are kept for the log only}}}\n                    the rows of (volume, view v) are s_v .. s_v + n - 1 in slice order (slice k = the k-th RSS slice, 3 mm apart)\n      boxes.json    {"format": "fm384_v1", "n_rows": N, "boxes": {<row>: [[class id, x0, y0, x1, y1, a0], ...]}} (rows without a box absent)\n                    x = column, y = row of the row\'s 384 view frame (pixel (r, c) covers [c, c+1) x [r, r+1)), clipped to [0, 384];\n                    a0 = the box\'s area in that frame BEFORE the clip (the presence rule\'s "original box area", >= the clipped area)\n      manifest.json any json object (input sha256, render code md5, the three check numbers): logged\n    Checks: the format tags, classes, shapes; every usable volume has exactly the views {0, 2, 3} (no view-0-only store: that would be\n    the scaffold_view0 degradation), row blocks inside N and disjoint; every box row inside a block, class 0..10, 0 <= x0 < x1 <= 384,\n    0 <= y0 < y1 <= 384, a0 >= 0.99 x the clipped area - 1 (rounding slack); one aux_holdout value per patient_id; >= min_volumes usable volumes; >= 1 holdout volume,\n    >= 1 training volume with boxes and >= 1 without; every class with a box in the training part."""\n\n    def __init__(self, d, min_volumes):\n        self.dir = os.path.abspath(d)\n        need = ("px.u8.npy", "index.json", "boxes.json", "manifest.json")\n        miss = [f_ for f_ in need if not os.path.isfile(os.path.join(self.dir, f_))]\n        if miss:\n            raise SystemExit("--fm-aux %s: missing %s" % (self.dir, miss))\n        self.md5 = {f_: file_md5(os.path.join(self.dir, f_)) for f_ in ("index.json", "boxes.json", "manifest.json")}\n        index = json.load(open(os.path.join(self.dir, "index.json"), encoding="utf-8"))\n        boxes = json.load(open(os.path.join(self.dir, "boxes.json"), encoding="utf-8"))\n        self.manifest = json.load(open(os.path.join(self.dir, "manifest.json"), encoding="utf-8"))\n        px = np.load(os.path.join(self.dir, "px.u8.npy"), mmap_mode="r")\n        bad = []\n        if index.get("format") != "fm384_v1" or boxes.get("format") != "fm384_v1":\n            bad.append("format tags %r / %r (want fm384_v1)" % (index.get("format"), boxes.get("format")))\n        if index.get("size") != FM_SIZE:\n            bad.append("size %r (want %d)" % (index.get("size"), FM_SIZE))\n        if list(index.get("classes") or []) != list(FM_CLASSES):\n            bad.append("classes %r (want %r)" % (index.get("classes"), list(FM_CLASSES)))\n        if px.dtype != np.uint8 or px.ndim != 3 or tuple(px.shape[1:]) != (FM_SIZE, FM_SIZE):\n            bad.append("px.u8.npy %s %s (want uint8 [N, %d, %d])" % (px.dtype, tuple(px.shape), FM_SIZE, FM_SIZE))\n        N = int(px.shape[0])\n        self.n_rows = N\n        if index.get("n_rows") != N or boxes.get("n_rows") != N:\n            bad.append("n_rows index %r / boxes %r vs px rows %d" % (index.get("n_rows"), boxes.get("n_rows"), N))\n        if bad:\n            raise SystemExit("--fm-aux %s refused: %s" % (self.dir, " | ".join(bad)))\n        vols_all = index.get("volumes") or {}\n        self.excluded = sorted(v for v, r in vols_all.items() if r.get("excluded"))\n        self.vols = sorted(v for v, r in vols_all.items() if not r.get("excluded"))\n        self.n, self.start, self.pid, self.hold = [], {}, [], []\n        spans = []\n        for vi, v in enumerate(self.vols):\n            r = vols_all[v]\n            n_ = r.get("n_slices")\n            views = r.get("views")\n            if not isinstance(n_, int) or n_ < 1:\n                bad.append("%s: n_slices %r" % (v, n_)); continue\n            try:\n                vd = {int(a_): int(b_) for a_, b_ in views}\n            except (TypeError, ValueError):\n                bad.append("%s: views %r (want [[0, start], [2, start], [3, start]])" % (v, views)); continue\n            if sorted(vd) != sorted(FM_VIEWS) or len(views) != len(FM_VIEWS):\n                bad.append("%s: views %s (want exactly %s)" % (v, sorted(vd), list(FM_VIEWS))); continue\n            for w_, s_ in vd.items():\n                if s_ < 0 or s_ + n_ > N:\n                    bad.append("%s view %d: rows %d..%d outside 0..%d" % (v, w_, s_, s_ + n_ - 1, N - 1))\n                self.start[(vi, w_)] = s_\n                spans.append((s_, s_ + n_, v, w_))\n            pid_ = r.get("patient_id")\n            if not isinstance(pid_, str) or not pid_:\n                bad.append("%s: patient_id %r" % (v, pid_))\n            if not isinstance(r.get("aux_holdout"), bool):\n                bad.append("%s: aux_holdout %r (want true / false)" % (v, r.get("aux_holdout")))\n            self.n.append(int(n_)); self.pid.append(pid_); self.hold.append(bool(r.get("aux_holdout")))\n            if len(bad) > 20:\n                break\n        spans.sort()\n        for (a0_, a1_, va, wa), (b0_, b1_, vb, wb) in zip(spans, spans[1:]):\n            if b0_ < a1_:\n                bad.append("row blocks overlap: %s view %d rows %d..%d and %s view %d rows %d..%d" % (va, wa, a0_, a1_ - 1, vb, wb, b0_, b1_ - 1))\n                break\n        if bad:\n            raise SystemExit("--fm-aux %s refused: %s" % (self.dir, " | ".join(bad[:20])))\n        self.n = np.array(self.n, np.int64)\n        self.hold = np.array(self.hold, bool)\n        row_vol = np.full(N, -1, np.int64)                  # row -> volume index (-1: not in a usable volume\'s block)\n        row_k = np.full(N, -1, np.int64)                    # row -> slice index\n        for (vi, w_), s_ in self.start.items():\n            row_vol[s_:s_ + self.n[vi]] = vi\n            row_k[s_:s_ + self.n[vi]] = np.arange(self.n[vi])\n        self.row_boxes = {}\n        per_cls = np.zeros((2, FM_A), np.int64)             # boxes (rows x boxes) per class: [training part, holdout]\n        boxed_k = [set() for _ in self.vols]\n        n_box = n_skip = 0\n        for key, lst in (boxes.get("boxes") or {}).items():\n            try:\n                row = int(key)\n            except ValueError:\n                bad.append("boxes: row key %r" % key); continue\n            if not 0 <= row < N:\n                bad.append("boxes: row %d outside 0..%d" % (row, N - 1)); continue\n            vi = int(row_vol[row])\n            if vi < 0:                                     # a row of an excluded volume (or of no block): never read\n                n_skip += 1\n                continue\n            if isinstance(lst, list) and not lst:          # an empty list = a row without a box\n                continue\n            arr = np.asarray(lst, np.float64)\n            if arr.ndim != 2 or arr.shape[1] != 6:\n                bad.append("boxes: row %d entry shape %s (want [n, 6] = class, x0, y0, x1, y1, a0)" % (row, arr.shape)); continue\n            c_ = arr[:, 0]\n            ok = (c_ == np.round(c_)) & (c_ >= 0) & (c_ < FM_A)\n            ok &= (arr[:, 1] >= 0) & (arr[:, 1] < arr[:, 3]) & (arr[:, 3] <= FM_SIZE) & (arr[:, 2] >= 0) & (arr[:, 2] < arr[:, 4]) & (arr[:, 4] <= FM_SIZE)\n            ok &= arr[:, 5] >= (arr[:, 3] - arr[:, 1]) * (arr[:, 4] - arr[:, 2]) * 0.99 - 1.0   # a0 >= the clipped area (rounding slack)\n            if not ok.all():\n                bad.append("boxes: row %d has malformed boxes %s" % (row, arr[~ok][:3].tolist())); continue\n            if len(arr):\n                self.row_boxes[row] = arr\n                boxed_k[vi].add(int(row_k[row]))\n                n_box += len(arr)\n                np.add.at(per_cls[int(self.hold[vi])], c_.astype(np.int64), 1)\n            if len(bad) > 20:\n                break\n        if bad:\n            raise SystemExit("--fm-aux %s refused: %s" % (self.dir, " | ".join(bad[:20])))\n        self.boxed_slices = [np.array(sorted(s_), np.int64) for s_ in boxed_k]\n        self.free_slices = [np.array(sorted(set(range(int(self.n[vi]))) - s_), np.int64) for vi, s_ in enumerate(boxed_k)]\n        self.train_boxed = [vi for vi in range(len(self.vols)) if not self.hold[vi] and len(self.boxed_slices[vi])]\n        self.train_neg = [vi for vi in range(len(self.vols)) if not self.hold[vi] and not len(self.boxed_slices[vi])]\n        self.holdout = [vi for vi in range(len(self.vols)) if self.hold[vi]]\n        pid_hold = collections.defaultdict(set)\n        for vi in range(len(self.vols)):\n            pid_hold[self.pid[vi]].add(bool(self.hold[vi]))\n        split_p = sorted(p_ for p_, s_ in pid_hold.items() if len(s_) > 1)\n        if split_p:\n            bad.append("patients on both sides of aux_holdout: %s" % split_p[:5])\n        if len(self.vols) < min_volumes:\n            bad.append("%d usable volumes < %d (--fm-aux-min-volumes; spec B5: >= 1,000)" % (len(self.vols), min_volumes))\n        if not self.holdout:\n            bad.append("no aux_holdout volume (the B5 diagnostic needs them)")\n        if not self.train_boxed or not self.train_neg:\n            bad.append("training volumes with boxes %d, without %d (both are drawn)" % (len(self.train_boxed), len(self.train_neg)))\n        if (per_cls[0] == 0).any():\n            bad.append("classes without a box in the training part: %s" % [FM_CLASSES[c] for c in np.flatnonzero(per_cls[0] == 0)])\n        if bad:\n            raise SystemExit("--fm-aux %s refused: %s" % (self.dir, " | ".join(bad[:20])))\n        self.per_cls, self.n_box, self.n_skip = per_cls, n_box, n_skip\n        # the holdout rule as information (the index\'s flag is authoritative): agreement with sha1(patient_id) % 10 == 0, two hash readings\n        h_full = np.array([int(hashlib.sha1(p_.encode()).hexdigest(), 16) % 10 == 0 for p_ in self.pid])\n        h_8 = np.array([int(hashlib.sha1(p_.encode()).hexdigest()[:8], 16) % 10 == 0 for p_ in self.pid])\n        self.hold_agree = (int((h_full == self.hold).sum()), int((h_8 == self.hold).sum()))\n\n    def summary(self):\n        return ("%s: %d usable volumes (%d excluded, skipped), %d patients; training %d with boxes + %d without, holdout %d (%d patients); "\n                "%d rows, %d boxes (rows x boxes; %d on excluded rows skipped); per class training / holdout %s; aux_holdout vs "\n                "sha1(patient_id) %% 10 == 0: %d / %d agree (full hex), %d / %d ([:8]); md5 %s; manifest keys %s" % (\n                    self.dir, len(self.vols), len(self.excluded), len(set(self.pid)), len(self.train_boxed), len(self.train_neg),\n                    len(self.holdout), len({self.pid[v] for v in self.holdout}), self.n_rows, self.n_box, self.n_skip,\n                    {FM_CLASSES[c]: (int(self.per_cls[0, c]), int(self.per_cls[1, c])) for c in range(FM_A)},\n                    self.hold_agree[0], len(self.vols), self.hold_agree[1], len(self.vols), self.md5, sorted(self.manifest)[:12]))\n\n    def window_rows(self, vi, view, k):\n        """the 3 rows of the window centred on slice k of (volume vi, view): k-1, k, k+1 clamped to the volume (phys step 1, window_rows)"""\n        s_, n_ = self.start[(vi, view)], int(self.n[vi])\n        return [s_ + min(n_ - 1, max(0, k + d_)) for d_ in (-1, 0, 1)]\n\n    def boxes_of(self, rows):\n        """the boxes [n, 6] of a window\'s rows (each distinct row once: a clamped border row is not counted twice)"""\n        got = [self.row_boxes[r] for r in sorted(set(rows)) if r in self.row_boxes]\n        return np.concatenate(got) if got else np.zeros((0, 6))\n\n\ndef fm_window_targets(boxes, crop, oy, ox, dy, dx, shift, flip):\n    """v9.8: the targets of one auxiliary window.  boxes [n, 6] = (class, x0, y0, x1, y1, a0) of its 3 rows in the 384 view frame.  The window\n    is cropped at (oy, ox) to crop x crop, translated by (dy, dx) with zero fill (out[y, x] = in[y + dy, x + dx], v9.5\'s shift_batch; shift =\n    its pad width N, 0 = no translation) and, with flip, mirrored left-right; every box goes through the same maps in the same order and is\n    clipped to the field after each (what leaves the field is gone; the entering border is empty).  Returns dense [A, G, G] float32 = per\n    class the covered fraction of every 32 x 32 cell by the union of the class\'s boxes (per pixel, fractional at box edges; G = crop / 32),\n    presence [A] float32 = 1 where some box of the class keeps >= 25% of its original area a0 inside the window, and the transformed\n    boxes [m, 6] (class, x0, y0, x1, y1, area kept) of the boxes that remain."""\n    G = crop // FM_CELL\n    dense = np.zeros((FM_A, G, G), np.float32)\n    pres = np.zeros(FM_A, np.float32)\n    if len(boxes) == 0:\n        return dense, pres, np.zeros((0, 6))\n    b = np.asarray(boxes, np.float64)\n    x0, x1 = np.clip(b[:, 1] - ox, 0, crop), np.clip(b[:, 3] - ox, 0, crop)\n    y0, y1 = np.clip(b[:, 2] - oy, 0, crop), np.clip(b[:, 4] - oy, 0, crop)\n    if shift:\n        x0, x1 = np.clip(x0 - dx, 0, crop), np.clip(x1 - dx, 0, crop)\n        y0, y1 = np.clip(y0 - dy, 0, crop), np.clip(y1 - dy, 0, crop)\n    if flip:\n        x0, x1 = crop - x1, crop - x0\n    area = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)\n    masks = {}\n    kept = []\n    for i in np.flatnonzero(area > 0):\n        c = int(b[i, 0])\n        m = masks.get(c)\n        if m is None:\n            m = masks[c] = np.zeros((crop, crop), np.float32)\n        c0, c1 = int(math.floor(x0[i])), int(math.ceil(x1[i]))\n        r0, r1 = int(math.floor(y0[i])), int(math.ceil(y1[i]))\n        cc, rr = np.arange(c0, c1), np.arange(r0, r1)\n        wx = np.minimum(x1[i], cc + 1) - np.maximum(x0[i], cc)\n        wy = np.minimum(y1[i], rr + 1) - np.maximum(y0[i], rr)\n        np.maximum(m[r0:r1, c0:c1], np.outer(wy, wx).astype(np.float32), out=m[r0:r1, c0:c1])\n        if area[i] >= FM_PRESENCE_MIN * b[i, 5]:\n            pres[c] = 1.0\n        kept.append((c, x0[i], y0[i], x1[i], y1[i], area[i]))\n    for c, m in masks.items():\n        dense[c] = m.reshape(G, FM_CELL, G, FM_CELL).mean((1, 3))\n    return dense, pres, np.array(kept, np.float64).reshape(-1, 6)\n\n\ndef fm_window_image(px, rows, crop, oy, ox, dy, dx, shift, flip):\n    """v9.8: the uint8 [3, crop, crop] image of one auxiliary window (channels = slices k-1, k, k+1): crop at (oy, ox), translate (zero fill,\n    v9.5\'s convention), mirror left-right -- the maps fm_window_targets applies to the boxes, in the same order."""\n    img = np.stack([np.asarray(px[r]) for r in rows])[:, oy:oy + crop, ox:ox + crop]\n    if shift:\n        img = np.pad(img, ((0, 0), (shift, shift), (shift, shift)))[:, shift + dy:shift + dy + crop, shift + dx:shift + dx + crop]\n    if flip:\n        img = img[:, :, ::-1]\n    return np.ascontiguousarray(img)\n\n\nclass FmAuxWindows(torch.utils.data.Dataset):\n    """v9.8: one auxiliary window per index; the index IS the window\'s spec (volume, view, slice, oy, ox, dy, dx, flip, gamma, contrast)\n    drawn in the main process by FmAuxSampler (or the fixed holdout list), so the workers draw nothing.  Returns (uint8 [3, crop, crop],\n    dense [A, G, G], presence [A], gamma, contrast); gamma / contrast are applied on the GPU (fm_photometric)."""\n\n    def __init__(self, store, crop, shift):\n        self.st, self.crop, self.shift = store, int(crop), int(shift)\n        self.px = None\n\n    def __len__(self):\n        return 1\n\n    def __getitem__(self, spec):\n        if self.px is None:\n            self.px = np.load(os.path.join(self.st.dir, "px.u8.npy"), mmap_mode="r")\n        vi, view, k, oy, ox, dy, dx, fl, gam, con = spec\n        rows = self.st.window_rows(vi, view, k)\n        img = fm_window_image(self.px, rows, self.crop, oy, ox, dy, dx, self.shift, fl)\n        dense, pres, _ = fm_window_targets(self.st.boxes_of(rows), self.crop, oy, ox, dy, dx, self.shift, fl)\n        return torch.from_numpy(img), torch.from_numpy(dense), torch.from_numpy(pres), float(gam), float(con)\n\n\ndef fm_collate(batch):\n    return (torch.stack([b_[0] for b_ in batch]), torch.stack([b_[1] for b_ in batch]), torch.stack([b_[2] for b_ in batch]),\n            torch.tensor([b_[3] for b_ in batch], dtype=torch.float32), torch.tensor([b_[4] for b_ in batch], dtype=torch.float32))\n\n\ndef fm_epoch_seeds(seed, fold, epoch):\n    """B3: the auxiliary streams of one (seed, fold, epoch): sampling 9800 + 17*seed + fold + 1000*epoch, augmentation the same + 2**32 (two\n    generators with one seed would emit the same numbers); fresh every epoch, so a resume at an epoch boundary repeats the batches."""\n    s_ = 9800 + 17 * int(seed) + int(fold) + 1000 * int(epoch)\n    return s_, s_ + (1 << 32)\n\n\ndef fm_fork_seed(seed, fold, epoch, step):\n    """B3: the global-stream seed inside the auxiliary step\'s fork_rng (drop path of the auxiliary forward), f(seed, fold, epoch, step)"""\n    return int(hashlib.sha1(("v9.8 fm-aux %d %d %d %d" % (seed, fold, epoch, step)).encode()).hexdigest()[:15], 16)\n\n\nclass FmAuxSampler:\n    """v9.8: one epoch of auxiliary batches (batch_sampler of FmAuxWindows): `steps` lists of M window specs, drawn in the main process\n    from two private generators.  Per step: FM_VOLS_PER_STEP slots; per slot (sampling stream) boxed = rand < 0.75, a volume uniform among\n    the training volumes with boxes (boxed) or without (otherwise), then M / 4 centre slices -- boxed: the first half uniform over the\n    volume\'s slices with a box (any class), the second half over its slices without one (all slices when it has none) -- and a view\n    uniform in {0, 2, 3} per window; then (augmentation stream) per window the crop offset oy, ox uniform on 0..384-crop, the translation\n    dy, dx uniform on -N..N (only when N = --shift-px > 0), flip = rand < --fm-aux-flip, gamma U(0.8, 1.25), contrast U(0.9, 1.1)\n    (augment()\'s ranges).  `digest` = sha1 over every spec yielded (the epoch\'s sequence; test 7)."""\n\n    def __init__(self, store, steps, M, crop, shift, flip_p, seeds):\n        self.st, self.steps, self.M, self.crop, self.shift, self.flip_p = store, int(steps), int(M), int(crop), int(shift), float(flip_p)\n        self.gs = torch.Generator().manual_seed(int(seeds[0]))\n        self.ga = torch.Generator().manual_seed(int(seeds[1]))\n        self.h = hashlib.sha1()\n        self.yielded = 0\n        assert self.M % (2 * FM_VOLS_PER_STEP) == 0\n\n    def __len__(self):\n        return self.steps\n\n    def digest(self):\n        return self.h.hexdigest()[:16]\n\n    def draw(self):\n        st, gs, ga, M = self.st, self.gs, self.ga, self.M\n        per = M // FM_VOLS_PER_STEP\n        base = []\n        for _ in range(FM_VOLS_PER_STEP):\n            boxed = float(torch.rand(1, generator=gs)) < FM_BOXED_FRAC\n            pool = st.train_boxed if boxed else st.train_neg\n            vi = pool[int(torch.randint(0, len(pool), (1,), generator=gs))]\n            if boxed:\n                pos, neg = st.boxed_slices[vi], st.free_slices[vi]\n                if not len(neg):\n                    neg = np.arange(int(st.n[vi]))\n                ks = np.concatenate([pos[torch.randint(0, len(pos), (per // 2,), generator=gs).numpy()],\n                                     neg[torch.randint(0, len(neg), (per - per // 2,), generator=gs).numpy()]])\n            else:\n                ks = torch.randint(0, int(st.n[vi]), (per,), generator=gs).numpy()\n            vw = torch.randint(0, len(FM_VIEWS), (per,), generator=gs).tolist()\n            base += [(int(vi), FM_VIEWS[v_], int(k_)) for v_, k_ in zip(vw, ks.tolist())]\n        lim = FM_SIZE - self.crop + 1\n        oy = torch.randint(0, lim, (M,), generator=ga).tolist()\n        ox = torch.randint(0, lim, (M,), generator=ga).tolist()\n        if self.shift > 0:\n            dy = torch.randint(-self.shift, self.shift + 1, (M,), generator=ga).tolist()\n            dx = torch.randint(-self.shift, self.shift + 1, (M,), generator=ga).tolist()\n        else:\n            dy = dx = [0] * M\n        fl = (torch.rand(M, generator=ga) < self.flip_p).tolist()\n        gam = (torch.rand(M, generator=ga) * 0.45 + 0.8).tolist()\n        con = (torch.rand(M, generator=ga) * 0.2 + 0.9).tolist()\n        return [b_ + (oy[j], ox[j], dy[j], dx[j], bool(fl[j]), gam[j], con[j]) for j, b_ in enumerate(base)]\n\n    def __iter__(self):\n        for _ in range(self.steps):\n            specs = self.draw()\n            self.h.update(repr(specs).encode())\n            self.yielded += 1\n            yield specs\n\n\ndef fm_photometric(img, gam, con):\n    """v9.8: augment()\'s gamma / contrast arithmetic, per window: img uint8 [M, 3, S, S] on the device, gam / con [M]"""\n    x = (img.float() / 255.).clamp_(0, 1).pow(gam.view(-1, 1, 1, 1))\n    x = ((x - 0.5) * con.view(-1, 1, 1, 1) + 0.5).clamp_(0, 1) * 255.\n    return x.round_().to(torch.uint8)\n\n\ndef fm_auc(y, s):\n    """ROC AUC (Mann-Whitney with average ranks for ties); nan when y has no positive or no negative"""\n    y = np.asarray(y, bool)\n    s = np.asarray(s, np.float64)\n    npos, nneg = int(y.sum()), int((~y).sum())\n    if npos == 0 or nneg == 0:\n        return float("nan")\n    _, inv, cnt = np.unique(s, return_inverse=True, return_counts=True)\n    avg = np.cumsum(cnt) - (cnt - 1) / 2.0\n    r = avg[inv]\n    return float((r[y].sum() - npos * (npos + 1) / 2.0) / (npos * nneg))\n\n\ndef fm_clip(model, max_norm):\n    """v9.8: ONE clip of the merged gradient over every parameter.  The norm of v9.7\'s parameters is computed exactly as v9.7\'s\n    clip_grad_norm_(model.parameters()) computes it (same function, same tensors, same order), the auxiliary head\'s separately, and they\n    are combined as sqrt(n_main^2 + n_aux^2) -- the L2 norm over all parameters -- or n_main itself when the head\'s gradient is 0 (lambda = 0:\n    then the clip is v9.7\'s bit for bit); every gradient (the head\'s included) is then scaled as clip_grad_norm_ scales it."""\n    from torch.nn.utils.clip_grad import _get_total_norm, _clip_grads_with_norm_\n    params = list(model.parameters())\n    main_g = [p.grad for n, p in model.named_parameters() if not n.startswith("aux_") and p.grad is not None]\n    aux_g = [p.grad for n, p in model.named_parameters() if n.startswith("aux_") and p.grad is not None]\n    n_main = _get_total_norm(main_g, 2.0, False, None)\n    if aux_g:\n        n_aux = _get_total_norm(aux_g, 2.0, False, None).to(n_main.device)\n        total = torch.where(n_aux == 0, n_main, torch.sqrt(n_main.square() + n_aux.square()))\n    else:\n        total = n_main\n    _clip_grads_with_norm_(params, max_norm, total, None)\n    return total\n\n\ndef dynamo_compiles():\n    """v9.8 (test 9): (frames dynamo compiled so far = counters[\'frames\'][\'ok\'], recompiles = distinct compile ids \'frame/k\' with k >= 1\n    in the compilation metrics, i.e. compiles of a frame that had been compiled before); (-1, -1) when the counters are unavailable"""\n    try:\n        import torch._dynamo.utils as du\n        ok = int(du.counters.get("frames", {}).get("ok", 0))\n        ids = {str(x_.compile_id) for x_ in du.get_compilation_metrics()}\n        rec = sum(1 for s_ in ids if re.match(r"^\\[?\\d+/(\\d+)", s_) and int(re.match(r"^\\[?\\d+/(\\d+)", s_).group(1)) > 0)\n        return ok, rec\n    except Exception:\n        return -1, -1\n\n\ndef fm_make_loader(store, batches, crop, shift, workers, seed_w):\n    """v9.8: a DataLoader over FmAuxWindows whose batch_sampler is `batches` (FmAuxSampler, or a fixed list of spec lists for the holdout\n    read-out with shift 0); its own workers and its own generator (never the global stream, never the main loader\'s generator)"""\n    return torch.utils.data.DataLoader(FmAuxWindows(store, crop, shift), batch_sampler=batches, num_workers=workers, collate_fn=fm_collate,\n                                       pin_memory=True, prefetch_factor=2 if workers > 0 else None, persistent_workers=False,\n                                       generator=torch.Generator().manual_seed(int(seed_w)))\n\n\ndef fm_aux_step(model, batch, img_main, enc_params, lam, crop, seed, k, ep, b_i, steps, acc, pw, log_every, dev):\n    """v9.8: one auxiliary step, called after the main loss.backward() of step b_i of epoch ep (fold k): the batch (uint8 windows, dense and\n    presence targets, gamma, contrast) -> fm_photometric -> model.enc (train mode, bf16 autocast, the compiled module under --compile) ->\n    aux_dense -> L_aux = BCE(dense, pos_weight 8) + BCE(presence) -> (lam * L_aux).backward() into the same .grad.  Everything inside\n    torch.random.fork_rng(current CUDA device) seeded with fm_fork_seed(seed, k, ep, b_i): the global CPU / CUDA states afterwards are the\n    ones before.  Asserts (test 10) the auxiliary and main batches\' spatial size == (crop, crop) and the encoder map / targets\' grid == G =\n    crop / 32.  Every log_every steps prints |lam grad L_aux| and |grad L_main| over enc_params (the main part = the .grad before this\n    backward).  acc [3] += (L_dense, L_presence, 1).  Returns (L_dense, L_presence)."""\n    xb, yd, yp, gam, con = batch\n    g_step = ep * steps + b_i\n    log_now = g_step % log_every == 0\n    fs = fm_fork_seed(seed, k, ep, b_i)\n    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):\n        torch.random.default_generator.manual_seed(fs)\n        torch.cuda.manual_seed(fs)\n        xg = fm_photometric(xb.to(dev, non_blocking=True), gam.to(dev, non_blocking=True), con.to(dev, non_blocking=True))\n        if tuple(xg.shape[-2:]) != tuple(img_main.shape[-2:]) or tuple(xg.shape[-2:]) != (crop, crop):   # test 10\n            raise AssertionError("v9.8: auxiliary batch %s vs main batch %s vs --crop %d: spatial sizes differ" % (\n                tuple(xg.shape), tuple(img_main.shape), crop))\n        with torch.autocast("cuda", dtype=torch.bfloat16):\n            xs = xg.float().div_(255.)\n            xs = ((xs - model.mean) / model.std).contiguous(memory_format=torch.channels_last)\n            fmap = model.enc(xs)\n        G = crop // FM_CELL\n        if fmap.ndim != 4 or tuple(fmap.shape[-2:]) != (G, G) or tuple(yd.shape[-2:]) != (G, G):   # test 10\n            raise AssertionError("v9.8: encoder map %s / dense target %s vs G = crop / 32 = %d" % (tuple(fmap.shape), tuple(yd.shape), G))\n        dz = model.aux_dense(fmap)                                                   # [M, A, G, G]\n        l_d = F.binary_cross_entropy_with_logits(dz, yd.to(dev, non_blocking=True), pos_weight=pw)\n        pz = torch.logsumexp(dz.flatten(2), -1) - math.log(G * G)                   # presence: log-mean-exp over the G^2 positions\n        l_p = F.binary_cross_entropy_with_logits(pz, yp.to(dev, non_blocking=True))\n        snap = [torch.zeros_like(p_) if p_.grad is None else p_.grad.detach().clone() for p_ in enc_params] if log_now else None\n        (lam * (l_d + l_p)).backward()\n    acc += torch.stack([l_d.detach().float(), l_p.detach().float(), torch.ones((), device=dev)])\n    if log_now:\n        with torch.no_grad():\n            n_m = torch.sqrt(sum(s_.float().square().sum() for s_ in snap))\n            n_x = torch.sqrt(sum(((p_.grad.float() if p_.grad is not None else torch.zeros_like(s_).float()) - s_.float()).square().sum()\n                                 for p_, s_ in zip(enc_params, snap)))\n        print("  fm-aux fold %d epoch %d step %d (fold step %d): |lambda grad L_aux| / |grad L_main| over the encoder %.4g (%.4g / %.4g); "\n              "L_dense %.4f L_presence %.4f" % (k, ep, b_i, g_step, float(n_x) / max(float(n_m), 1e-30), float(n_x), float(n_m),\n                                                 float(l_d.detach()), float(l_p.detach())), flush=True)\n    return l_d.detach(), l_p.detach()\n\n\n@torch.no_grad()\ndef fm_holdout_eval(model, store, hold_specs, crop, M, workers, dev):\n    """v9.8: per-class presence AUC on the store\'s aux_holdout volumes (the B5 diagnostic; never used to choose anything): every slice of\n    every holdout volume in the views {0, 2, 3}, centre crop (center()\'s offset), no translation / flip / photometric jitter, batches of M\n    windows (the last padded with zeros) through model.enc in eval mode; window level = sigmoid(presence logit) vs the window\'s presence\n    target (the 25% rule), volume level = the max over the volume\'s windows vs any window present.  Inside fork_rng; the module\'s train /\n    eval mode is restored."""\n    was = model.training\n    o_ = (FM_SIZE - crop) // 2\n    specs = [(vi, w_, k_, o_, o_, 0, 0, False, 1.0, 1.0) for vi, w_, k_ in hold_specs]\n    G = crop // FM_CELL\n    P, Y = [], []\n    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):\n        model.eval()\n        for xb, yd, yp, _, _ in fm_make_loader(store, [specs[i:i + M] for i in range(0, len(specs), M)], crop, 0, workers, 0):\n            n_ = len(xb)\n            if n_ < M:\n                xb = torch.cat([xb, xb.new_zeros((M - n_,) + tuple(xb.shape[1:]))])\n            with torch.autocast("cuda", dtype=torch.bfloat16):\n                xs = xb.to(dev, non_blocking=True).float().div_(255.)\n                xs = ((xs - model.mean) / model.std).contiguous(memory_format=torch.channels_last)\n                fmap = model.enc(xs)\n            dz = model.aux_dense(fmap)[:n_]\n            P.append(torch.sigmoid(torch.logsumexp(dz.flatten(2), -1) - math.log(G * G)).float().cpu())\n            Y.append(yp)\n    model.train(was)\n    P, Y = torch.cat(P).numpy(), torch.cat(Y).numpy() > 0.5\n    win = [fm_auc(Y[:, c], P[:, c]) for c in range(FM_A)]\n    vid = np.array([s_[0] for s_ in specs])\n    vols = sorted(set(vid.tolist()))\n    Pv = np.stack([P[vid == v_].max(0) for v_ in vols])\n    Yv = np.stack([Y[vid == v_].any(0) for v_ in vols])\n    vol = [fm_auc(Yv[:, c], Pv[:, c]) for c in range(FM_A)]\n    return {"window_auc": dict(zip(FM_CLASSES, win)), "volume_auc": dict(zip(FM_CLASSES, vol)), "windows": len(specs), "volumes": len(vols),\n            "window_pos": dict(zip(FM_CLASSES, Y.sum(0).astype(int).tolist())), "volume_pos": dict(zip(FM_CLASSES, Yv.sum(0).astype(int).tolist()))}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--px", required=True)\n    ap.add_argument("--px-extra", default=None, help="add-on pixel cache (px_cache --views v2add) read next to --px; its windows join every study")\n    ap.add_argument("--px-override", default=None, help="override cache (sidefix): studies listed there are read from it instead of --px")\n    ap.add_argument("--targets", required=True)\n    ap.add_argument("--extra", default=None)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--arms", default="E_prior")\n    ap.add_argument("--backbone", default="resnet34")\n    ap.add_argument("--init", default="F:/rsnaknee/pretrained/resnet34-imagenet.pth")\n    ap.add_argument("--seeds", type=int, default=1)\n    ap.add_argument("--folds", type=int, default=5)\n    ap.add_argument("--epochs", type=int, default=10)\n    ap.add_argument("--batch", type=int, default=8)\n    ap.add_argument("--train-windows", type=int, default=48)\n    ap.add_argument("--crop", type=int, default=256)\n    ap.add_argument("--slices", type=int, default=3, help="adjacent slices per window = input channels (odd; CNN backbones only beyond 3)")\n    ap.add_argument("--slice-jitter", type=float, default=0.0, help="probability that a training window uses neighbours two slices apart")\n    ap.add_argument("--ema", type=float, default=0.0, help="EMA decay of the trainable weights (0 = off); EMA weights are evaluated and saved")\n    ap.add_argument("--bn-recal-batches", type=int, default=0, help="v7: BatchNorm backbones only (CoAtNet, ResNet): after --ema copies the "\n                    "averaged weights in, recompute the BatchNorm running statistics for them over this many training batches "\n                    "(centre crop, no augmentation) before evaluating and saving; also applied by --reeval-run.  0 = off "\n                    "(backbones without BatchNorm are never touched)")\n    ap.add_argument("--include-review-holdout", action="store_true", help="add the 120 review-holdout studies (full report targets, "\n                    "fold -3) to the training pool; they were held out for the report-reader evaluation, not for image training")\n    ap.add_argument("--grad-ckpt", action="store_true", help="gradient checkpointing in the timm encoder (keeps 208 windows/study on 32 GB cards)")\n    ap.add_argument("--syn-decomp", action="store_true", help="v9.2 (10-10): synovitis logit = specific part + beta * stop-grad(effusion logit); "\n                    "beta a scalar parameter (init 0); writes oof_synspec_<arm>_s<seed>.npy next to the OOF.  Off = v9.1 bit for bit")\n    ap.add_argument("--grad-ckpt-blocks", type=int, default=0, help="v9.1 (10-09): recompute only the first N encoder blocks in backward (ViT "\n                    "encoders with .blocks): memory between --grad-ckpt (all blocks) and none, about N/12 of its extra compute; same maths")\n    ap.add_argument("--phys-mm", type=float, default=0.0, help="window spacing in mm of anatomy: the covering stride and the K-slice "\n                    "neighbour step are multiplied by round(phys_mm / slice spacing), so a 0.4 mm 3-D stack is sampled like a 3.5 mm "\n                    "2-D series (0 = per-slice, the old behaviour)")\n    ap.add_argument("--rare-boost", action="store_true", help="oversample studies with rare report states (MCL>=2/acute, fracture, high PF OA, Baker>=moderate, contusion); needs --state-targets")\n    ap.add_argument("--lr", type=float, default=3e-4)\n    ap.add_argument("--freeze-blocks", type=int, default=16, help="ofl_l backbone: number of leading transformer blocks kept frozen (of 24)")\n    ap.add_argument("--llrd", type=float, default=1.0, help="ViT backbones (ofl_l, timm vit_*): layer-wise learning-rate decay per block from the top (1 = off); block i of n gets lr * llrd**(n - 1 - i), embeddings lr * llrd**n")\n    ap.add_argument("--head-lr", type=float, default=1e-3)\n    ap.add_argument("--wd", type=float, default=0.02)\n    ap.add_argument("--workers", type=int, default=6)\n    ap.add_argument("--eval-workers", type=int, default=3)\n    ap.add_argument("--forward-only", action="store_true", help="one batch forward/backward, NO optimizer step, then exit")\n    ap.add_argument("--compile", action="store_true", help="torch.compile the encoder in place (dynamic shapes); same maths, 1.4x (ConvNeXt-small) / 1.9x (DINOv3-B) on a PRO 6000; weight names unchanged")\n    ap.add_argument("--resume", action="store_true", help="continue every fold from its end-of-epoch checkpoint in <out>/ckpt when present")\n    ap.add_argument("--ckpt-every", type=int, default=1, help="write the fold checkpoint every N epochs (0 = never); removed when the fold ends")\n    ap.add_argument("--stop-after-epoch", type=int, default=-1, help="testing: exit right after the checkpoint of this epoch")\n    ap.add_argument("--eval-stride", type=int, default=3, help="evaluation windows: every k-th slice as centre (3 = the validated default; 1 = every slice)")\n    ap.add_argument("--train-sampling", default="uniform", choices=["uniform", "cover"],\n                    help="uniform = --train-windows random windows of the study; cover = every series/view at --cover-stride with a random phase (capped at --train-windows)")\n    ap.add_argument("--cover-stride", type=int, default=3)\n    ap.add_argument("--state-targets", default=None, help="state_targets_<tag>.csv from make_state_targets.py (report-state construct)")\n    ap.add_argument("--state-spec", default=None, help="state_spec_<tag>.json (categories, prior vectors, binaries)")\n    ap.add_argument("--state-weight", type=float, default=0.5, help="weight of the state/binary losses next to the arm loss")\n    ap.add_argument("--spatial-attn", action="store_true", help="per-class spatial attention inside each window (encoder keeps its "\n                    "grid; class queries start at zero = plain average), then per-class window attention over class-specific tokens")\n    ap.add_argument("--pair-rank", type=float, default=0.0, help="weight of the matched-pair ranking loss on the 8 pair families "\n                    "(positive vs hard negative of a class with the same co-finding signature); needs --state-targets; 0 = off")\n    ap.add_argument("--pair-frac", type=float, default=0.33, help="share of training batches that open with a matched pair (--pair-rank)")\n    ap.add_argument("--pair-extra", action="store_true", help="v9: add the within-class grade families Effusion, Baker\'s, Contusion, "\n                    "Fracture (higher grade over lower grade of the same structure) to --pair-rank")\n    ap.add_argument("--oa-area-rank", action="store_true", help="v9.3 R4d: per OA class a family that ranks high_grade_large above "\n                    "high_grade_small on the extent head\'s log1p(area/100) (needs --extent and a spec with high_grade_small = opusv11c), "\n                    "and the extent-head soft target " + str(OA_SMALL_TARGET) + " for high_grade_small cells")\n    ap.add_argument("--oa-sign-decoy", action="store_true", help="v9.3 R4d: per OA class a family that ranks high grade with NO bone sign "\n                    "written above low_grade_chondral WITH a sign written (osteophytes / subchondral marrow / bone-on-bone); signature 0")\n    ap.add_argument("--acl-acuity-decoy", action="store_true", help="v9.3 R4d: a family that ranks chronic ACL tears (and age-unstated tears "\n                    "with no contusion and effusion <= small) above ACL normal / signal / low-grade WITH a contusion or effusion >= moderate; "\n                    "needs --lig-age")\n    ap.add_argument("--mcl-boundary", action="store_true", help="v9.3 R4d: a family that ranks MCL high_grade_partial_tear / complete_tear "\n                    "(not chronic / graft / no_tear per --lig-age) above partial_tear_or_grade2; needs --lig-age")\n    ap.add_argument("--lig-age", default=None, help="v9.3: csv with columns study, class, age (chronic / acute / age_unstated / graft / "\n                    "no_tear) from the E8 ligament re-read (e8/lig_age_A.csv) or the adjudicated r4d_inputs/ligament_age.csv; a cell absent "\n                    "from the csv = age_unstated")\n    ap.add_argument("--save-area", action="store_true", help="v9.3 read-out (output only, no effect on training): write oof_area_<arm>_s<seed>.npy "\n                    "([pool, 12] extent-head area proxy, OA columns meaningful; needs --extent) next to the OOF, also under --reeval-run")\n    ap.add_argument("--save-state-oof", action="store_true", help="v9.7 read-out (output only, no effect on training or on the class "\n                    "outputs): the state head\'s per-class category probabilities (softmax over the class\'s valid slots, as "\n                    "Net.state_evidence uses them) and binary probabilities for every held-out and gold study a fold model predicts -> "\n                    "oof_state_<arm>_s<seed>.npy + .json (layout) and preds_state_<arm>_s<seed>.json (gold), also under --reeval-run; "\n                    "needs --state-targets.  Off (default) = v9.6 bit for bit")\n    ap.add_argument("--seed-base", type=int, default=0, help="first seed (seeds seed_base .. seed_base+seeds-1); a replicate of a recipe")\n    ap.add_argument("--extent", action="store_true", help="M4: OA defect extent per series and plane (area proxy) into the OA logits")\n    ap.add_argument("--extent-weight", type=float, default=0.2, help="weight of the extent supervision (>= 1 cm^2 stated vs normal/low grade)")\n    ap.add_argument("--extent-classes", default="Medial OA,Lateral OA,PF OA", help="v9.4: the OA classes whose logit receives the extent "\n                    "head\'s area term (M4 ablation: \'Medial OA,Lateral OA\' leaves PF OA\'s logit without it; the area, its supervision and "\n                    "the area families are unchanged).  Default = all three = v9.3")\n    ap.add_argument("--mirror", type=float, default=0.0, help="v9.4 (spec_trainer_v94_mirror_20261010.md): probability that a training "\n                    "batch is mirrored into the virtual contralateral knee -- coronal / axial windows flipped left-right, sagittal windows "\n                    "reversed along the slice axis (channels reversed, rel_pos negated), coronal view ids 2 <-> 3 -- with the model outputs "\n                    "re-indexed Medial <-> Lateral (menisci, OA) before every loss and MCL not trained on that batch.  One draw per batch; "\n                    "training forward only; 0 = off (v9.3 bit for bit)")\n    ap.add_argument("--shift-px", type=int, default=0, help="v9.5 (spec_trainer_v95_shift_20261010.md): translation jitter of N pixels -- once "\n                    "per training batch, after the crop and before the --aug v2 affine, every window of the batch is translated by the "\n                    "same integer (dy, dx), each uniform on -N..N, with zero fill (pad N zeros, crop back at N+dy, N+dx); drawn from its "\n                    "own stream (9500 + 17*seed + fold), so every other draw is unchanged.  Gives --crop 384 the +-16 px translation that "\n                    "--crop 352 takes from its random crop offset.  Training forward only; 0 = off (v9.4 bit for bit)")\n    ap.add_argument("--warm-start", default=None, help="v9.6 (spec_r5_repair_training_20261010.md): repair training -- fold k starts from "\n                    "DIR/models/<arm>_s<seed>_f<k>.pt (the whole Net state dict, strict=True, loaded before the optimizer and the EMA are "\n                    "built, so the EMA starts from it); the optimizer and the LR schedule start fresh over --epochs.  DIR\'s run_config must "\n                    "equal this run in --folds, --holdout-fold, backbone, crop, slices, spatial attention, extent (+ classes) and syn-decomp, "\n                    "its state spec must have this run\'s categories and binaries (prior / base values may differ: the state head\'s spec "\n                    "buffers are re-set from this run\'s spec after the load), and pool_uids.json the pool (refused at start-up otherwise).  "\n                    "Off (default) = v9.5 bit for bit")\n    ap.add_argument("--warm-lr-scale", type=float, default=1.0, help="v9.6: with --warm-start, every parameter group\'s learning rate "\n                    "(encoder, layer-wise groups, head) times s (suggested 0.3); the schedule\'s shape is unchanged (warm-up + cosine over "\n                    "--epochs).  Needs --warm-start")\n    ap.add_argument("--holdout-fold", type=int, default=None,\n                    help="clean ruler fold for pseudo-label rounds: this hash fold is excluded from EVERY training set (no model is trained "\n                         "for it); its predictions are the mean of the other folds\' models, so a teacher->student chain evaluated on it "\n                         "with local_ruler --folds-only is leak-free")\n    ap.add_argument("--reeval-run", default=None, help="predict-only: load models/<arm>_s<seed>_f<k>.pt from this run directory and rewrite the out-of-fold "\n                    "predictions into --out with the current --eval-stride (no training)")\n    ap.add_argument("--aug", default="crop", choices=["crop", "v2"], help="v2 = crop + per-study rotation/scale + gamma/contrast jitter (no flips: laterality)")\n    ap.add_argument("--aug-grid-fp32", action="store_true", help="v9.6: --aug v2 only -- the rotation / scale sampling grid (affine_grid) "\n                    "and its bilinear resampling (grid_sample) in float32 with autocast off for those two calls (inside the bf16 autocast "\n                    "region affine_grid\'s matmul runs in bf16: ~0.7 px grid error); same random draws, same order.  Off (default) = v9.5 bit for bit")\n    ap.add_argument("--draw-seed", action="store_true", help="v9.6 (SCAN_A D1): every TRAINING draw seeds its window choice (cover phase, "\n                    "slice jitter, window subset) with [seed, epoch, index, nonce], nonce = the draw\'s position in the epoch, so a study drawn "\n                    "k times in one epoch (the rare-boost / pair samplers draw with replacement) gets k window sets; the samplers\' drawn "\n                    "index sequence is unchanged; evaluation and BN recalibration keep v9.5\'s seeding.  Off (default) = v9.5 bit for bit")\n    ap.add_argument("--rare-boost-names", action="store_true", help="v9.6 (SCAN_A D4): the --rare-boost terms by state NAME: MCL +2 only for "\n                    "partial_tear_or_grade2 / high_grade_partial_tear / complete_tear (or the acute binary) -- v9.5\'s index rule (>= 2) also "\n                    "boosted chronic_or_healed_change; the Fracture / PF OA / Baker\'s / Contusion terms by the names of v9.5\'s indices "\n                    "(asserted at start-up to select the same studies).  Needs --rare-boost.  Off (default) = v9.5 bit for bit")\n    ap.add_argument("--aug-res", type=float, default=0.0, help="v2 only: probability that a training study is down-sampled in-plane by "\n                    "1.25-2x and up-sampled back (simulates 0.33-0.5 mm scanners); 0 = off, random stream unchanged")\n    ap.add_argument("--only-folds", default=None, help="diagnostic: comma list of folds to train (others skipped); outputs are written with a partial_ prefix and must not be used as OOF")\n    ap.add_argument("--check-targets", action="store_true", help="pre-launch check on CPU: build the pool and every arm\'s targets and the "\n                    "state targets with the training code path, print a summary, and exit (no data loading, no model, no GPU)")\n    ap.add_argument("--fm-aux", default=None, help="v9.8 (spec_trainer_v98_fmaux_20261011.md B): the fm384_v1 store DIR of the fastMRI+ "\n                    "lesion-box auxiliary task -- every step M windows through the same encoder into the aux_* head (dense coverage BCE "\n                    "pos_weight 8 + presence BCE), lambda * L_aux back-propagated into the same gradients before the one clip / AdamW / "\n                    "EMA step; aux windows cropped to --crop (+ --shift-px), G = crop / 32.  ConvNeXt with --spatial-attn only.  Off "\n                    "(default) = v9.7 bit for bit")\n    ap.add_argument("--fm-aux-weight", type=float, default=0.3, help="v9.8: lambda of the auxiliary loss (pre-registered 0.3; 0 = the "\n                    "isolation test: the auxiliary step runs, the main path is v9.7\'s bit for bit)")\n    ap.add_argument("--fm-aux-windows", type=int, default=96, help="v9.8: M auxiliary windows per step (4 volumes x M/4; a multiple of 8; "\n                    ">= 0.25 x --batch x --train-windows)")\n    ap.add_argument("--fm-aux-flip", type=float, default=0.5, help="v9.8: probability that an auxiliary window (and its boxes) is mirrored "\n                    "left-right")\n    ap.add_argument("--fm-aux-workers", type=int, default=2, help="v9.8: loader workers of the auxiliary batches (their own, not the "\n                    "main loader\'s)")\n    ap.add_argument("--fm-aux-log-every", type=int, default=100, help="v9.8: every N steps log |lambda grad L_aux| / |grad L_main| over the "\n                    "encoder parameters (before the clip)")\n    ap.add_argument("--fm-aux-min-volumes", type=int, default=1000, help="v9.8: the store must hold at least this many usable volumes "\n                    "(spec B5: >= 1,000; lower only for tests on a synthetic store)")\n    ap.add_argument("--i-have-user-authorisation-to-finetune", action="store_true")\n    a = ap.parse_args()\n    if not a.forward_only and not a.check_targets and not a.i_have_user_authorisation_to_finetune:\n        raise SystemExit("refusing to update an encoder: the user has only authorised light heads on cached features. "\n                         "Use --forward-only, or pass the authorisation flag after the user says so.")\n    if not 0.0 <= a.mirror <= 1.0:                       # v9.4\n        raise SystemExit("--mirror is a probability: 0 <= p <= 1")\n    if not 0 <= a.shift_px < a.crop:                     # v9.5\n        raise SystemExit("--shift-px is a translation in pixels: 0 <= N < --crop")\n    if a.aug_grid_fp32 and a.aug != "v2":                # v9.6: the flag changes the v2 affine only; never silently ignored\n        raise SystemExit("--aug-grid-fp32 changes the --aug v2 affine grid: needs --aug v2")\n    if a.rare_boost_names and not a.rare_boost:          # v9.6: the flag changes --rare-boost\'s weights only\n        raise SystemExit("--rare-boost-names changes the --rare-boost weights: needs --rare-boost")\n    if a.warm_start is None and a.warm_lr_scale != 1.0:  # v9.6: the scale applies only to a warm start; never silently ignored\n        raise SystemExit("--warm-lr-scale needs --warm-start")\n    if a.warm_start is not None:\n        if not (math.isfinite(a.warm_lr_scale) and a.warm_lr_scale > 0):\n            raise SystemExit("--warm-lr-scale must be a positive number")\n        if a.reeval_run:\n            raise SystemExit("--warm-start trains from a source run\'s fold models; --reeval-run only predicts: not both")\n        if a.backbone == OFL_BACKBONE:\n            raise SystemExit("--warm-start: ofl_l fold models are deploy state dicts without the frozen public weights, so the whole-Net "\n                             "strict load is not possible for them (not implemented)")\n    ext_classes = tuple(s.strip() for s in a.extent_classes.split(","))      # v9.4: the default tuple is the three OA classes = v9.3\n    for l in ext_classes:\n        if l not in OA_L:\n            raise SystemExit("--extent-classes: %r is not one of %s" % (l, OA_L))\n    if set(ext_classes) != set(OA_L) and not a.extent:\n        raise SystemExit("--extent-classes changes the extent head\'s logit term: needs --extent")\n    # ---- v9.8: the --fm-aux flags (a companion flag without --fm-aux is refused, never silently ignored)\n    fm_flags = ("fm_aux_weight", "fm_aux_windows", "fm_aux_flip", "fm_aux_workers", "fm_aux_log_every", "fm_aux_min_volumes")\n    if a.fm_aux is None:\n        set_fm = ["--" + f_.replace("_", "-") for f_ in fm_flags if getattr(a, f_) != ap.get_default(f_)]\n        if set_fm:\n            raise SystemExit("%s change the auxiliary task: need --fm-aux" % ", ".join(set_fm))\n    else:\n        fm_bad = []\n        if a.reeval_run:\n            fm_bad.append("--reeval-run only predicts (the auxiliary task trains)")\n        if "convnext" not in a.backbone:\n            fm_bad.append("backbone %s: the auxiliary path is implemented for ConvNeXt (no BatchNorm: an auxiliary batch would move "\n                          "BatchNorm statistics -- CoAtNet / ResNet refused; ViT / OFL encoders return no [N, d, G, G] map)" % a.backbone)\n        if not a.spatial_attn:\n            fm_bad.append("needs --spatial-attn (the encoder must return its stride-32 map)")\n        if a.crop % FM_CELL or not FM_CELL <= a.crop <= FM_SIZE:\n            fm_bad.append("--crop %d: the auxiliary windows (rendered at %d) are cropped to it, so it must be a multiple of %d in %d..%d" % (\n                a.crop, FM_SIZE, FM_CELL, FM_CELL, FM_SIZE))\n        if not (math.isfinite(a.fm_aux_weight) and a.fm_aux_weight >= 0):\n            fm_bad.append("--fm-aux-weight must be a finite number >= 0")\n        if a.fm_aux_windows <= 0 or a.fm_aux_windows % (2 * FM_VOLS_PER_STEP):\n            fm_bad.append("--fm-aux-windows %d: a positive multiple of %d (%d volumes x an even number of windows: half on slices with a "\n                          "box)" % (a.fm_aux_windows, 2 * FM_VOLS_PER_STEP, FM_VOLS_PER_STEP))\n        elif a.fm_aux_windows < 0.25 * a.batch * a.train_windows:\n            fm_bad.append("--fm-aux-windows %d < 0.25 x %d main windows per step (spec B5)" % (a.fm_aux_windows, a.batch * a.train_windows))\n        if not 0.0 <= a.fm_aux_flip <= 1.0:\n            fm_bad.append("--fm-aux-flip is a probability")\n        if a.fm_aux_workers < 0 or a.fm_aux_log_every < 1 or a.fm_aux_min_volumes < 1:\n            fm_bad.append("--fm-aux-workers >= 0, --fm-aux-log-every >= 1, --fm-aux-min-volumes >= 1")\n        if fm_bad:\n            raise SystemExit("--fm-aux refused: " + " | ".join(fm_bad))\n    # ---- v9.8 E2: an --out that already holds a finished run is refused before ANY file is written (v9.7 rewrote its run_config.json\n    # and pool_uids.json first and only then skipped or refused the done arm); --check-targets writes nothing and is not affected\n    if not a.check_targets and os.path.isdir(a.out):\n        done_m = sorted(f_ for f_ in os.listdir(a.out) if FM_DONE_RE.match(f_) and os.path.isfile(os.path.join(a.out, f_)))\n        if done_m:\n            raise SystemExit("--out %s already holds a finished run (done marker %s): refused before writing any file; use a fresh --out" % (\n                a.out, ", ".join(done_m)))\n    fm_store = None\n    if a.fm_aux is not None:                             # v9.8: the store, checked whole before anything is written or built\n        fm_store = FmStore(a.fm_aux, a.fm_aux_min_volumes)\n        print("v9.8 fm-aux: store %s" % fm_store.summary(), flush=True)\n        print("v9.8 fm-aux: lambda %g, M %d windows per step (%d volumes x %d; boxed volumes %.0f%%, half their windows on slices with a box), "\n              "views %s uniform, crop %d (offset uniform on 0..%d) -> G %d, shift %s, flip p %.2f, gamma / contrast as augment(); dense BCE "\n              "pos_weight %g + presence (log-mean-exp over %d positions, >= %.0f%% of a box\'s area) BCE; head aux_* (LayerNorm %s -> linear "\n              "-> %d); %d loader workers; |lambda grad aux| / |grad main| every %d steps; holdout presence AUC every epoch" % (\n                  a.fm_aux_weight, a.fm_aux_windows, FM_VOLS_PER_STEP, a.fm_aux_windows // FM_VOLS_PER_STEP, 100 * FM_BOXED_FRAC,\n                  list(FM_VIEWS), a.crop, FM_SIZE - a.crop, a.crop // FM_CELL, ("-%d..%d px (--shift-px)" % (a.shift_px, a.shift_px)) if a.shift_px\n                  else "none (--shift-px 0)", a.fm_aux_flip, FM_POS_WEIGHT, (a.crop // FM_CELL) ** 2, 100 * FM_PRESENCE_MIN, "d", FM_A,\n                  a.fm_aux_workers, a.fm_aux_log_every), flush=True)\n    os.makedirs(a.out, exist_ok=True)\n    dev = torch.device("cpu") if a.check_targets else torch.device("cuda")\n    torch.backends.cudnn.benchmark = True\n    index = json.load(open(os.path.join(a.px, "index.json"), encoding="utf-8"))\n    done = {u for u, v in index["done"].items() if v[1] > 0}\n    rows = list(csv.DictReader(open(a.targets, encoding="utf-8")))\n    by_uid = {r["StudyInstanceUID"]: r for r in rows}\n    if a.extra:\n        for r in csv.DictReader(open(a.extra, encoding="utf-8")):\n            if r["StudyInstanceUID"] in by_uid:\n                by_uid[r["StudyInstanceUID"]].update({k: v for k, v in r.items() if k != "StudyInstanceUID"})\n    gold = [r["StudyInstanceUID"] for r in rows if r["is_gold"] == "1" and r["StudyInstanceUID"] in done]\n    # a non-gold study whose report labels were blanked (e.g. the report describes another body part) carries no target:\n    # it leaves the pool, exactly like a gold row (logged below; bad_report_studies.json lists the blanked studies)\n    blanked = {r["StudyInstanceUID"] for r in rows if r["is_gold"] == "0" and any(r.get(L + "|prior", "") == "" for L in LABELS)}\n    pool = sorted(r["StudyInstanceUID"] for r in rows if r["is_gold"] == "0" and r["StudyInstanceUID"] in done\n                  and r["StudyInstanceUID"] not in blanked\n                  and ((r["review_holdout"] == "0" and int(r["fold"]) != -3) or a.include_review_holdout))\n    assert not (set(pool) & set(gold))\n    for u in gold:\n        assert all(by_uid[u][L + "|strict"] == "" and by_uid[u][L + "|prior"] == "" for L in LABELS)\n    K = a.folds\n    pool_fold = np.array([int(hashlib.sha1(u.encode()).hexdigest()[:8], 16) % K for u in pool])\n    print("pool %d gold %d folds %s | non-gold studies with blanked report labels left out: %d" % (\n        len(pool), len(gold), np.bincount(pool_fold, minlength=K).tolist(), len(blanked & done)), flush=True)\n\n    def col(uids, suffix, default):\n        out = np.full((len(uids), C), default, np.float32)\n        for i, u in enumerate(uids):\n            for j, L in enumerate(LABELS):\n                v = by_uid[u].get(L + "|" + suffix, "")\n                if v != "":\n                    out[i, j] = float(v)\n        return torch.from_numpy(out).to(dev)\n\n    specs = {}\n    for item in a.arms.split(","):\n        if "=" in item:\n            name, spec = item.split("=", 1)\n            specs[name] = tuple(spec.split(":"))\n\n    def arm_tensors(arm):\n        if arm == "S_strict":\n            return "bce", col(pool, "strict", 0.), col(pool, "strict_w", 0.)\n        if arm == "P_presence":\n            return "bce", col(pool, "presence", 0.), col(pool, "presence_w", 0.)\n        # v8: an explicit definition (name=kind:column[:weight_column]) wins over the built-in arm of the same name, so\n        # "E_lrprior=lr:lr_prior:w" keeps the arm name (and every downstream file name) while adding cell weights\n        spec = specs.get(arm) or {"E_prior": ("bce", "prior"), "E_lrprior": ("lr", "lr_prior")}[arm]\n        kind, column = spec[0], spec[1]\n        if len(spec) > 2 and spec[2]:                       # name=kind:column:weight_column  (0 = cell carries no information)\n            w = col(pool, spec[2], float("nan") if kind == "lr" else 0.)\n            t = col(pool, column, float("nan") if kind == "lr" else 0.)\n            if kind == "lr":                                # v8: every pool cell needs a target and a weight (log lr)\n                assert not torch.isnan(t).any() and not torch.isnan(w).any(), "lr arm with weights: a cell lacks lr or weight"\n                assert bool((w >= 0).all()), "negative cell weight"\n            return kind, t, w\n        t = col(pool, column, float("nan"))\n        assert not torch.isnan(t).any()\n        return kind, t, torch.ones_like(t)\n\n    bias0 = torch.logit(col(pool, "prior", float("nan")).mean(0).clamp(1e-3, 1 - 1e-3))\n\n    state_spec = None\n    if a.state_targets:\n        state_spec = json.load(open(a.state_spec, encoding="utf-8"))\n        srows = {r["StudyInstanceUID"]: r for r in csv.DictReader(open(a.state_targets, encoding="utf-8"))}\n        st_idx = torch.full((len(pool), C), -1, dtype=torch.long)\n        st_w = torch.zeros((len(pool), C))\n        bin_names = [(b, l) for b, labs in state_spec["binaries"].items() for l in labs]\n        bin_t = torch.full((len(pool), len(bin_names)), float("nan"))\n        for i, u in enumerate(pool):\n            r = srows.get(u)\n            if r is None:\n                continue\n            for c, l in enumerate(LABELS):\n                v = r.get(l + "|state", "")\n                if v != "":\n                    st_idx[i, c] = int(v); st_w[i, c] = float(r.get(l + "|sw") or 1.0)\n            for j, (b, l) in enumerate(bin_names):\n                v = r.get(l + "|" + b, "")\n                if v != "":\n                    bin_t[i, j] = float(v)\n        st_idx, st_w, bin_t = st_idx.to(dev), st_w.to(dev), bin_t.to(dev)\n        print("state targets: known cells per class %s | binaries known %s" % (\n            (st_idx >= 0).sum(0).tolist(), (~torch.isnan(bin_t)).sum(0).tolist()), flush=True)\n\n    # ---- v9.7: --save-state-oof -- the read-out\'s column layout from this run\'s state spec (refused here, before any data is loaded)\n    st_layout = None\n    if a.save_state_oof:\n        if state_spec is None:\n            raise SystemExit("--save-state-oof reads out the state head: needs --state-targets / --state-spec")\n        st_layout = state_layout(state_spec)\n        print("v9.7 save-state-oof: the state head\'s read-out of every held-out and gold study a fold model predicts -> %soof_state_<arm>_s<seed>"\n              ".npy [%d pool rows x %d columns: %s; binaries %s] + .json (layout), %spreds_state_<arm>_s<seed>.json (gold, per fold model); "\n              "per class softmax over the class\'s valid slots (Net.state_evidence\'s p), per binary sigmoid" % (\n                  "partial_" if a.only_folds else "", len(pool), st_layout["width"],\n                  ", ".join("%s %d" % (c_["class"], c_["stop"] - c_["start"]) for c_ in st_layout["columns"] if "categories" in c_),\n                  ["%s/%s" % (c_["binary"], c_["class"]) for c_ in st_layout["columns"] if "binary" in c_], "partial_" if a.only_folds else ""),\n              flush=True)\n\n    # ---- v9.6: warm start -- the source run must have this run\'s Net structure on the same hash folds and the same pool, and every fold\n    # model to be loaded must exist: refused here, before any data is loaded or any model is built (each file is loaded at its fold\'s start)\n    warm_rec = None\n    if a.warm_start is not None:\n        wdir = os.path.abspath(a.warm_start)\n        for f_ in ("run_config.json", "pool_uids.json"):\n            if not os.path.isfile(os.path.join(wdir, f_)):\n                raise SystemExit("--warm-start %s refused: no %s there" % (wdir, f_))\n        src_cfg = json.load(open(os.path.join(wdir, "run_config.json"), encoding="utf-8"))\n        warm_bad = []\n        for f_ in WARM_FIELDS:\n            mine, theirs = getattr(a, f_), src_cfg.get(f_, ap.get_default(f_))\n            if f_ == "extent_classes":                   # a set of class names (the order does not change the mask)\n                mine, theirs = sorted(s.strip() for s in mine.split(",")), sorted(s.strip() for s in str(theirs).split(","))\n            if mine != theirs:\n                warm_bad.append("%s: this run %r, source %r" % (f_, mine, theirs))\n        # the state head exists iff --state-targets is given (then built from --state-spec): same presence; and the source\'s spec must have\n        # this run\'s CATEGORIES (per class the same states in the same order: the state head\'s slots) and BINARIES (the same names over the\n        # same classes in the same order: the binary heads\' rows).  Its prior / base values may differ (R5: a re-calibrated spec) -- the three\n        # spec buffers are re-set from this run\'s spec after every load (warm_load)\n        sp_mine = a.state_spec if a.state_targets else None\n        sp_src = src_cfg.get("state_spec") if src_cfg.get("state_targets") else None\n        spec_note = "no state head"\n        if (sp_mine is None) != (sp_src is None):\n            warm_bad.append("state head: this run %s, source %s" % ("on (" + str(sp_mine) + ")" if sp_mine else "off", "on (" + str(sp_src) + ")" if sp_src else "off"))\n        elif sp_mine is not None:\n            try:\n                spec_src = json.load(open(sp_src, encoding="utf-8"))\n            except (OSError, ValueError):\n                spec_src = None\n            if spec_src is None:\n                warm_bad.append("state spec: the source\'s %s cannot be read, so its categories / binaries cannot be compared" % sp_src)\n            else:\n                cat_diff = [l for l in LABELS if spec_src.get("categories", {}).get(l) != state_spec["categories"].get(l)]\n                if cat_diff:\n                    warm_bad.append("state spec categories differ for %s (this run %s, source %s)" % (cat_diff, sp_mine, sp_src))\n                if list(spec_src.get("binaries", {}).items()) != list(state_spec["binaries"].items()):\n                    warm_bad.append("state spec binaries differ (this run %s %s, source %s %s)" % (\n                        sp_mine, list(state_spec["binaries"].items()), sp_src, list(spec_src.get("binaries", {}).items())))\n                vals_same = all(spec_src.get(f_, {}).get(l) == state_spec[f_].get(l) for f_ in ("prior", "base") for l in LABELS)\n                spec_note = "%s: categories and binaries == the source\'s (%s); prior / base values %s" % (\n                    os.path.basename(sp_mine), os.path.basename(sp_src), "identical" if vals_same else\n                    "DIFFER -- state_prior / state_valid / state_base_logit are re-set from this run\'s spec after every load")\n        src_pool = json.load(open(os.path.join(wdir, "pool_uids.json"), encoding="utf-8"))\n        if src_pool != pool:\n            warm_bad.append("pool: this run %d studies, source %d (%d of this run\'s not in the source, %d of the source\'s not in this run)" % (\n                len(pool), len(src_pool), len(set(pool) - set(src_pool)), len(set(src_pool) - set(pool))))\n        only_w = None if not a.only_folds else {int(x) for x in a.only_folds.split(",")}\n        warm_need = [os.path.join(wdir, "models", "%s_s%d_f%d.pt" % (arm_, seed_, k_)) for seed_ in range(a.seed_base, a.seed_base + a.seeds)\n                for arm_ in [x.split("=", 1)[0] for x in a.arms.split(",") if x] for k_ in range(K)\n                if (only_w is None or k_ in only_w) and k_ != a.holdout_fold]\n        warm_miss = [p_ for p_ in warm_need if not os.path.isfile(p_)]\n        if warm_miss:\n            warm_bad.append("fold models missing: %s" % ", ".join(warm_miss))\n        # this run must never write over its source: not the same directory, and no file already in --out (top level, models/, ckpt/)\n        # may be the same file as one of the source\'s (a merged run directory holds hard links to the per-fold originals; this run opens\n        # run_config.json, pool_uids.json, preds, OOF, fold models and checkpoints for writing)\n        if os.path.realpath(a.out) == os.path.realpath(wdir):\n            warm_bad.append("--out is the warm-start directory itself (its run_config, pool and fold models would be overwritten)")\n        src_ids, out_ids = file_ids(wdir), file_ids(a.out)\n        warm_same = sorted(out_ids[i_] for i_ in set(out_ids) & set(src_ids))\n        if warm_same:\n            warm_bad.append("file(s) in --out are the same files as the source\'s (hard links?): %s" % ", ".join(warm_same))\n        # this run\'s spec must be well formed (one prior per category): the state head\'s valid slots come from the prior lists\n        if state_spec is not None:\n            bad_len = [l for l in LABELS if len(state_spec["prior"][l]) != len(state_spec["categories"][l])]\n            if bad_len:\n                warm_bad.append("state spec %s: the prior list length differs from the category count for %s" % (sp_mine, bad_len))\n        if warm_bad:\n            raise SystemExit("--warm-start %s refused: %s" % (wdir, " | ".join(warm_bad)))\n        # the record a warm checkpoint carries (+ the source fold model\'s md5): a resume needs the same source, scale and state spec file\n        warm_rec = {"dir": wdir, "lr_scale": a.warm_lr_scale, "state_spec_md5": file_md5(a.state_spec) if a.state_targets else None}\n        print("v9.6 warm-start: source %s matches this run (K %d, holdout fold %s, %s, crop %d, slices %d, spatial %s, extent %s, syn-decomp %s, "\n              "pool %d); state spec %s; %d fold model(s) to load; LR x %g; optimizer, LR schedule (over --epochs %d) and EMA start fresh" % (\n                  wdir, K, a.holdout_fold, a.backbone, a.crop, a.slices, a.spatial_attn, a.extent, a.syn_decomp, len(pool), spec_note,\n                  len(warm_need), a.warm_lr_scale, a.epochs), flush=True)\n\n    # ---- v9.8 E1: --reeval-run predicts with the source\'s fold models; their state head was trained under the source\'s state spec, so\n    # this run\'s --state-spec must be that very file content (md5), else the state columns would be read out under the wrong labels.\n    # Refused at start-up (before anything is written), with both paths and md5s.  Also moved here from reeval() (E2: every start-up check\n    # before run_config.json): the source\'s pool must be this run\'s pool.\n    if a.reeval_run and not a.check_targets:\n        rdir = os.path.abspath(a.reeval_run)\n        if not os.path.isfile(os.path.join(rdir, "pool_uids.json")):\n            raise SystemExit("--reeval-run %s refused: no pool_uids.json there" % rdir)\n        if json.load(open(os.path.join(rdir, "pool_uids.json"), encoding="utf-8")) != pool:\n            raise SystemExit("pool of --reeval-run differs from the current pool")\n        r_cfg_p = os.path.join(rdir, "run_config.json")\n        r_cfg = json.load(open(r_cfg_p, encoding="utf-8")) if os.path.isfile(r_cfg_p) else None\n        sp_this = os.path.abspath(a.state_spec) if a.state_targets else None\n        if r_cfg is None:\n            if sp_this is not None:\n                raise SystemExit("--reeval-run %s refused: no run_config.json there, so the state spec its fold models were trained with "\n                                 "(this run: %s, md5 %s) cannot be checked" % (rdir, sp_this, file_md5(sp_this)))\n        else:\n            sp_src = r_cfg.get("state_spec") if r_cfg.get("state_targets") else None\n            if (sp_this is None) != (sp_src is None):\n                raise SystemExit("--reeval-run %s refused: state head %s in the source run (state_spec %s) but %s in this run (--state-spec "\n                                 "%s)" % (rdir, "on" if sp_src else "off", sp_src, "on" if sp_this else "off", sp_this))\n            if sp_this is not None:\n                if not os.path.isfile(sp_src):\n                    raise SystemExit("--reeval-run %s refused: the source run\'s state spec %s (run_config.json) does not exist, so its md5 "\n                                     "cannot be compared with this run\'s %s (md5 %s)" % (rdir, sp_src, sp_this, file_md5(sp_this)))\n                m_src, m_this = file_md5(sp_src), file_md5(sp_this)\n                if m_src != m_this:\n                    raise SystemExit("--reeval-run %s refused: the source run\'s state spec %s (md5 %s) is not this run\'s --state-spec %s "\n                                     "(md5 %s): the state columns would carry the wrong labels" % (rdir, sp_src, m_src, sp_this, m_this))\n\n    # ---- v9.6: --rare-boost-names (SCAN_A D4): the name rules under this run\'s spec, checked here (also under --check-targets); the\n    # masks of both rule sets over the pool are kept for make_loader, which cross-checks them against v9.5\'s rare_weights per training set\n    rare_masks_idx = rare_masks_name = pool_row = None\n    if a.rare_boost_names:\n        if state_spec is None:\n            raise SystemExit("--rare-boost-names needs --state-targets (the rare-boost terms read the report states)")\n        cats_rb = state_spec["categories"]\n        miss_rb = [(l, n_) for l, _, ns in RARE_NAME_TERMS for n_ in ns if n_ not in cats_rb[l]]\n        if miss_rb:\n            raise SystemExit("--rare-boost-names: state names missing from the spec %s: %s" % (a.state_spec, miss_rb))\n        bn_rb = {(b, l): j for j, (b, l) in enumerate(bin_names)}\n        acute_rb = (bin_t[:, bn_rb[("acute", "MCL")]].cpu().numpy() == 1.0) if ("acute", "MCL") in bn_rb else np.zeros(len(pool), bool)\n        st_rb = st_idx.cpu().numpy()\n        rare_masks_idx = rare_term_masks(st_rb, cats_rb, acute_rb, False)\n        rare_masks_name = rare_term_masks(st_rb, cats_rb, acute_rb, True)\n        diff_rb = [l for l in ("Fracture", "PF OA", "Baker\'s", "Contusion") if not np.array_equal(rare_masks_idx[l], rare_masks_name[l])]\n        if diff_rb:\n            raise SystemExit("--rare-boost-names: under %s the name rules of %s select other studies than v9.5\'s index rules (%s)" % (\n                a.state_spec, diff_rb, {l: (int(rare_masks_idx[l].sum()), int(rare_masks_name[l].sum())) for l in diff_rb}))\n        pool_row = {u: i for i, u in enumerate(pool)}\n        w_rb_idx, w_rb_name = rare_weights_of(rare_masks_idx), rare_weights_of(rare_masks_name)\n        mcl_c = LABELS.index("MCL")\n        lost_rb = rare_masks_idx["MCL"] & ~rare_masks_name["MCL"]\n        print("v9.6 rare-boost-names: terms by state name %s; Fracture / PF OA / Baker\'s / Contusion select the same pool studies as v9.5\'s "\n              "index rules (%s); MCL boosted by v9.5 but not by name: %d pool studies (their MCL states %s; gained: %d); rare weight "\n              "changes for %d of %d pool studies (mean weight %.4f -> %.4f)" % (\n                  {l: list(ns) for l, _, ns in RARE_NAME_TERMS},\n                  {l: int(rare_masks_name[l].sum()) for l in ("Fracture", "PF OA", "Baker\'s", "Contusion")}, int(lost_rb.sum()),\n                  dict(collections.Counter(cats_rb["MCL"][int(s_)] for s_ in st_rb[lost_rb, mcl_c])),\n                  int((rare_masks_name["MCL"] & ~rare_masks_idx["MCL"]).sum()), int((w_rb_idx != w_rb_name).sum()), len(pool),\n                  float(w_rb_idx.mean()), float(w_rb_name.mean())), flush=True)\n    if a.draw_seed:                                      # v9.6: start-up line of the per-draw window streams\n        print("v9.6 draw-seed: every training draw seeds its windows with [seed, epoch, index, nonce = position in the epoch] (repeat "\n              "draws of a study in one epoch get their own windows); the samplers\' index sequence is unchanged; evaluation and BN "\n              "recalibration keep the (seed, epoch, index) seeding", flush=True)\n\n    # ---- v9.4: anatomical mirror -- P over the classes (and the binaries) and the MCL binaries, fixed for the run; the spec check\n    # (identical state category lists of the exchanged classes) refuses here, before any data is loaded\n    mir_perm = mir_bin_perm = mir_mcl_bins = None\n    if a.mirror > 0:\n        perm_, bperm_ = mirror_perm(state_spec, bin_names if state_spec is not None else None)\n        mir_perm = torch.tensor(perm_, device=dev)\n        mir_bin_perm = None if bperm_ is None else torch.tensor(bperm_, device=dev)\n        mir_mcl_bins = [j for j, (b, l) in enumerate(bin_names) if l == "MCL"] if state_spec is not None else []\n        print("v9.4 mirror: probability %.2f per training batch; output permutation %s%s; MCL not trained on mirrored batches (main / "\n              "state / pair weights 0%s)" % (a.mirror, perm_, "" if bperm_ is None else ", binaries " + str(bperm_),\n                                            "" if not mir_mcl_bins else ", binaries %s unknown" % mir_mcl_bins), flush=True)\n    if a.shift_px > 0:                                   # v9.5: start-up line of the translation jitter\n        print("v9.5 shift: one integer translation (dy, dx), each uniform on -%d..%d, per training batch (every study / window / plane / "\n              "channel of the batch), zero fill, after the crop%s; own stream 9500 + 17*seed + fold; never in evaluation" % (\n                  a.shift_px, a.shift_px, " and before the v2 affine" if a.aug == "v2" else " (--aug crop: no affine)"), flush=True)\n    if a.aug_grid_fp32:                                  # v9.6: start-up line of the float32 affine grid\n        print("v9.6 aug-grid-fp32: the --aug v2 rotation / scale grid (affine_grid) and its bilinear resampling (grid_sample) run in float32 "\n              "with autocast off for those two calls; every random draw and every other operation unchanged", flush=True)\n\n    # ---- P1 pair families (v9.3: built by build_pair_families, indexed by family; v9.2 = the same v9 list indexed by class)\n    pair_fams, pair_info, fam_class, fam_q = [], {}, [], []\n    new_fam = a.oa_area_rank or a.oa_sign_decoy or a.acl_acuity_decoy or a.mcl_boundary\n    if (a.pair_extra or new_fam) and not a.pair_rank > 0:\n        raise SystemExit("--pair-extra / --oa-area-rank / --oa-sign-decoy / --acl-acuity-decoy / --mcl-boundary need --pair-rank > 0")\n    if a.oa_area_rank and not a.extent:\n        raise SystemExit("--oa-area-rank ranks the extent head\'s area: needs --extent")\n    lig_age = load_lig_age(a.lig_age, pool) if a.lig_age else None\n    if (a.acl_acuity_decoy or a.mcl_boundary) and lig_age is None:\n        raise SystemExit("--acl-acuity-decoy / --mcl-boundary need --lig-age <csv> (columns study, class, age)")\n    if a.pair_rank > 0:\n        if state_spec is None:\n            raise SystemExit("--pair-rank needs --state-targets")\n        pair_fams, role, sig, pw, fam_class = build_pair_families(\n            state_spec, st_idx.cpu().numpy(), bin_t.cpu().numpy(), bin_names, a.pair_extra, a.oa_area_rank, a.oa_sign_decoy,\n            a.acl_acuity_decoy, a.mcl_boundary, lig_age)\n        fam_q = [fm["quantity"] for fm in pair_fams]\n        pair_info = {"role": role, "sig": sig}\n        role_t, sig_t, pw_t = torch.from_numpy(role).to(dev), torch.from_numpy(sig).to(dev), torch.from_numpy(pw).to(dev)\n        print("pair families: " + "; ".join("%s pos %d neg %d" % (fm["name"], int((role[:, f] == 1).sum()), int((role[:, f] == -1).sum()))\n                                              for f, fm in enumerate(pair_fams)), flush=True)\n        if new_fam:\n            print("v9.3 families over the pool: " + " | ".join(family_summary(pair_fams, role, sig)), flush=True)\n\n    ext_y = None\n    if a.extent:\n        if state_spec is None:\n            raise SystemExit("--extent needs --state-targets")\n        ext_y = torch.full((len(pool), C), float("nan"))\n        for l in ("Medial OA", "Lateral OA", "PF OA"):\n            c, cats_ = LABELS.index(l), state_spec["categories"][l]\n            pos_i, neg_i = cats_.index("high_grade_large"), {cats_.index("normal"), cats_.index("low_grade_chondral")}\n            si = st_idx[:, c].cpu()\n            ext_y[si == pos_i, c] = 1.0\n            for k_ in neg_i:\n                ext_y[si == k_, c] = 0.0\n            if a.oa_area_rank:                   # v9.3: the small high-grade lesion as the head\'s "< 1 cm^2 leaning" example\n                ext_y[si == cats_.index("high_grade_small"), c] = OA_SMALL_TARGET\n        ext_y = ext_y.to(dev)\n        print("extent supervision: positives (>= 1 cm^2 stated) %s, negatives %s" % (\n            [int((ext_y[:, LABELS.index(l)] == 1).sum()) for l in ("Medial OA", "Lateral OA", "PF OA")],\n            [int((ext_y[:, LABELS.index(l)] == 0).sum()) for l in ("Medial OA", "Lateral OA", "PF OA")]), flush=True)\n        if a.oa_area_rank:\n            print("extent supervision v9.3: high_grade_small cells at soft target %.1f: %s" % (OA_SMALL_TARGET,\n                  [int((ext_y[:, LABELS.index(l)] == OA_SMALL_TARGET).sum()) for l in OA_L]), flush=True)\n\n    def extent_loss(area, sel):\n        """>= 1 cm^2 stated (1) vs normal / low grade (0), weighted by the report certainty like the state loss (audit 5)."""\n        y = ext_y[sel]\n        known = ~torch.isnan(y)\n        if not bool(known.any()):\n            return area.new_zeros(())\n        w_ = st_w[sel].float()[known]\n        logit_ = 2.0 * (torch.log1p(area.float()) - math.log(101.0))\n        l_ = F.binary_cross_entropy_with_logits(logit_[known], y[known], reduction="none")\n        return (l_ * w_).sum() / w_.sum().clamp_min(1e-6)\n\n    def pair_buckets(uids):\n        """v9.3: pair_buckets_of over the training studies\' rows (v9.2\'s bucket list for the v9 family list)."""\n        idx = {u: i for i, u in enumerate(pool)}\n        pi = np.array([idx[u] for u in uids])\n        return pair_buckets_of(pair_info["role"], pair_info["sig"], pi)\n\n    def pair_loss(z, sel, area=None, cw=None):\n        """v9.3: pair_loss_of on the batch\'s rows; area = the extent head\'s [B, C] area for the \'area\' families (None otherwise).\n        v9.4: cw = the batch\'s cell weights when they differ from st_w[sel] (a mirrored batch: MCL column 0)."""\n        return pair_loss_of(z, area, role_t[sel], sig_t[sel], pw_t[sel], st_w[sel] if cw is None else cw, fam_class, fam_q)\n\n    if a.check_targets:\n        for item in a.arms.split(","):\n            kind, t, w = arm_tensors(item.split("=")[0])\n            print("check-targets arm %s (%s): %d x %d cells, finite %s, weighted cells %d, mean target per class %s" % (\n                item.split("=")[0], kind, t.shape[0], t.shape[1], bool(torch.isfinite(t).all()), int((w > 0).sum()),\n                [round(float(v), 3) for v in t.mean(0)]), flush=True)\n        print("check-targets: bias0 finite %s; holdout fold %s has %d pool studies" % (\n            bool(torch.isfinite(bias0).all()), a.holdout_fold, int((pool_fold == a.holdout_fold).sum()) if a.holdout_fold is not None else 0))\n        if a.pair_rank > 0:                       # v9.3: every family\'s sizes and buckets over the whole pool and over each fold\'s training set\n            print("check-targets families (pool): " + " | ".join(family_summary(pair_fams, pair_info["role"], pair_info["sig"])), flush=True)\n            for k in range(K):\n                tr_ = np.flatnonzero((pool_fold != k) & (pool_fold != a.holdout_fold)) if a.holdout_fold is not None else np.flatnonzero(pool_fold != k)\n                bk_ = pair_buckets_of(pair_info["role"], pair_info["sig"], tr_)\n                print("check-targets fold %d training set %d studies: %d buckets, probability sum %.4f" % (k, len(tr_), len(bk_), sum(b[2] for b in bk_)), flush=True)\n        bad = []\n        for item in a.arms.split(","):\n            kind, t, w = arm_tensors(item.split("=")[0])\n            if not bool(torch.isfinite(t).all()):\n                bad.append("non-finite targets in arm " + item)\n            if kind == "lr" and bool((t <= 0).any()):\n                bad.append("non-positive likelihood ratios in arm " + item)\n            if kind == "bce" and bool(((t < 0) | (t > 1)).any()):\n                bad.append("probability targets outside [0, 1] in arm " + item)\n        if not bool(torch.isfinite(bias0).all()):\n            bad.append("non-finite bias0 (a blank or non-numeric prior in the pool)")\n        if state_spec is not None:\n            kc = torch.tensor([len(state_spec["categories"][l]) for l in LABELS], device=st_idx.device)\n            if bool((st_idx >= kc[None]).any()):\n                bad.append("state index outside its class\'s category list")\n        if bad:\n            print("check-targets FAILED: " + "; ".join(bad), flush=True)\n            raise SystemExit(1)\n        print("check-targets OK", flush=True)\n        raise SystemExit(0)\n\n    def rare_weights(uids):\n        """1 + 2[MCL state>=2 or acute] + 2[fracture state>=1] + [PF OA high] + [Baker>=moderate] + [traumatic edema/contusion], cap 4."""\n        idx = {u: i for i, u in enumerate(pool)}\n        L = {l: c for c, l in enumerate(LABELS)}\n        bn = {(b, l): j for j, (b, l) in enumerate(bin_names)}\n        w = []\n        for u in uids:\n            i = idx[u]\n            st = st_idx[i].tolist()\n            v = 1.0\n            v += 2.0 * (st[L["MCL"]] >= 2 or (("acute", "MCL") in bn and float(bin_t[i, bn[("acute", "MCL")]]) == 1.0))\n            v += 2.0 * (st[L["Fracture"]] >= 1)\n            v += 1.0 * (st[L["PF OA"]] in (3, 4))\n            v += 1.0 * (st[L["Baker\'s"]] in (3, 4, 5))\n            v += 1.0 * (st[L["Contusion"]] in (4, 5))\n            w.append(min(v, 4.0))\n        return torch.tensor(w, dtype=torch.double)\n\n    def make_loader(uids, mode, seed, shuffle, nonce=False):\n        """v9.6: nonce=True (only run()\'s training loader, under --draw-seed) makes the training sampler yield (index, nonce) draws"""\n        ds = Studies(a.px, uids, a.train_windows, mode, seed, eval_stride=a.eval_stride,\n                     train_sampling=a.train_sampling, cover_stride=a.cover_stride, px_extra=a.px_extra, slices=a.slices,\n                     slice_jitter=a.slice_jitter if mode == "train" else 0.0, phys_mm=a.phys_mm, px_override=a.px_override,\n                     extent_meta=a.extent or a.mirror > 0)      # v9.4: the mirror reads the plane column (the model ignores the extra columns)\n        train = mode == "train"                     # evaluation: one study per batch, few workers, no pinned copies\n        sampler = None\n        if train and a.rare_boost:\n            if state_spec is None:\n                raise SystemExit("--rare-boost needs --state-targets")\n            w = rare_weights(uids)\n            if a.rare_boost_names:                   # v9.6 (SCAN_A D4): the name-rule weights; v9.5\'s (w) must equal the index-rule replica\n                rows_rb = np.array([pool_row[u] for u in uids])\n                w_idx_rb = rare_weights_of({l: m_[rows_rb] for l, m_ in rare_masks_idx.items()})\n                if not np.array_equal(w_idx_rb, w.numpy()):\n                    raise SystemExit("--rare-boost-names: the index-rule replica differs from rare_weights on this training set")\n                w_name_rb = torch.tensor(rare_weights_of({l: m_[rows_rb] for l, m_ in rare_masks_name.items()}), dtype=torch.double)\n                print("v9.6 rare-boost-names: %d of %d training studies change weight (MCL by name; mean %.4f -> %.4f)" % (\n                    int((w_name_rb != w).sum()), len(uids), float(w.mean()), float(w_name_rb.mean())), flush=True)\n                w = w_name_rb\n            sampler = torch.utils.data.WeightedRandomSampler(w, num_samples=len(uids), replacement=True,\n                                                             generator=torch.Generator().manual_seed(seed + 7))\n            if nonce:                                # v9.6 --draw-seed: the same sampler (weights, generator), yielding (index, nonce)\n                sampler = NonceWeightedSampler(sampler.weights, num_samples=len(uids), replacement=True, generator=sampler.generator)\n            shuffle = False\n            print("rare-boost: mean weight %.2f, boosted studies %d of %d" % (float(w.mean()), int((w > 1).sum()), len(w)), flush=True)\n        if train and a.pair_rank > 0:               # matched pairs on top of the same base distribution\n            bk = pair_buckets(uids)\n            bs = PairBatchSampler(len(uids), a.batch, None if sampler is None else sampler.weights, bk, a.pair_frac, seed + 7)\n            if nonce:                                # v9.6 --draw-seed: the same sampler (fresh generator, same seed), yielding (index, nonce)\n                bs = NoncePairBatchSampler(len(uids), a.batch, None if sampler is None else sampler.weights, bk, a.pair_frac, seed + 7)\n            print("pair-rank: %d buckets over %d families, %.0f%% of batches drawn as a matched pair (others: base draws, weighted with replacement)" % (\n                len(bk), len(pair_fams), 100 * a.pair_frac), flush=True)\n            return ds, torch.utils.data.DataLoader(ds, batch_sampler=bs, num_workers=a.workers, collate_fn=collate,\n                                                   persistent_workers=False, pin_memory=True, prefetch_factor=2,\n                                                   generator=torch.Generator().manual_seed(seed))\n        if nonce and train and sampler is None and shuffle:   # v9.6 --draw-seed, plain shuffle: the RandomSampler the DataLoader would\n            g_ns = torch.Generator().manual_seed(seed)        # build (same generator object, which also seeds the workers), yielding (index, nonce)\n            return ds, torch.utils.data.DataLoader(ds, batch_size=a.batch, sampler=NonceRandomSampler(ds, generator=g_ns),\n                                                   num_workers=a.workers, collate_fn=collate, drop_last=True, persistent_workers=False,\n                                                   pin_memory=True, prefetch_factor=2, generator=g_ns)\n        return ds, torch.utils.data.DataLoader(ds, batch_size=a.batch if train else 1, shuffle=shuffle, sampler=sampler,\n                                               num_workers=a.workers if train else min(a.workers, a.eval_workers),\n                                               collate_fn=collate, drop_last=train, persistent_workers=False,\n                                               pin_memory=train, prefetch_factor=2 if train else 1,\n                                               generator=torch.Generator().manual_seed(seed))\n\n    def augment(img, gen, shgen=None, shifts=None):\n        B, W, _, S, _ = img.shape\n        oy = int(torch.randint(0, S - a.crop + 1, (1,), generator=gen))\n        ox = int(torch.randint(0, S - a.crop + 1, (1,), generator=gen))\n        img = img[..., oy:oy + a.crop, ox:ox + a.crop]\n        if shgen is not None:                   # v9.5: one translation per batch from its own stream, zero fill, before the affine\n            dy, dx = shift_draw(a.shift_px, shgen)\n            img = shift_batch(img, dy, dx, a.shift_px)\n            if shifts is not None:              # the epoch read-out\'s record\n                shifts.append((dy, dx))\n        if a.aug != "v2":\n            return img\n        # per-study rotation (+-10 deg) and scale (0.9-1.1) through one affine grid; gamma 0.8-1.25; contrast 0.9-1.1.\n        # No flips: a horizontal flip swaps medial/lateral, a vertical one anterior/posterior.\n        Cc = img.shape[2]\n        x = img.float().reshape(B * W, Cc, a.crop, a.crop)\n        ang = (torch.rand(B, generator=gen) * 20 - 10) * math.pi / 180\n        sc = torch.rand(B, generator=gen) * 0.2 + 0.9\n        cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc\n        theta = torch.stack([torch.stack([cos, -sin, torch.zeros(B)], 1), torch.stack([sin, cos, torch.zeros(B)], 1)], 1)\n        theta = theta.repeat_interleave(W, 0).to(x.device)\n        if a.aug_grid_fp32:                     # v9.6: grid + bilinear resampling in float32, autocast off for these two calls only\n            with torch.autocast("cuda", enabled=False):   # (in the bf16 region affine_grid\'s matmul runs in bf16: ~0.7 px grid error)\n                grid = F.affine_grid(theta.float(), x.shape, align_corners=False)\n                x = F.grid_sample(x.float(), grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n        else:                                   # v9.5\'s two lines, unchanged\n            grid = F.affine_grid(theta, x.shape, align_corners=False)\n            x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n        if a.aug_res > 0:                       # per-study in-plane resolution loss (coarse scanners: 0.33-0.5 mm pixels)\n            hit = torch.rand(B, generator=gen) < a.aug_res\n            fac = torch.rand(B, generator=gen) * 0.75 + 1.25\n            if bool(hit.any()):\n                xs = x.view(B, W, Cc, a.crop, a.crop)\n                for b in torch.nonzero(hit).flatten().tolist():\n                    s_ = max(8, int(round(a.crop / float(fac[b]))))\n                    lo = F.interpolate(xs[b], size=(s_, s_), mode="area")\n                    xs[b] = F.interpolate(lo, size=(a.crop, a.crop), mode="bilinear", align_corners=False)\n                x = xs.view(B * W, Cc, a.crop, a.crop)\n        gamma = (torch.rand(B, generator=gen) * 0.45 + 0.8).repeat_interleave(W, 0).to(x.device).view(-1, 1, 1, 1)\n        contrast = (torch.rand(B, generator=gen) * 0.2 + 0.9).repeat_interleave(W, 0).to(x.device).view(-1, 1, 1, 1)\n        x = (x / 255.).clamp_(0, 1).pow(gamma)\n        x = ((x - 0.5) * contrast + 0.5).clamp_(0, 1) * 255.\n        return x.round_().to(torch.uint8).reshape(B, W, Cc, a.crop, a.crop)\n\n    def center(img):\n        S = img.shape[-1]\n        o = (S - a.crop) // 2\n        return img[..., o:o + a.crop, o:o + a.crop]\n\n    @torch.no_grad()                                  # evaluation only: no autograd graph (same outputs, less time and memory)\n    def predict(model, uids, seed, want_spec=False, want_area=False, want_state=False):\n        model.eval()\n        ds, dl = make_loader(uids, "eval", seed, False)\n        out = torch.zeros((len(uids), C), device=dev)\n        spec = torch.zeros((len(uids),), device=dev)\n        area = torch.zeros((len(uids), C), device=dev)           # v9.3 --save-area: the extent head\'s area proxy per study\n        if want_state:                                            # v9.7 --save-state-oof: the state head\'s read-out per study\n            state_slots_check(model, st_layout)\n            pst = torch.full((len(uids), st_layout["width"]), float("nan"), device=dev)\n        for img, meta, mask, idx in dl:\n            with torch.autocast("cuda", dtype=torch.bfloat16):\n                if want_state:                  # v9.7: the same forward, also returning the state / binary logits it computed (z unchanged)\n                    z, sl_, bl_ = model(center(img.to(dev, non_blocking=True)), meta.to(dev), mask.to(dev), chunk=PREDICT_CHUNK, return_aux=True)\n                    ps_ = state_probs(model, sl_, bl_)\n                else:\n                    z = model(center(img.to(dev, non_blocking=True)), meta.to(dev), mask.to(dev), chunk=PREDICT_CHUNK)\n            out[idx.to(dev)] = z.float()\n            if want_state:\n                pst[idx.to(dev)] = ps_\n            if want_spec and getattr(model, "syn_decomp", False):      # v9.2: the synovium-specific part of the last batch\n                spec[idx.to(dev)] = model._syn_spec\n            if want_area and getattr(model, "extent", False):          # v9.3: [B, C] area of the last batch (OA columns meaningful)\n                area[idx.to(dev)] = model._ext_area.float()\n        res = [out] + ([spec] if want_spec else []) + ([area] if want_area else []) + ([pst] if want_state else [])\n        return res[0] if len(res) == 1 else tuple(res)\n\n    @torch.no_grad()\n    def bn_recalibrate(model, uids, seed, n_batches):\n        """--ema copies only the trainable parameters into the shipped model, so BatchNorm running statistics still describe\n        the live weights (CoAtNet screen 10-08: holdout 0.61, predictions nearly constant).  Recompute them for the shipped\n        weights: cumulative averages over n_batches training batches, centre crop, no augmentation.  The whole model is in\n        train mode so that no zero padding enters the statistics (Net.forward pads only in eval mode or without BatchNorm);\n        the head\'s dropout comes after the encoder and does not reach them.  Returns the number of windows seen."""\n        bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]\n        if not bns or n_batches <= 0:\n            return 0\n        mom = [m.momentum for m in bns]\n        for m in bns:\n            m.reset_running_stats()\n            m.momentum = None\n        model.train()\n        ds_r, dl_r = make_loader(uids, "train", seed, True)\n        ds_r.slice_jitter = 0.0                      # audit 10-08 defect 1: no slice-jitter augmentation in the statistics\n        seen = used = 0\n        stance = torch.compiler.set_stance("force_eager") if hasattr(torch.compiler, "set_stance") else contextlib.nullcontext()\n        with stance:                                 # audit defect 2: never through a compiled encoder (guards, recompiles)\n            for i, (img, meta, mask, idx) in enumerate(dl_r):\n                if i >= n_batches:\n                    break\n                with torch.autocast("cuda", dtype=torch.bfloat16):\n                    model(center(img.to(dev, non_blocking=True)), meta.to(dev), mask.to(dev))\n                seen += int(mask.sum())\n                used += 1\n        for m, v in zip(bns, mom):\n            m.momentum = v\n        model.eval()\n        return used, seen\n\n    # ---- v9.8 --fm-aux: thin wrappers of the module-level fm_make_loader / fm_aux_step / fm_holdout_eval with this run\'s settings\n    def fm_loader(batches, seed_w):\n        return fm_make_loader(fm_store, batches, a.crop, a.shift_px if isinstance(batches, FmAuxSampler) else 0, a.fm_aux_workers, seed_w)\n\n    def fm_step(model, batch, img_main, enc_params, seed, k, ep, b_i, steps, acc, pw):\n        return fm_aux_step(model, batch, img_main, enc_params, a.fm_aux_weight, a.crop, seed, k, ep, b_i, steps, acc, pw,\n                           a.fm_aux_log_every, dev)\n\n    def fm_eval(model, hold_specs):\n        return fm_holdout_eval(model, fm_store, hold_specs, a.crop, a.fm_aux_windows, a.fm_aux_workers, dev)\n\n    def fm_auc_str(d_):\n        return " ".join("%s %.3f" % (c_, v_) for c_, v_ in d_.items())\n\n    def reeval(arm, seed):\n        only = None if not a.only_folds else {int(x) for x in a.only_folds.split(",")}\n        pre = "partial_" if only else ""\n        path = os.path.join(a.out, pre + "preds_%s_s%d.json" % (arm, seed))\n        if os.path.exists(path):\n            if a.save_state_oof and not all(os.path.isfile(p_) for p_ in state_files(a.out, pre, arm, seed)):   # v9.7: never a silent skip\n                raise SystemExit("--save-state-oof: %s exists (arm %s seed %d is done) but its state read-out does not (made without the flag); "\n                                 "re-predict into a fresh --out" % (path, arm, seed))\n            return\n        src = json.load(open(os.path.join(a.reeval_run, "pool_uids.json"), encoding="utf-8"))\n        if src != pool:\n            raise SystemExit("pool of --reeval-run differs from the current pool")\n        z_gold = torch.zeros((len(gold), C), device=dev)\n        z_pool = torch.zeros((len(pool), C), device=dev)\n        area_pool = torch.zeros((len(pool), C), device=dev)    # v9.3 --save-area\n        t_arm = time.time()\n        H = a.holdout_fold\n        if a.save_state_oof:                            # v9.7: NaN = a row not predicted here; the holdout fold\'s rows accumulate / (K - 1)\n            st_pool = torch.full((len(pool), st_layout["width"]), float("nan"), device=dev)\n            if H is not None:\n                st_pool[torch.from_numpy(np.flatnonzero(pool_fold == H)).to(dev)] = 0.0\n            st_gold = {}                                # fold k -> fold model k\'s read-out of every gold study\n        for k in range(K):\n            if only is not None and k not in only:\n                continue\n            if H is not None and k == H:                # no model is trained for the holdout fold\n                continue\n            ho = np.flatnonzero(pool_fold == k)\n            model = Net(a.backbone, a.init, freeze_blocks=a.freeze_blocks, crop=a.crop, state_spec=state_spec, in_chans=a.slices, spatial=a.spatial_attn, extent=a.extent, syn_decomp=a.syn_decomp, extent_classes=ext_classes).to(dev).to(memory_format=torch.channels_last)\n            sd = torch.load(os.path.join(a.reeval_run, "models", "%s_s%d_f%d.pt" % (arm, seed, k)), map_location="cpu")\n            if a.backbone == OFL_BACKBONE:\n                model.load_deploy_state_dict(sd)\n            else:\n                model.load_state_dict(sd)\n            if model.has_bn and a.bn_recal_batches > 0:\n                tr = np.flatnonzero((pool_fold != k) & (pool_fold != H)) if H is not None else np.flatnonzero(pool_fold != k)\n                b_, w_ = bn_recalibrate(model, [pool[i] for i in tr], 9100 + 17 * seed + k, a.bn_recal_batches)\n                print("  reeval fold %d: BatchNorm statistics recalibrated over %d batches (%d windows)" % (k, b_, w_), flush=True)\n            if a.save_state_oof:                        # v9.7: the same gold forward, plus fold model k\'s state read-out\n                zg_, sg_ = predict(model, gold, seed, want_state=True)\n                z_gold += zg_ / (K if H is None else K - 1)\n                st_gold[k] = sg_.cpu().numpy()\n            else:\n                z_gold += predict(model, gold, seed) / (K if H is None else K - 1)\n            if a.save_state_oof:                        # v9.7: the same predictions (+ the area proxy when asked), plus the state read-out\n                res_ = predict(model, [pool[i] for i in ho], seed, want_area=a.save_area, want_state=True)\n                z_pool[torch.from_numpy(ho).to(dev)] = res_[0]\n                if a.save_area:\n                    area_pool[torch.from_numpy(ho).to(dev)] = res_[1]\n                st_pool[torch.from_numpy(ho).to(dev)] = res_[-1]\n            elif a.save_area:                           # v9.3: same predictions, plus the area proxy\n                zp_, ar_ = predict(model, [pool[i] for i in ho], seed, want_area=True)\n                z_pool[torch.from_numpy(ho).to(dev)] = zp_\n                area_pool[torch.from_numpy(ho).to(dev)] = ar_\n            else:\n                z_pool[torch.from_numpy(ho).to(dev)] = predict(model, [pool[i] for i in ho], seed)\n            if H is not None:                       # the clean holdout fold, as in training (mean of the fold models)\n                hh = np.flatnonzero(pool_fold == H)\n                if a.save_state_oof:                    # v9.7: as below, plus each fold model\'s state read-out / (K - 1)\n                    res_ = predict(model, [pool[i] for i in hh], seed, want_area=a.save_area, want_state=True)\n                    z_pool[torch.from_numpy(hh).to(dev)] += res_[0] / (K - 1)\n                    if a.save_area:\n                        area_pool[torch.from_numpy(hh).to(dev)] += res_[1] / (K - 1)\n                    st_pool[torch.from_numpy(hh).to(dev)] += res_[-1] / (K - 1)\n                elif a.save_area:\n                    zh_, ah_ = predict(model, [pool[i] for i in hh], seed, want_area=True)\n                    z_pool[torch.from_numpy(hh).to(dev)] += zh_ / (K - 1)\n                    area_pool[torch.from_numpy(hh).to(dev)] += ah_ / (K - 1)\n                else:\n                    z_pool[torch.from_numpy(hh).to(dev)] += predict(model, [pool[i] for i in hh], seed) / (K - 1)\n            print("  reeval %s seed %d fold %d done (eval stride %d) %.0fs" % (arm, seed, k, a.eval_stride, time.time() - t_arm), flush=True)\n            del model\n            torch.cuda.empty_cache()\n        np.save(os.path.join(a.out, pre + "oof_%s_s%d.npy" % (arm, seed)), z_pool.cpu().numpy().astype(np.float32))\n        if a.save_area:                                      # v9.3 read-out: the extent head\'s area proxy per pool study\n            np.save(os.path.join(a.out, pre + "oof_area_%s_s%d.npy" % (arm, seed)), area_pool.cpu().numpy().astype(np.float32))\n        if a.save_state_oof:                                 # v9.7 read-out, written before the preds json (the done marker)\n            write_state_readout(a.out, pre, arm, seed, st_layout, a.state_spec, len(pool), st_pool.cpu().numpy(), gold,\n                                [int(by_uid[u]["fold"]) for u in gold], st_gold, K, H, only, a.reeval_run)\n        out = {"arm": arm, "seed": seed, "reeval_of": a.reeval_run, "eval_stride": a.eval_stride, "gold_uids": gold, "labels": LABELS,\n               "holdout_fold": H,\n               "only_folds": sorted(only) if only else None,\n               "gold_fold": [int(by_uid[u]["fold"]) for u in gold],\n               "gold_logits": [[round(float(x), 5) for x in r] for r in z_gold.cpu().numpy()],\n               "receipt": [{"arm": arm, "kind": "reeval", "pool_studies": len(pool), "gold_in_training": 0, "trained": False}]}\n        json.dump(out, open(path, "w"))\n\n    def run(arm, seed):\n        if a.reeval_run:\n            return reeval(arm, seed)\n        only = None if not a.only_folds else {int(x) for x in a.only_folds.split(",")}\n        path = os.path.join(a.out, ("partial_" if only else "") + "preds_%s_s%d.json" % (arm, seed))\n        if os.path.exists(path):\n            if a.save_state_oof and not all(os.path.isfile(p_) for p_ in state_files(a.out, "partial_" if only else "", arm, seed)):   # v9.7\n                raise SystemExit("--save-state-oof: %s exists (arm %s seed %d is done) but its state read-out does not (made without the flag); "\n                                 "write it with --reeval-run of this run into a fresh --out" % (path, arm, seed))\n            return\n        kind, tgt, wgt = arm_tensors(arm)\n        cell_w = kind == "lr" and len(specs.get(arm, ())) > 2 and bool(specs[arm][2])   # v8\n        z_gold = torch.zeros((len(gold), C), device=dev)\n        z_pool = torch.zeros((len(pool), C), device=dev)\n        spec_pool = torch.zeros((len(pool),), device=dev)      # v9.2\n        area_pool = torch.zeros((len(pool), C), device=dev)    # v9.3 --save-area\n        logs, steps_total = [], 0\n        warm_md5 = {}                                         # v9.6: fold -> md5 of the source fold model it started from\n        fm_res = {}                                           # v9.8: fold -> final holdout read-out and B5 gate (--fm-aux)\n        t_arm = time.time()\n        H = a.holdout_fold\n        if a.save_state_oof:                            # v9.7: NaN = a row not predicted here; the holdout fold\'s rows accumulate / (K - 1)\n            st_pool = torch.full((len(pool), st_layout["width"]), float("nan"), device=dev)\n            if H is not None:\n                st_pool[torch.from_numpy(np.flatnonzero(pool_fold == H)).to(dev)] = 0.0\n            st_gold = {}                                # fold k -> fold model k\'s read-out of every gold study\n        for k in range(K):\n            if only is not None and k not in only:\n                print("  fold %d skipped (--only-folds)" % k, flush=True)\n                continue\n            if H is not None and k == H:\n                print("  fold %d is the clean holdout: no model trained for it" % k, flush=True)\n                continue\n            tr = np.flatnonzero((pool_fold != k) & (pool_fold != H)) if H is not None else np.flatnonzero(pool_fold != k)\n            ho = np.flatnonzero(pool_fold == k)\n            torch.manual_seed(1000 + 17 * seed)          # same init for every fold: heads start identical, so fold models can be souped\n            gen = torch.Generator().manual_seed(5000 + 17 * seed + k)\n            mgen = torch.Generator().manual_seed(9400 + 17 * seed + k) if a.mirror > 0 else None   # v9.4: the mirror draws\' own stream (crops / jitter stay v9.3\'s)\n            # v9.5: the shift draws\' own stream (every other draw stays v9.4\'s); not \'sgen\', which the checkpoint block uses for the sampler\n            shgen = torch.Generator().manual_seed(9500 + 17 * seed + k) if a.shift_px > 0 else None\n            model = Net(a.backbone, a.init, freeze_blocks=a.freeze_blocks, crop=a.crop, state_spec=state_spec, in_chans=a.slices, spatial=a.spatial_attn, extent=a.extent, syn_decomp=a.syn_decomp, extent_classes=ext_classes,\n                        fm_aux=FM_A if fm_store is not None else 0).to(dev).to(memory_format=torch.channels_last)   # v9.8: + the aux_* head with --fm-aux\n            torch.manual_seed(1000 + 17 * seed + k)      # fold-specific stream for dropout / drop path / augmentation\n            if a.grad_ckpt:                              # 32 GB cards: recompute encoder activations in backward (same maths, ~1.3x compute)\n                model.enc.set_grad_checkpointing(True)\n            elif a.grad_ckpt_blocks > 0:                 # v9.1: only the first N blocks are recomputed (96 GB card at crop 384: N=6 fits)\n                partial_grad_checkpointing(model.enc, a.grad_ckpt_blocks)\n            if a.grad_ckpt_blocks > 0 and not a.grad_ckpt:\n                print("  v9.1 partial gradient checkpointing: first %d encoder blocks recomputed in backward" % a.grad_ckpt_blocks, flush=True)\n            if a.compile:                                # fused kernels; Module.compile keeps state_dict keys (checkpoints, deploy unchanged)\n                import torch._dynamo as _dynamo          # aliased: a bare \'import torch...\' here would make torch a local name\n                _dynamo.config.cache_size_limit = max(32, _dynamo.config.cache_size_limit)\n                model.enc.compile(dynamic=True)\n            with torch.no_grad():\n                model.bias.copy_(bias0)\n            if warm_rec is not None:                     # v9.6: the source run\'s fold-k model, whole and strict, after the bias0 copy (which\n                # it overwrites) and before the parameter groups, the optimizer and the EMA exist (the EMA below clones the loaded weights)\n                wpath = os.path.join(warm_rec["dir"], "models", "%s_s%d_f%d.pt" % (arm, seed, k))\n                n_w, warm_md5[k], warm_chg = warm_load(model, wpath)\n                # this run\'s spec buffers as now set: a --resume below must find exactly these in the checkpoint\'s weights\n                warm_spec = {n_: b_.detach().clone() for n_, b_ in model.named_buffers() if n_ in WARM_SPEC_BUFFERS}\n                if warm_chg:                             # the state head\'s spec buffers now hold this run\'s spec: per-class change log\n                    print("  v9.6 warm-start fold %d: state_prior / state_valid / state_base_logit re-set from this run\'s spec (%s) after the "\n                          "load; per class (%s): max |change| state_prior %s; changed valid slots %s; |change| state_base_logit %s; loaded "\n                          "state_gate %s" % (\n                              k, os.path.basename(a.state_spec), ", ".join(LABELS),\n                              ["%.4g" % v_ for v_ in warm_chg["state_prior"]], warm_chg["state_valid"],\n                              ["%.4g" % v_ for v_ in warm_chg["state_base_logit"]],\n                              ["%.3g" % v_ for v_ in model.state_gate.detach().float().cpu().tolist()]), flush=True)\n            enc = [p for n, p in model.named_parameters() if n.startswith("enc.") and p.requires_grad]\n            head = [p for n, p in model.named_parameters() if not n.startswith("enc.")]\n            groups = [{"params": enc, "lr": a.lr}]\n            vit_mod = model.enc.vit if a.backbone == OFL_BACKBONE else (model.enc if hasattr(model.enc, "blocks") else None)\n            if vit_mod is not None and a.llrd != 1.0:\n                nb = len(vit_mod.blocks)\n                groups, seen = [], set()\n                for i, blk in enumerate(vit_mod.blocks):\n                    ps = [p for p in blk.parameters() if p.requires_grad]\n                    if ps:\n                        groups.append({"params": ps, "lr": a.lr * (a.llrd ** (nb - 1 - i))}); seen.update(id(p) for p in ps)\n                rest = [p for p in enc if id(p) not in seen]          # final norm (top) and, when unfrozen, embeddings (bottom)\n                top = [p for n, p in vit_mod.named_parameters() if n.startswith("norm.") and p.requires_grad]\n                bottom = [p for p in rest if id(p) not in {id(q) for q in top}]\n                if top:\n                    groups.append({"params": top, "lr": a.lr})\n                if bottom:\n                    groups.append({"params": bottom, "lr": a.lr * (a.llrd ** nb)})\n            if k == (sorted(only)[0] if only else 0) and seed == 0:\n                print("trainable encoder params %.1fM, head params %.2fM, frozen encoder params %.1fM" % (   # v9.8: the aux_* head counted on its own line\n                    sum(p.numel() for p in enc) / 1e6, sum(p.numel() for n, p in model.named_parameters() if not n.startswith(("enc.", "aux_"))) / 1e6,\n                    sum(p.numel() for n, p in model.named_parameters() if n.startswith("enc.") and not p.requires_grad) / 1e6), flush=True)\n            opt = torch.optim.AdamW(groups + [{"params": head, "lr": a.head_lr}], weight_decay=a.wd)\n            if warm_rec is not None:                     # v9.6: every group (encoder, layer-wise groups, head) x s, before the LR schedule\n                lr_w = [g_["lr"] for g_ in opt.param_groups]                 # records it as the group\'s initial LR\n                for g_ in opt.param_groups:\n                    g_["lr"] = g_["lr"] * a.warm_lr_scale\n                print("  v9.6 warm-start %s seed %d fold %d: %s loaded strict (%d tensors, md5 %s); EMA starts from the loaded weights; "\n                      "LR of the %d parameter groups x %g: %s -> %s" % (arm, seed, k, wpath, n_w, warm_md5[k], len(lr_w), a.warm_lr_scale,\n                                                                      ["%.3g" % v_ for v_ in lr_w], ["%.3g" % g_["lr"] for g_ in opt.param_groups]), flush=True)\n            ema = None\n            if a.ema > 0:                               # EMA over everything that trains (frozen parts are shared anyway)\n                ema = {n_: p_.detach().clone() for n_, p_ in model.named_parameters() if p_.requires_grad}\n            ds, dl = make_loader([pool[i] for i in tr], "train", 9000 + 17 * seed + k, True, nonce=a.draw_seed)   # v9.6: + nonce\n            total = len(dl) * a.epochs\n            warm = max(1, int(.05 * total))\n            sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else\n                                                      .5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))\n            tr_t = torch.from_numpy(tr).to(dev)\n            log = []\n            fm_rec = fm_hist = fm_pw = fm_hold = None\n            if fm_store is not None:                     # v9.8: the fold\'s auxiliary set-up (the record a checkpoint must match on resume)\n                if model.has_bn:\n                    raise SystemExit("--fm-aux refused: %s has BatchNorm layers (the auxiliary batch would move their statistics)" % a.backbone)\n                if sorted(n_ for n_, _ in model.named_parameters() if n_.startswith("aux_")) != ["aux_b", "aux_ln_b", "aux_ln_w", "aux_w"]:\n                    raise AssertionError("v9.8: the auxiliary head must be exactly aux_ln_w, aux_ln_b, aux_w, aux_b")\n                fm_rec = {"dir": fm_store.dir, "md5": fm_store.md5, "weight": a.fm_aux_weight, "windows": a.fm_aux_windows, "flip": a.fm_aux_flip,\n                          "crop": a.crop, "shift_px": a.shift_px}\n                fm_hist = []\n                fm_pw = torch.tensor(FM_POS_WEIGHT, device=dev)\n                fm_hold = [(vi, w_, k_) for vi in fm_store.holdout for w_ in FM_VIEWS for k_ in range(int(fm_store.n[vi]))]\n                print("  v9.8 fm-aux %s seed %d fold %d: %d auxiliary batches of %d windows per epoch (one per step) through model.enc "\n                      "(%s); head aux_* %d parameters in the head group (lr %g); holdout read-out %d windows of %d volumes" % (\n                          arm, seed, k, len(dl), a.fm_aux_windows, "compiled" if a.compile else "eager",\n                          sum(p_.numel() for n_, p_ in model.named_parameters() if n_.startswith("aux_")), a.head_lr, len(fm_hold),\n                          len(fm_store.holdout)), flush=True)\n            # resumable state, written atomically at the end of every --ckpt-every epochs: weights, optimizer, LR schedule, EMA,\n            # and every random stream that decides the next epoch (augmentation, sampler, loader, CPU and GPU global streams)\n            ckpt_path = os.path.join(a.out, "ckpt", "%s_s%d_f%d.pt" % (arm, seed, k))\n            start_ep = 0\n            if a.resume and os.path.exists(ckpt_path):\n                ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)\n                if ck.get("epochs") != a.epochs or ck.get("train_studies") != len(tr):\n                    raise SystemExit("checkpoint %s does not match this run (epochs / training set)" % ckpt_path)\n                warm_now = None if warm_rec is None else dict(warm_rec, md5=warm_md5[k])\n                if ck.get("warm_start") != warm_now:     # v9.6: a checkpoint continues only the warm start it came from (none = none), and\n                    # only while the source fold model is the file it started from (md5)\n                    raise SystemExit("checkpoint %s does not match this run (warm start %s vs this run %s)" % (ckpt_path, ck.get("warm_start"), warm_now))\n                for key_, on_ in (("aug_grid_fp32", a.aug_grid_fp32), ("draw_seed", a.draw_seed), ("rare_boost_names", a.rare_boost_names)):\n                    if bool(ck.get(key_, False)) != on_:   # v9.6: one grid precision / draw seeding / boost rule per fold\n                        raise SystemExit("checkpoint %s does not match this run (--%s %s vs this run %s)" % (\n                            ckpt_path, key_.replace("_", "-"), bool(ck.get(key_, False)), on_))\n                if ck.get("fm_aux") != fm_rec:            # v9.8: a fold continues only the auxiliary set-up it started with (none = none)\n                    raise SystemExit("checkpoint %s does not match this run (--fm-aux %s vs this run %s)" % (ckpt_path, ck.get("fm_aux"), fm_rec))\n                if fm_rec is not None:\n                    fm_hist = list(ck.get("fm_hist") or [])\n                model.load_state_dict(ck["model"])\n                if warm_rec is not None:                 # v9.6: the resumed weights must carry this run\'s spec buffers (the checkpoint\'s\n                    bufs_ = dict(model.named_buffers())  # would otherwise silently replace the spec this run was launched with)\n                    stale = [n_ for n_, v_ in warm_spec.items() if not torch.equal(bufs_[n_], v_)]\n                    if stale:\n                        raise SystemExit("checkpoint %s holds other %s than this run\'s state spec (%s) builds: it was written under another "\n                                         "spec; resume with that spec or restart the fold" % (ckpt_path, stale, a.state_spec))\n                opt.load_state_dict(ck["opt"])\n                sched.load_state_dict(ck["sched"])\n                if ema is not None:\n                    ema = {n_: v.to(dev) for n_, v in ck["ema"].items()}\n                gen.set_state(ck["gen"])\n                if ck.get("sampler_gen") is not None:\n                    (dl.batch_sampler.generator if isinstance(dl.batch_sampler, PairBatchSampler) else dl.sampler.generator).set_state(ck["sampler_gen"])\n                if ck.get("loader_gen") is not None:\n                    dl.generator.set_state(ck["loader_gen"])\n                if mgen is not None and ck.get("mirror_gen") is not None:   # v9.4: the mirror stream continues where it stopped\n                    mgen.set_state(ck["mirror_gen"])\n                if shgen is not None and ck.get("shift_gen") is not None:   # v9.5: the shift stream continues where it stopped\n                    shgen.set_state(ck["shift_gen"])\n                torch.set_rng_state(ck["cpu_rng"])\n                torch.cuda.set_rng_state_all(ck["cuda_rng"])\n                log = list(ck["log"])\n                start_ep = int(ck["epoch"]) + 1\n                print("  resumed %s seed %d fold %d after epoch %d (%s)" % (arm, seed, k, ck["epoch"], ckpt_path), flush=True)\n            for ep in range(start_ep, a.epochs):\n                ds.epoch = ep\n                model.train()\n                tot = n = 0\n                pr_batches = pr_terms = 0\n                mir_batches = 0                             # v9.4\n                shift_ep = []                               # v9.5: the epoch\'s (dy, dx) draws (read-out only)\n                ext_acc = torch.zeros(8, device=dev)        # v9.3: + the high_grade_small cells\' mean log1p(A/100) (entries 6, 7)\n                if fm_store is not None:                    # v9.8: this epoch\'s auxiliary batches: fresh generators (seeded from seed, fold,\n                    # epoch), one batch per step, own loader; built inside fork_rng (the global CPU stream is not consumed)\n                    with torch.random.fork_rng(devices=[]):\n                        fm_seeds = fm_epoch_seeds(seed, k, ep)\n                        fm_samp = FmAuxSampler(fm_store, len(dl), a.fm_aux_windows, a.crop, a.shift_px, a.fm_aux_flip, fm_seeds)\n                        fm_it = iter(fm_loader(fm_samp, fm_seeds[1] + 1))\n                    fm_acc = torch.zeros(3, device=dev)\n                    fm_b = 0\n                for img, meta, mask, idx in dl:\n                    sel = tr_t[idx.to(dev)]\n                    with torch.autocast("cuda", dtype=torch.bfloat16):   # augment stays inside the autocast region, as in v9.3 (aug v2\'s grid maths)\n                        img_d, meta_d = augment(img.to(dev, non_blocking=True), gen, shgen, shift_ep), meta.to(dev)   # v9.5: + the shift stream\n                        # v9.4: one draw per batch (both studies, so both halves of a matched pair, share it): the virtual contralateral\n                        # knee goes through the same forward; the outputs are re-indexed to the canonical class order before every loss\n                        flip = mgen is not None and float(torch.rand(1, generator=mgen)) < a.mirror\n                        if flip:\n                            img_d, meta_d = mirror_batch(img_d, meta_d)\n                            mir_batches += 1\n                        z, sl, bl = model(img_d, meta_d, mask.to(dev), return_aux=True)\n                    area = model._ext_area                                          # [B, C] with --extent, else None\n                    w_main = wgt[sel] if (kind == "bce" or cell_w) else None\n                    sti_b, stw_b, bint_b = (st_idx[sel], st_w[sel], bin_t[sel]) if state_spec is not None else (None, None, None)\n                    if flip:\n                        z, sl, bl, area, w_main, sti_b, stw_b, bint_b = mirror_outputs(z, sl, bl, area, w_main, sti_b, stw_b, bint_b,\n                                                                                       mir_perm, mir_bin_perm, mir_mcl_bins)\n                    loss = bce_soft(z, tgt[sel], w_main) if kind == "bce" else lr_loss(z, tgt[sel], w_main)\n                    if state_spec is not None:\n                        loss = loss + a.state_weight * state_loss(sl, bl, sti_b, stw_b, bint_b)\n                    if a.extent:\n                        ext_l = extent_loss(area, sel)\n                        if a.extent_weight > 0:\n                            loss = loss + a.extent_weight * ext_l\n                        with torch.no_grad():                     # epoch log of the extent path (audit 6), no sync per step\n                            la_, y_ = torch.log1p(area.float() / 100.0), ext_y[sel]\n                            ext_acc += torch.stack([(la_ * (y_ == 1)).sum(), (y_ == 1).sum().float(), (la_ * (y_ == 0)).sum(),\n                                                    (y_ == 0).sum().float(), ext_l.detach().float(), torch.ones((), device=dev),\n                                                    (la_ * (y_ == OA_SMALL_TARGET)).sum(), (y_ == OA_SMALL_TARGET).sum().float()])\n                    if a.pair_rank > 0:\n                        pl, pc = pair_loss(z, sel, area if a.oa_area_rank else None, stw_b)   # v9.3: area families read the extent head\n                        loss = loss + a.pair_rank * pl\n                        pr_batches += int(pc > 0); pr_terms += pc\n                    opt.zero_grad(set_to_none=True)\n                    loss.backward()\n                    if fm_store is not None:                # v9.8: the auxiliary step, after the main backward, into the same .grad\n                        with torch.random.fork_rng(devices=[]):   # the fetch (loader iterator) never touches the global CPU stream\n                            fm_batch = next(fm_it)\n                        fm_ld, fm_lp = fm_step(model, fm_batch, img_d, enc, seed, k, ep, fm_b, len(dl), fm_acc, fm_pw)\n                        fm_b += 1\n                    if a.forward_only:\n                        g = sum(float(p.grad.abs().sum()) for p in model.parameters() if p.grad is not None)\n                        print("forward-only check: loss %.4f, gradient mass %.3e, windows %d, optimizer steps 0" % (\n                            float(loss), g, int(mask.sum())), flush=True)\n                        if fm_store is not None:\n                            print("forward-only check v9.8 fm-aux: L_dense %.4f L_presence %.4f (lambda %g), %d auxiliary windows; the "\n                                  "gradient mass above includes lambda * grad L_aux" % (float(fm_ld), float(fm_lp), a.fm_aux_weight,\n                                                                                         a.fm_aux_windows), flush=True)\n                        return\n                    if fm_store is None:\n                        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)\n                    else:                                   # v9.8: one clip of the merged gradient (v9.7\'s when lambda = 0)\n                        fm_clip(model, 5.)\n                    opt.step()\n                    sched.step()\n                    if ema is not None:\n                        with torch.no_grad():\n                            d_ = min(a.ema, (1 + sched.last_epoch) / (10 + sched.last_epoch))   # warm EMA early on\n                            for n_, p_ in model.named_parameters():\n                                if n_ in ema:\n                                    ema[n_].mul_(d_).add_(p_.detach(), alpha=1 - d_)\n                    tot += float(loss.detach()) * len(idx)\n                    n += len(idx)\n                log.append(round(tot / max(1, n), 5))\n                print("  %s seed %d fold %d epoch %d loss %.4f  %.0fs" % (arm, seed, k, ep, log[-1], time.time() - t_arm), flush=True)\n                if a.syn_decomp:                         # v9.2 read-out (audit N1): beta live and EMA, every epoch\n                    print("  syn-decomp epoch %d: beta %.4f (ema %s)" % (ep, float(model.syn_beta.detach().cpu()),\n                          ("%.4f" % float(ema["syn_beta"].cpu())) if ema is not None and "syn_beta" in ema else "n/a"), flush=True)\n                if a.extent:\n                    ea = ext_acc.tolist()\n                    print("  extent epoch %d: auxiliary loss %.4f (weight %.2f) | gamma MOA/LOA/PFOA %s | mean log1p(A/100) on >= 1 cm^2 stated %.3f vs normal/low grade %.3f" % (\n                        ep, ea[4] / max(1.0, ea[5]), a.extent_weight,\n                        [round(float(model.ext_gamma[LABELS.index(l)]), 3) for l in ("Medial OA", "Lateral OA", "PF OA")],\n                        ea[0] / max(1.0, ea[1]), ea[2] / max(1.0, ea[3])), flush=True)\n                    if a.oa_area_rank:\n                        print("  extent v9.3 epoch %d: mean log1p(A/100) on high_grade_small (target %.1f) %.3f over %d cells" % (\n                            ep, OA_SMALL_TARGET, ea[6] / max(1.0, ea[7]), int(ea[7])), flush=True)\n                if a.pair_rank > 0:\n                    print("  pair-rank epoch %d: %d of %d batches had at least one matched-pair term (%d class terms; includes base batches that happened to pair)" % (ep, pr_batches, len(dl), pr_terms), flush=True)\n                if a.mirror > 0:                         # v9.4 read-out: the mirrored share of the epoch\'s batches\n                    print("  mirror epoch %d: %d of %d batches mirrored (p %.2f)" % (ep, mir_batches, len(dl), a.mirror), flush=True)\n                if a.shift_px > 0:                       # v9.5 read-out: the epoch\'s batch translations\n                    dys, dxs = [sh[0] for sh in shift_ep] or [0], [sh[1] for sh in shift_ep] or [0]\n                    print("  shift epoch %d: %d of %d batches translated (N %d): dy %d..%d, dx %d..%d, mean |dy| %.2f |dx| %.2f" % (\n                        ep, len(shift_ep), len(dl), a.shift_px, min(dys), max(dys), min(dxs), max(dxs),\n                        sum(abs(v) for v in dys) / len(dys), sum(abs(v) for v in dxs) / len(dxs)), flush=True)\n                if fm_store is not None:                 # v9.8 read-out: the epoch\'s auxiliary losses, sample digest, holdout AUC, compiles\n                    with torch.random.fork_rng(devices=[]):\n                        fm_left = next(fm_it, None)\n                    if fm_left is not None or fm_samp.yielded != len(dl) or fm_b != len(dl):\n                        raise AssertionError("v9.8: epoch %d used %d auxiliary batches, the sampler yielded %d, the loader had more: %s" % (\n                            ep, fm_b, fm_samp.yielded, fm_left is not None))\n                    fa = fm_acc.tolist()\n                    ev_ = fm_eval(model, fm_hold)\n                    cmp_ = dynamo_compiles() if a.compile else None\n                    fm_hist.append({"epoch": ep, "l_dense": fa[0] / max(1.0, fa[2]), "l_presence": fa[1] / max(1.0, fa[2]), "steps": int(fa[2]),\n                                    "digest": fm_samp.digest(), "holdout": ev_, "dynamo": cmp_})\n                    print("  fm-aux epoch %d: L_dense %.5f L_presence %.5f over %d steps (lambda %g); sample digest %s (seeds %s)" % (\n                        ep, fm_hist[-1]["l_dense"], fm_hist[-1]["l_presence"], int(fa[2]), a.fm_aux_weight, fm_samp.digest(),\n                        list(fm_seeds)), flush=True)\n                    print("  fm-aux epoch %d holdout presence AUC (live weights; %d windows / %d volumes), window level: %s | volume level: %s" % (\n                        ep, ev_["windows"], ev_["volumes"], fm_auc_str(ev_["window_auc"]), fm_auc_str(ev_["volume_auc"])), flush=True)\n                    if cmp_ is not None:\n                        print("  fm-aux epoch %d dynamo: %d frames compiled, %d recompiles (test 9: <= 4 at the end of epoch 0, no growth after)" % (\n                            ep, cmp_[0], cmp_[1]), flush=True)\n                if a.ckpt_every > 0 and (ep + 1) % a.ckpt_every == 0 and ep + 1 < a.epochs:\n                    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)\n                    sgen = getattr(dl.sampler, "generator", None) if isinstance(dl.sampler, torch.utils.data.WeightedRandomSampler) else None\n                    if isinstance(dl.batch_sampler, PairBatchSampler):\n                        sgen = dl.batch_sampler.generator\n                    ck_ = {"epoch": ep, "epochs": a.epochs, "train_studies": len(tr), "log": log,\n                           "model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),\n                           "ema": None if ema is None else {n_: v.detach().cpu() for n_, v in ema.items()},\n                           "gen": gen.get_state(), "sampler_gen": None if sgen is None else sgen.get_state(),\n                           "loader_gen": None if dl.generator is None else dl.generator.get_state(),\n                           "cpu_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all()}\n                    if mgen is not None:                 # v9.4: the key exists only with --mirror (the checkpoint is otherwise v9.3\'s)\n                        ck_["mirror_gen"] = mgen.get_state()\n                    if shgen is not None:                # v9.5: the key exists only with --shift-px > 0 (otherwise v9.4\'s key set)\n                        ck_["shift_gen"] = shgen.get_state()\n                    if warm_rec is not None:             # v9.6: the key exists only with --warm-start (otherwise v9.5\'s key set)\n                        ck_["warm_start"] = dict(warm_rec, md5=warm_md5[k])\n                    for key_, on_ in (("aug_grid_fp32", a.aug_grid_fp32), ("draw_seed", a.draw_seed), ("rare_boost_names", a.rare_boost_names)):\n                        if on_:                          # v9.6: each key exists only with its flag (otherwise v9.5\'s key set)\n                            ck_[key_] = True\n                    if fm_rec is not None:               # v9.8: the keys exist only with --fm-aux (otherwise v9.7\'s key set); the auxiliary\n                        ck_["fm_aux"] = fm_rec           # generators restart from their epoch seeds, so they need no state here\n                        ck_["fm_hist"] = fm_hist\n                    torch.save(ck_, ckpt_path + ".tmp")\n                    os.replace(ckpt_path + ".tmp", ckpt_path)\n                if a.stop_after_epoch >= 0 and ep == a.stop_after_epoch:\n                    print("  stop-after-epoch %d: exiting (test)" % ep, flush=True)\n                    sys.exit(3)\n            steps_total += total\n            if ema is not None:                         # evaluate and ship the EMA weights\n                with torch.no_grad():\n                    for n_, p_ in model.named_parameters():\n                        if n_ in ema:\n                            p_.copy_(ema[n_])\n                if model.has_bn and a.bn_recal_batches > 0:\n                    b_, w_ = bn_recalibrate(model, [pool[i] for i in tr], 9100 + 17 * seed + k, a.bn_recal_batches)\n                    print("  BatchNorm statistics recalibrated for the EMA weights: %d batches, %d windows" % (b_, w_), flush=True)\n                elif model.has_bn:\n                    print("  WARNING: BatchNorm backbone with --ema and no --bn-recal-batches: the saved model pairs EMA weights with "\n                          "the live weights\' BatchNorm statistics (see the 10-08 CoAtNet screen)", flush=True)\n            if fm_store is not None:                     # v9.8: the B5 diagnostic with the shipped weights (EMA when --ema), written per fold\n                ev_ = fm_eval(model, fm_hold)\n                gate = {c_: (ev_["window_auc"][c_], thr_, bool(ev_["window_auc"][c_] >= thr_)) for c_, thr_ in FM_GATE}\n                fm_res[k] = {"final_holdout": ev_, "gate": gate, "gate_pass": all(g_[2] for g_ in gate.values())}\n                print("  fm-aux fold %d final holdout presence AUC (%s weights; %d windows / %d volumes), window level: %s | volume level: %s" % (\n                    k, "EMA" if ema is not None else "live", ev_["windows"], ev_["volumes"], fm_auc_str(ev_["window_auc"]),\n                    fm_auc_str(ev_["volume_auc"])), flush=True)\n                print("  fm-aux fold %d B5 gate (the head learned; diagnostic, not a mechanism verdict): %s -> %s" % (\n                    k, ", ".join("%s %.3f >= %.2f %s" % (c_, v_[0], v_[1], "ok" if v_[2] else "NOT MET") for c_, v_ in gate.items()),\n                    "PASS" if fm_res[k]["gate_pass"] else "FAIL (stop and check the implementation / data; never read as \'mechanism "\n                                                         "without effect\')"), flush=True)\n                json.dump({"arm": arm, "seed": seed, "fold": k, "kind": "v9.8 --fm-aux auxiliary-task log", "store": fm_rec,\n                           "classes": list(FM_CLASSES), "epochs": fm_hist, "final_holdout": ev_, "final_weights": "EMA" if ema is not None else "live",\n                           "gate": gate, "gate_pass": fm_res[k]["gate_pass"], "pos_weight": FM_POS_WEIGHT, "presence_min_area": FM_PRESENCE_MIN,\n                           "volumes_per_step": FM_VOLS_PER_STEP, "boxed_volume_share": FM_BOXED_FRAC},\n                          open(os.path.join(a.out, "fm_aux_%s_s%d_f%d.json" % (arm, seed, k)), "w"), indent=1)\n            os.makedirs(os.path.join(a.out, "models"), exist_ok=True)\n            torch.save(model.deploy_state_dict(),\n                       os.path.join(a.out, "models", "%s_s%d_f%d.pt" % (arm, seed, k)))\n            if os.path.exists(ckpt_path):                 # the fold is complete: its resumable state is no longer needed\n                os.remove(ckpt_path)\n            if a.save_state_oof:                        # v9.7: the same gold forward, plus fold model k\'s state read-out\n                zg_, sg_ = predict(model, gold, seed, want_state=True)\n                z_gold += zg_ / (K if H is None else K - 1)\n                st_gold[k] = sg_.cpu().numpy()\n            else:\n                z_gold += predict(model, gold, seed) / (K if H is None else K - 1)\n            if a.save_state_oof:                        # v9.7: the same held-out forward, plus the state read-out\n                zp_, sp_, ar_, ps_ = predict(model, [pool[i] for i in ho], seed, want_spec=True, want_area=True, want_state=True)\n                st_pool[torch.from_numpy(ho).to(dev)] = ps_\n            else:\n                zp_, sp_, ar_ = predict(model, [pool[i] for i in ho], seed, want_spec=True, want_area=True)\n            z_pool[torch.from_numpy(ho).to(dev)] = zp_\n            if a.syn_decomp:\n                spec_pool[torch.from_numpy(ho).to(dev)] = sp_\n            if a.save_area:\n                area_pool[torch.from_numpy(ho).to(dev)] = ar_\n            if H is not None:\n                hh = np.flatnonzero(pool_fold == H)\n                if a.save_state_oof:                    # v9.7: as below, plus each fold model\'s state read-out / (K - 1)\n                    res_ = predict(model, [pool[i] for i in hh], seed, want_area=a.save_area, want_state=True)\n                    z_pool[torch.from_numpy(hh).to(dev)] += res_[0] / (K - 1)\n                    if a.save_area:\n                        area_pool[torch.from_numpy(hh).to(dev)] += res_[1] / (K - 1)\n                    st_pool[torch.from_numpy(hh).to(dev)] += res_[-1] / (K - 1)\n                elif a.save_area:                       # v9.3: same predictions, plus the area proxy\n                    zh_, ah_ = predict(model, [pool[i] for i in hh], seed, want_area=True)\n                    z_pool[torch.from_numpy(hh).to(dev)] += zh_ / (K - 1)\n                    area_pool[torch.from_numpy(hh).to(dev)] += ah_ / (K - 1)\n                else:\n                    z_pool[torch.from_numpy(hh).to(dev)] += predict(model, [pool[i] for i in hh], seed) / (K - 1)\n            if fm_store is not None and a.compile:        # v9.8 (test 9): the dynamo count once the fold\'s predictions are made too\n                cmp_ = dynamo_compiles()\n                print("  fm-aux fold %d dynamo after the fold\'s predictions: %d frames compiled, %d recompiles" % (k, cmp_[0], cmp_[1]), flush=True)\n            logs.append([log[0], log[-1]])\n            del model, opt\n            torch.cuda.empty_cache()\n        np.save(os.path.join(a.out, ("partial_" if only else "") + "oof_%s_s%d.npy" % (arm, seed)), z_pool.cpu().numpy().astype(np.float32))\n        if a.syn_decomp:                                     # v9.2 read-out: the synovium-specific logit per pool study\n            np.save(os.path.join(a.out, ("partial_" if only else "") + "oof_synspec_%s_s%d.npy" % (arm, seed)), spec_pool.cpu().numpy().astype(np.float32))\n        if a.save_area:                                      # v9.3 read-out: the extent head\'s area proxy per pool study\n            np.save(os.path.join(a.out, ("partial_" if only else "") + "oof_area_%s_s%d.npy" % (arm, seed)), area_pool.cpu().numpy().astype(np.float32))\n        if a.save_state_oof:                                 # v9.7 read-out, written before the preds json (the done marker)\n            write_state_readout(a.out, "partial_" if only else "", arm, seed, st_layout, a.state_spec, len(pool), st_pool.cpu().numpy(), gold,\n                                [int(by_uid[u]["fold"]) for u in gold], st_gold, K, H, only, None)\n        out = {"arm": arm, "seed": seed, "epochs": a.epochs, "nested": False, "gold_uids": gold, "only_folds": sorted(only) if only else None,\n               "holdout_fold": H,\n               "gold_fold": [int(by_uid[u]["fold"]) for u in gold], "labels": LABELS,\n               "gold_logits": [[round(float(x), 5) for x in r] for r in z_gold.cpu().numpy()],\n               "loss_first_last": logs, "optimizer_steps": steps_total, "train_seconds": round(time.time() - t_arm, 1),\n               "receipt": [{"arm": arm, "kind": kind, "pool_studies": len(pool), "gold_in_training": 0,\n                            "init_seed": 1000 + 17 * seed, "note": "fold models share one initialisation (soup-compatible)"}]}\n        if warm_rec is not None:                             # v9.6: provenance of every fold model\'s starting point\n            out["receipt"][0].update(warm_start=warm_rec["dir"], warm_lr_scale=warm_rec["lr_scale"],\n                                     warm_state_spec_md5=warm_rec["state_spec_md5"],\n                                     warm_fold_md5={str(k_): v_ for k_, v_ in sorted(warm_md5.items())},\n                                     note="v9.6 --warm-start: fold k continued from the source run\'s fold-k model; optimizer, LR schedule and EMA fresh")\n        if fm_store is not None:                             # v9.8: the auxiliary task\'s provenance and the per-fold B5 gate\n            out["receipt"][0]["fm_aux"] = {"store": fm_store.dir, "md5": fm_store.md5, "weight": a.fm_aux_weight, "windows": a.fm_aux_windows,\n                                           "flip": a.fm_aux_flip, "crop": a.crop, "shift_px": a.shift_px,\n                                           "gate_pass": {str(k_): v_["gate_pass"] for k_, v_ in sorted(fm_res.items())},\n                                           "note": "v9.8 --fm-aux: fastMRI+ lesion-box auxiliary task on the shared encoder (training only; the fold "\n                                                   "models carry no aux_* key); per-fold log fm_aux_<arm>_s<seed>_f<k>.json"}\n        json.dump(out, open(path, "w"))\n\n    if not a.reeval_run:                                     # v9.8 E2: the arms\' targets are checked before run_config.json is written\n        for arm_ in [x.split("=", 1)[0] for x in a.arms.split(",") if x]:   # (v9.7 checked them in run(), after writing it)\n            try:\n                arm_tensors(arm_)\n            except (AssertionError, KeyError, IndexError, ValueError) as e:\n                raise SystemExit("arm %s refused before anything was written: its targets fail the training checks (%s: %s)" % (\n                    arm_, type(e).__name__, e))\n    cfg = {k: getattr(a, k) for k in vars(a)}\n    cfg.update({"pool_studies": len(pool), "gold_studies": len(gold), "feature_studies": len(done), "labels": LABELS,\n                "scaffold": "compact fine-tuned image model on the pixel cache"})\n    json.dump(cfg, open(os.path.join(a.out, "run_config.json"), "w"), indent=1)\n    json.dump(pool, open(os.path.join(a.out, "pool_uids.json"), "w"))\n    for seed in range(a.seed_base, a.seed_base + a.seeds):\n        for arm in [x.split("=", 1)[0] for x in a.arms.split(",") if x]:\n            run(arm, seed)\n            if a.forward_only:\n                return\n    print("ALL DONE", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'compact_infer.py': '"""Compact-model inference worker (Kaggle or local): raw DICOM -> image scores for one list of studies.\n\nPath, identical to how the compact model was trained and evaluated:\n  raw DICOM  --mainline training.native_cache.prepare (one study at a time)-->  native uint16 series\n             --px_cache.StudySlices.build (the SAME file that built the pixel cache)-->  288 px slices\n             --compact_train evaluation windows (every third slice as centre, both views, 3 adjacent slices)-->\n             --centre crop 256, fold models of compact_train.Net, mean logit-->  12 probabilities\nNothing is trained here.  No report, no label, no test-time fitting.  Failed or unreached studies are\nlisted as fallbacks for the controller.\n\nv3 (10-10, R4e): (1) a fold model trained with trainer v9.2+ --syn-decomp (state-dict key syn_beta) is rebuilt with\nNet(..., syn_decomp=True) -- needs a pinned compact_train.py whose Net has syn_decomp (v9.3); (2) --crop2: group 2 is built\nwith its own crop and fed its own centre crop of the same decoded windows; (3) --compile: torch.compile of every fold\nmodel\'s encoder (mode \'default\', dynamic=False: the 10-08 T4 benchmark kaggle/inferbench); before the first study is scored\nevery encoder batch shape is compiled on synthetic windows (broken-kernel bound) and the first real study runs compiled and\neager: its study-level probabilities must agree within --compile-tol, or else an fp32 forward must show the compiled path\nno further from the fp32 model than eager; any error or failed gate puts every model back on its eager encoder (logged as\nCOMPILE_FALLBACK, recorded in the receipt); (4) --trainer-md5: stop when the imported compact_train.py is not the bundle\'s\npinned trainer.  Without these flags and without syn_beta in the state dicts the computation is v2\'s.\nv3b (10-10): two groups are mixed in logit space, p = sigmoid((1 - w2) z1 + w2 z2) of the fold-mean logits (v3 mixed probabilities); predictions.csv probabilities are written with %.9g (v3 %.6f rounded ~3 % of cells to 0 / 1).\n\nv4 (10-10, fluid stacker): optional --fluid-stacker <json> (analysis/fit_fluid_stacker.py on the box).  Each study\'s D2 fluid\nvolume is measured in the loader worker from the rows px_cache.StudySlices.build has just rendered for the models (no extra\ndecode; the 10-09 definition of scratch_fluid_1009/fluid_measure.py, see fluid_d2 below), and for every class the json marks\n"adopt" the output logit is replaced by the stacker\'s logit b0 + b_z * z + b_lv * log1p(mL) + b_miss * miss.  A study whose\nvolume cannot be measured (no fluid-sensitive sagittal series, missing region view, any exception) takes the missing path\n(lv = 0, miss = 1), never a crash.\nv4b (10-10) = v3b + v4\'s fluid additions; z = the class\'s deployed v3b line mix (1 - w2) z1 + w2 z2 of the groups\' fold-mean\nlogits (one group: its fold-mean logit), the same tensor operations as score(); the json must be combine "logit" and its line\nweights must equal (1 - w2, w2) of every adopted class (SystemExit otherwise).  Without --fluid-stacker the output is v3b\'s.\nv4c (10-11) = v4b + (1) --flip-tta: the trainer\'s v9.4 --mirror at inference -- every fold model also scores the mirrored windows\n(compact_train_v97.mirror_batch, copied: coronal / axial windows flipped on the width axis, sagittal windows reversed along the slice\naxis with rel_pos negated, coronal bilateral views 2 <-> 3), its 12 logits re-indexed by P (Medial <-> Lateral Meniscus, Medial <->\nLateral OA; compact_train_v97.mirror_perm), and the two passes averaged in logit space per fold model; MCL = the plain pass only (the\ntrainer drops MCL from every mirrored batch); (2) --state-stacker <json> (analysis/fit_state_stacker.py, one model group): for the\nclasses the json adopts, the output logit is b0 + coef . [z, the class\'s state-head probabilities, its binaries (, lv, miss)], the\nstate read-out = compact_train_v97.state_probs of every fold model\'s forward, averaged over the fold models (the trainer\'s gold-mean\nconvention), with --flip-tta averaged over the two passes like the logits (MCL\'s block and binaries from the plain pass); a class is\nreplaced by at most one stacker (state or fluid); (3) --dump-state: the per-model state read-out in --dump-logits.  None of them\ngiven = v4b byte for byte.\n"""\nimport argparse, contextlib, csv, hashlib, inspect, io, json, math, os, shutil, sys, time\nfrom pathlib import Path\nsys.dont_write_bytecode = True\nHERE = Path(__file__).resolve().parent\nimport numpy as np\nimport torch\n\nUIDK = "StudyInstanceUID"\nFORCED_SIDE = {}        # study uid -> "L" / "R": the native render uses this side instead of tags / sagittal IPP x\nCOMPILE_SANITY_DP = 0.05   # v3 --compile, synthetic windows (random pixels): compiled vs eager |dp| above this = a broken kernel\n\n\ndef compile_gate(d_final, tol, err_c=None, err_e=None):\n    """v3 --compile acceptance on the first real study\'s study-level probabilities.  d_final = max |p_compiled - p_eager|;\n    err_c / err_e = max |p - p_fp32| of the compiled / eager path (an fp32 forward of the same study, asked for only when\n    d_final > tol).  Returns (True | False | None = needs the fp32 reference, reason)."""\n    if d_final <= tol:\n        return True, "compiled == eager within %.3g (max |dp| %.3g)" % (tol, d_final)\n    if err_c is None or err_e is None:\n        return None, "max |dp| compiled vs eager %.3g > %.3g: fp32 reference needed" % (d_final, tol)\n    if err_c <= err_e + tol:\n        return True, ("compiled vs eager max |dp| %.3g > %.3g, but compiled is no further from the fp32 model than eager "\n                      "(max |dp| vs fp32: compiled %.3g, eager %.3g)" % (d_final, tol, err_c, err_e))\n    return False, ("compiled vs eager max |dp| %.3g > %.3g and compiled is further from the fp32 model than eager "\n                   "(compiled %.3g, eager %.3g)" % (d_final, tol, err_c, err_e))\n\n\n# ---- v4: D2 fluid volume (training-time definition: scratch_fluid_1009/fluid_measure.py + fluid_io.py on the box, 10-09,\n# sagittal part; the same operations in the same order, on the rows StudySlices.build returns instead of the px384_v4 cache rows)\n#   series  among the study\'s Sagittal blocks whose META fluid == 1 (fluid-sensitive), the one with the lowest median of its\n#           full-field view (id 0) rows over pixels > 10 (255 when none); ties keep the first in block order\n#   region  Effusion: view 4 (100 mm patellofemoral local), Baker\'s: view 5 (100 mm posterior local) of that series\n#   mask    region rows >= 229 (the series\' native p99 in the series_companded uint8 mapping), then per slice a morphological\n#           opening with an elliptical disk of radius max(1, round(1.5 mm / pixel_mm)) px (keeps pockets >= ~3 mm thick)\n#   volume  sum over slices of (mask pixels x pixel_mm^2) x slice spacing / 1000 -> mL; pixel_mm = extent_130 x 130 / 384 per\n#           slice row, spacing = expm1(log_spacing) of the series\' first row (float32 meta, as the cache stores it)\nFLUID_SIZE = 384                     # the definition and the stacker were fitted on the 384 px cache (px384_v4)\nFLUID_P99 = 229\nFLUID_OPEN_MM = 1.5\nFLUID_REGION = {"eff_sag_p99_open": 4, "bak_sag_p99_open": 5}\nFLUID_COL = {"log_spacing": 5, "extent_130": 6, "fluid": 7, "fatsat": 8}          # px_cache.META_COLS positions\n\n\ndef _fluid_disk(r):\n    import cv2\n    r = max(1, int(round(r)))\n    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))\n\n\ndef _fluid_opened(mask, pix_mm):\n    import cv2\n    out = np.zeros_like(mask)\n    for k in range(len(mask)):\n        if mask[k].any():\n            out[k] = cv2.morphologyEx(mask[k], cv2.MORPH_OPEN, _fluid_disk(FLUID_OPEN_MM / pix_mm[k]))\n    return out\n\n\ndef _fluid_ml(mask, pix_mm, spacing):\n    return float((mask.reshape(len(mask), -1).sum(1) * pix_mm ** 2).sum() * spacing / 1000.0)\n\n\ndef fluid_d2(px, meta, blocks):\n    """D2 volumes (mL) of one study from its rendered rows: px [N, 384, 384] uint8, meta [N, 12] float32, blocks = the\n    StudySlices.build blocks ({"plane", "n_slices", "views": [[view_id, first row], ...], "series"}); px may be any object\n    whose [a:b] slicing returns those rows.  Returns {column: mL or None, ...}; never raises."""\n    rec = {"status": "ok", "eff_sag_p99_open": None, "bak_sag_p99_open": None, "n_sag_fluid_series": 0, "series": None,\n           "median": None, "n": None, "spacing": None}\n    try:\n        best = None\n        for b in blocks:\n            if b["plane"] != "Sagittal":\n                continue\n            n = int(b["n_slices"])\n            views, first = {}, None\n            for v, st in b["views"]:\n                m = np.asarray(meta[st])\n                if first is None:\n                    first = m\n                ext = np.asarray(meta[st:st + n, FLUID_COL["extent_130"]], np.float64) * 130.0      # mm spanned by the 384 px side\n                views[int(v)] = (int(st), ext / FLUID_SIZE)                                         # per-slice pixel size (mm)\n            if first is None or int(first[FLUID_COL["fluid"]]) != 1:\n                continue\n            rec["n_sag_fluid_series"] += 1\n            st0, _ = views[0]\n            full = np.asarray(px[st0:st0 + n])\n            fg = full[full > 10]\n            med = float(np.median(fg)) if fg.size else 255.0\n            if best is None or med < best[0]:\n                best = (med, b, views, n, float(np.expm1(first[FLUID_COL["log_spacing"]])))\n        if best is None:\n            rec["status"] = "no fluid-sensitive sagittal series"\n            return rec\n        med, b, views, n, spacing = best\n        rec.update(series=b.get("series"), median=med, n=n, spacing=spacing)\n        missing = []\n        for col, v in FLUID_REGION.items():\n            if v not in views:\n                missing.append(col)\n                continue\n            st, pix = views[v]\n            vol = np.asarray(px[st:st + n])\n            m99 = (vol >= FLUID_P99).astype(np.uint8)\n            rec[col] = _fluid_ml(_fluid_opened(m99, pix), pix, spacing)\n        if missing:\n            rec["status"] = "no region view for " + ",".join(missing)\n    except Exception as exc:                      # the missing path, never a crash\n        rec.update(status="error: " + repr(exc)[:200], eff_sag_p99_open=None, bak_sag_p99_open=None)\n    return rec\n\n\ndef fluid_stacker_load(path, n_groups):\n    """the adopted classes of an analysis/fit_fluid_stacker.py json -> {"classes": {label index: coefficients}, ...}."""\n    raw = Path(path).read_bytes()\n    st = json.loads(raw.decode("utf-8"))\n    feat = st["feature"]\n    if feat.get("combine") != "logit":                # v4b: the deployed mix is in logit space (v4 also took "prob")\n        raise SystemExit("--fluid-stacker %s: feature.combine %r; v4b mixes the lines in logit space, the stacker\'s z must be "\n                         "combine logit" % (path, feat.get("combine")))\n    w = [float(x) for x in feat["weights"]]\n    if len(w) != n_groups or len(feat["lines"]) != n_groups:\n        raise SystemExit("--fluid-stacker %s was fitted on %d model lines, this run has %d model groups" % (path, len(w), n_groups))\n    if abs(sum(w) - 1.0) > 1e-9:\n        raise SystemExit("--fluid-stacker %s: line weights %s do not sum to 1" % (path, w))\n    cls = {}\n    for name, r in st["classes"].items():\n        if not r.get("adopt"):\n            continue\n        if r["volume_column"] not in FLUID_REGION or FLUID_REGION[r["volume_column"]] != int(r["view_id"]):\n            raise SystemExit("--fluid-stacker %s: %s uses volume %s / view %s, not a D2 column of this worker" % (\n                path, name, r["volume_column"], r["view_id"]))\n        co = {k: float(r[k]) for k in ("b0", "b_z", "b_lv", "b_miss")}\n        if not all(math.isfinite(v) for v in co.values()):\n            raise SystemExit("--fluid-stacker %s: non-finite coefficient for %s" % (path, name))\n        wc = [float(x) for x in (feat.get("weights_per_class") or {}).get(name, w)]   # v4b: per-class line weights if the fit had them\n        if len(wc) != n_groups or abs(sum(wc) - 1.0) > 1e-9:\n            raise SystemExit("--fluid-stacker %s: line weights %s of %s do not fit %d model groups" % (path, wc, name, n_groups))\n        cls[name] = dict(co, volume_column=r["volume_column"], weights=wc)\n    return {"path": str(path), "sha256": hashlib.sha256(raw).hexdigest(), "combine": feat["combine"], "weights": w, "classes": cls,\n            "lines": [l.get("run") for l in feat["lines"]], "created": st.get("created")}\n\n\ndef fluid_stack(stk, labels, zmix, fl):\n    """stacker logits of the adopted classes for one study.  v4b: zmix = the deployed line mix per class [12] (float64; v3b\'s\n    (1 - w2) z1 + w2 z2 of the groups\' fold-mean logits, one group: its fold-mean logit); fl = fluid_d2 record (None = not\n    measured).  Returns ({label: z_out}, z_feature [12])."""\n    z = np.asarray(zmix, np.float64)\n    out = {}\n    for name, co in stk["classes"].items():\n        v = None if fl is None else fl.get(co["volume_column"])\n        miss = v is None or not math.isfinite(float(v)) or float(v) < 0\n        lv = 0.0 if miss else float(np.log1p(float(v)))\n        x = np.array([z[labels.index(name)], lv, 1.0 if miss else 0.0], np.float64)\n        out[name] = co["b0"] + float(x @ np.array([co["b_z"], co["b_lv"], co["b_miss"]], np.float64))\n    return out, z\n\n\n# ---- v4c --flip-tta: the trainer\'s v9.4 mirror (compact_train_v97.py: MIRROR_PAIRS line 944, mirror_perm lines 947-971,\n# mirror_batch lines 973-993), copied; the only difference: no autograd assert (it guards the trainer\'s evaluation paths)\nMIRROR_PAIRS = (("Medial Meniscus", "Lateral Meniscus"), ("Medial OA", "Lateral OA"))   # the class pairs P exchanges; the rest fixed\n\n\ndef mirror_perm(labels, spec=None, bin_names=None):\n    """compact_train_v97.mirror_perm with LABELS passed in: the class permutation P (its own inverse) and, with a state spec and the\n    model\'s bin_names, the binary permutation; refuses a spec whose paired classes differ in their category lists."""\n    swap = {}\n    for m_, l_ in MIRROR_PAIRS:\n        swap[m_], swap[l_] = l_, m_\n    perm = [labels.index(swap.get(l, l)) for l in labels]\n    bin_perm = None\n    if spec is not None:\n        for m_, l_ in MIRROR_PAIRS:\n            if spec["categories"][m_] != spec["categories"][l_]:\n                raise SystemExit("--flip-tta needs identical state category lists for %s and %s; the spec has %s vs %s" % (\n                    m_, l_, spec["categories"][m_], spec["categories"][l_]))\n        if bin_names is not None:\n            bin_perm = []\n            for b, l in bin_names:\n                partner = (b, swap.get(l, l))\n                if partner not in bin_names:\n                    raise SystemExit("--flip-tta: the binary %s exists for %s but not for %s" % (b, l, swap[l]))\n                bin_perm.append(bin_names.index(partner))\n    return perm, bin_perm\n\n\ndef all_models_v4c(models, models2):\n    return list(models) + list(models2)\n\n\ndef mirror_windows(img, meta):\n    """compact_train_v97.mirror_batch: img [B, W, K, S, S], meta [B, W, >= 8] with the plane in column 7 (0 sagittal / 1 coronal / 2\n    axial).  Coronal and axial windows are flipped on the width axis; a sagittal window is reversed along the slice axis (its K slice\n    channels) and its rel_pos (column 2) negated; the coronal bilateral locals change sides: view id 2 <-> 3 (column 1)."""\n    assert meta.shape[-1] >= 8, "v4c flip TTA needs the plane column (meta column 7)"\n    sag = meta[..., 7].round() == 0\n    img = torch.where(sag[:, :, None, None, None], img.flip(2), img.flip(-1))\n    meta = meta.clone()\n    meta[..., 2] = torch.where(sag, -meta[..., 2], meta[..., 2])\n    v = meta[..., 1]\n    meta[..., 1] = torch.where(v == 2, torch.full_like(v, 3.), torch.where(v == 3, torch.full_like(v, 2.), v))\n    return img, meta\n\n\n# ---- v4c --state-stacker: the trainer\'s v9.7 state read-out (compact_train_v97.py: state_layout lines 1139-1157, state_probs 1174-1179)\ndef state_layout(spec, labels):\n    """compact_train_v97.state_layout with LABELS passed in: per class a block of K_c columns (its categories, slots 0 .. K_c-1), then\n    one column per binary head (the spec\'s binaries dict order, each binary\'s class list order = Net.bin_names)."""\n    bad = [l for l in labels if len(spec["prior"][l]) != len(spec["categories"][l])]\n    if bad:\n        raise SystemExit("--state-stacker: the state spec\'s prior list length differs from its category count for %s" % bad)\n    cols, j = [], 0\n    for l in labels:\n        k_ = len(spec["categories"][l])\n        cols.append({"class": l, "categories": list(spec["categories"][l]), "start": j, "stop": j + k_})\n        j += k_\n    for b, labs in spec["binaries"].items():\n        for l in labs:\n            cols.append({"binary": b, "class": l, "column": j})\n            j += 1\n    return {"columns": cols, "width": j}\n\n\ndef state_readout(model, sl, bl):\n    """compact_train_v97.state_probs: per class p = sl.softmax(-1) (the -inf invalid slots get 0) restricted to its K_c valid slots, then\n    the sigmoid of every binary logit; float32 [B, D] in the layout of the trainer\'s oof_state file."""\n    p = sl.softmax(-1)\n    return torch.cat([p[:, c, :kc] for c, kc in enumerate(model.state_k)] + [torch.sigmoid(bl)], 1).float()\n\n\ndef state_stacker_load(path, labels, layout, spec_md5=None):\n    """the adopted classes of an analysis/fit_state_stacker.py json -> per class the feature list, column indices and raw-space\n    coefficients; refuses a json whose layout is not this bundle\'s (state spec + Net binaries) or whose features do not match it, and\n    (spec_md5 given) a json fitted on models of another state spec: two specs can share the layout (rr4e and rr5 do) and differ in\n    their priors."""\n    raw = Path(path).read_bytes()\n    st = json.loads(raw.decode("utf-8"))\n    if st.get("kind") != "state_stacker_v1":\n        raise SystemExit("--state-stacker %s: kind %r, expected state_stacker_v1" % (path, st.get("kind")))\n    feat = st["feature"]\n    if [float(x) for x in feat.get("weights", [])] != [1.0] or len(feat.get("lines", [])) != 1:\n        raise SystemExit("--state-stacker %s: fitted on %s model lines (weights %s); v4c applies it to one model group" % (\n            path, len(feat.get("lines", [])), feat.get("weights")))\n    if st["layout"]["columns"] != layout["columns"] or int(st["layout"]["width"]) != layout["width"]:\n        raise SystemExit("--state-stacker %s: its state layout is not this bundle\'s (state spec / binaries differ)" % path)\n    if spec_md5 is not None and st.get("state_spec_md5") != spec_md5:\n        raise SystemExit("--state-stacker %s: fitted on models trained with state spec md5 %s; this bundle\'s spec has md5 %s" % (\n            path, st.get("state_spec_md5"), spec_md5))\n    blocks = {c_["class"]: c_ for c_ in layout["columns"] if "categories" in c_}\n    bins = [c_ for c_ in layout["columns"] if "binary" in c_]\n    cls = {}\n    for name, r in st["classes"].items():\n        if not r.get("adopt"):\n            continue\n        if name not in labels:\n            raise SystemExit("--state-stacker: unknown class %r" % name)\n        b_ = blocks[name]\n        want = ["z"] + ["state:" + c for c in b_["categories"]] + ["bin:" + c_["binary"] for c_ in bins if c_["class"] == name]\n        cols = [None] + list(range(b_["start"], b_["stop"])) + [c_["column"] for c_ in bins if c_["class"] == name]\n        if r.get("volume_column"):\n            if r["volume_column"] not in FLUID_REGION:\n                raise SystemExit("--state-stacker %s: %s uses volume %s, not a D2 column of this worker" % (path, name, r["volume_column"]))\n            want += ["lv", "miss"]\n            cols += [None, None]\n        keep = [i for i, f in enumerate(want) if f not in set(r.get("dropped") or [])]\n        want, cols = [want[i] for i in keep], [cols[i] for i in keep]\n        if list(r["features"]) != want:\n            raise SystemExit("--state-stacker %s: %s features %s are not the layout\'s %s" % (path, name, r["features"], want))\n        coef = [float(x) for x in r["coef"]]\n        if len(coef) != len(want) or not all(math.isfinite(x) for x in coef + [float(r["b0"])]):\n            raise SystemExit("--state-stacker %s: %s has %d coefficients for %d features (or non-finite ones)" % (path, name, len(coef), len(want)))\n        cls[name] = {"features": want, "columns": cols, "coef": coef, "b0": float(r["b0"]), "volume_column": r.get("volume_column")}\n    return {"path": str(path), "sha256": hashlib.sha256(raw).hexdigest(), "classes": cls, "created": st.get("created"),\n            "needs_volume": any(v["volume_column"] for v in cls.values()), "lines": feat.get("lines")}\n\n\nclass CStudies(torch.utils.data.Dataset):\n    def __init__(self, data, split, uids, tmp, code, size, eval_stride=3, views="v1", slices=3, anchor_weights=None, phys_mm=0.0,\n                 side_model=None, side_threshold=2.0, side_max_x=0.0, side_position_mm=20.0, extent_meta=False, fluid=False):\n        self.data, self.split, self.uids = str(data), split, list(uids)\n        self.fluid = bool(fluid)         # v4: measure the D2 fluid volumes on the rendered rows (--fluid-stacker)\n        # extent (M4) fold models read three more meta columns: series index, plane, slices in the series -- the same cache\n        # columns (0, 2, 10) compact_train.Studies appends with extent_meta\n        self.extent_meta = bool(extent_meta)\n        self.tmp, self.code, self.size = str(tmp), str(code), int(size)\n        self.eval_stride, self.views, self.slices = int(eval_stride), str(views), int(slices)\n        self.phys_mm = float(phys_mm)\n        self.anchor_weights = None if anchor_weights is None else str(anchor_weights)\n        if self.views in ("v3k", "v4") and not self.anchor_weights:\n            raise ValueError("views v3k need --anchor-weights (SKM-TEA checkpoint) to rebuild the knee anchors at inference")\n        self.native = None\n        # image side check (bundles that ship a side model): studies without one explicit DICOM laterality tag are\n        # scored by the orientation classifier on their canonical coronal/axial slices; a confidently mirrored study is\n        # re-rendered with the other side (native render, geometry, anchors and views all follow).  None = off.\n        self.side_model = None if side_model is None else str(side_model)\n        self.side_threshold, self.side_max_x = float(side_threshold), float(side_max_x)\n        self.side_position_mm = float(side_position_mm)\n        self.side_net = None\n        self.source_directory = True     # False: read DICOMs from the chunked competition archive under <data>/download\n\n    def __len__(self):\n        return len(self.uids)\n\n    def _setup(self):\n        for p in (self.code, str(HERE)):\n            if p not in sys.path:\n                sys.path.insert(0, p)\n        import probe_infer as PI\n        self.PI = PI\n        self.native = PI.install_native(True)\n        import px_cache\n        import compact_train\n        self.builder = px_cache.StudySlices(self.tmp, [], self.size, self.views)   # (self.slices is the window depth K)\n        self.window_rows = compact_train.window_rows          # the trainer\'s own window list (eval mode)\n        self.bank = None\n        if self.views in ("v3k", "v4"):\n            # knee-centred views: per-series geometry from the DICOM headers (mainline geometry bank, the code that\n            # built the training geometry) + SKM-TEA joint anchors (knee_anchor_bank_v2, the code of the training bank)\n            import training.geometry_bank as GB\n            import knee_anchor_bank_v2 as KB\n            self.GB, self.KB = GB, KB\n            dev = "cuda" if torch.cuda.is_available() else "cpu"\n            if dev == "cpu":\n                torch.set_num_threads(2)\n            self.bank = KB.AnchorBank(KB.load_model(self.anchor_weights, dev), dev)\n        import training.dicom_geometry as G\n        if not getattr(G, "_forced_side_wrapped", False):\n            import dataclasses\n            orig = G.decide_study_side\n\n            def decide(study, tag_values, sagittal_x, _orig=orig):\n                r = _orig(study, tag_values, sagittal_x)\n                f = FORCED_SIDE.get(str(study))\n                return r if f is None else dataclasses.replace(r, side=f, source="image_side_check")\n            G.decide_study_side = decide\n            G._forced_side_wrapped = True\n        if self.side_model:\n            import side_common as SC\n            self.SC = SC\n            sdev = "cuda" if torch.cuda.is_available() else "cpu"\n            net = SC.build_side_net()\n            net.load_state_dict(torch.load(self.side_model, map_location="cpu", weights_only=True), strict=True)\n            self.side_net, self.side_dev = net.eval().to(sdev), sdev\n\n    def _prepare(self, uid, work):\n        with contextlib.redirect_stdout(io.StringIO()):\n            self.native.prepare(self.data, work, split=self.split, study_ids=[uid], source_directory=self.source_directory)\n        if self.bank is not None or self.side_net is not None:\n            with contextlib.redirect_stdout(io.StringIO()):\n                self.GB.build(root=self.data, manifest=str(work / "manifest.json"), output=str(work / "geometry"),\n                              split=self.split, source_directory=self.source_directory)\n\n    def _native_and_geometry(self, uid, work):\n        """native render + geometry bank for one study; with a side model, the image side check (may re-render)."""\n        self.PI.OneStudySource.current = uid\n        self._prepare(uid, work)\n        if self.side_net is None:\n            return None\n        key = hashlib.sha256(uid.encode()).hexdigest()[:24]\n        with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\n            study = json.load(fh)\n        sides = {s.get("side") for s in study["series"]}\n        cur = next(iter(sides)) if len(sides) == 1 else None\n        status = set()\n        for s in study["series"]:\n            gp = work / "geometry" / (hashlib.sha256((uid + "/" + s["SeriesInstanceUID"]).encode()).hexdigest()[:24] + ".json")\n            if gp.exists():\n                status.add(json.loads(gp.read_text(encoding="utf-8")).get("laterality_status"))\n        sag = [float(np.median(s["positions"])) for s in study["series"] if s.get("plane") == "Sagittal" and s.get("positions")]\n        x = float(np.median(sag)) if sag else None\n        note = {"side": cur, "status": "|".join(sorted(str(v) for v in status)), "x": None if x is None else round(x, 1),\n                "score": None, "flipped": False}\n        if status == {"explicit_study_tags"} and cur in ("L", "R"):\n            # one explicit DICOM laterality tag stands unless the knee lies clearly on the other side of the scanner\n            # (10-07: 18 training studies had the tag contradicted by both the position and the images)\n            pos = None if x is None or abs(x) < self.side_position_mm else ("L" if x >= 0 else "R")\n            if pos is None or pos == cur:\n                return note\n            note["tag_contradicted_by_position"] = True\n        if self.side_max_x > 0 and cur in ("L", "R") and x is not None and abs(x) >= self.side_max_x:\n            return note                                   # far from the scanner centre: the sagittal IPP x rule holds\n        ims = []\n        for s in study["series"]:\n            if s.get("plane") not in self.SC.SIDE_PLANES:\n                continue\n            p = Path(s["path"])\n            if not p.exists():\n                p = work / "series" / os.path.basename(str(s["path"]).replace(chr(92), "/"))\n            if p.exists():\n                ims.append(self.SC.side_inputs(np.load(p, mmap_mode="r")))\n        ims = [v for v in ims if len(v)]\n        if not ims:\n            return note\n        xb = torch.from_numpy(np.concatenate(ims)).float().div_(255.)[:, None]\n        with torch.inference_mode():\n            z = self.side_net(((xb - 0.45) / 0.25).to(self.side_dev)).float().squeeze(1)\n        score = float(z.mean())\n        note["score"] = round(score, 3)\n        if score < -self.side_threshold:\n            new = "R" if cur in (None, "L") else "L"\n            FORCED_SIDE[uid] = new\n            shutil.rmtree(work, ignore_errors=True)\n            self._prepare(uid, work)\n            with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\n                got = {s.get("side") for s in json.load(fh)["series"]}\n            if got != {new}:\n                raise ValueError("forced side %s not applied (%s)" % (new, got))\n            note.update(flipped=True, new_side=new)\n        return note\n\n    def rows(self, uid, forced_side=None):\n        """training-cache override builder: the same native -> geometry -> anchors -> views path for one study, with an\n        optional forced side; returns (StudySlices.build output, set of sides in the rendered study)."""\n        if self.native is None:\n            self._setup()\n        key = hashlib.sha256(uid.encode()).hexdigest()[:24]\n        work = Path(self.tmp) / ("r_" + key[:16])\n        if forced_side is not None:\n            FORCED_SIDE[uid] = forced_side\n        try:\n            self.PI.OneStudySource.current = uid\n            with contextlib.redirect_stdout(io.StringIO()):\n                self.native.prepare(self.data, work, split=self.split, study_ids=[uid], source_directory=self.source_directory)\n                self.GB.build(root=self.data, manifest=str(work / "manifest.json"), output=str(work / "geometry"),\n                              split=self.split, source_directory=self.source_directory)\n            self.builder.cache = str(work)\n            with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\n                study = json.load(fh)\n            if self.bank is not None:\n                rec = self.bank.record(uid, study, geo_dir=str(work / "geometry"))\n                (work / "anchors").mkdir(exist_ok=True)\n                with open(work / "anchors" / (self.KB.study_key(uid) + ".json"), "w", encoding="utf-8") as fh:\n                    json.dump(rec, fh)\n                self.builder.anchors = str(work / "anchors")\n            return self.builder.build(uid), {s.get("side") for s in study["series"]}\n        finally:\n            FORCED_SIDE.pop(uid, None)\n            shutil.rmtree(work, ignore_errors=True)\n\n    def __getitem__(self, i):\n        uid = self.uids[i]\n        t0 = time.perf_counter()\n        work = Path(self.tmp) / ("s_" + hashlib.sha256(uid.encode()).hexdigest()[:16])\n        t1 = t0\n        tf, fl = 0.0, None                                    # v4: fluid measurement seconds / record\n        try:\n            if self.native is None:\n                self._setup()\n            side_note = self._native_and_geometry(uid, work)\n            t1 = time.perf_counter()\n            self.builder.cache = str(work)\n            if self.bank is not None:\n                key = self.KB.study_key(uid)\n                with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\n                    study = json.load(fh)\n                rec = self.bank.record(uid, study, geo_dir=str(work / "geometry"))\n                (work / "anchors").mkdir(exist_ok=True)\n                with open(work / "anchors" / (key + ".json"), "w", encoding="utf-8") as fh:\n                    json.dump(rec, fh)\n                self.builder.anchors = str(work / "anchors")\n                anchor_note = {"method": rec["method"], "why": rec["why"][:4]}\n            else:\n                anchor_note = None\n            t2 = time.perf_counter()\n            item = self.builder.build(uid)\n            px, meta = item["px"].numpy(), item["meta"].numpy()\n            if self.fluid:                                    # v4: D2 on the rows just rendered (never raises)\n                t3 = time.perf_counter()\n                fl = fluid_d2(px, meta, item["blocks"])\n                tf = time.perf_counter() - t3\n            blocks = [{"n_slices": b["n_slices"], "views": [(0, v, st) for v, st in b["views"]]} for b in item["blocks"]]\n            cand = self.window_rows(blocks, lambda f, r: meta[r], "eval", self.slices, self.eval_stride, phys_mm=self.phys_mm)\n            if not cand:\n                raise ValueError("no usable window")\n            rows = np.array(cand, np.int64)                 # (file, centre row, K slice rows)\n            img = np.stack([px[rows[:, 2 + d]] for d in range(self.slices)], 1)\n            m = meta[rows[:, 1]]\n            meta6 = np.stack([m[:, 1], m[:, 3], m[:, 4], m[:, 5], m[:, 6], m[:, 7]], 1).astype(np.float32)\n            if self.extent_meta:\n                meta6 = np.concatenate([meta6, np.stack([m[:, 0], m[:, 2], m[:, 10]], 1).astype(np.float32)], 1)\n            out = {"uid": uid, "error": None, "images": torch.from_numpy(img), "meta": torch.from_numpy(meta6), "anchor": anchor_note,\n                   "side": side_note}\n        except Exception as exc:\n            t2 = locals().get("t2", t1)\n            out = {"uid": uid, "error": repr(exc)[:300], "images": torch.zeros((0, self.slices, self.size, self.size), dtype=torch.uint8),\n                   "meta": torch.zeros((0, 9 if self.extent_meta else 6), dtype=torch.float32), "anchor": None}\n        finally:\n            shutil.rmtree(work, ignore_errors=True)\n        out["seconds"] = [round(t1 - t0, 3), round(time.perf_counter() - t2 - tf, 3), round(t2 - t1, 3)]   # decode, render, anchor\n        if self.fluid:\n            out["fluid"] = fl\n            out["fluid_s"] = round(tf, 4)\n        return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--split", default="test")\n    ap.add_argument("--code", required=True)\n    ap.add_argument("--models", required=True, help="folder with <arm>_s<seed>_f<fold>.pt state dicts")\n    ap.add_argument("--arm", required=True)\n    ap.add_argument("--backbone", default="resnet34")\n    ap.add_argument("--init", default=None, help="ofl_l backbone: the public OFL-L weights (safetensors) the fold models are rebuilt on")\n    ap.add_argument("--freeze-blocks", type=int, default=16)\n    ap.add_argument("--size", type=int, default=288)\n    ap.add_argument("--crop", type=int, default=256)\n    ap.add_argument("--views", default="v1", help="px_cache view set the fold models were trained with (v1 / v2)")\n    ap.add_argument("--state-spec", default=None, help="state_spec json of the report-state construct the fold models were trained with")\n    ap.add_argument("--slices", type=int, default=3, help="adjacent slices per window the fold models were trained with")\n    ap.add_argument("--anchor-weights", default=None, help="views v3k: the SKM-TEA checkpoint (anatomy.pt of the code dataset) for the knee anchors")\n    ap.add_argument("--phys-mm", type=float, default=0.0, help="window spacing in mm the fold models were trained with (compact_train --phys-mm)")\n    ap.add_argument("--eval-stride", type=int, default=3, help="window centres: every k-th slice (3 = training-time validation default; 1 = every slice)")\n    ap.add_argument("--assignment", default=None)\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--tmp", default=None)\n    ap.add_argument("--amp", default="auto", choices=["auto", "fp16", "bf16", "fp32"])\n    ap.add_argument("--chunk", type=int, default=None, help="encoder windows per step (default 192; 96 when a CoAtNet group "\n                    "is loaded: CoAtNet-2 at 192 does not fit a 16 GB T4)")\n    ap.add_argument("--workers", type=int, default=2)\n    ap.add_argument("--prefetch", type=int, default=1)\n    ap.add_argument("--item-timeout", type=float, default=900.)\n    ap.add_argument("--deadline", type=float, default=0.)\n    ap.add_argument("--dump-logits", default=None)\n    ap.add_argument("--models2", default=None, help="second model group (another architecture), run on the same windows")\n    ap.add_argument("--arm2", default=None, help="file prefix of the second group (default: --arm)")\n    ap.add_argument("--backbone2", default=None)\n    ap.add_argument("--w2", default=None, help="weight of group 2 per class: one number or 12 comma-separated (label order); "\n                    "v3b: logit p = (1 - w) z_group1 + w z_group2 (z = fold-mean logit of the group)")\n    ap.add_argument("--side-model", default=None, help="orientation classifier (side_common.build_side_net state dict); off when absent")\n    ap.add_argument("--side-threshold", type=float, default=2.0, help="flip a study only when its mean logit < -threshold")\n    ap.add_argument("--side-max-x", type=float, default=0.0, help="> 0: skip the check for untagged studies this far (mm) from x = 0")\n    ap.add_argument("--side-position-mm", type=float, default=20.0, help="a DICOM tag is re-checked only when the knee lies this far on the other side")\n    ap.add_argument("--crop2", type=int, default=None, help="v3: centre crop of group 2 (--models2), cut from the same decoded "\n                    "windows; group 2 is built with this crop (default: --crop)")\n    ap.add_argument("--compile", action="store_true", help="v3: torch.compile every fold model\'s encoder (checked against eager, "\n                    "eager fallback on any error or a difference above --compile-tol)")\n    ap.add_argument("--compile-mode", default="default", help="v3: torch.compile mode (inductor backend); \'default\' = the 10-08 T4 benchmark")\n    ap.add_argument("--compile-backend", default="inductor", help="v3: torch.compile backend; inductor on Kaggle (other values are "\n                    "for local plumbing tests on machines without Triton, e.g. aot_eager)")\n    ap.add_argument("--compile-tol", type=float, default=1e-3, help="v3: largest accepted |p_compiled - p_eager| of the first real "\n                    "study\'s study-level probabilities; above it the compiled path is kept only when an fp32 forward shows it is no "\n                    "further from the fp32 model than eager (+ this tolerance), else every model runs eager")\n    ap.add_argument("--trainer-md5", default=None, help="v3: md5 of the bundle\'s pinned compact_train.py; the worker stops when "\n                    "the imported compact_train.py differs")\n    ap.add_argument("--fluid-stacker", default=None, help="v4: coefficient json of analysis/fit_fluid_stacker.py; the classes it "\n                    "marks adopt get the stacker logit (model logit + D2 fluid volume measured on the rendered rows)")\n    ap.add_argument("--fluid-dump", default=None, help="v4: per-study fluid / stacker csv (default <output>/fluid_stacker.csv)")\n    ap.add_argument("--flip-tta", action="store_true", help="v4c: every fold model also scores the mirrored windows (the trainer\'s v9.4 "\n                    "--mirror transform); outputs re-indexed (Medial <-> Lateral Meniscus / OA), the two passes averaged in logit space, "\n                    "MCL from the plain pass")\n    ap.add_argument("--state-stacker", default=None, help="v4c: coefficient json of analysis/fit_state_stacker.py (one model group); the "\n                    "classes it adopts get b0 + coef . [z, state probabilities, binaries (, lv, miss)]")\n    ap.add_argument("--state-dump", default=None, help="v4c: per-study state-stacker csv (default <output>/state_stacker.csv)")\n    ap.add_argument("--dump-state", action="store_true", help="v4c: with --dump-logits, also the per-model state read-out (and the "\n                    "mirrored pass with --flip-tta)")\n    a = ap.parse_args()\n    if a.chunk is None:\n        a.chunk = 96 if any("coatnet" in (b or "") for b in (a.backbone, a.backbone2)) else 192\n    out = Path(a.output)\n    out.mkdir(parents=True, exist_ok=True)\n    tmp = Path(a.tmp or (out / "tmp"))\n    tmp.mkdir(parents=True, exist_ok=True)\n    t_start = time.time()\n    if a.assignment:\n        uids = json.loads(Path(a.assignment).read_text(encoding="utf-8"))\n    else:\n        with open(Path(a.data) / (a.split + ".csv"), encoding="utf-8") as fh:\n            uids = [r[UIDK] for r in csv.DictReader(fh)]\n    if len(uids) != len(set(uids)):\n        raise SystemExit("duplicate study ids")\n\n    dev = torch.device("cuda")\n    for p in (a.code, str(HERE)):\n        if p not in sys.path:\n            sys.path.insert(0, p)\n    import timm\n    import compact_train as CT\n    LABELS = CT.LABELS\n    trainer_md5 = hashlib.md5(Path(CT.__file__).read_bytes()).hexdigest()       # v3: the trainer module actually imported\n    if a.trainer_md5 and trainer_md5 != a.trainer_md5.lower():\n        raise SystemExit("compact_train.py at %s has md5 %s, the bundle pins %s" % (CT.__file__, trainer_md5, a.trainer_md5))\n    cap = torch.cuda.get_device_capability()[0]\n    amp = a.amp if a.amp != "auto" else ("bf16" if cap >= 8 else "fp16")\n    state_spec = json.load(open(a.state_spec, encoding="utf-8")) if a.state_spec else None\n    crop2 = a.crop if a.crop2 is None else int(a.crop2)                          # v3: group 2\'s own centre crop\n    if a.crop2 is not None and not 0 < crop2 <= a.size:\n        raise SystemExit("--crop2 %d: must be in (0, --size %d]" % (crop2, a.size))\n    net_has_syn = "syn_decomp" in inspect.signature(CT.Net.__init__).parameters\n\n    def load_group(folder, arm, backbone, crop):\n        fs = sorted(Path(folder).glob(arm + "_s*_f*.pt"))\n        if not fs:\n            raise SystemExit("no model files for arm %s in %s" % (arm, folder))\n        ms, n_sp = [], 0\n        for p in fs:\n            sd = torch.load(p, map_location="cpu", weights_only=True)\n            extent = any(k.split(".")[-1] in ("ext_u", "ext_view", "ext_gamma") for k in sd)   # M4 fold model (OA extent)\n            if extent:\n                N_EXTENT[0] += 1\n            spatial = any(k == "sp_q" or k.endswith(".sp_q") for k in sd)       # P2 fold model (per-class spatial attention)\n            n_sp += int(spatial)\n            # v3: trainer v9.2+ --syn-decomp adds one scalar parameter, Net.syn_beta (z_syn = z_spec + beta * sg(z_eff))\n            syn = any(k == "syn_beta" or k.endswith(".syn_beta") for k in sd)\n            N_SYN[0] += int(syn)\n            kw = {}\n            if spatial:\n                kw["spatial"] = True\n            if extent:\n                kw["extent"] = True\n            if syn:\n                if not net_has_syn:\n                    raise SystemExit("%s was trained with --syn-decomp (key syn_beta) but the imported compact_train.py (%s, md5 %s) "\n                                     "has no Net(syn_decomp=...): pin trainer v9.3 (compact_train_v93.py)" % (p.name, CT.__file__, trainer_md5))\n                kw["syn_decomp"] = True\n            m = CT.Net(backbone, a.init if backbone == CT.OFL_BACKBONE else None, freeze_blocks=a.freeze_blocks, crop=crop,\n                       state_spec=state_spec, in_chans=a.slices, **kw)\n            if backbone == CT.OFL_BACKBONE:\n                m.load_deploy_state_dict(sd)          # fold model = public weights + trained tail/head\n            else:\n                m.load_state_dict(sd, strict=True)\n            ms.append(m.eval().requires_grad_(False).to(dev).to(memory_format=torch.channels_last))\n        return fs, ms, n_sp\n\n    N_EXTENT = [0]\n    N_SYN = [0]\n    files, models, n_spatial = load_group(a.models, a.arm, a.backbone, a.crop)\n    n_syn1 = N_SYN[0]\n    files2, models2, n_spatial2, W2 = [], [], 0, None\n    if a.models2:\n        if not a.backbone2 or a.backbone2 == CT.OFL_BACKBONE or a.backbone == CT.OFL_BACKBONE:\n            raise SystemExit("--models2 needs --backbone2 (timm backbones only, no OFL shared trunk)")\n        files2, models2, n_spatial2 = load_group(a.models2, a.arm2 or a.arm, a.backbone2, crop2)\n        w = [float(x) for x in (a.w2 or "0.5").split(",")]\n        w = w * len(LABELS) if len(w) == 1 else w\n        if len(w) != len(LABELS) or not all(0.0 <= x <= 1.0 for x in w):\n            raise SystemExit("--w2: one weight or %d weights in [0, 1]" % len(LABELS))\n        W2 = torch.tensor(w, dtype=torch.float64, device=dev)\n    shared_trunk = a.backbone == CT.OFL_BACKBONE and len(models) > 1\n    if shared_trunk:\n        # all fold models share the frozen trunk bit-for-bit; keep one copy and run it once per window batch\n        sd0 = models[0].enc.state_dict()\n        for m in models[1:]:\n            sdm = m.enc.state_dict()\n            for k in models[0].enc.frozen_keys:\n                if not torch.equal(sdm[k], sd0[k]):\n                    raise SystemExit("fold models do not share the frozen trunk: " + k)\n    # ---- v4 --fluid-stacker: checked before any study is read (a mismatch is a configuration error, not a per-study failure)\n    STK = None\n    if a.fluid_stacker:\n        if a.size != FLUID_SIZE or a.views != "v4":\n            raise SystemExit("--fluid-stacker needs --size %d --views v4 (the D2 definition reads views 0/4/5 of the 384 px rows); "\n                             "got --size %d --views %s" % (FLUID_SIZE, a.size, a.views))\n        STK = fluid_stacker_load(a.fluid_stacker, 1 + int(bool(models2)))\n        for name in STK["classes"]:\n            if name not in LABELS:\n                raise SystemExit("--fluid-stacker: unknown class %r" % name)\n        for name, co in STK["classes"].items():       # v4b: the stacker was fitted on the line mix this bundle deploys\n            j = LABELS.index(name)\n            want = [1.0 - float(W2[j]), float(W2[j])] if models2 else [1.0]\n            if len(co["weights"]) != len(want) or max(abs(x - y) for x, y in zip(co["weights"], want)) > 1e-12:\n                raise SystemExit("--fluid-stacker %s: %s was fitted on the line mix %s, this bundle deploys %s (--w2): refit the stacker "\n                                 "on the deployed mix or deploy its weights" % (STK["path"], name, co["weights"], want))\n        print("FLUID_READY", json.dumps({"json": STK["path"], "sha256": STK["sha256"][:16], "combine": STK["combine"],\n                                         "weights": STK["weights"], "adopted": {k: {c: round(v[c], 5) for c in ("b0", "b_z", "b_lv", "b_miss")}\n                                                                                 for k, v in STK["classes"].items()},\n                                         "volume": {k: v["volume_column"] for k, v in STK["classes"].items()},\n                                         "z": "v3b line mix", "weights_per_class": {k: v["weights"] for k, v in STK["classes"].items()}}), flush=True)\n    # ---- v4c --flip-tta / --state-stacker / --dump-state: checked before any study is read\n    FLIP, SST, AUX = None, None, False\n    if a.flip_tta:\n        if shared_trunk or a.backbone == CT.OFL_BACKBONE:\n            raise SystemExit("--flip-tta: timm backbones only (not the OFL shared trunk)")\n        if N_EXTENT[0] == 0:\n            raise SystemExit("--flip-tta needs the plane column, which the worker adds to the window meta for extent fold models only")\n        if state_spec is not None and any(tuple(m.bin_names) != tuple(models[0].bin_names) for m in all_models_v4c(models, models2)):\n            raise SystemExit("--flip-tta: the fold models\' binary heads differ")\n        perm, bin_perm = mirror_perm(LABELS, state_spec, list(models[0].bin_names) if state_spec is not None else None)\n        FLIP = {"perm": perm, "bin_perm": bin_perm, "mcl": LABELS.index("MCL")}\n        print("FLIP_TTA_READY", json.dumps({"perm": perm, "mcl_plain_pass_only": True}), flush=True)\n    if a.state_stacker or a.dump_state:\n        if state_spec is None:\n            raise SystemExit("--state-stacker / --dump-state read out the state head: the bundle has no state spec")\n        if shared_trunk or a.backbone == CT.OFL_BACKBONE:\n            raise SystemExit("--state-stacker / --dump-state: timm backbones only")\n        LAYOUT = state_layout(state_spec, LABELS)\n        for m in all_models_v4c(models, models2):     # compact_train_v97.state_slots_check: the head\'s slots are the layout\'s\n            valid = m.state_valid.detach().cpu()\n            for c, b_ in enumerate([c_ for c_ in LAYOUT["columns"] if "categories" in c_]):\n                kc = b_["stop"] - b_["start"]\n                if b_["class"] != LABELS[c] or m.state_k[c] != kc or not bool(valid[c, :kc].all()) or bool(valid[c, kc:].any()):\n                    raise SystemExit("--state-stacker: a fold model\'s state slots for %s are not the spec\'s %d categories" % (LABELS[c], kc))\n            if [(c_["binary"], c_["class"]) for c_ in LAYOUT["columns"] if "binary" in c_] != [tuple(x) for x in m.bin_names]:\n                raise SystemExit("--state-stacker: a fold model\'s binaries are not the spec\'s")\n        AUX = True\n    MCL_STATE_COLS = []\n    if AUX:\n        MCL_STATE_COLS = [i for c_ in LAYOUT["columns"] if c_["class"] == "MCL"\n                          for i in (range(c_["start"], c_["stop"]) if "categories" in c_ else [c_["column"]])]\n    if a.state_stacker:\n        if models2:\n            raise SystemExit("--state-stacker: one model group only (the json is fitted on one line\'s z and state read-out)")\n        SST = state_stacker_load(a.state_stacker, LABELS, LAYOUT, hashlib.md5(Path(a.state_spec).read_bytes()).hexdigest())\n        both = sorted(set(SST["classes"]) & set(STK["classes"] if STK is not None else []))\n        if both:\n            raise SystemExit("--state-stacker and --fluid-stacker both adopt %s: a class is replaced by one stacker (fit it jointly)" % both)\n        if SST["needs_volume"] and (a.size != FLUID_SIZE or a.views != "v4"):\n            raise SystemExit("--state-stacker with volume terms needs --size %d --views v4" % FLUID_SIZE)\n        print("STATE_STACKER_READY", json.dumps({"json": SST["path"], "sha256": SST["sha256"][:16], "adopted": {\n            k: {"features": len(v["features"]), "volume": v["volume_column"]} for k, v in SST["classes"].items()}}), flush=True)\n    print("MODEL_READY", json.dumps({"gpu": torch.cuda.get_device_name(), "amp": amp, "models": len(models),\n                                     "models2": len(models2), "spatial_models": n_spatial + n_spatial2, "extent_models": N_EXTENT[0],\n                                     "syn_decomp_models": N_SYN[0], "crop": a.crop, "crop2": crop2 if models2 else None,\n                                     "compile_requested": bool(a.compile), "trainer_md5": trainer_md5,\n                                     "studies": len(uids), "seconds": round(time.time() - t_start, 1)}), flush=True)\n    off = (a.size - a.crop) // 2\n    off2 = (a.size - crop2) // 2\n\n    def _forward(m, img, meta, mask, mode):\n        """one fold model on a study\'s windows (v2\'s run_model, unchanged)."""\n        if mode == "fp32":\n            return m(img, meta, mask, chunk=a.chunk).float()\n        with torch.autocast("cuda", dtype=torch.float16 if mode == "fp16" else torch.bfloat16):\n            return m(img, meta, mask, chunk=a.chunk).float()\n\n    # ---- v3 --compile: each fold model\'s encoder through torch.compile; EAGER holds the eager encoder of every model that\n    # currently runs a compiled one (empty = the v2 path, bit for bit)\n    COMPILE = {"requested": bool(a.compile), "active": False, "mode": a.compile_mode, "backend": a.compile_backend,\n               "tol_prob": a.compile_tol, "checked": False, "fallback": None, "shapes": None, "check_seconds": None,\n               "decision": None, "study_dprob": None, "fp32_err_compiled": None, "fp32_err_eager": None,\n               "max_abs_dlogit": None, "max_abs_dprob": None, "per_group": None, "graphs": None}\n    EAGER = {}\n    all_models = list(models) + list(models2)\n\n    def set_eager(reason):\n        for m in all_models:\n            if id(m) in EAGER:\n                m.enc = EAGER[id(m)]\n        EAGER.clear()\n        COMPILE["active"] = False\n        COMPILE["fallback"] = str(reason)[:400]\n        try:\n            import torch._dynamo as _dynamo\n            _dynamo.reset()\n        except Exception:\n            pass\n        print("COMPILE_FALLBACK", json.dumps({"reason": COMPILE["fallback"], "every model runs its eager encoder": True}), flush=True)\n\n    @contextlib.contextmanager\n    def eager_encoder(m):\n        """this fold model on its eager encoder for the duration (the fp32 rescue, the eager side of the compile check)."""\n        enc = m.enc\n        if id(m) in EAGER:\n            m.enc = EAGER[id(m)]\n        try:\n            yield\n        finally:\n            if id(m) in EAGER:\n                m.enc = enc\n\n    def _forward_aux(m, img, meta, mask, mode):\n        """v4c: _forward plus the state / binary logits of the same forward (Net.forward return_aux)."""\n        if mode == "fp32":\n            z, sl, bl = m(img, meta, mask, chunk=a.chunk, return_aux=True)\n        else:\n            with torch.autocast("cuda", dtype=torch.float16 if mode == "fp16" else torch.bfloat16):\n                z, sl, bl = m(img, meta, mask, chunk=a.chunk, return_aux=True)\n        return z.float(), sl.float(), bl.float()\n\n    def run_model_aux(m, img, meta, mask, mode):\n        """v4c: run_model with the aux outputs (same compiled / eager handling)."""\n        if not EAGER:\n            return _forward_aux(m, img, meta, mask, mode)\n        if mode == "fp32":\n            with eager_encoder(m):\n                return _forward_aux(m, img, meta, mask, mode)\n        try:\n            return _forward_aux(m, img, meta, mask, mode)\n        except Exception as exc:\n            set_eager("compiled forward failed: " + repr(exc))\n            return _forward_aux(m, img, meta, mask, mode)\n\n    def run_model(m, img, meta, mask, mode):\n        if not EAGER:                                         # no compiled encoder installed: v2\n            return _forward(m, img, meta, mask, mode)\n        if mode == "fp32":                                    # the fp32 rescue of a non-finite autocast output stays eager\n            with eager_encoder(m):\n                return _forward(m, img, meta, mask, mode)\n        try:\n            return _forward(m, img, meta, mask, mode)\n        except Exception as exc:                              # e.g. a recompile that fails: eager from here on, this call again\n            set_eager("compiled forward failed: " + repr(exc))\n            return _forward(m, img, meta, mask, mode)\n\n    if a.compile:\n        if shared_trunk or a.backbone == CT.OFL_BACKBONE:\n            COMPILE["fallback"] = "not applied: OFL shared-trunk path (timm backbones only)"\n            print("COMPILE_SKIPPED", json.dumps({"reason": COMPILE["fallback"]}), flush=True)\n        else:\n            try:\n                import torch._dynamo as _dynamo\n                # every (encoder, batch shape) is one compiled entry; 6 shapes x 8 fold models must never hit the limit (a hit\n                # silently runs that frame eagerly: no wrong result, only lost speed)\n                for name_, lim_ in (("cache_size_limit", 64), ("recompile_limit", 64), ("accumulated_cache_size_limit", 2048),\n                                    ("accumulated_recompile_limit", 2048)):\n                    if hasattr(_dynamo.config, name_):\n                        setattr(_dynamo.config, name_, max(lim_, int(getattr(_dynamo.config, name_))))\n                ckw = dict(dynamic=False, mode=a.compile_mode) if a.compile_backend == "inductor" else dict(dynamic=False, backend=a.compile_backend)\n                for m in all_models:\n                    EAGER[id(m)] = m.enc\n                    m.enc = torch.compile(EAGER[id(m)], **ckw)   # as kaggle/inferbench: m.enc = torch.compile(enc, mode=..., dynamic=False)\n                COMPILE["active"] = True\n            except Exception as exc:\n                set_eager("torch.compile setup failed: " + repr(exc))\n\n    def final_prob(z1, z2):\n        """study-level probabilities from per-model logits [n, C], formed exactly as score() forms them."""\n        if z2 is None:\n            return torch.sigmoid(z1.mean(0).double())\n        z = (1 - W2) * z1.mean(0).double() + W2 * z2.mean(0).double()         # v3b: the two groups mixed in logit space\n        return torch.sigmoid(z).clamp(1e-12, 1 - 1e-12)\n\n    def compare(zc, ze, gw, key):\n        """per-model compiled vs eager: finiteness must agree; the largest |dlogit| / |dp| goes into gw."""\n        if not torch.equal(torch.isfinite(zc), torch.isfinite(ze)):\n            raise ValueError("compiled and eager outputs differ in finiteness")\n        fin = torch.isfinite(zc) & torch.isfinite(ze)\n        if bool(fin.any()):\n            gw[key + "_dlogit"] = max(gw.get(key + "_dlogit", 0.0), float((zc - ze).abs()[fin].max()))\n            gw[key + "_dprob"] = max(gw.get(key + "_dprob", 0.0),\n                                     float((torch.sigmoid(zc.double()) - torch.sigmoid(ze.double())).abs()[fin].max()))\n\n    def compile_check(item):\n        """Before the first study is scored: (1) compile every encoder batch shape this run can produce (Net.forward pads the\n        windows to a multiple of 32 and cuts them into --chunk steps) on synthetic windows, each fold model\'s compiled output\n        against its eager output there only as a broken-kernel bound (finiteness agrees, |dp| <= COMPILE_SANITY_DP); (2) the\n        first real study through every fold model compiled and eager: the study-level probabilities (what the submission\n        holds) must agree within --compile-tol; if they do not, an fp32 forward of the same study decides -- the compiled path\n        is kept only when it is no further from the fp32 model than the eager path we deploy (+ --compile-tol).  Any error or\n        a failed gate puts every model back on its eager encoder (COMPILE_FALLBACK)."""\n        t0 = time.perf_counter()\n        COMPILE["checked"] = True\n        try:\n            if a.chunk <= 0 or a.chunk % 32:\n                raise ValueError("--compile needs --chunk to be a positive multiple of 32 (got %d)" % a.chunk)\n            # studies of 32, 64, ..., chunk windows produce every encoder batch a run can see: chunk-sized steps plus a\n            # last step of 32 .. chunk windows (Net.forward pads to a multiple of 32)\n            sizes = list(range(32, a.chunk + 1, 32))\n            gen = torch.Generator().manual_seed(20261010)\n            ncol = int(item["meta"].shape[1])\n            full = item["images"].to(dev, non_blocking=True)\n            meta_r = item["meta"].to(dev)[None]\n            mask_r = torch.ones((1, full.shape[0]), dtype=torch.bool, device=dev)\n            groups = [("group1", models, a.crop, off)] + ([("group2", models2, crop2, off2)] if models2 else [])\n            per_group, real = [], {}\n            d_final = err_c = err_e = None\n            with torch.inference_mode():\n                for gname, ms, cg, og in groups:\n                    gw = {"group": gname, "crop": cg}\n                    for s in sizes:\n                        img_s = torch.randint(0, 256, (1, s, a.slices, cg, cg), generator=gen, dtype=torch.uint8).to(dev)\n                        meta_s = torch.zeros((1, s, ncol), dtype=torch.float32, device=dev)\n                        mask_s = torch.ones((1, s), dtype=torch.bool, device=dev)\n                        for m in ms:\n                            zc = _forward(m, img_s, meta_s, mask_s, amp)\n                            with eager_encoder(m):\n                                ze = _forward(m, img_s, meta_s, mask_s, amp)\n                            compare(zc, ze, gw, "synthetic")\n                    if not gw.get("synthetic_dprob", 0.0) <= COMPILE_SANITY_DP:\n                        raise ValueError("%s synthetic windows: compiled vs eager |dp| %.3g > %.2g (a broken compiled kernel?)" % (\n                            gname, gw["synthetic_dprob"], COMPILE_SANITY_DP))\n                    img_r = full[None, :, :, og:og + cg, og:og + cg]                     # the first real study\n                    zc_l, ze_l = [], []\n                    for m in ms:\n                        zc = _forward(m, img_r, meta_r, mask_r, amp)[0]\n                        with eager_encoder(m):\n                            ze = _forward(m, img_r, meta_r, mask_r, amp)[0]\n                        compare(zc, ze, gw, "study")\n                        zc_l.append(zc)\n                        ze_l.append(ze)\n                    real[gname] = (torch.stack(zc_l), torch.stack(ze_l), img_r, ms)\n                    per_group.append({k: (round(v, 7) if isinstance(v, float) else v) for k, v in gw.items()})\n                pc = final_prob(real["group1"][0], real["group2"][0] if models2 else None)\n                pe = final_prob(real["group1"][1], real["group2"][1] if models2 else None)\n                d_final = float((pc - pe).abs().max())\n                ok, why = compile_gate(d_final, a.compile_tol)\n                if ok is None:                                # tie-break: an fp32 forward of the same study, eager encoders\n                    z32 = {}\n                    for gname, (_, _, img_r, ms) in real.items():\n                        zz = []\n                        for m in ms:\n                            with eager_encoder(m):\n                                zz.append(_forward(m, img_r, meta_r, mask_r, "fp32")[0])\n                        z32[gname] = torch.stack(zz)\n                    p32 = final_prob(z32["group1"], z32.get("group2"))\n                    err_c, err_e = float((pc - p32).abs().max()), float((pe - p32).abs().max())\n                    ok, why = compile_gate(d_final, a.compile_tol, err_c, err_e)\n            torch.cuda.synchronize()\n            COMPILE.update(shapes=sizes, per_group=per_group, decision=why, study_dprob=round(d_final, 7),\n                           fp32_err_compiled=None if err_c is None else round(err_c, 7),\n                           fp32_err_eager=None if err_e is None else round(err_e, 7),\n                           max_abs_dlogit=max(g.get("study_dlogit", 0.0) for g in per_group),\n                           max_abs_dprob=max(g.get("study_dprob", 0.0) for g in per_group))\n            try:\n                from torch._dynamo.utils import counters as _counters\n                COMPILE["graphs"] = int(_counters["stats"]["unique_graphs"])\n            except Exception:\n                pass\n            if not ok:\n                set_eager("compile check: " + why)\n            else:\n                print("COMPILE_READY", json.dumps({k: COMPILE[k] for k in ("mode", "backend", "shapes", "decision", "study_dprob",\n                                                                        "fp32_err_compiled", "fp32_err_eager", "per_group", "graphs")}), flush=True)\n        except Exception as exc:\n            set_eager("compile check failed: " + repr(exc))\n        COMPILE["check_seconds"] = round(time.perf_counter() - t0, 1)\n\n    def heads_from_tokens(m, tok, meta, mask):\n        """compact_train.Net.head: encoder output -> class logits (the same code the trainer used, state construct included)."""\n        return m.head(tok, meta, mask)\n\n    def run_shared(img, meta, mask, mode):\n        """OFL fold models: frozen trunk once per window chunk, then every fold\'s trained tail + head."""\n        x = img[mask]\n        m0 = models[0]\n        tails = [[] for _ in models]\n        step = a.chunk or len(x)\n        ctx = contextlib.nullcontext() if mode == "fp32" else torch.autocast("cuda", dtype=torch.float16 if mode == "fp16" else torch.bfloat16)\n        with ctx:\n            for s0 in range(0, len(x), step):\n                xs = x[s0:s0 + step].float().div_(255.)\n                xs = ((xs - m0.mean) / m0.std).contiguous(memory_format=torch.channels_last)\n                tr = m0.enc.trunk(xs)\n                for i, m in enumerate(models):\n                    tails[i].append(m.enc.tail(tr))\n            zs = [heads_from_tokens(m, torch.cat(tails[i]), meta, mask) for i, m in enumerate(models)]\n        return torch.stack([z[0].float() for z in zs])\n\n    EXTRA = {}                                                # v4c: per-study side outputs of score_v4c (dump, state stacker)\n\n    def one_pass(m, img, meta, mask):\n        """v4c: one fold model\'s forward with the v4b fp32 rescue; (z [1, 12], state read-out [1, D] or None)."""\n        if AUX:\n            z, sl, bl = run_model_aux(m, img, meta, mask, amp)\n            if not torch.isfinite(z).all():\n                z, sl, bl = run_model_aux(m, img, meta, mask, "fp32")\n            return z, sl, bl\n        z = run_model(m, img, meta, mask, amp)\n        if not torch.isfinite(z).all():\n            z = run_model(m, img, meta, mask, "fp32")\n        return z, None, None\n\n    def score_v4c(ms, img, meta, mask, group=1):\n        """v4c: per fold model the plain pass (and with --flip-tta the mirrored pass, re-indexed by P and averaged in logit space, MCL\n        from the plain pass); the state read-out of the same forwards (the mirrored one re-indexed by P / bin_perm, averaged, MCL\'s\n        block and binaries from the plain pass).  Returns the per-model logits the study is scored with; EXTRA keeps the rest."""\n        out, plain, mirr, st_dep, st_plain, st_mirr = [], [], [], [], [], []\n        if FLIP is not None:\n            img_m, meta_m = mirror_windows(img, meta)\n        for m in ms:\n            z, sl, bl = one_pass(m, img, meta, mask)\n            st = state_readout(m, sl, bl) if sl is not None else None\n            if FLIP is None:\n                out.append(z[0])\n                if st is not None:\n                    st_dep.append(st[0]); st_plain.append(st[0])\n                continue\n            zm, slm, blm = one_pass(m, img_m, meta_m, mask)\n            P = FLIP["perm"]\n            zt = 0.5 * (z[0] + zm[0][P])\n            zt[FLIP["mcl"]] = z[0][FLIP["mcl"]]\n            out.append(zt)\n            plain.append(z[0]); mirr.append(zm[0])\n            if st is not None:\n                stm = state_readout(m, slm[:, P], blm[:, FLIP["bin_perm"]])\n                stt = 0.5 * (st[0] + stm[0])\n                stt[MCL_STATE_COLS] = st[0][MCL_STATE_COLS]\n                st_dep.append(stt); st_plain.append(st[0]); st_mirr.append(state_readout(m, slm, blm)[0])\n        EXTRA["group%d" % group] = {"plain": plain, "mirror": mirr, "state": st_dep, "state_plain": st_plain, "state_mirror": st_mirr}\n        return out\n\n    @torch.inference_mode()\n    def score(images, meta):\n        full = images.to(dev, non_blocking=True)\n        img = full[None, :, :, off:off + a.crop, off:off + a.crop]\n        # v3 --crop2: group 2\'s own centre crop of the same decoded windows (the same tensor when the crops agree)\n        img2 = img if crop2 == a.crop else full[None, :, :, off2:off2 + crop2, off2:off2 + crop2]\n        meta = meta.to(dev)[None]\n        mask = torch.ones((1, img.shape[1]), dtype=torch.bool, device=dev)\n        if shared_trunk:\n            zs = run_shared(img, meta, mask, amp)\n            if not torch.isfinite(zs).all():\n                zs = run_shared(img, meta, mask, "fp32")\n            return zs.mean(0), zs\n        if FLIP is not None or AUX:                           # v4c: flip TTA / state read-out (v4b\'s loops below otherwise)\n            EXTRA.clear()\n            zs = torch.stack(score_v4c(models, img, meta, mask))\n            if not models2:\n                return zs.mean(0), zs\n            zs2 = torch.stack(score_v4c(models2, img2, meta, mask, group=2))\n        else:\n            zs = []\n            for m in models:\n                z = run_model(m, img, meta, mask, amp)\n                if not torch.isfinite(z).all():\n                    z = run_model(m, img, meta, mask, "fp32")\n                zs.append(z[0])\n            zs = torch.stack(zs)\n            if not models2:\n                return zs.mean(0), zs\n            zs2 = []\n            for m in models2:                                 # second architecture on the same windows (one decode)\n                z = run_model(m, img2, meta, mask, amp)\n                if not torch.isfinite(z).all():\n                    z = run_model(m, img2, meta, mask, "fp32")\n                zs2.append(z[0])\n            zs2 = torch.stack(zs2)\n        z = (1 - W2) * zs.mean(0).double() + W2 * zs2.mean(0).double()        # v3b: the two groups mixed in logit space\n        pf = torch.sigmoid(z).clamp(1e-12, 1 - 1e-12)\n        return torch.log(pf / (1 - pf)), torch.cat([zs, zs2])\n\n    fh = open(out / "predictions.csv", "w", encoding="utf-8", newline="")\n    wr = csv.writer(fh)\n    wr.writerow([UIDK] + LABELS)\n    fallback, stats = {}, {"decode_s": 0., "render_s": 0., "gpu_s": 0., "windows": 0, "wait_s": 0., "rows": 0, "anchor_s": 0.}\n    per_study, dump = [], {}\n    anchor_methods = {}\n    side_counts = {"checked": 0, "flipped": 0}\n    # v4: per-study fluid / stacker record\n    FL = {"studies": 0, "with_volume": {}, "missing": {}, "status": {}, "fluid_s": 0., "stack_fallback": {}, "first": []}\n    if STK is not None:\n        FL["with_volume"] = {k: 0 for k in STK["classes"]}\n        FL["missing"] = {k: 0 for k in STK["classes"]}\n        ffh = open(a.fluid_dump or (out / "fluid_stacker.csv"), "w", encoding="utf-8", newline="")\n        fwr = csv.writer(ffh)\n        fwr.writerow([UIDK, "status", "n_sag_fluid_series", "series", "median", "n", "spacing", "eff_sag_p99_open", "bak_sag_p99_open",\n                      "fluid_s"] + [c + s for c in STK["classes"] for s in ("|z_feature", "|z_out", "|p_model", "|p_out")])\n\n    SS = {"studies": 0, "stack_fallback": {}, "first": []}       # v4c: per-study state-stacker record\n    if SST is not None:\n        sfh = open(a.state_dump or (out / "state_stacker.csv"), "w", encoding="utf-8", newline="")\n        swr = csv.writer(sfh)\n        swr.writerow([UIDK] + [c + s for c in SST["classes"] for s in ("|z_feature", "|z_out", "|p_model", "|p_out")])\n\n    def apply_state_stacker(uid, item, zs, p):\n        """v4c: overwrite the adopted classes\' probabilities with the state stacker\'s; anything unexpected keeps the model\'s value."""\n        SS["studies"] += 1\n        zf = zs.mean(0).double().cpu().numpy()                # the deployed group-1 logit (one group), as score() returns it\n        stv = torch.stack(EXTRA["group1"]["state"]).double().mean(0).cpu().numpy()   # arithmetic mean over the fold models\n        row = [uid]\n        try:\n            fl = item.get("fluid")\n            for name, co in SST["classes"].items():\n                j = LABELS.index(name)\n                x = []\n                for f, col in zip(co["features"], co["columns"]):\n                    if f == "z":\n                        x.append(float(zf[j]))\n                    elif f in ("lv", "miss"):\n                        v = None if fl is None else fl.get(co["volume_column"])\n                        miss = v is None or not math.isfinite(float(v)) or float(v) < 0\n                        x.append((0.0 if miss else float(np.log1p(float(v)))) if f == "lv" else (1.0 if miss else 0.0))\n                    else:\n                        x.append(float(stv[col]))\n                z_out = co["b0"] + float(np.dot(np.asarray(x, np.float64), np.asarray(co["coef"], np.float64)))\n                p_model = float(p[j])\n                if math.isfinite(z_out):\n                    p[j] = 1.0 / (1.0 + math.exp(-z_out)) if z_out >= 0 else math.exp(z_out) / (1.0 + math.exp(z_out))\n                else:\n                    SS["stack_fallback"][uid] = "non-finite stacker logit for " + name\n                row += ["%.17g" % zf[j], "%.17g" % z_out, "%.17g" % p_model, "%.17g" % float(p[j])]\n        except Exception as exc:                              # keep the model\'s probabilities for this study\n            SS["stack_fallback"][uid] = repr(exc)[:200]\n            row = [uid] + [""] * (4 * len(SST["classes"]))\n        swr.writerow(row)\n        sfh.flush()\n\n    def apply_stacker(uid, item, zs, p):\n        """v4: overwrite the adopted classes\' probabilities with the stacker\'s; anything unexpected keeps the v3 value."""\n        fl = item.get("fluid")\n        st = "not measured" if fl is None else fl.get("status", "?")\n        FL["studies"] += 1\n        FL["status"][st] = FL["status"].get(st, 0) + 1\n        FL["fluid_s"] += float(item.get("fluid_s", 0.0))\n        n1 = len(models)\n        # v4b: z = the deployed v3b line mix, the same tensor operations as score() (one group: its fold-mean logit)\n        if models2:\n            zmix = ((1 - W2) * zs[:n1].mean(0).double() + W2 * zs[n1:].mean(0).double()).cpu().numpy()\n        else:\n            zmix = zs.mean(0).double().cpu().numpy()\n        row_extra = []\n        try:\n            zo, zf = fluid_stack(STK, LABELS, zmix, fl)\n            for name, z_out in zo.items():\n                j = LABELS.index(name)\n                col = STK["classes"][name]["volume_column"]\n                has = fl is not None and fl.get(col) is not None\n                FL["with_volume" if has else "missing"][name] += 1\n                p_model = float(p[j])\n                if math.isfinite(z_out):\n                    p[j] = 1.0 / (1.0 + math.exp(-z_out)) if z_out >= 0 else math.exp(z_out) / (1.0 + math.exp(z_out))\n                else:\n                    FL["stack_fallback"][uid] = "non-finite stacker logit for " + name\n                row_extra += ["%.17g" % zf[j], "%.17g" % z_out, "%.17g" % p_model, "%.17g" % float(p[j])]   # v4b: full precision\n        except Exception as exc:                      # keep the v3 probabilities for this study\n            FL["stack_fallback"][uid] = repr(exc)[:200]\n            row_extra = [""] * (4 * len(STK["classes"]))\n        f = fl or {}\n        fwr.writerow([uid, st, f.get("n_sag_fluid_series", ""), f.get("series", ""), f.get("median", ""), f.get("n", ""),\n                      f.get("spacing", ""), "" if f.get("eff_sag_p99_open") is None else repr(f["eff_sag_p99_open"]),\n                      "" if f.get("bak_sag_p99_open") is None else repr(f["bak_sag_p99_open"]), item.get("fluid_s", "")] + row_extra)\n        ffh.flush()\n        if len(FL["first"]) < 50:\n            FL["first"].append([uid, st, f.get("eff_sag_p99_open"), f.get("bak_sag_p99_open"), item.get("fluid_s")])\n\n    def handle(item):\n        uid = item["uid"]\n        if item["error"] or not len(item["images"]):\n            fallback[uid] = item["error"] or "no usable window"\n            return\n        if EAGER and not COMPILE["checked"]:\n            compile_check(item)                       # v3: once, before the first study is scored; not counted in gpu_s\n        tg = time.perf_counter()\n        try:\n            zmean, zs = score(item["images"], item["meta"])\n        except Exception as exc:\n            fallback[uid] = "gpu: " + repr(exc)[:200]\n            return\n        if not torch.isfinite(zmean).all():\n            fallback[uid] = "non-finite score"\n            return\n        p = torch.sigmoid(zmean).double().cpu().numpy()\n        dt = time.perf_counter() - tg\n        if STK is not None:\n            ts = time.perf_counter()\n            apply_stacker(uid, item, zs, p)           # v4: adopted classes only; the other classes keep v3b\'s numbers\n            FL["stack_s"] = FL.get("stack_s", 0.0) + time.perf_counter() - ts\n        if SST is not None:                           # v4c: the state stacker (never the same class as the fluid stacker)\n            apply_state_stacker(uid, item, zs, p)\n        wr.writerow([uid] + ["%.9g" % float(x) for x in p])\n        fh.flush()\n        stats["rows"] += 1\n        stats["decode_s"] += item["seconds"][0]\n        stats["render_s"] += item["seconds"][1]\n        stats["anchor_s"] += item["seconds"][2] if len(item["seconds"]) > 2 else 0.\n        if item.get("side") and item["side"].get("score") is not None:\n            side_counts["checked"] += 1\n            side_counts["flipped"] += int(bool(item["side"].get("flipped")))\n        if item.get("anchor"):\n            anchor_methods[item["anchor"]["method"]] = anchor_methods.get(item["anchor"]["method"], 0) + 1\n        stats["gpu_s"] += dt\n        stats["windows"] += len(item["images"])\n        per_study.append([uid, len(item["images"]), item["seconds"][0], item["seconds"][1], round(dt, 3)])\n        if a.dump_logits:\n            if FLIP is None and not a.dump_state:\n                dump[uid] = zs.float().cpu().numpy().tolist()\n            else:                                     # v4c: the scored per-model logits plus the passes / state read-out\n                g = EXTRA.get("group1", {})\n                rec_ = {"z": zs.float().cpu().numpy().tolist()}\n                if FLIP is not None:\n                    rec_["z_plain"] = [v.float().cpu().tolist() for v in g.get("plain", [])] + [v.float().cpu().tolist() for v in EXTRA.get("group2", {}).get("plain", [])]\n                    rec_["z_mirror_raw"] = [v.float().cpu().tolist() for v in g.get("mirror", [])] + [v.float().cpu().tolist() for v in EXTRA.get("group2", {}).get("mirror", [])]\n                if a.dump_state:\n                    rec_["state_plain"] = [v.float().cpu().tolist() for v in g.get("state_plain", [])]\n                    if FLIP is not None:\n                        rec_["state_mirror_raw"] = [v.float().cpu().tolist() for v in g.get("state_mirror", [])]\n                dump[uid] = rec_\n\n    pos = 0\n    timed_out = False\n    while pos < len(uids) and not timed_out:\n        ds = CStudies(a.data, a.split, uids[pos:], tmp, a.code, a.size, a.eval_stride, a.views, a.slices, a.anchor_weights, a.phys_mm,\n                      a.side_model, a.side_threshold, a.side_max_x, a.side_position_mm, extent_meta=N_EXTENT[0] > 0,\n                      fluid=STK is not None or (SST is not None and SST["needs_volume"]))\n        loader = torch.utils.data.DataLoader(ds, batch_size=None, shuffle=False, num_workers=a.workers,\n                                             prefetch_factor=a.prefetch if a.workers else None, persistent_workers=False,\n                                             multiprocessing_context="spawn" if a.workers else None,\n                                             timeout=a.item_timeout if a.workers else 0)\n        try:\n            mark = time.perf_counter()\n            for item in loader:\n                stats["wait_s"] += time.perf_counter() - mark\n                handle(item)\n                pos += 1\n                if pos % 20 == 0 or pos == len(uids) or pos <= 3:\n                    el = time.time() - t_start\n                    prog = {"done": pos, "of": len(uids), "scored": stats["rows"], "fallback": len(fallback),\n                            "elapsed_s": round(el, 1), "s_per_study": round(el / max(pos, 1), 2),\n                            "decode_s": round(stats["decode_s"], 1), "render_s": round(stats["render_s"], 1),\n                            "gpu_s": round(stats["gpu_s"], 1), "anchor_s": round(stats["anchor_s"], 1),\n                            "anchors": anchor_methods, "windows": stats["windows"]}\n                    if STK is not None:                   # v4\n                        prog.update(fluid_s=round(FL["fluid_s"], 2), fluid_missing=FL["missing"])\n                    print(json.dumps(prog), flush=True)\n                if a.deadline and time.time() > a.deadline:\n                    timed_out = True\n                    break\n                mark = time.perf_counter()\n        except Exception as exc:\n            if pos < len(uids):\n                fallback[uids[pos]] = "loader: " + repr(exc)[:200]\n            pos += 1\n        del loader\n    for u in uids[pos:]:\n        fallback[u] = "not reached before the deadline"\n    fh.close()\n    if STK is not None:\n        ffh.close()\n    if SST is not None:\n        sfh.close()\n    shutil.rmtree(tmp, ignore_errors=True)\n    if a.dump_logits:\n        Path(a.dump_logits).write_text(json.dumps(dump), encoding="utf-8")\n    receipt = {"status": "PASS", "arm": a.arm, "models": [p.name for p in files], "backbone": a.backbone, "shared_trunk": bool(shared_trunk), "init": a.init,\n               "spatial_models": n_spatial, "extent_models": N_EXTENT[0], "group2": None if not models2 else {"models": [p.name for p in files2], "backbone": a.backbone2,\n                                                                                   "spatial_models": n_spatial2, "w2": W2.tolist(),\n                                                                                   "syn_decomp_models": N_SYN[0] - n_syn1, "crop": crop2},\n               "syn_decomp_models": n_syn1, "crop": a.crop, "crop2": crop2 if models2 else None, "line_mix": "logit", "compile": COMPILE,\n               "trainer_md5": trainer_md5, "trainer_file": str(CT.__file__), "worker_file": Path(__file__).name,\n               "studies": len(uids), "scored": stats["rows"], "fallback": fallback, "timed_out": timed_out, "amp": amp,\n               "gpu": torch.cuda.get_device_name(), "torch": torch.__version__, "timm": timm.__version__,\n               "seconds": round(time.time() - t_start, 1), "stats": {k: round(v, 1) for k, v in stats.items()},\n               "per_study": per_study[:50], "anchor_methods": anchor_methods, "views": a.views, "phys_mm": a.phys_mm,\n               "side_check": None if not a.side_model else dict(side_counts, threshold=a.side_threshold, max_x=a.side_max_x, position_mm=a.side_position_mm,\n                                                                  model_sha256=hashlib.sha256(Path(a.side_model).read_bytes()).hexdigest()),\n               "anchor_weights_sha256": (hashlib.sha256(Path(a.anchor_weights).read_bytes()).hexdigest() if a.anchor_weights else None),\n               # v3: the worker and trainer entries hash the files actually running / imported (same as HERE/<name> in the notebook)\n               "sha256": {n: hashlib.sha256(({"compact_train.py": Path(CT.__file__), "compact_infer.py": Path(__file__).resolve()}.get(n)\n                                              or (HERE / n)).read_bytes()).hexdigest()\n                          for n in ("px_cache.py", "compact_train.py", "compact_infer.py", "probe_infer.py")\n                          + (("side_common.py",) if a.side_model else ())\n                          + (("knee_anchor_bank_v2.py",) if a.views in ("v3k", "v4") else ())},\n               "model": "compact fine-tuned image model on the pixel cache"}\n    if FLIP is not None:                              # v4c\n        receipt["flip_tta"] = {"perm": FLIP["perm"], "bin_perm": FLIP["bin_perm"], "mcl": "plain pass only",\n                               "transform": "compact_train_v97.mirror_batch (coronal/axial width flip; sagittal slice reversal + rel_pos "\n                                            "negated; coronal views 2 <-> 3)", "combine": "per fold model 0.5 (z + P z_mirror), logit space"}\n    if SST is not None:                               # v4c\n        receipt["state_stacker"] = {"json": SST["path"], "sha256": SST["sha256"], "created": SST["created"], "lines": SST["lines"],\n                                    "classes": {k: {"features": v["features"], "b0": v["b0"], "coef": v["coef"], "volume_column": v["volume_column"]}\n                                                for k, v in SST["classes"].items()},\n                                    "state_combine": "arithmetic mean over the fold models of state_probs" + (" (flip TTA: mean of the two passes)" if FLIP else ""),\n                                    "studies": SS["studies"], "stack_fallback": SS["stack_fallback"]}\n    if STK is not None:                               # v4\n        receipt["fluid_stacker"] = {"json": STK["path"], "sha256": STK["sha256"], "created": STK["created"], "combine": STK["combine"],\n                                    "weights": STK["weights"], "lines": STK["lines"], "z": "v3b line mix",\n                                    "weights_per_class": {k: v["weights"] for k, v in STK["classes"].items()},\n                                    "classes": {k: {c: v[c] for c in ("b0", "b_z", "b_lv", "b_miss", "volume_column")} for k, v in STK["classes"].items()},\n                                    "studies": FL["studies"], "with_volume": FL["with_volume"], "missing": FL["missing"], "status": FL["status"],\n                                    "fluid_s": round(FL["fluid_s"], 2), "stack_s": round(FL.get("stack_s", 0.0), 3),\n                                    "fluid_s_per_study": round(FL["fluid_s"] / max(1, FL["studies"]), 4),\n                                    "stack_fallback": FL["stack_fallback"], "first": FL["first"]}\n    (out / "receipt.json").write_text(json.dumps(receipt, indent=1), encoding="utf-8")\n    print("WORKER_DONE", json.dumps({k: receipt[k] for k in ("studies", "scored", "timed_out", "seconds", "amp")}),\n          "fallback", len(fallback), "compile", json.dumps({k: COMPILE[k] for k in ("requested", "active", "fallback")}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'probe_infer.py': '"""probe_ inference worker (Kaggle or local): raw DICOM -> label-free image scores for one list of studies.\n\nPath, identical to how the probe features were cached:\n  raw DICOM  --mainline training.native_cache.prepare (one study at a time)-->  native uint16 series\n             --fe_extract.StudyWindows.build (the SAME file that built the feature cache)-->  384 px windows\n             --frozen OrthoFoundation-L public SSL weights-->  CLS + 6x6 patch cells --PCA-256 (saved pca.npz)-->\n             --saved probe heads (probe_train.Head, K fold models x seeds), mean logit-->  12 probabilities\nNothing is trained here.  No report, no label, no test-time adaptation.  A study that cannot be decoded, or\nthat is not reached before the deadline, is listed as a fallback (the controller fills it with the mean score).\nEverything here is a probe_ scaffold (frozen encoder + light heads); it is not the mainline model.\n"""\nimport argparse, contextlib, csv, hashlib, inspect, io, json, os, shutil, sys, time\nfrom pathlib import Path\nfrom types import SimpleNamespace\nsys.dont_write_bytecode = True\nHERE = Path(__file__).resolve().parent\nimport numpy as np\nimport torch\n\nUIDK = "StudyInstanceUID"\n\n\nclass OneStudySource:\n    """Directory reader that lists only the study being prepared (the stock one lists every DICOM of the split)."""\n    current = None\n\n    def __init__(self, root, split):\n        self.root = Path(root)\n        self.split = split\n\n    def read(self, item):\n        return (self.root / (item.filename if hasattr(item, "filename") else item)).read_bytes()\n\n    def infolist(self):\n        base = self.root / (self.split + "_series") / OneStudySource.current\n        return [SimpleNamespace(filename=p.relative_to(self.root).as_posix()) for p in sorted(base.glob("*/*.dcm"))]\n\n    def close(self):\n        pass\n\n\ndef quantize_series_slicewise(images):\n    """Same global float32 extrema and per-voxel arithmetic as training.native_cache.quantize_series, without\n    the whole-volume float32 copy (taken from the mainline\'s accepted Kaggle memory patch)."""\n    lo = min(float(np.asarray(x, np.float32).min()) for x in images)\n    hi = max(float(np.asarray(x, np.float32).max()) for x in images)\n    scale = max(hi - lo, 1e-6)\n    encoded = np.empty((len(images), *images[0].shape), dtype=np.uint16)\n    for z, image in enumerate(images):\n        volume = np.asarray(image, np.float32)\n        encoded[z] = np.rint((volume - lo) / scale * 65535).clip(0, 65535).astype(np.uint16)\n    window = np.percentile(encoded[:, ::4, ::4], [1, 99]).astype(float).tolist()\n    return encoded, {"minimum": lo, "maximum": hi, "max_absolute_quantization_error_bound": scale / 131070,\n                     "display_window_uint16": window}\n\n\ndef install_native(lean=True):\n    import training.native_cache as native\n    if lean:\n        src = inspect.getsource(native.prepare)\n        for before, after in [\n            ("pydicom.dcmread(io.BytesIO(archive.read(i)))",\n             "pydicom.dcmread(io.BytesIO(archive.read(i)), stop_before_pixels=True)"),\n            ("geom.pixels_from_dataset(objects[sid][name],path=name)",\n             "geom.pixels_from_dataset(pydicom.dcmread(io.BytesIO(archive.read(name))),path=name)")]:\n            if src.count(before) != 1:\n                raise RuntimeError("training.native_cache.prepare differs from the frozen source")\n            src = src.replace(before, after)\n        exec(compile(src, "probe_infer:prepare", "exec"), native.__dict__)\n        native.quantize_series = quantize_series_slicewise\n    native.DirectorySource = OneStudySource\n    return native\n\n\nclass Studies(torch.utils.data.Dataset):\n    def __init__(self, data, split, uids, tmp, code, lean=True):\n        self.data, self.split, self.uids = str(data), split, list(uids)\n        self.tmp, self.code, self.lean = str(tmp), str(code), lean\n        self.native = None\n        self.windows = None\n\n    def __len__(self):\n        return len(self.uids)\n\n    def _setup(self):\n        if self.code not in sys.path:\n            sys.path.insert(0, self.code)\n        if str(HERE) not in sys.path:\n            sys.path.insert(0, str(HERE))\n        self.native = install_native(self.lean)\n        import fe_extract\n        self.windows = fe_extract.StudyWindows(self.tmp, [])\n\n    def __getitem__(self, i):\n        uid = self.uids[i]\n        t0 = time.perf_counter()\n        work = Path(self.tmp) / ("s_" + hashlib.sha256(uid.encode()).hexdigest()[:16])\n        t1 = t0\n        try:\n            if self.native is None:\n                self._setup()\n            OneStudySource.current = uid\n            with contextlib.redirect_stdout(io.StringIO()):\n                self.native.prepare(self.data, work, split=self.split, study_ids=[uid], source_directory=True)\n            t1 = time.perf_counter()\n            self.windows.cache = str(work)\n            item = self.windows.build(uid)\n            item["error"] = None\n        except Exception as exc:\n            item = {"uid": uid, "error": repr(exc)[:300], "images": torch.zeros((0, 3, 384, 384), dtype=torch.uint8),\n                    "meta": torch.zeros((0, 12), dtype=torch.float32), "series": [], "removed": 0}\n        finally:\n            shutil.rmtree(work, ignore_errors=True)\n        item["seconds"] = [round(t1 - t0, 3), round(time.perf_counter() - t1, 3)]\n        return item\n\n\ndef load_heads(heads_dir, arm, hidden, dev):\n    if str(HERE) not in sys.path:\n        sys.path.insert(0, str(HERE))\n    from probe_train import Head, LABELS\n    if Path(heads_dir).is_file():                     # one file: {"<arm>_s<seed>_<fold>.pt": state_dict}\n        blob = torch.load(heads_dir, map_location="cpu", weights_only=True)\n        items = sorted((k, v) for k, v in blob.items() if k.startswith(arm + "_s"))\n    else:\n        items = [(p.name, torch.load(p, map_location="cpu", weights_only=True))\n                 for p in sorted(Path(heads_dir).glob(arm + "_s*_*.pt"))]\n    if not items:\n        raise SystemExit("no head files for arm " + arm)\n    heads = []\n    for name, state in items:\n        h = Head(hidden)\n        h.load_state_dict(state, strict=True)\n        heads.append(h.eval().requires_grad_(False).to(dev))\n    return heads, LABELS, [name for name, _ in items]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True, help="folder with <split>.csv, <split>_series.csv, <split>_series/")\n    ap.add_argument("--split", default="test")\n    ap.add_argument("--code", required=True, help="folder that contains the mainline \'training\' package")\n    ap.add_argument("--weights", required=True)\n    ap.add_argument("--pca", required=True)\n    ap.add_argument("--norm", required=True)\n    ap.add_argument("--heads", required=True)\n    ap.add_argument("--arm", required=True)\n    ap.add_argument("--hidden", type=int, default=256)\n    ap.add_argument("--assignment", default=None, help="json list of study ids; default = every id of <split>.csv")\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--tmp", default=None)\n    ap.add_argument("--amp", default="auto", choices=["auto", "fp16", "bf16", "fp32"])\n    ap.add_argument("--batch", type=int, default=64)\n    ap.add_argument("--workers", type=int, default=2)\n    ap.add_argument("--prefetch", type=int, default=1)\n    ap.add_argument("--item-timeout", type=float, default=900., help="seconds to wait for one study from the loader")\n    ap.add_argument("--deadline", type=float, default=0., help="unix time after which no new study is started")\n    ap.add_argument("--dump-features", default=None)\n    ap.add_argument("--stock-native", action="store_true", help="use the unpatched native builder (parity checks)")\n    a = ap.parse_args()\n    out = Path(a.output)\n    out.mkdir(parents=True, exist_ok=True)\n    tmp = Path(a.tmp or (out / "tmp"))\n    tmp.mkdir(parents=True, exist_ok=True)\n    t_start = time.time()\n    if a.assignment:\n        uids = json.loads(Path(a.assignment).read_text(encoding="utf-8"))\n    else:\n        with open(Path(a.data) / (a.split + ".csv"), encoding="utf-8") as fh:\n            uids = [r[UIDK] for r in csv.DictReader(fh)]\n    if len(uids) != len(set(uids)):\n        raise SystemExit("duplicate study ids")\n\n    dev = torch.device("cuda")\n    if a.code not in sys.path:\n        sys.path.insert(0, a.code)\n    if str(HERE) not in sys.path:\n        sys.path.insert(0, str(HERE))\n    import fe_extract as FE\n    import timm\n    from safetensors.torch import load_file\n    cap = torch.cuda.get_device_capability()[0]\n    amp = a.amp if a.amp != "auto" else ("bf16" if cap >= 8 else "fp16")\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.backends.cudnn.allow_tf32 = True\n    model = timm.create_model("vit_large_patch16_dinov3_qkvb", pretrained=False, num_classes=0, global_pool="",\n                              img_size=FE.IMG)\n    model.load_state_dict(load_file(a.weights), strict=True)\n    model.rope = None\n    model.eval().requires_grad_(False).to(dev)\n    prefix = int(getattr(model, "num_prefix_tokens", 5))\n    mean = torch.tensor([.485, .456, .406], device=dev).view(1, 3, 1, 1)\n    std = torch.tensor([.229, .224, .225], device=dev).view(1, 3, 1, 1)\n    g = FE.IMG // 16\n    blk = g // FE.GRID\n    z = np.load(a.pca)\n    pca_mean, pca_comp = torch.from_numpy(z["mean"]).to(dev), torch.from_numpy(z["components"]).to(dev)\n    zn = np.load(a.norm)\n    c_mu, c_sd = torch.from_numpy(zn["mu"]).to(dev), torch.from_numpy(zn["sd"]).to(dev)\n    heads, LABELS, head_names = load_heads(a.heads, a.arm, a.hidden, dev)\n    print("MODEL_READY", json.dumps({"gpu": torch.cuda.get_device_name(), "amp": amp, "heads": len(heads), "studies": len(uids), "seconds": round(time.time() - t_start, 1)}), flush=True)\n    retries = {"fp32_batches": 0}\n\n    def forward_tokens(x, mode):\n        if mode == "fp32":\n            return model.forward_features(x).float()\n        dt = torch.float16 if mode == "fp16" else torch.bfloat16\n        with torch.autocast("cuda", dtype=dt):\n            return model.forward_features(x).float()\n\n    @torch.inference_mode()\n    def encode(images_u8):\n        out_cls, out_cells = [], []\n        for chunk in images_u8.split(a.batch):\n            x = chunk.to(dev, non_blocking=True).float().div_(255.)\n            x = (x - mean) / std\n            tok = forward_tokens(x, amp)\n            if not torch.isfinite(tok).all():\n                tok = forward_tokens(x, "fp32")\n                retries["fp32_batches"] += 1\n            b = tok.shape[0]\n            if tok.shape[1] != prefix + g * g:\n                raise ValueError("unexpected token count %s" % (tuple(tok.shape),))\n            patch = tok[:, prefix:].reshape(b, FE.GRID, blk, FE.GRID, blk, tok.shape[-1]).mean((2, 4))\n            out_cls.append(tok[:, 0])\n            out_cells.append(patch.reshape(b, FE.GRID * FE.GRID, tok.shape[-1]))\n        return torch.cat(out_cls), torch.cat(out_cells)\n\n    @torch.inference_mode()\n    def score(cls, cells, meta):\n        cls16 = cls.half()                                             # the cache stores float16\n        cells16 = ((cells - pca_mean) @ pca_comp).half()\n        x_cells = ((cells16.float() - c_mu) / c_sd).clamp_(-8, 8).half().float()\n        x_cls = cls16.float()\n        meta = meta.to(dev)\n        slot = meta[:, 1].long().clamp(0, 9)[None]\n        view = meta[:, 3].long().clamp(0, 5)[None]\n        geom = meta[:, 4:8][None]\n        mask = torch.ones((1, len(cls)), dtype=torch.bool, device=dev)\n        zs = torch.stack([h(x_cls[None], x_cells[None], slot, view, geom, mask)[0] for h in heads])\n        return zs.mean(0), zs, cls16, cells16\n\n    pred_path = out / "predictions.csv"\n    fh = open(pred_path, "w", encoding="utf-8", newline="")\n    wr = csv.writer(fh)\n    wr.writerow([UIDK] + LABELS)\n    fallback, stats = {}, {"decode_s": 0., "render_s": 0., "gpu_s": 0., "windows": 0, "wait_s": 0., "rows": 0}\n    per_study = []\n    if a.dump_features:\n        Path(a.dump_features).mkdir(parents=True, exist_ok=True)\n\n    def handle(item):\n        uid = item["uid"]\n        if item["error"] or not len(item["images"]):\n            fallback[uid] = item["error"] or "no usable window"\n            return\n        tg = time.perf_counter()\n        try:\n            cls, cells = encode(item["images"])\n            zmean, zs, cls16, cells16 = score(cls, cells, item["meta"])\n        except Exception as exc:\n            fallback[uid] = "gpu: " + repr(exc)[:200]\n            return\n        if not torch.isfinite(zmean).all():\n            fallback[uid] = "non-finite score"\n            return\n        p = torch.sigmoid(zmean).double().cpu().numpy()\n        dt = time.perf_counter() - tg\n        wr.writerow([uid] + ["%.6f" % float(x) for x in p])\n        fh.flush()\n        stats["rows"] += 1\n        stats["decode_s"] += item["seconds"][0]\n        stats["render_s"] += item["seconds"][1]\n        stats["gpu_s"] += dt\n        stats["windows"] += len(cls)\n        per_study.append([uid, len(cls), item["seconds"][0], item["seconds"][1], round(dt, 3)])\n        if a.dump_features:\n            np.savez(os.path.join(a.dump_features, hashlib.sha256(uid.encode()).hexdigest()[:24] + ".npz"), uid=uid,\n                     cls=cls16.cpu().numpy(), cells=cells16.cpu().numpy(), meta=item["meta"].numpy(),\n                     logits=zs.float().cpu().numpy(), series=np.array(item["series"]))\n\n    pos = 0\n    timed_out = False\n    while pos < len(uids) and not timed_out:\n        ds = Studies(a.data, a.split, uids[pos:], tmp, a.code, lean=not a.stock_native)\n        # spawn: the same start method as the Windows runs this was tested with (no fork-after-CUDA state);\n        # timeout: a stuck decode must surface as an error for that study, never as a silent hang\n        loader = torch.utils.data.DataLoader(ds, batch_size=None, shuffle=False, num_workers=a.workers,\n                                             prefetch_factor=a.prefetch if a.workers else None, persistent_workers=False,\n                                             multiprocessing_context="spawn" if a.workers else None,\n                                             timeout=a.item_timeout if a.workers else 0)\n        try:\n            mark = time.perf_counter()\n            for item in loader:\n                stats["wait_s"] += time.perf_counter() - mark\n                handle(item)\n                pos += 1\n                if pos % 20 == 0 or pos == len(uids) or pos <= 3:\n                    el = time.time() - t_start\n                    print(json.dumps({"done": pos, "of": len(uids), "scored": stats["rows"], "fallback": len(fallback),\n                                      "elapsed_s": round(el, 1), "s_per_study": round(el / max(pos, 1), 2),\n                                      "decode_s": round(stats["decode_s"], 1), "render_s": round(stats["render_s"], 1),\n                                      "gpu_s": round(stats["gpu_s"], 1), "windows": stats["windows"]}), flush=True)\n                if a.deadline and time.time() > a.deadline:\n                    timed_out = True\n                    break\n                mark = time.perf_counter()\n        except Exception as exc:                       # a loader worker died: skip that study, restart after it\n            if pos < len(uids):\n                fallback[uids[pos]] = "loader: " + repr(exc)[:200]\n            pos += 1\n        del loader\n    for u in uids[pos:]:\n        fallback[u] = "not reached before the deadline"\n    fh.close()\n    shutil.rmtree(tmp, ignore_errors=True)\n    receipt = {"status": "PASS", "arm": a.arm, "heads": head_names, "studies": len(uids), "scored": stats["rows"],\n               "fallback": fallback, "timed_out": timed_out, "amp": amp, "fp32_retry_batches": retries["fp32_batches"],\n               "gpu": torch.cuda.get_device_name(), "torch": torch.__version__, "timm": timm.__version__,\n               "seconds": round(time.time() - t_start, 1), "stats": {k: round(v, 1) for k, v in stats.items()},\n               "per_study": per_study[:50],\n               "native": "stock" if a.stock_native else "lean (header pass, slice-wise quantise)",\n               "sha256": {n: hashlib.sha256((HERE / n).read_bytes()).hexdigest()\n                          for n in ("fe_extract.py", "probe_train.py", "probe_infer.py")},\n               "scaffold": "probe_: frozen encoder + light heads"}\n    (out / "receipt.json").write_text(json.dumps(receipt, indent=1), encoding="utf-8")\n    print("WORKER_DONE", json.dumps({k: receipt[k] for k in ("studies", "scored", "timed_out", "seconds", "amp")}),\n          "fallback", len(fallback), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'probe_dual.py': '"""probe_ controller: split the test studies over the visible GPUs, run probe_infer.py workers, merge.\n\nNever fails the notebook for a single bad study or a dead worker: whatever a worker wrote is used, every\nother study gets the per-class mean of the scored studies (0.5 if nothing was scored) and is counted in the\nreceipt.  Output columns and row order follow sample_submission.csv.\nv3b (10-10): submission.csv probabilities are written with %.9g (probe_dual.py: %.6f); nothing else differs.\n"""\nimport argparse, csv, json, os, subprocess, sys, time\nfrom pathlib import Path\nsys.dont_write_bytecode = True\nHERE = Path(__file__).resolve().parent\nUIDK = "StudyInstanceUID"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--split", default="test")\n    ap.add_argument("--code", required=True)\n    ap.add_argument("--bundle", required=True, help="folder with model.safetensors, pca.npz, feature_norm.npz, heads/")\n    ap.add_argument("--arm", required=True)\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--amp", default="auto")\n    ap.add_argument("--loader-workers", type=int, default=2)\n    ap.add_argument("--deadline", type=float, default=0.)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--weights", default=None, help="default: <bundle>/model.safetensors")\n    ap.add_argument("--worker", default="probe_infer.py", help="probe_infer.py or compact_infer.py")\n    ap.add_argument("--worker-args", default="[]", help="json list of extra worker arguments (compact worker)")\n    ap.add_argument("--max-fallback-fraction", type=float, default=0.03)\n    a = ap.parse_args()\n    t0 = time.time()\n    data, out, bundle = Path(a.data), Path(a.output), Path(a.bundle)\n    out.mkdir(parents=True, exist_ok=True)\n    with open(data / (a.split + ".csv"), encoding="utf-8") as fh:\n        ids = [r[UIDK] for r in csv.DictReader(fh)]\n    if a.limit:\n        ids = ids[:a.limit]\n    if len(ids) != len(set(ids)):\n        raise SystemExit("duplicate test ids")\n    try:\n        names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],\n                                        text=True).strip().splitlines()\n    except Exception:\n        names = ["unknown"]\n    n_gpu = max(1, len(names))\n\n    def cost(uid):                       # number of DICOM files; scheduling only\n        n = 0\n        try:\n            for s in os.scandir(data / (a.split + "_series") / uid):\n                if s.is_dir():\n                    n += sum(1 for _ in os.scandir(s.path))\n        except OSError:\n            pass\n        return n\n\n    costs = {u: cost(u) for u in ids}\n    order = sorted(ids, key=lambda u: (-costs[u], u))\n    parts = [order[i::n_gpu] for i in range(n_gpu)]\n    # inside a worker: keep the big studies from arriving back to back\n    parts = [p[::2] + p[1::2] for p in parts]\n    heads_path = str(bundle / "heads.pt") if (bundle / "heads.pt").is_file() else str(bundle / "heads")\n    procs = []\n    for gi, part in enumerate(parts):\n        wdir = out / ("worker%d" % gi)\n        wdir.mkdir(exist_ok=True)\n        assign = wdir / "ids.json"\n        assign.write_text(json.dumps(part), encoding="utf-8")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gi), OMP_NUM_THREADS="1", MKL_NUM_THREADS="1",\n                   OPENBLAS_NUM_THREADS="1", MALLOC_ARENA_MAX="2", PYTHONDONTWRITEBYTECODE="1")\n        cmd = [sys.executable, "-u", str(HERE / a.worker), "--data", str(data), "--split", a.split,\n               "--code", a.code, "--arm", a.arm, "--assignment", str(assign), "--output", str(wdir), "--amp", a.amp,\n               "--workers", str(a.loader_workers), "--deadline", str(a.deadline)]\n        if a.worker == "probe_infer.py":\n            cmd += ["--weights", a.weights or str(bundle / "model.safetensors"), "--pca", str(bundle / "pca.npz"),\n                    "--norm", str(bundle / "feature_norm.npz"), "--heads", heads_path]\n        else:\n            cmd += ["--models", str(bundle)] + [str(x) for x in json.loads(a.worker_args)]\n        log = open(wdir / "log.txt", "w", encoding="utf-8")\n        procs.append((gi, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log, wdir, part))\n        print("WORKER_START", gi, "studies", len(part), "dicom files", sum(costs[u] for u in part), flush=True)\n    last = 0\n    while any(p.poll() is None for _, p, _, _, _ in procs):\n        time.sleep(5)\n        if time.time() - last > 60:\n            last = time.time()\n            for gi, p, _, wdir, part in procs:\n                try:\n                    tail = [l for l in (wdir / "log.txt").read_text(encoding="utf-8", errors="replace").splitlines()\n                            if l.startswith("{")][-1:]\n                except OSError:\n                    tail = []\n                print("PROGRESS worker", gi, "of", len(part), tail[0] if tail else "(starting)", flush=True)\n    codes = {}\n    for gi, p, log, wdir, part in procs:\n        log.close()\n        codes[gi] = p.returncode\n        if p.returncode != 0:\n            print("WORKER_FAILED", gi, "exit", p.returncode, flush=True)\n            print((wdir / "log.txt").read_text(encoding="utf-8", errors="replace")[-3000:], flush=True)\n\n    with open(data / "sample_submission.csv", encoding="utf-8") as fh:\n        rd = csv.reader(fh)\n        header = next(rd)\n        sample_ids = [r[0] for r in rd]\n    if a.limit:\n        sample_ids = [u for u in sample_ids if u in set(ids)]\n    if set(sample_ids) != set(ids):\n        raise SystemExit("test.csv and sample_submission.csv list different studies")\n    labels = header[1:]\n    pred, receipts = {}, []\n    for gi, p, log, wdir, part in procs:\n        f = wdir / "predictions.csv"\n        if f.exists():\n            with open(f, encoding="utf-8") as fh:\n                for r in csv.DictReader(fh):\n                    try:\n                        v = [float(r[L]) for L in labels]\n                    except (KeyError, ValueError, TypeError):\n                        continue\n                    if all(0. <= x <= 1. for x in v):\n                        pred[r[UIDK]] = v\n        try:\n            receipts.append(json.loads((wdir / "receipt.json").read_text(encoding="utf-8")))\n        except Exception:\n            receipts.append({"status": "NO_RECEIPT", "worker": gi})\n    scored = [u for u in sample_ids if u in pred]\n    if scored:\n        mean = [sum(pred[u][j] for u in scored) / len(scored) for j in range(len(labels))]\n    else:\n        mean = [0.5] * len(labels)\n    missing = [u for u in sample_ids if u not in pred]\n    with open(out / "submission.csv", "w", encoding="utf-8", newline="") as fh:\n        wr = csv.writer(fh)\n        wr.writerow(header)\n        for u in sample_ids:\n            wr.writerow([u] + ["%.9g" % x for x in pred.get(u, mean)])\n    reasons = {}\n    for r in receipts:\n        for u, why in (r.get("fallback") or {}).items():\n            reasons[u] = why\n    total = {"decode_s": 0., "render_s": 0., "gpu_s": 0., "windows": 0., "rows": 0.}\n    for r in receipts:\n        for k in total:\n            total[k] += float((r.get("stats") or {}).get(k, 0.))\n    n = max(1., total["rows"])\n    result = {"status": "PASS" if scored else "ALL_FALLBACK", "arm": a.arm, "rows": len(sample_ids),\n              "scored": len(scored), "fallback_studies": len(missing),\n              "fallback_reasons": {u: reasons.get(u, "worker did not report") for u in missing[:40]},\n              "worker_exit_codes": codes, "gpus": names, "seconds": round(time.time() - t0, 1),\n              "per_study_mean": {"decode_s": round(total["decode_s"] / n, 3), "render_s": round(total["render_s"] / n, 3),\n                                 "gpu_s": round(total["gpu_s"] / n, 3), "windows": round(total["windows"] / n, 1)},\n              "workers": receipts, "worker": a.worker}\n    (out / "submission_receipt.json").write_text(json.dumps(result, indent=1), encoding="utf-8")\n    if len(missing) > a.max_fallback_fraction * len(sample_ids):\n        raise SystemExit("PROBE_TOO_MANY_FALLBACKS %d of %d (limit %.0f%%); see submission_receipt.json" % (\n            len(missing), len(sample_ids), 100 * a.max_fallback_fraction))\n    print("CONTROLLER_DONE", json.dumps({k: result[k] for k in ("status", "rows", "scored", "fallback_studies",\n                                                                "seconds", "per_study_mean", "worker_exit_codes")}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'knee_anchor_bank_v2.py': '"""Knee anchor bank v2: the knee joint centre (and patella centre) of every study in world coordinates, projected into\neach series\' pixel grid.  Library (AnchorBank, used by compact_infer.py at Kaggle inference) + CLI (the 5090 bank).\n\nPer study: every aligned sagittal series yields a candidate = centroid of the SKM-TEA joint tissues (any of femoral /\ntibial cartilage / meniscus confidently present: max over the three channels > 0.5) on 7 slices ~4 mm apart around the\nmiddle of the stack.  A candidate is kept when its mass >= MASS_MIN pixels, it projects inside the central 15-85 % of\nits own series\' rows and columns, and it lies within PRIOR_MM of the slab prior.  Candidates within AGREE_MM are\nmass-averaged; otherwise the heaviest wins (\'disagree\' recorded).  With no candidate the anchor is the slab prior.\nSlab prior = centre of the intersection of the orthogonal stacks: LR from the sagittal stack, AP from the coronal stack,\nSI from the sagittal+coronal field centres (header geometry only, no model).\nFixes over v1 (found on the 58 gold montages): v1\'s \'sum of three probabilities > 0.5\' mask fired on calf tissue in\nlarge-FOV scans (joint at row 0.91-0.96 accepted); v1 took only the first fluid-sensitive series; v1 rejected\noblique-sagittal protocols (cos < 0.97) wholesale; v1\'s fallback was the sagittal field centre.\nNon-fluid-sensitive sagittal series (T1 / PD without fat suppression) give ~zero mass: expected, not a flip error\n(checked on 136 gold series x 4 flip combinations).\n\nCLI (5090):  python knee_anchor_bank_v2.py --out F:/rsnaknee/anchors_v2 [--uids list.json] [--limit N]\nOutput anchors/<sha24(study)>.json: {"study", "version": 2, "joint_lps", "patella_lps", "slab_prior_lps", "method",\n  "why": [...], "series": {sid: {"joint": [row_frac, col_frac], "patella": [...] | null, "in_field", "dist_mm"}}}\nNo label is read anywhere here."""\nimport argparse, csv, hashlib, json, os, sys, time\nimport numpy as np, cv2, torch, torch.nn.functional as F\n\nGEO = "F:/rsnaknee/geometry"            # 5090 defaults (CLI); the library takes explicit directories\nCACHE = "F:/rsnaknee/cache_native"\nLO, HI = 0.15, 0.85          # plausibility window (fractions of the source series\' rows / cols)\nAGREE_MM = 20.0\nPRIOR_MM = 35.0              # a segmentation candidate farther than this from the slab prior is rejected (g35: 85 mm)\nOBLIQUE_COS = 0.85           # v1 used 0.97 (14 deg): oblique-sagittal knee protocols were rejected wholesale\nSPAN_MM = 4.0                # spacing between the segmented slices (3-D stacks otherwise contribute a 3 mm slab)\nMASS_MIN = 250               # v1: 400\nN_SLICES = 7\n\n\ndef key_for(uid, sid):\n    return hashlib.sha256((uid + "/" + sid).encode()).hexdigest()[:24]\n\n\ndef study_key(uid):\n    return hashlib.sha256(uid.encode()).hexdigest()[:24]\n\n\ndef load_model(weights, dev):\n    """the SKM-TEA V-Net through the mainline loader (needs the `training` package on sys.path)."""\n    from training.anatomy import load_segmenter\n    return load_segmenter(weights, dev)\n\n\ndef project(aff, X):\n    """aff [Z,3,3] (col,row,1 -> LPS mm); X [3] LPS -> (z, col, row, |distance to that plane|) of the nearest slice."""\n    o = aff[:, :, 2]\n    n = np.cross(aff[0, :, 0], aff[0, :, 1]); n /= np.linalg.norm(n)\n    d = (o - X[None]) @ n\n    z = int(np.argmin(np.abs(d)))\n    M = aff[z, :, :2]\n    cr, *_ = np.linalg.lstsq(M, X - o[z], rcond=None)\n    return z, float(cr[0]), float(cr[1]), float(abs(d[z]))\n\n\ndef centroid(prob, thr=0.5):\n    m = prob > thr\n    if m.sum() < 40:\n        return None, 0\n    ys, xs = np.nonzero(m)\n    w = prob[m]\n    return (float((ys * w).sum() / w.sum()), float((xs * w).sum() / w.sum())), int(m.sum())\n\n\nclass AnchorBank:\n    def __init__(self, model, dev, geo_dir=None):\n        self.model, self.dev, self.geo_dir = model, dev, geo_dir\n\n    def geometry(self, uid, sid, geo_dir=None):\n        p = os.path.join(geo_dir or self.geo_dir, key_for(uid, sid) + ".json")\n        return json.load(open(p)) if os.path.exists(p) else None\n\n    @torch.no_grad()\n    def segment(self, series, geo, n_slices=N_SLICES):\n        from training.anatomy_runtime import native_volume\n        vol = native_volume(series["path"])\n        mean, std = float(vol.mean()), max(float(vol.std()), 1.)\n        aff = np.asarray(geo["affines_col_row_origin_lps_mm"], float)\n        if abs(aff[0, 1, 0]) / np.linalg.norm(aff[0, :, 0]) < OBLIQUE_COS or abs(aff[0, 2, 1]) / np.linalg.norm(aff[0, :, 1]) < OBLIQUE_COS:\n            return None, "oblique"\n        flip_x = aff[0, 1, 0] < 0; flip_y = aff[0, 2, 1] > 0          # same convention as training.anatomy.segment_series\n        m = len(vol) // 2\n        dz = float(np.linalg.norm(aff[1, :, 2] - aff[0, :, 2])) if len(aff) > 1 else 4.\n        step = max(1, int(round(SPAN_MM / max(dz, 1e-3))))\n        zs = sorted({min(len(vol) - 1, max(0, m + k * step)) for k in range(-(n_slices // 2), n_slices // 2 + 1)})\n        out = []\n        for z in zs:\n            x = (np.asarray(vol[z], np.float32) - mean) / std\n            if flip_x: x = x[:, ::-1]\n            if flip_y: x = x[::-1]\n            sy, sx = series["spacing_mm"][z]\n            shape = (max(16, round(x.shape[1] * sx / .3125)), max(16, round(x.shape[0] * sy / .3125)))\n            x = cv2.resize(np.ascontiguousarray(x), shape, interpolation=cv2.INTER_LINEAR)\n            t = torch.from_numpy(x.copy())[None, None].to(self.dev)\n            h, w = t.shape[-2:]; t = F.pad(t, (0, (-w) % 16, 0, (-h) % 16), mode="replicate")\n            p = self.model(t).sigmoid()[..., :h, :w]\n            p = F.interpolate(p, size=vol.shape[-2:], mode="bilinear", align_corners=False)[0].float().cpu().numpy()\n            if flip_x: p = p[:, :, ::-1]\n            if flip_y: p = p[:, ::-1]\n            out.append((z, np.ascontiguousarray(p)))\n        return out, "ok"\n\n    def candidate(self, s, geo):\n        aff = np.asarray(geo["affines_col_row_origin_lps_mm"], float)\n        segs, why = self.segment(s, geo)\n        if segs is None:\n            return None, why\n        pts, wts, ppts, pwts = [], [], [], []\n        for z, p in segs:\n            joint = np.max(p[1:4], axis=0)                       # any joint tissue confidently present\n            c, n = centroid(joint, 0.5)\n            if c is not None:\n                pts.append(aff[z] @ np.array([c[1], c[0], 1.])); wts.append(n)\n            cp, npx = centroid(p[0], 0.5)\n            if cp is not None and npx >= 60:\n                ppts.append(aff[z] @ np.array([cp[1], cp[0], 1.])); pwts.append(npx)\n        mass, pmass = int(sum(wts)), int(sum(pwts))\n        if mass < MASS_MIN:\n            return None, "mass<%d(%d)" % (MASS_MIN, mass)\n        X = np.average(np.array(pts), axis=0, weights=np.array(wts, float))\n        Xp = np.average(np.array(ppts), axis=0, weights=np.array(pwts, float)) if pmass >= 120 else None\n        if Xp is not None and not (Xp[1] < X[1] and Xp[2] > X[2] - 10):   # patella: anterior (smaller LPS y), not below the joint\n            Xp = None\n        h, w = s["native_shape"][-2:]\n        z, c, r, _ = project(aff, X)\n        rf, cf = r / max(1, h - 1), c / max(1, w - 1)\n        if not (LO <= rf <= HI and LO <= cf <= HI):\n            return None, "implausible(rf=%.2f,cf=%.2f)" % (rf, cf)\n        return dict(X=X, Xp=Xp, mass=mass, pmass=pmass, sid=s["SeriesInstanceUID"]), "ok"\n\n    def slab_prior(self, uid, d, geo_dir=None):\n        per = {}\n        for s in d["series"]:\n            geo = self.geometry(uid, s["SeriesInstanceUID"], geo_dir)\n            if geo is None or not geo["quality"]["aligned"]:\n                continue\n            aff = np.asarray(geo["affines_col_row_origin_lps_mm"], float)\n            h, w = s["native_shape"][-2:]; z = len(aff) // 2\n            per.setdefault(s["plane"], []).append(aff[z] @ np.array([(w - 1) / 2., (h - 1) / 2., 1.]))\n        if not per:\n            return None\n        G = np.zeros(3)\n        for ax, planes in enumerate((("Sagittal",), ("Coronal",), ("Sagittal", "Coronal"))):\n            cs = [c for pl in planes for c in per.get(pl, [])] or [c for pl in per for c in per[pl]]\n            G[ax] = float(np.median([c[ax] for c in cs]))\n        return G\n\n    def record(self, uid, d, geo_dir=None):\n        """anchor record of one study from its native-cache study dict `d` and the geometry directory."""\n        sags = [s for s in d["series"] if s["plane"] == "Sagittal" and os.path.exists(s["path"])]\n        sags.sort(key=lambda s: -(s.get("fluid_sensitive") or 0))\n        rec = {"study": uid, "version": 2, "joint_lps": None, "patella_lps": None, "method": None, "why": [], "series": {}}\n        G = self.slab_prior(uid, d, geo_dir)\n        rec["slab_prior_lps"] = None if G is None else [float(v) for v in G]\n        cands = []\n        for s in sags:\n            geo = self.geometry(uid, s["SeriesInstanceUID"], geo_dir)\n            if geo is None or not geo["quality"]["aligned"]:\n                rec["why"].append("no_geo" if geo is None else "not_aligned"); continue\n            try:\n                c, why = self.candidate(s, geo)\n            except Exception as exc:                                   # one bad series must not lose the study\n                c, why = None, "error:" + repr(exc)[:80]\n            if c is None:\n                rec["why"].append(why); continue\n            if G is not None and np.linalg.norm(c["X"] - G) > PRIOR_MM:\n                rec["why"].append("far_from_prior(%.0fmm)" % np.linalg.norm(c["X"] - G)); continue\n            cands.append(c)\n        X = Xp = None\n        if cands:\n            cands.sort(key=lambda c: -c["mass"])\n            best = cands[0]\n            close = [c for c in cands if np.linalg.norm(c["X"] - best["X"]) <= AGREE_MM]\n            if len(close) >= 2:\n                X = np.average(np.array([c["X"] for c in close]), axis=0, weights=np.array([c["mass"] for c in close], float))\n                rec["method"] = "skm_tea_joint_avg%d" % len(close)\n            else:\n                X = best["X"]; rec["method"] = "skm_tea_joint"\n                if len(cands) >= 2:\n                    rec["why"].append("disagree(%.0fmm)" % np.linalg.norm(cands[1]["X"] - best["X"]))\n            pc = [c for c in close if c["Xp"] is not None] or [c for c in cands if c["Xp"] is not None]\n            if pc:\n                Xp = max(pc, key=lambda c: c["pmass"])["Xp"]\n        elif G is not None:\n            X = G; rec["method"] = "slab_prior"\n        else:\n            rec["why"].append("no_geometry")\n        if X is not None:\n            rec["joint_lps"] = [float(v) for v in X]\n            if Xp is not None:\n                rec["patella_lps"] = [float(v) for v in Xp]\n            for s in d["series"]:\n                geo = self.geometry(uid, s["SeriesInstanceUID"], geo_dir)\n                if geo is None:\n                    continue\n                aff = np.asarray(geo["affines_col_row_origin_lps_mm"], float)\n                h, w = s["native_shape"][-2:]\n                z, c, r, dist = project(aff, X)\n                entry = {"joint": [r / max(1, h - 1), c / max(1, w - 1)], "in_field": bool(0 <= r < h and 0 <= c < w),\n                         "dist_mm": round(dist, 1), "patella": None}\n                if Xp is not None:\n                    zp, cp, rp, dp = project(aff, Xp)\n                    entry["patella"] = [rp / max(1, h - 1), cp / max(1, w - 1)]\n                rec["series"][s["SeriesInstanceUID"]] = entry\n        return rec\n\n\ndef plausible_v1(rec, d):\n    """does an existing record pass the v2 gate?  (joint fractions of every sagittal series inside the window)"""\n    if not str(rec.get("method", "")).startswith("skm_tea_joint"):\n        return False\n    for s in d["series"]:\n        if s["plane"] != "Sagittal":\n            continue\n        e = rec["series"].get(s["SeriesInstanceUID"])\n        if e is None or not e["in_field"]:\n            return False\n        rf, cf = e["joint"]\n        if not (LO <= rf <= HI and LO <= cf <= HI):\n            return False\n    return True\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", default="F:/rsnaknee/anchors_v2")\n    ap.add_argument("--geo", default=GEO)\n    ap.add_argument("--cache", default=CACHE)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--uids", default=None)\n    ap.add_argument("--only-rework", default=None, help="existing anchor dir: process only fallback / implausible records")\n    ap.add_argument("--weights", default="F:/rsnaknee/pretrained/skm_tea/model-raw.ckpt")\n    ap.add_argument("--code-root", default="D:/rsnaknee", help="directory holding the mainline `training` package")\n    a = ap.parse_args()\n    if a.code_root and os.path.isdir(a.code_root) and a.code_root not in sys.path:\n        sys.path.insert(0, a.code_root)\n    os.makedirs(a.out, exist_ok=True)\n    dev = "cuda"\n    bank = AnchorBank(load_model(a.weights, dev), dev, a.geo)\n    uids = [r["StudyInstanceUID"] for r in csv.DictReader(open(os.path.join(a.cache, "patients.csv")))]\n    if a.uids: uids = json.load(open(a.uids))\n    if a.limit: uids = uids[:a.limit]\n    t0 = time.time(); stats = {"ok": 0, "ok_avg": 0, "disagree": 0, "fallback": 0, "no_geometry": 0, "patella": 0, "skipped_v1_ok": 0}\n    why_all = {}\n    for i, uid in enumerate(uids):\n        outp = os.path.join(a.out, study_key(uid) + ".json")\n        d = json.load(open(os.path.join(a.cache, "studies", study_key(uid) + ".json"), encoding="utf-8"))\n        if a.only_rework:\n            v1p = os.path.join(a.only_rework, study_key(uid) + ".json")\n            if os.path.exists(v1p) and plausible_v1(json.load(open(v1p)), d):\n                stats["skipped_v1_ok"] += 1\n                continue\n        elif os.path.exists(outp) and json.load(open(outp)).get("version") == 2:\n            continue\n        rec = bank.record(uid, d)\n        m = rec["method"] or ""\n        if m.startswith("skm_tea_joint"):\n            stats["ok"] += 1\n            if "avg" in m: stats["ok_avg"] += 1\n        elif m == "slab_prior":\n            stats["fallback"] += 1\n        else:\n            stats["no_geometry"] += 1\n        if rec["patella_lps"] is not None: stats["patella"] += 1\n        for w in rec["why"]:\n            k = w.split("(")[0]; why_all[k] = why_all.get(k, 0) + 1\n            if k == "disagree": stats["disagree"] += 1\n        json.dump(rec, open(outp, "w"), indent=0)\n        if (i + 1) % 100 == 0:\n            print(i + 1, stats, "%.0fs" % (time.time() - t0), flush=True)\n    print("done", stats, "why:", why_all, "%.0fs" % (time.time() - t0), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'side_common.py': '"""Image-based knee side check, shared by the training-cache fix and Kaggle inference.\n\nThe native cache stores coronal and axial series in a canonical frame where the study\'s side has already been applied:\nlateral should sit at +column for both knees (a right knee is mirrored as the last pixel operation, see\ntraining/dicom_geometry.flip_for_plane).  When the side is wrong, every coronal/axial image of the study is mirrored.\nside_inputs() turns one canonical native volume into a few normalised 2-D images; the classifier says whether lateral is\nat +column (correct orientation) or at -column (mirrored, i.e. the study\'s side is wrong)."""\nimport numpy as np\nimport cv2\n\nSIDE_SIZE = 160\nSIDE_FRACS = (0.4, 0.5, 0.6)\nSIDE_PLANES = ("Coronal", "Axial")\n\n\ndef side_inputs(vol):\n    """canonical native volume [n, H, W] (any dtype) -> uint8 [k, SIDE_SIZE, SIDE_SIZE], k <= 3 middle slices."""\n    n = int(vol.shape[0])\n    if n < 1:\n        return np.zeros((0, SIDE_SIZE, SIDE_SIZE), np.uint8)\n    idx = sorted({int(round(f * (n - 1))) for f in SIDE_FRACS})\n    out = []\n    for i in idx:\n        im = np.asarray(vol[i], np.float32)\n        lo, hi = np.percentile(im[::4, ::4], [1, 99])\n        im = np.clip((im - lo) / max(float(hi - lo), 1e-6), 0, 1)\n        im = cv2.resize(im, (SIDE_SIZE, SIDE_SIZE), interpolation=cv2.INTER_AREA)\n        out.append(np.rint(im * 255).astype(np.uint8))\n    return np.stack(out)\n\n\ndef build_side_net():\n    import timm, torch.nn as nn\n    net = timm.create_model("resnet34", pretrained=False, num_classes=1, in_chans=1)\n    return net\n'}

import os, sys, json, shutil, subprocess, zipfile, time, hashlib
from pathlib import Path
T0 = NB_T0   # [blend] the 8.3 h deadline counts from the notebook start (the Raptor stage ran first)
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], check=False)
subprocess.run(['df', '-h', '/dev/shm', '/kaggle/working', '/tmp'], check=False)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
TMP = Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else Path('/tmp')
TMP = TMP / 'probe'
TMP.mkdir(parents=True, exist_ok=True)

def find_mount(slug, required):
    cands = [INPUT / slug, INPUT / 'competitions' / slug]
    cands += list(INPUT.glob('datasets/*/' + slug)) + list(INPUT.glob('datasets/*/' + slug + '/*'))
    cands += list(INPUT.glob('*/' + slug)) + list(INPUT.glob(slug + '/*'))
    found = []
    for p in cands:
        if (p / required).is_file() and p not in found:
            found.append(p)
    if len(found) != 1:
        raise RuntimeError('Cannot resolve %s: %s' % (slug, found))
    return found[0]

CODE_SRC = find_mount(CODE_SLUG, 'receipt.json')
DATA = find_mount(COMPETITION, 'test.csv')

# mainline decode + rendering code: byte-identical to what built the training caches
CODE = TMP / 'mainline_code'
if (CODE_SRC / 'cloud_code.zip').is_file():
    with zipfile.ZipFile(CODE_SRC / 'cloud_code.zip') as z:
        z.extractall(CODE)
else:
    shutil.copytree(CODE_SRC / 'cloud_code', CODE, dirs_exist_ok=True)
for name, digest in CODE_SHA.items():
    assert hashlib.sha256((CODE / 'training' / name).read_bytes()).hexdigest() == digest, ('mainline module differs', name)

# pinned wheels (same versions as the runs that produced the training caches)
WHEELS = TMP / 'wheels'
WHEELS.mkdir(exist_ok=True)
if (CODE_SRC / 'dependencies.zip').is_file():
    with zipfile.ZipFile(CODE_SRC / 'dependencies.zip') as z:
        for n in z.namelist():
            if n.endswith('.whl'):
                (WHEELS / Path(n).name).write_bytes(z.read(n))
else:
    for p in CODE_SRC.rglob('*.whl'):
        shutil.copy(p, WHEELS / p.name)
wheels = sorted(str(p) for p in WHEELS.glob('*.whl'))
assert wheels, 'no wheels found'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', *wheels], check=True)

SRC = TMP / 'probe_src'
SRC.mkdir(exist_ok=True)
for name, text in SOURCES.items():
    (SRC / name).write_bytes(text.encode('utf-8'))
    assert hashlib.sha256((SRC / name).read_bytes()).hexdigest() == SOURCE_SHA[name], name
BUNDLE = TMP / 'bundle'
BUNDLE.mkdir(exist_ok=True)

M_SRC = find_mount(C_SLUG, 'manifest.json')
manifest = json.loads((M_SRC / 'manifest.json').read_text())
assert manifest['bundle_id'] == BUNDLE_ID, (manifest['bundle_id'], BUNDLE_ID)
for name in manifest['models'] + (manifest.get('models2') or []):
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('state_spec'):
    name = manifest['state_spec']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('side_model'):
    name = manifest['side_model']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('fluid_stacker'):
    name = manifest['fluid_stacker']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
if manifest.get('state_stacker'):
    name = manifest['state_stacker']
    assert hashlib.sha256((M_SRC / name).read_bytes()).hexdigest() == manifest['sha256'][name], name
    shutil.copy(M_SRC / name, BUNDLE / name)
WORKER = 'compact_infer.py'
WORKER_ARGS = ['--backbone', manifest['backbone'], '--size', str(manifest['size']), '--crop', str(manifest['crop']),
               '--eval-stride', str(manifest.get('eval_stride', 3)), '--views', manifest.get('views', 'v1')]
if manifest.get('state_spec'):
    WORKER_ARGS += ['--state-spec', str(BUNDLE / manifest['state_spec'])]
if manifest.get('side_model'):
    # image side check: studies without one trustworthy laterality tag are re-rendered when the orientation classifier
    # says they are mirrored (same rule as the S5 training-cache fix)
    WORKER_ARGS += ['--side-model', str(BUNDLE / manifest['side_model']), '--side-threshold', str(manifest.get('side_threshold', 2.0)),
                    '--side-position-mm', str(manifest.get('side_position_mm', 20.0)), '--side-max-x', str(manifest.get('side_max_x', 0.0))]
WORKER_ARGS += ['--slices', str(manifest.get('slices', 3)), '--phys-mm', str(manifest.get('phys_mm', 0.0))]
if manifest.get('models2'):
    # second architecture on the same windows (one decode per study), per-class weights fixed in the bundle
    WORKER_ARGS += ['--models2', str(BUNDLE), '--arm2', manifest['arm2'], '--backbone2', manifest['backbone2'],
                    '--w2', ','.join(str(x) for x in manifest['w2'])]
    if manifest.get('crop2') is not None:
        # v3: the second group's own centre crop of the same decoded windows
        WORKER_ARGS += ['--crop2', str(manifest['crop2'])]
if manifest.get('trainer_md5'):
    # v3: the embedded compact_train.py must be the trainer this bundle was built for (and the worker re-checks what it imports)
    assert hashlib.md5((SRC / 'compact_train.py').read_bytes()).hexdigest() == manifest['trainer_md5'], 'pinned trainer differs from the bundle manifest (md5)'
    assert SOURCE_SHA['compact_train.py'] == manifest['src_sha256']['compact_train.py'], 'pinned trainer differs from the bundle manifest (sha256)'
    WORKER_ARGS += ['--trainer-md5', manifest['trainer_md5']]
if manifest.get('compile'):
    # v3: torch.compile of the fold models' encoders (checked against eager inside the worker; eager fallback, logged)
    WORKER_ARGS += ['--compile']
if manifest.get('fluid_stacker'):
    # v4b: D2 fluid volume measured on the rendered rows; the classes the json adopts get the stacker's logit
    WORKER_ARGS += ['--fluid-stacker', str(BUNDLE / manifest['fluid_stacker'])]
if manifest.get('flip_tta') or FLIP_TTA:
    # v4c: every fold model also scores the mirrored windows; the two passes are averaged in logit space (MCL: plain pass)
    # [d957] FLIP_TTA: this notebook's switch, at the manifest key's position (the R6 manifest predates flip TTA: no flip_tta key)
    WORKER_ARGS += ['--flip-tta']
if manifest.get('state_stacker'):
    # v4c: the classes the json adopts get b0 + coef . [z, state probabilities, binaries (, lv, miss)]
    WORKER_ARGS += ['--state-stacker', str(BUNDLE / manifest['state_stacker'])]
if manifest.get('views') in ('v3k', 'v4'):
    # knee-centred views: the anchors are rebuilt from the test DICOM headers + the SKM-TEA checkpoint of the code dataset
    assert hashlib.sha256((CODE_SRC / 'anatomy.pt').read_bytes()).hexdigest() == manifest['anchor_weights_sha256'], 'anatomy.pt differs from the training anchor checkpoint'
    WORKER_ARGS += ['--anchor-weights', str(CODE_SRC / 'anatomy.pt')]
if manifest['backbone'] == 'ofl_l':
    # the fold models hold only the trained tail; rebuild the frozen part from the public OFL-L weights
    W_SRC = find_mount(W_SLUG, 'weight_parts.json')
    parts = json.loads((W_SRC / 'weight_parts.json').read_text())
    h = hashlib.sha256()
    with (BUNDLE / 'model.safetensors').open('wb') as dst:
        for item in parts['parts']:
            blob = (W_SRC / item['name']).read_bytes()
            assert len(blob) == item['bytes'] and hashlib.sha256(blob).hexdigest() == item['sha256'], item['name']
            dst.write(blob)
            h.update(blob)
    assert h.hexdigest() == parts['assembled_sha256'] == manifest['init_weights_sha256'], 'encoder weights differ'
    del blob
    WORKER_ARGS += ['--init', str(BUNDLE / 'model.safetensors'), '--freeze-blocks', str(manifest['freeze_blocks'])]
print('ASSETS_READY', str(CODE_SRC), str(M_SRC), str(DATA), 'flip_tta', '--flip-tta' in WORKER_ARGS, 'seconds %.1f' % (time.time() - T0), flush=True)


In [ ]:
import hashlib, importlib.metadata, os, subprocess
import pandas as pd
roots = [p.parent for p in Path('/kaggle/input').rglob('train.csv')
         if (p.parent/'train_series.csv').exists() and (p.parent/'train_series').is_dir()]
if len(roots) != 1:
    raise RuntimeError('competition train mount is ambiguous: ' + str(roots))
root = roots[0]
split = json.loads((OWN/'splits.json').read_text())
if split['status'] != 'READY':
    raise RuntimeError('predeclared coverage gate failed')
ids = [r['id'] for k in ('fit','tune','confirmation') for r in split[k]]
evaluation = TMP/'own_report_eval'
evaluation.mkdir(exist_ok=False)
pd.DataFrame({'StudyInstanceUID': ids}).to_csv(evaluation/'test.csv', index=False)
series = pd.read_csv(root/'train_series.csv', dtype=str)
series = series[series.StudyInstanceUID.isin(ids)]
if set(series.StudyInstanceUID) != set(ids):
    raise RuntimeError('selected study series coverage mismatch')
series.to_csv(evaluation/'test_series.csv', index=False)
os.symlink(root/'train_series', evaluation/'test_series', target_is_directory=True)
assets = {'out': str(OWN/'results'), 'started_epoch': NB_T0, 'source': str(SRC), 'bundle': str(BUNDLE),
          'code': str(CODE), 'anatomy': str(CODE_SRC/'anatomy.pt'), 'train': str(root/'train.csv'),
          'eval_root': str(evaluation), 'cache': str(TMP/'own_report_features'), 'splits': str(OWN/'splits.json'),
          'split_sha256': hashlib.sha256((OWN/'splits.json').read_bytes()).hexdigest()}
(OWN/'assets.json').write_text(json.dumps(assets, indent=2))
environment = {p: importlib.metadata.version(p) for p in ('torch','torchvision','timm','numpy','pydicom')}
environment['source_notebook_sha256'] = '16a9d953972d9d82717f279f75e57e798d3902f93d1071c30f3caa67326e3601'
(OWN/'environment.json').write_text(json.dumps(environment, indent=2))
try:
    subprocess.run([sys.executable, str(OWN/'train.py'), '--assets', str(OWN/'assets.json')], check=True,
                   timeout=max(1., NB_T0 + 10800 - time.time()))
except subprocess.TimeoutExpired:
    (OWN/'results').mkdir(exist_ok=True)
    (OWN/'results'/'budget_rejection.json').write_text(json.dumps({'stage': 'hard_wall_deadline', 'incomplete_candidate': True}))
    print('OWN_REPORT_HARD_BUDGET_STOP_NO_SUBMISSION', flush=True)
if list(Path('/kaggle/working').rglob('*.csv')):
    raise RuntimeError('forbidden CSV output in working directory')
print('PRIVATE_OWN_REPORT_NOTEBOOK_COMPLETE_NO_SUBMISSION', flush=True)
